<a href="https://colab.research.google.com/github/dchav12/New-stable-/blob/main/NCAA_Player_Prop_hit_rater.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving NCAA prop 3:1.csv to NCAA prop 3:1.csv


In [ ]:
import pandas as pd
import numpy as np

############################################
# PROP ENGINE v1 — Hitrate28 Edition
############################################

BASE_WEIGHTS = {
    "L10": 0.25,
    "L5": 0.15,
    "2025": 0.25,
    "HM_AW": 0.15,
    "H2H": 0.05,
    "DVP": 0.15
}

L5_SHRINK = 0.75
L10_SHRINK = 0.90
DVP_DELTA_CAP = 0.15
PROB_CLAMP = (0.02, 0.98)

BANKROLL_UNIT = 0.01
KELLY_CAP = 0.05

############################################
# HELPERS
############################################

def pct_to_float(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x) if x <= 1 else float(x)/100
    x = str(x).replace("%","").strip()
    if x in ["--","-","–","—",""]:
        return np.nan
    val = float(x)
    return val/100 if val > 1 else val

def american_to_prob(odds):
    if odds > 0:
        return 100/(odds+100)
    return -odds/(-odds+100)

def american_to_decimal(odds):
    if odds > 0:
        return 1 + odds/100
    return 1 + 100/(-odds)

def shrink(p, factor):
    if pd.isna(p):
        return np.nan
    return 0.5 + (p - 0.5) * factor

def clamp(p):
    return max(PROB_CLAMP[0], min(PROB_CLAMP[1], p))

############################################
# LOAD HITRATE28
############################################

df = pd.read_csv("Hitrate28.csv")

# Fix potential column drift
df.rename(columns={
    "IM PROB %":"IM_PROB_PCT",
    "HM/AW":"HM_AW",
    "DVP L10.1":"DVP_L10_2",
    "DVP L10":"DVP_L10",
    "DVP L5":"DVP_L5",
    "DVP HM/AW":"DVP_HM_AW"
}, inplace=True)

############################################
# Normalize Percentages
############################################

pct_cols = ["IM_PROB_PCT","L5","L10","2025","HM_AW","H2H",
            "DVP_L5","DVP_L10","DVP_L10_2","DVP_HM_AW"]

for c in pct_cols:
    if c in df.columns:
        df[c] = df[c].apply(pct_to_float)

############################################
# Odds Handling
############################################

# Split ODDS column if needed
if "ODDS_VALUE" not in df.columns:
    df["ODDS_VALUE"] = df["ODDS"].astype(str).str.extract(r'(-?\d+)')

df["AMERICAN_ODDS"] = df["ODDS_VALUE"].astype(float)
df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"] = df["AMERICAN_ODDS"].apply(american_to_decimal)

############################################
# DVP Composite (capped)
############################################

dvp_cols = ["DVP_L5","DVP_L10","DVP_L10_2","DVP_HM_AW"]
df["DVP_RAW"] = df[dvp_cols].mean(axis=1)

df["DVP"] = df["DVP_RAW"].apply(
    lambda x: 0.5 + np.clip(x-0.5, -DVP_DELTA_CAP, DVP_DELTA_CAP)
    if not pd.isna(x) else np.nan
)

############################################
# Shrink Recent Form
############################################

df["L5"] = df["L5"].apply(lambda x: shrink(x, L5_SHRINK))
df["L10"] = df["L10"].apply(lambda x: shrink(x, L10_SHRINK))

############################################
# MODEL PROBABILITY
############################################

model_probs = []

for _, r in df.iterrows():

    features = {
        "L10": r["L10"],
        "L5": r["L5"],
        "2025": r["2025"],
        "HM_AW": r["HM_AW"],
        "H2H": r["H2H"],
        "DVP": r["DVP"]
    }

    valid = {k:v for k,v in features.items() if not pd.isna(v)}

    if len(valid) == 0:
        model_probs.append(np.nan)
        continue

    weight_sum = sum(BASE_WEIGHTS[k] for k in valid.keys())
    prob = sum((BASE_WEIGHTS[k]/weight_sum) * valid[k] for k in valid.keys())

    model_probs.append(clamp(prob))

df["MODEL_PROB"] = model_probs

############################################
# EDGE / EV / KELLY
############################################

df["EDGE"] = df["MODEL_PROB"] - df["BOOK_PROB"]

df["EV_1U"] = df["MODEL_PROB"]*(df["DEC_ODDS"]-1) - (1-df["MODEL_PROB"])

def kelly(row):
    b = row["DEC_ODDS"] - 1
    p = row["MODEL_PROB"]
    q = 1 - p
    f = (b*p - q)/b
    return max(0, min(f, KELLY_CAP))

df["KELLY_FULL"] = df.apply(kelly, axis=1)
df["KELLY_HALF"] = df["KELLY_FULL"]/2
df["KELLY_FULL_UNITS"] = df["KELLY_FULL"]/BANKROLL_UNIT
df["KELLY_HALF_UNITS"] = df["KELLY_HALF"]/BANKROLL_UNIT

############################################
# OUTPUTS
############################################

top_likely = df.sort_values("MODEL_PROB", ascending=False).head(10)
top_ev = df.sort_values("EV_1U", ascending=False).head(10)

print("TOP 10 MOST LIKELY")
display(top_likely[["PLAYER","PROP","OUTCOME","MODEL_PROB","EDGE","KELLY_HALF_UNITS"]])

print("\nTOP 10 HIGHEST EV")
display(top_ev[["PLAYER","PROP","OUTCOME","MODEL_PROB","EV_1U","KELLY_HALF_UNITS"]])

FileNotFoundError: [Errno 2] No such file or directory: 'Hitrate28.csv'

In [ ]:
     import pandas as pd
df = pd.read_csv("Hitrate28.csv")
print(df.columns.tolist())

['LAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS', 'AVG', 'IM PROB %', 'L5', 'L10', '2025', 'HM/AW', 'H2H', 'DVP L5', 'DVP L10', 'DVP L10.1', 'DVP HM/AW']


In [ ]:
print(df.columns.tolist())

['LAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS', 'AVG', 'IM PROB %', 'L5', 'L10', '2025', 'HM/AW', 'H2H', 'DVP L5', 'DVP L10', 'DVP L10.1', 'DVP HM/AW']


In [ ]:
df = pd.read_csv("Hitrate28.csv")

# Normalize columns hard
df.columns = (
    df.columns
      .str.strip()
      .str.replace(" ", "_")
      .str.replace("/", "_")
      .str.replace("%", "PCT")
)

# Force PLAYER column detection
for col in df.columns:
    if col.lower() in ["layer", "player"]:
        df.rename(columns={col: "PLAYER"}, inplace=True)

print(df.columns.tolist())

['PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS', 'AVG', 'IM_PROB_PCT', 'L5', 'L10', '2025', 'HM_AW', 'H2H', 'DVP_L5', 'DVP_L10', 'DVP_L10.1', 'DVP_HM_AW']


In [ ]:
import numpy as np
import pandas as pd

# ---------- helpers ----------
def pct_to_float(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.floating)):
        return float(x) if x <= 1 else float(x)/100
    s = str(x).strip().replace("%","")
    if s in ["--","-","–","—",""]:
        return np.nan
    try:
        v = float(s)
        return v/100 if v > 1 else v
    except:
        return np.nan

def american_to_prob(odds):
    if pd.isna(odds):
        return np.nan
    odds = float(odds)
    return 100/(odds+100) if odds > 0 else (-odds)/((-odds)+100)

def american_to_decimal(odds):
    if pd.isna(odds):
        return np.nan
    odds = float(odds)
    return 1 + odds/100 if odds > 0 else 1 + 100/(-odds)

def shrink(p, factor):
    if pd.isna(p):
        return np.nan
    return 0.5 + (p - 0.5) * factor

def clamp(p, lo=0.02, hi=0.98):
    if pd.isna(p):
        return np.nan
    return max(lo, min(hi, float(p)))

def ev_per_1u(p, dec):
    if pd.isna(p) or pd.isna(dec):
        return np.nan
    return float(p*(dec-1) - (1-p))

def kelly_frac(p, dec):
    if pd.isna(p) or pd.isna(dec) or dec <= 1:
        return np.nan
    b = dec - 1
    q = 1 - p
    f = (b*p - q)/b
    return max(0.0, float(f))

# ---------- config (Prop Engine v1) ----------
BASE_WEIGHTS = {"L10":0.25, "L5":0.15, "2025":0.25, "HM_AW":0.15, "H2H":0.05, "DVP":0.15}
L5_SHRINK = 0.75
L10_SHRINK = 0.90
DVP_DELTA_CAP = 0.15
KELLY_CAP = 0.05        # cap to 5% bankroll
UNIT_SIZE = 0.01        # 1u = 1% bankroll
MAX_UNITS = 5.0

# ---------- column fixes (safe) ----------
df.columns = [c.strip() for c in df.columns]

# Standardize any straggler column names
rename_map = {
    "IM PROB %":"IM_PROB_PCT",
    "HM/AW":"HM_AW",
    "DVP L5":"DVP_L5",
    "DVP L10":"DVP_L10",
    "DVP L10.1":"DVP_L10_2",
    "DVP HM/AW":"DVP_HM_AW",
}
df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns}, inplace=True)

# ---------- normalize percentage columns ----------
pct_cols = ["IM_PROB_PCT","L5","L10","2025","HM_AW","H2H","DVP_L5","DVP_L10","DVP_L10_2","DVP_HM_AW"]
for c in pct_cols:
    if c in df.columns:
        df[c] = df[c].apply(pct_to_float)
    else:
        df[c] = np.nan

# ---------- odds handling ----------
# Your file sometimes has ODDS like "BET\n -176" -> extract the number
if "ODDS_VALUE" in df.columns:
    df["AMERICAN_ODDS"] = pd.to_numeric(df["ODDS_VALUE"], errors="coerce")
else:
    df["AMERICAN_ODDS"] = df["ODDS"].astype(str).str.extract(r'(-?\d+)')[0].astype(float)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

# ---------- DVP composite with cap ----------
dvp_cols = ["DVP_L5","DVP_L10","DVP_L10_2","DVP_HM_AW"]
df["DVP_RAW"] = df[dvp_cols].mean(axis=1, skipna=True)
df["DVP"] = df["DVP_RAW"].apply(lambda x: np.nan if pd.isna(x) else 0.5 + np.clip(x-0.5, -DVP_DELTA_CAP, DVP_DELTA_CAP))

# ---------- shrink recent form to avoid L5 overfit ----------
df["L5_S"]  = df["L5"].apply(lambda x: shrink(x, L5_SHRINK))
df["L10_S"] = df["L10"].apply(lambda x: shrink(x, L10_SHRINK))

# ---------- model probability (renormalize weights if missing) ----------
model_probs = []
for _, r in df.iterrows():
    feats = {
        "L10": r["L10_S"],
        "L5": r["L5_S"],
        "2025": r["2025"],
        "HM_AW": r["HM_AW"],
        "H2H": r["H2H"],
        "DVP": r["DVP"]
    }
    valid = {k:v for k,v in feats.items() if not pd.isna(v)}
    if not valid:
        model_probs.append(np.nan)
        continue
    wsum = sum(BASE_WEIGHTS[k] for k in valid.keys())
    p = sum((BASE_WEIGHTS[k]/wsum) * valid[k] for k in valid.keys())
    model_probs.append(clamp(p))

df["MODEL_PROB"] = model_probs

# ---------- edge / EV / Kelly ----------
df["EDGE"] = df["MODEL_PROB"] - df["BOOK_PROB"]
df["EV_1U"] = df.apply(lambda x: ev_per_1u(x["MODEL_PROB"], x["DEC_ODDS"]), axis=1)

df["KELLY_FULL_FRAC"] = df.apply(lambda x: kelly_frac(x["MODEL_PROB"], x["DEC_ODDS"]), axis=1)
df["KELLY_FULL_FRAC"] = df["KELLY_FULL_FRAC"].clip(lower=0.0, upper=KELLY_CAP)
df["KELLY_HALF_FRAC"] = 0.5 * df["KELLY_FULL_FRAC"]

df["KELLY_FULL_UNITS"] = (df["KELLY_FULL_FRAC"]/UNIT_SIZE).clip(upper=MAX_UNITS)
df["KELLY_HALF_UNITS"] = (df["KELLY_HALF_FRAC"]/UNIT_SIZE).clip(upper=MAX_UNITS)

# ---------- outputs ----------
top_likely = df.sort_values(["MODEL_PROB","EDGE"], ascending=[False, False]).head(10)
top_ev     = df.sort_values(["EV_1U","EDGE"], ascending=[False, False]).head(10)

print("TOP 10 MOST LIKELY")
display(top_likely[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","MODEL_PROB","BOOK_PROB","EDGE","EV_1U","KELLY_HALF_UNITS"]])

print("\nTOP 10 HIGHEST EV")
display(top_ev[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","MODEL_PROB","BOOK_PROB","EDGE","EV_1U","KELLY_HALF_UNITS"]])

# optional: save outputs
df.to_csv("prop_engine_v1_full_output.csv", index=False)
top_likely.to_csv("prop_engine_v1_top10_most_likely.csv", index=False)
top_ev.to_csv("prop_engine_v1_top10_highest_ev.csv", index=False)
print("\nSaved: prop_engine_v1_full_output.csv, prop_engine_v1_top10_most_likely.csv, prop_engine_v1_top10_highest_ev.csv")

TOP 10 MOST LIKELY


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,MODEL_PROB,BOOK_PROB,EDGE,EV_1U,KELLY_HALF_UNITS
12,Josh Hubbard,MIZZ at MSST,Rebounds,u3.5,-143.0,0.759000,0.588477,0.170523,0.289769,2.5
0,Barrington Hargress,COLO at HOU,Points,o9.5,-176.0,0.756158,0.637681,0.118477,0.185793,2.5
10,Kevin Overton,MISS at AUB,Rebounds,u4.5,-162.0,0.751000,0.618321,0.132679,0.214580,2.5
9,Oziyah Sellers,VILL at SJU,Rebounds,u3.5,-170.0,0.740300,0.629630,0.110670,0.175771,2.5
39,Brayden Burries,KU at ARIZ,Rebounds,u5.5,-145.0,0.738750,0.591837,0.146913,0.248233,2.5
14,Rienk Mast,NEB at USC,Three Pointers Made,u1.5,-197.0,0.737053,0.663300,0.073753,0.111191,2.5
13,Brenen Lorient,BYU at WVU,Three Pointers Made,u0.5,-190.0,0.728579,0.655172,0.073407,0.112042,2.5
2,Brenen Lorient,BYU at WVU,Points,o9.5,-164.0,0.725737,0.621212,0.104525,0.168259,2.5
15,Jalen Haralson,NCST at ND,Rebounds,u4.5,-157.0,0.721526,0.610895,0.110631,0.181097,2.5
17,Amari Allen,ALA at TENN,Points,o9.5,-190.0,0.717105,0.655172,0.061933,0.094529,2.5



TOP 10 HIGHEST EV


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,MODEL_PROB,BOOK_PROB,EDGE,EV_1U,KELLY_HALF_UNITS
7,Jayden Epps,MIZZ at MSST,Points,u14.5,-104.0,0.713350,0.509804,0.203546,0.399263,2.5
55,Bryce Lindsay,VILL at SJU,Rebounds,u1.5,126.0,0.609950,0.442478,0.167472,0.378487,2.5
24,Jaden Bradley,KU at ARIZ,Three Pointers Made,u0.5,123.0,0.614850,0.448430,0.166420,0.371116,2.5
81,Isaiah Johnson,COLO at HOU,Points,o14.5,128.0,0.598737,0.438596,0.160140,0.365120,2.5
23,Sebastian Williams-Adams,MISS at AUB,Rebounds,u3.5,120.0,0.604400,0.454545,0.149855,0.329680,2.5
12,Josh Hubbard,MIZZ at MSST,Rebounds,u3.5,-143.0,0.759000,0.588477,0.170523,0.289769,2.5
92,Ezra Ausar,NEB at USC,Rebounds,u7.5,-102.0,0.645211,0.504950,0.140260,0.277770,2.5
90,AJ Dybantsa,BYU at WVU,Rebounds,u7.5,108.0,0.613421,0.480769,0.132652,0.275916,2.5
69,Zuby Ejiofor,VILL at SJU,Points,u15.5,-108.0,0.652500,0.519231,0.133269,0.256667,2.5
32,Ja'Kobi Gillespie,ALA at TENN,Points,u20.5,-120.0,0.683550,0.545455,0.138095,0.253175,2.5



Saved: prop_engine_v1_full_output.csv, prop_engine_v1_top10_most_likely.csv, prop_engine_v1_top10_highest_ev.csv


In [ ]:
import pandas as pd
import numpy as np
import itertools

# ---------- Parlay settings ----------
PARLAY_POOL_SOURCE = "EV"     # "EV" or "LIKELY"
PARLAY_POOL_N = 14            # pool size used for combo search
PARLAY_LEGS = [2,3,4]
TOP_PARLAYS_EACH = 15

AVOID_SAME_PLAYER = True
AVOID_SAME_GAME = True   # set False if you want same-game allowed

# ---------- helpers ----------
def expected_value_per_1u(p, dec_odds):
    if pd.isna(p) or pd.isna(dec_odds):
        return np.nan
    return float(p*(dec_odds-1) - (1-p))

def kelly_fraction(p, dec_odds):
    if pd.isna(p) or pd.isna(dec_odds) or dec_odds <= 1:
        return np.nan
    b = dec_odds - 1
    q = 1 - p
    f = (b*p - q)/b
    return max(0.0, float(f))

# ---------- choose pool ----------
if PARLAY_POOL_SOURCE == "LIKELY":
    pool = df.sort_values(["MODEL_PROB","EDGE"], ascending=[False, False]).head(PARLAY_POOL_N).copy()
else:
    pool = df.sort_values(["EV_1U","EDGE"], ascending=[False, False]).head(PARLAY_POOL_N).copy()

idxs = list(pool.index)

# ---------- build parlays (independence assumption v1) ----------
parlay_rows = {2:[],3:[],4:[]}

for legs in PARLAY_LEGS:
    for combo in itertools.combinations(idxs, legs):
        sub = pool.loc[list(combo)]

        if AVOID_SAME_PLAYER and sub["PLAYER"].nunique() < legs:
            continue
        if AVOID_SAME_GAME and sub["GAME"].nunique() < legs:
            continue

        joint_p = float(np.prod(sub["MODEL_PROB"].values))
        joint_book_p = float(np.prod(sub["BOOK_PROB"].values))
        joint_dec = float(np.prod(sub["DEC_ODDS"].values))

        ev = expected_value_per_1u(joint_p, joint_dec)
        edge = joint_p - joint_book_p
        kelly = kelly_fraction(joint_p, joint_dec)

        parlay_rows[legs].append({
            "LEGS": legs,
            "PARLAY_MODEL_PROB": joint_p,
            "PARLAY_BOOK_PROB": joint_book_p,
            "PARLAY_EDGE": edge,
            "PARLAY_DEC_ODDS": joint_dec,
            "PARLAY_EV_1U": ev,
            "PARLAY_KELLY_FULL": kelly,
            "LEGS_DETAIL": " | ".join(
                f'{r["PLAYER"]} ({r["GAME"]}) {r["PROP"]} {r["OUTCOME"]} @ {int(r["AMERICAN_ODDS"])}'
                for _, r in sub.iterrows()
            )
        })

# ---------- format + display + save ----------
for legs in PARLAY_LEGS:
    parlays_df = pd.DataFrame(parlay_rows[legs])
    if parlays_df.empty:
        print(f"\nNo valid {legs}-leg parlays found with current constraints.")
        continue

    parlays_df["PARLAY_MODEL_PROB_PCT"] = (parlays_df["PARLAY_MODEL_PROB"]*100).round(2)
    parlays_df["PARLAY_BOOK_PROB_PCT"] = (parlays_df["PARLAY_BOOK_PROB"]*100).round(2)
    parlays_df["PARLAY_EDGE_PCTPTS"] = (parlays_df["PARLAY_EDGE"]*100).round(2)
    parlays_df["PARLAY_EV_1U"] = parlays_df["PARLAY_EV_1U"].round(4)

    parlays_df = parlays_df.sort_values(["PARLAY_EV_1U","PARLAY_EDGE"], ascending=[False, False]).head(TOP_PARLAYS_EACH)

    print(f"\nTOP {TOP_PARLAYS_EACH} — {legs}-LEG PARLAYS (Pool={PARLAY_POOL_SOURCE}, N={PARLAY_POOL_N})")
    display(parlays_df[[
        "LEGS_DETAIL",
        "PARLAY_MODEL_PROB_PCT",
        "PARLAY_BOOK_PROB_PCT",
        "PARLAY_EDGE_PCTPTS",
        "PARLAY_EV_1U",
        "PARLAY_DEC_ODDS"
    ]])

    out_name = f"prop_engine_v1_parlays_{legs}leg.csv"
    parlays_df.to_csv(out_name, index=False)
    print(f"Saved: {out_name}")


TOP 15 — 2-LEG PARLAYS (Pool=EV, N=14)


,LEGS_DETAIL,PARLAY_MODEL_PROB_PCT,PARLAY_BOOK_PROB_PCT,PARLAY_EDGE_PCTPTS,PARLAY_EV_1U,PARLAY_DEC_ODDS
0,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,43.51,22.56,20.95,0.9289,4.433077
1,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,43.86,22.86,21.00,0.9186,4.374231
2,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,42.71,22.36,20.35,0.9102,4.472308
12,Bryce Lindsay (VILL at SJU) Rebounds u1.5 @ 12...,37.50,19.84,17.66,0.8901,5.039800
13,Bryce Lindsay (VILL at SJU) Rebounds u1.5 @ 12...,36.52,19.41,17.11,0.8818,5.152800
23,Jaden Bradley (KU at ARIZ) Three Pointers Made...,36.81,19.67,17.15,0.8717,5.084400
3,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,43.11,23.17,19.94,0.8606,4.315385
14,Bryce Lindsay (VILL at SJU) Rebounds u1.5 @ 12...,36.87,20.11,16.75,0.8329,4.972000
24,Jaden Bradley (KU at ARIZ) Three Pointers Made...,37.16,20.38,16.78,0.8231,4.906000
32,Isaiah Johnson (COLO at HOU) Points o14.5 @ 12...,36.19,19.94,16.25,0.8152,5.016000


Saved: prop_engine_v1_parlays_2leg.csv

TOP 15 — 3-LEG PARLAYS (Pool=EV, N=14)


,LEGS_DETAIL,PARLAY_MODEL_PROB_PCT,PARLAY_BOOK_PROB_PCT,PARLAY_EDGE_PCTPTS,PARLAY_EV_1U,PARLAY_DEC_ODDS
0,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,26.75,10.12,16.64,1.6447,9.885762
1,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,26.05,9.89,16.16,1.6331,10.107415
10,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,26.26,10.03,16.23,1.6191,9.973246
61,Bryce Lindsay (VILL at SJU) Rebounds u1.5 @ 12...,22.45,8.70,13.75,1.5802,11.490744
2,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,26.30,10.25,16.04,1.5648,9.752769
11,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,26.51,10.39,16.12,1.5511,9.623308
18,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,25.81,10.16,15.65,1.5399,9.839077
62,Bryce Lindsay (VILL at SJU) Rebounds u1.5 @ 12...,22.67,9.02,13.65,1.5132,11.087560
69,Bryce Lindsay (VILL at SJU) Rebounds u1.5 @ 12...,22.07,8.82,13.25,1.5022,11.336160
112,Jaden Bradley (KU at ARIZ) Three Pointers Made...,22.25,8.94,13.31,1.4888,11.185680


Saved: prop_engine_v1_parlays_3leg.csv

TOP 15 — 4-LEG PARLAYS (Pool=EV, N=14)


,LEGS_DETAIL,PARLAY_MODEL_PROB_PCT,PARLAY_BOOK_PROB_PCT,PARLAY_EDGE_PCTPTS,PARLAY_EV_1U,PARLAY_DEC_ODDS
0,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,16.02,4.44,11.58,2.6103,22.539536
1,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,16.17,4.60,11.57,2.5166,21.748675
7,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,15.75,4.50,11.25,2.5012,22.236314
41,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,15.87,4.56,11.31,2.4825,21.941142
172,Bryce Lindsay (VILL at SJU) Rebounds u1.5 @ 12...,13.57,3.96,9.62,2.4308,25.279637
2,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,17.26,5.11,12.15,2.3793,19.577685
3,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,16.41,4.86,11.55,2.3744,20.562384
8,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,16.81,5.00,11.81,2.3645,20.016646
9,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,15.98,4.76,11.22,2.3597,21.023424
42,Jayden Epps (MIZZ at MSST) Points u14.5 @ -104...,16.94,5.06,11.88,2.3465,19.750938


Saved: prop_engine_v1_parlays_4leg.csv


In [ ]:
import pandas as pd
import numpy as np
import requests
import re

# --- Load your base sheet ---
df = pd.read_csv("Hitrate28.csv")
df.columns = [c.strip() for c in df.columns]
if "LAYER" in df.columns and "PLAYER" not in df.columns:
    df = df.rename(columns={"LAYER":"PLAYER"})

# --- ESPN endpoints ---
SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"
SUMMARY    = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary"

# Slate date(s) - set these
DATES = ["20260228"]  # YYYYMMDD

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def normalize_game_str(s):
    # Your format is like "MIZZ at MSST" or "COLO at HOU"
    return re.sub(r"\s+", " ", s.strip().upper())

# ---- 1) Pull scoreboard events for the slate date(s) ----
events = []
for d in DATES:
    sb = get_json(SCOREBOARD, params={"dates": d})
    for ev in sb.get("events", []):
        comp = ev["competitions"][0]
        home = comp["competitors"][0]
        away = comp["competitors"][1]
        # ESPN doesn't guarantee ordering in all sports; so detect by "homeAway"
        for c in comp["competitors"]:
            if c["homeAway"] == "home":
                home = c
            elif c["homeAway"] == "away":
                away = c

        events.append({
            "date": d,
            "eventId": ev["id"],
            "home_abbrev": home["team"].get("abbreviation","").upper(),
            "away_abbrev": away["team"].get("abbreviation","").upper(),
            "home_name": home["team"].get("displayName",""),
            "away_name": away["team"].get("displayName",""),
        })

events_df = pd.DataFrame(events)
events_df["GAME_KEY"] = events_df["away_abbrev"] + " AT " + events_df["home_abbrev"]

# ---- 2) Match your df rows to ESPN eventId via GAME ----
df["GAME_NORM"] = df["GAME"].apply(normalize_game_str)
# Convert "MIZZ at MSST" -> "MIZZ AT MSST"
df["GAME_KEY"] = df["GAME_NORM"].str.replace(" AT ", " AT ", regex=False)

df = df.merge(
    events_df[["eventId","GAME_KEY","date","home_abbrev","away_abbrev","home_name","away_name"]],
    on="GAME_KEY",
    how="left"
)

print("Rows missing eventId:", df["eventId"].isna().sum())

# ---- 3) Pull boxscores for matched eventIds and extract player stat lines ----
player_rows = []
unique_events = df["eventId"].dropna().astype(str).unique().tolist()

for eid in unique_events:
    summ = get_json(SUMMARY, params={"event": eid})
    # Boxscore player stats typically live under boxscore -> players
    box = summ.get("boxscore", {})
    players = box.get("players", [])
    for team_block in players:
        team = team_block.get("team", {})
        team_abbrev = team.get("abbreviation","").upper()
        for stat_group in team_block.get("statistics", []):
            # stat_group has "athletes" with "stats"
            for ath in stat_group.get("athletes", []):
                athlete = ath.get("athlete", {})
                name = athlete.get("displayName","")
                # Stats array meaning depends on group; use "labels" if present
                labels = stat_group.get("labels", [])
                stats  = ath.get("stats", [])
                row = {"eventId": eid, "team_abbrev": team_abbrev, "PLAYER_ESPN": name}
                for i, lab in enumerate(labels):
                    row[lab] = stats[i] if i < len(stats) else None
                player_rows.append(row)

box_df = pd.DataFrame(player_rows)

# ---- 4) Join boxscore rows to your prop rows by player name (basic v1 match) ----
def norm_name(n):
    return re.sub(r"[^A-Z ]","", str(n).upper()).strip()

df["PLAYER_NORM"] = df["PLAYER"].apply(norm_name)
box_df["PLAYER_NORM"] = box_df["PLAYER_ESPN"].apply(norm_name)

# Keep only box rows for the same eventId
df = df.merge(
    box_df,
    on=["eventId","PLAYER_NORM"],
    how="left",
    suffixes=("","_box")
)

print("Rows missing ESPN player match:", df["PLAYER_ESPN"].isna().sum())

# Save enriched base
df.to_csv("prop_engine_enriched_base.csv", index=False)
print("Saved: prop_engine_enriched_base.csv")

Rows missing eventId: 21
Rows missing ESPN player match: 92
Saved: prop_engine_enriched_base.csv


In [ ]:
# --- Rebuild GAME matching using team name fallback ---

def build_game_keys(events_df):
    rows = []
    for _, r in events_df.iterrows():
        rows.append({
            "eventId": r["eventId"],
            "GAME_KEY_ABBR": f'{r["away_abbrev"]} AT {r["home_abbrev"]}',
            "GAME_KEY_NAME": f'{r["away_name"].upper()} AT {r["home_name"].upper()}'
        })
    return pd.DataFrame(rows)

game_keys = build_game_keys(events_df)

df["GAME_NORM"] = df["GAME"].str.upper().str.strip()
df["GAME_NORM"] = df["GAME_NORM"].str.replace(" AT ", " AT ")

# Try abbreviation match first
df = df.merge(
    game_keys[["eventId","GAME_KEY_ABBR"]],
    left_on="GAME_NORM",
    right_on="GAME_KEY_ABBR",
    how="left"
)

# If still missing, try name match
missing = df["eventId"].isna()
df.loc[missing, "eventId"] = df[missing].merge(
    game_keys,
    left_on="GAME_NORM",
    right_on="GAME_KEY_NAME",
    how="left"
)["eventId_y"].values

print("Rows missing eventId after fallback:", df["eventId"].isna().sum())

KeyError: 'eventId'

In [ ]:
print("df columns:", df.columns.tolist())


df columns: ['PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS', 'AVG', 'IM PROB %', 'L5', 'L10', '2025', 'HM/AW', 'H2H', 'DVP L5', 'DVP L10', 'DVP L10.1', 'DVP HM/AW', 'GAME_NORM', 'GAME_KEY', 'eventId_x', 'date', 'home_abbrev', 'away_abbrev', 'home_name', 'away_name', 'PLAYER_NORM', 'team_abbrev', 'PLAYER_ESPN', 'MIN', 'PTS', 'FG', '3PT', 'FT', 'REB', 'AST', 'TO', 'STL', 'BLK', 'OREB', 'DREB', 'PF', 'eventId_y', 'GAME_KEY_ABBR']


In [ ]:
import pandas as pd
import requests
import re

SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"
DATES = ["20260228"]  # set slate date YYYYMMDD

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def normalize_game_str(s):
    return re.sub(r"\s+", " ", s.strip().upper())

# Pull scoreboard
events = []
for d in DATES:
    sb = get_json(SCOREBOARD, params={"dates": d})
    for ev in sb.get("events", []):
        comp = ev["competitions"][0]
        home = None
        away = None
        for c in comp["competitors"]:
            if c["homeAway"] == "home":
                home = c
            if c["homeAway"] == "away":
                away = c
        events.append({
            "date": d,
            "eventId": ev["id"],
            "home_abbrev": home["team"].get("abbreviation","").upper(),
            "away_abbrev": away["team"].get("abbreviation","").upper(),
            "home_name": home["team"].get("displayName",""),
            "away_name": away["team"].get("displayName",""),
        })

events_df = pd.DataFrame(events)
events_df["GAME_KEY"] = events_df["away_abbrev"] + " AT " + events_df["home_abbrev"]

# Prepare df GAME_KEY
df["GAME_NORM"] = df["GAME"].apply(normalize_game_str)
df["GAME_KEY"] = df["GAME_NORM"].str.replace(" AT ", " AT ", regex=False)

# Merge in eventId
df = df.merge(
    events_df[["eventId","GAME_KEY","date","home_abbrev","away_abbrev","home_name","away_name"]],
    on="GAME_KEY",
    how="left"
)

print("eventId created. Missing eventId:", df["eventId"].isna().sum())

eventId created. Missing eventId: 21


In [ ]:
# If merge created eventId_x/eventId_y, consolidate them into eventId
if "eventId" not in df.columns:
    if "eventId_x" in df.columns and "eventId_y" in df.columns:
        df["eventId"] = df["eventId_x"].fillna(df["eventId_y"])
    elif "eventId_x" in df.columns:
        df["eventId"] = df["eventId_x"]
    elif "eventId_y" in df.columns:
        df["eventId"] = df["eventId_y"]

print("Missing eventId:", df["eventId"].isna().sum())

Missing eventId: 21


In [ ]:
# Build fallback game keys using both abbrev + full name
game_keys = events_df.copy()
game_keys["GAME_KEY_ABBR"] = game_keys["away_abbrev"] + " AT " + game_keys["home_abbrev"]
game_keys["GAME_KEY_NAME"] = game_keys["away_name"].str.upper() + " AT " + game_keys["home_name"].str.upper()

# Build a "name style" version of your GAME field
df["GAME_KEY_NAME"] = df["GAME"].str.upper().str.strip()

# Fill missing eventId using name match
missing = df["eventId"].isna()
df.loc[missing, "eventId"] = df.loc[missing].merge(
    game_keys[["eventId","GAME_KEY_NAME"]],
    left_on="GAME_KEY_NAME",
    right_on="GAME_KEY_NAME",
    how="left"
)["eventId"].values

print("Missing eventId after fallback:", df["eventId"].isna().sum())

MergeError: Passing 'suffixes' which cause duplicate columns {'eventId_x'} is not allowed.

In [ ]:
import pandas as pd
import numpy as np
import re

# --- 0) Normalize df columns and consolidate eventId if it got suffixed ---
df.columns = [c.strip() for c in df.columns]

# If previous merges created eventId_x / eventId_y, consolidate
if "eventId" not in df.columns:
    if "eventId_x" in df.columns and "eventId_y" in df.columns:
        df["eventId"] = df["eventId_x"].fillna(df["eventId_y"])
    elif "eventId_x" in df.columns:
        df["eventId"] = df["eventId_x"]
    elif "eventId_y" in df.columns:
        df["eventId"] = df["eventId_y"]
    else:
        df["eventId"] = np.nan

# --- 1) Build GAME keys on BOTH sides ---
def normalize_game_str(s):
    return re.sub(r"\s+", " ", str(s).strip().upper())

df["GAME_NORM"] = df["GAME"].apply(normalize_game_str)  # ex: "MIZZ AT MSST"

# events_df must already exist from scoreboard pull
events_df["GAME_KEY_ABBR"] = events_df["away_abbrev"].str.upper() + " AT " + events_df["home_abbrev"].str.upper()
events_df["GAME_KEY_NAME"] = events_df["away_name"].str.upper() + " AT " + events_df["home_name"].str.upper()

# --- 2) Build lookup dicts (no merges = no suffix errors) ---
abbr_map = dict(zip(events_df["GAME_KEY_ABBR"], events_df["eventId"]))
name_map = dict(zip(events_df["GAME_KEY_NAME"], events_df["eventId"]))

# --- 3) Fill missing eventId from abbreviation key first ---
missing = df["eventId"].isna()
df.loc[missing, "eventId"] = df.loc[missing, "GAME_NORM"].map(abbr_map)

# --- 4) Fill remaining missing from full-name key (only helps if df GAME is full names) ---
missing2 = df["eventId"].isna()
df.loc[missing2, "eventId"] = df.loc[missing2, "GAME_NORM"].map(name_map)

print("Missing eventId after dict fallback:", df["eventId"].isna().sum())

Missing eventId after dict fallback: 21


In [ ]:
print("Missing eventId:", df["eventId"].isna().sum())

Missing eventId: 21


In [ ]:
# Show unmatched rows/games
unmatched = df[df["eventId"].isna()].copy()
print("Unmatched rows:", len(unmatched))
print("\nUnique unmatched GAME strings:")
print("\n".join(sorted(unmatched["GAME_NORM"].unique())))

Unmatched rows: 21

Unique unmatched GAME strings:
GTWN AT XAV
MISS AT AUB
MIZZ AT MSST
NCST AT ND
TEX AT TXAM
UCLA AT MINN
WIS AT WASH


In [ ]:
import difflib

# Build a set of ESPN abbreviations we can match to
espn_abbrs = sorted(set(events_df["home_abbrev"].str.upper()).union(set(events_df["away_abbrev"].str.upper())))

def token_suggestions(token, n=5):
    token = token.upper().strip()
    return difflib.get_close_matches(token, espn_abbrs, n=n, cutoff=0.0)

# Extract tokens from unmatched games: "AAA AT BBB" -> ["AAA","BBB"]
tokens = set()
for g in sorted(df[df["eventId"].isna()]["GAME_NORM"].unique()):
    if " AT " in g:
        a, h = g.split(" AT ")
        tokens.add(a.strip())
        tokens.add(h.strip())

tokens = sorted(tokens)

for t in tokens:
    print(t, "->", token_suggestions(t, n=6))

AUB -> ['UK', 'KU', 'WVU', 'VAN', 'UVA', 'USC']
GTWN -> ['TENN', 'GONZ', 'VT', 'WVU', 'VAN', 'UNC']
MINN -> ['TENN', 'CONN', 'VAN', 'UNC', 'SMC', 'NEB']
MISS -> ['ISU', 'USC', 'SMC', 'SLU', 'SJU', 'VILL']
MIZZ -> ['ARIZ', 'SMC', 'ISU', 'VILL', 'GONZ', 'CLEM']
MSST -> ['VT', 'USC', 'TTU', 'SMC', 'SLU', 'SJU']
NCST -> ['UNC', 'VT', 'VAN', 'USC', 'TTU', 'SMC']
ND -> ['VAN', 'UNC', 'NEB', 'DUQ', 'TENN', 'GONZ']
TEX -> ['TENN', 'VT', 'TTU', 'NEB', 'DUKE', 'CLEM']
TXAM -> ['VT', 'VAN', 'UVA', 'TTU', 'SMC', 'FLA']
UCLA -> ['UVA', 'USC', 'UNC', 'FLA', 'ALA', 'COLO']
WASH -> ['WVU', 'VAN', 'UVA', 'USC', 'SMC', 'SLU']
WIS -> ['ISU', 'WVU', 'USC', 'SMC', 'SLU', 'SJU']
XAV -> ['VT', 'WVU', 'VAN', 'UVA', 'FLA', 'ARK']


In [ ]:
# Fill this using the suggestions you see above
ALIAS = {
    # examples (DO NOT assume these are correct for your slate — use suggestions output)
    # "MIZZ": "MIZ",
    # "MSST": "MSST",
    # "TXAM": "TAMU",
    # "ARIZ": "ARIZ",
    # "VILL": "NOVA",
}

def apply_alias_to_game(game_norm):
    # "AAA AT BBB"
    if " AT " not in game_norm:
        return game_norm
    away, home = game_norm.split(" AT ")
    away2 = ALIAS.get(away.strip(), away.strip())
    home2 = ALIAS.get(home.strip(), home.strip())
    return f"{away2} AT {home2}"

# Apply alias-adjusted game key
df["GAME_KEY_ALIAS"] = df["GAME_NORM"].apply(apply_alias_to_game)

# Build ESPN game key -> eventId map
events_df["GAME_KEY_ABBR"] = events_df["away_abbrev"].str.upper() + " AT " + events_df["home_abbrev"].str.upper()
abbr_map = dict(zip(events_df["GAME_KEY_ABBR"], events_df["eventId"]))

# Re-fill missing eventId using alias key
missing = df["eventId"].isna()
df.loc[missing, "eventId"] = df.loc[missing, "GAME_KEY_ALIAS"].map(abbr_map)

print("Missing eventId after alias mapping:", df["eventId"].isna().sum())

Missing eventId after alias mapping: 21


In [ ]:
import pandas as pd
import numpy as np
import re
from difflib import SequenceMatcher

# df must already have: GAME (your string), eventId (currently NaN for 21), and events_df from scoreboard pull
# events_df must have: eventId, away_abbrev, home_abbrev, away_name, home_name

def norm(s):
    return re.sub(r"\s+", " ", str(s).upper().strip())

def sim(a, b):
    a, b = norm(a), norm(b)
    if a == b:
        return 1.0
    return SequenceMatcher(None, a, b).ratio()

def parse_game(game_str):
    g = norm(game_str).replace(" VS ", " AT ")
    if " AT " in g:
        away, home = g.split(" AT ", 1)
        return away.strip(), home.strip()
    return None, None

# Pre-normalize events_df strings for scoring
events_df = events_df.copy()
events_df["AWAY_ABBR_N"] = events_df["away_abbrev"].astype(str).apply(norm)
events_df["HOME_ABBR_N"] = events_df["home_abbrev"].astype(str).apply(norm)
events_df["AWAY_NAME_N"] = events_df["away_name"].astype(str).apply(norm)
events_df["HOME_NAME_N"] = events_df["home_name"].astype(str).apply(norm)

def best_event_for_game(game_str):
    away_tok, home_tok = parse_game(game_str)
    if away_tok is None:
        return None, 0.0

    best_id = None
    best_score = -1

    for _, e in events_df.iterrows():
        # Score away/home against BOTH abbrev and name
        away_score = max(sim(away_tok, e["AWAY_ABBR_N"]), sim(away_tok, e["AWAY_NAME_N"]))
        home_score = max(sim(home_tok, e["HOME_ABBR_N"]), sim(home_tok, e["HOME_NAME_N"]))

        # Combine (weighted slightly toward abbrev/name match)
        score = 0.5 * away_score + 0.5 * home_score

        if score > best_score:
            best_score = score
            best_id = e["eventId"]

    return best_id, best_score

# Apply only to rows still missing
missing_mask = df["eventId"].isna()
candidates = df.loc[missing_mask, "GAME"].apply(best_event_for_game)

df.loc[missing_mask, "eventId_candidate"] = candidates.apply(lambda x: x[0])
df.loc[missing_mask, "eventId_score"] = candidates.apply(lambda x: x[1])

# Use a threshold so we don't assign garbage
THRESH = 0.78  # if your abbreviations are very different, lower to 0.72
assign_mask = missing_mask & (df["eventId_score"] >= THRESH)
df.loc[assign_mask, "eventId"] = df.loc[assign_mask, "eventId_candidate"]

print("Missing eventId after fuzzy match:", df["eventId"].isna().sum())

# Show what is still missing + the top suggestion
still = df[df["eventId"].isna()][["GAME"]].copy()
if len(still) > 0:
    still["best_candidate_eventId"] = df.loc[df["eventId"].isna(), "eventId_candidate"].values
    still["best_score"] = df.loc[df["eventId"].isna(), "eventId_score"].values
    print("\nStill missing (showing up to 25):")
    display(still.head(25))

Missing eventId after fuzzy match: 21

Still missing (showing up to 25):


,GAME,best_candidate_eventId,best_score
3,UCLA at MINN,401808271,0.535714
7,MIZZ at MSST,401822962,0.267857
10,MISS at AUB,401822962,0.291667
12,MIZZ at MSST,401822962,0.267857
15,NCST at ND,401820774,0.366667
23,MISS at AUB,401822962,0.291667
33,TEX at TXAM,401825548,0.300000
37,TEX at TXAM,401825548,0.300000
38,TEX at TXAM,401825548,0.300000
41,TEX at TXAM,401825548,0.300000


In [ ]:
print("ESPN events pulled:", len(events_df))
print("Unique games in your sheet:", df['GAME'].nunique())

ESPN events pulled: 15
Unique games in your sheet: 20


In [ ]:
import pandas as pd
import requests
from datetime import datetime, timedelta

SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

# --- pick a window of dates around "today" ---
# If you know the slate date, set CENTER_DATE = "20260228"
CENTER_DATE = None  # or "YYYYMMDD"

def yyyymmdd(dt):
    return dt.strftime("%Y%m%d")

if CENTER_DATE:
    center = datetime.strptime(CENTER_DATE, "%Y%m%d")
else:
    center = datetime.now()

date_list = [yyyymmdd(center + timedelta(days=k)) for k in [-2, -1, 0, 1, 2]]

events = []
for d in date_list:
    sb = get_json(SCOREBOARD, params={"dates": d})
    for ev in sb.get("events", []):
        comp = ev["competitions"][0]
        home = next(c for c in comp["competitors"] if c["homeAway"] == "home")
        away = next(c for c in comp["competitors"] if c["homeAway"] == "away")
        events.append({
            "date": d,
            "eventId": ev["id"],
            "home_abbrev": home["team"].get("abbreviation","").upper(),
            "away_abbrev": away["team"].get("abbreviation","").upper(),
            "home_name": home["team"].get("displayName",""),
            "away_name": away["team"].get("displayName",""),
        })

events_df = pd.DataFrame(events).drop_duplicates(subset=["eventId"]).reset_index(drop=True)

print("Dates pulled:", date_list)
print("ESPN events pulled:", len(events_df))
print("Unique games in your sheet:", df["GAME"].nunique())

Dates pulled: ['20260226', '20260227', '20260228', '20260301', '20260302']
ESPN events pulled: 22
Unique games in your sheet: 20


In [ ]:
import re
from difflib import SequenceMatcher
import numpy as np

def norm(s):
    return re.sub(r"\s+", " ", str(s).upper().strip())

def sim(a, b):
    a, b = norm(a), norm(b)
    if a == b:
        return 1.0
    return SequenceMatcher(None, a, b).ratio()

def parse_game(game_str):
    g = norm(game_str)
    if " AT " in g:
        away, home = g.split(" AT ", 1)
        return away.strip(), home.strip()
    return None, None

events_df2 = events_df.copy()
events_df2["AWAY_ABBR_N"] = events_df2["away_abbrev"].apply(norm)
events_df2["HOME_ABBR_N"] = events_df2["home_abbrev"].apply(norm)
events_df2["AWAY_NAME_N"] = events_df2["away_name"].apply(norm)
events_df2["HOME_NAME_N"] = events_df2["home_name"].apply(norm)

def best_event_for_game(game_str):
    away_tok, home_tok = parse_game(game_str)
    if away_tok is None:
        return None, 0.0
    best_id, best_score = None, -1
    for _, e in events_df2.iterrows():
        away_score = max(sim(away_tok, e["AWAY_ABBR_N"]), sim(away_tok, e["AWAY_NAME_N"]))
        home_score = max(sim(home_tok, e["HOME_ABBR_N"]), sim(home_tok, e["HOME_NAME_N"]))
        score = 0.5*away_score + 0.5*home_score
        if score > best_score:
            best_score = score
            best_id = e["eventId"]
    return best_id, best_score

# Ensure eventId exists
if "eventId" not in df.columns:
    df["eventId"] = np.nan

missing_mask = df["eventId"].isna()
cands = df.loc[missing_mask, "GAME"].apply(best_event_for_game)

df.loc[missing_mask, "eventId_candidate"] = cands.apply(lambda x: x[0])
df.loc[missing_mask, "eventId_score"] = cands.apply(lambda x: x[1])

THRESH = 0.72
assign = missing_mask & (df["eventId_score"] >= THRESH)
df.loc[assign, "eventId"] = df.loc[assign, "eventId_candidate"]

print("Missing eventId after expanded fuzzy match:", df["eventId"].isna().sum())

still = df[df["eventId"].isna()][["GAME","eventId_candidate","eventId_score"]].drop_duplicates()
if len(still):
    print("\nStill missing (unique games):")
    display(still)

Missing eventId after expanded fuzzy match: 21

Still missing (unique games):


,GAME,eventId_candidate,eventId_score
3,UCLA at MINN,401808271,0.535714
7,MIZZ at MSST,401825544,0.321429
10,MISS at AUB,401825550,0.485714
15,NCST at ND,401820774,0.366667
33,TEX at TXAM,401825548,0.300000
50,WIS at WASH,401820821,0.483333
85,GTWN at XAV,401829265,0.297619


In [ ]:
import pandas as pd
import requests
from datetime import datetime, timedelta

SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def yyyymmdd(dt):
    return dt.strftime("%Y%m%d")

# If you know your slate date, set it here (recommended)
CENTER_DATE = None   # e.g. "20260228"

if CENTER_DATE:
    center = datetime.strptime(CENTER_DATE, "%Y%m%d")
else:
    center = datetime.now()

date_list = [yyyymmdd(center + timedelta(days=k)) for k in range(-14, 15)]

events = []
for d in date_list:
    sb = get_json(SCOREBOARD, params={"dates": d})
    for ev in sb.get("events", []):
        comp = ev["competitions"][0]
        home = next(c for c in comp["competitors"] if c["homeAway"] == "home")
        away = next(c for c in comp["competitors"] if c["homeAway"] == "away")
        events.append({
            "date": d,
            "eventId": ev["id"],
            "home_abbrev": home["team"].get("abbreviation",""),
            "away_abbrev": away["team"].get("abbreviation",""),
            "home_name": home["team"].get("displayName",""),
            "away_name": away["team"].get("displayName",""),
        })

events_df = pd.DataFrame(events).drop_duplicates(subset=["eventId"]).reset_index(drop=True)
print("Dates pulled:", date_list[0], "to", date_list[-1])
print("ESPN events pulled:", len(events_df))
print("Unique games in your sheet:", df["GAME"].nunique())

Dates pulled: 20260214 to 20260314
ESPN events pulled: 370
Unique games in your sheet: 20


In [ ]:
import numpy as np
import re

def norm_team_token(s):
    # upper, remove non-alphanum (handles TA&M -> TAM)
    return re.sub(r"[^A-Z0-9]", "", str(s).upper().strip())

def parse_game_tokens(game_str):
    g = str(game_str).upper().strip()
    if " AT " in g:
        a, h = g.split(" AT ", 1)
        return norm_team_token(a), norm_team_token(h)
    return None, None

# Strong starter alias map for your sheet style -> ESPN style tokens
# You can extend this later, but these cover your missing list.
ALIAS = {
    "MIZZ": "MIZ",     # Missouri
    "TXAM": "TAM",     # Texas A&M (TA&M -> TAM)
    "GTWN": "GU",      # Georgetown
}

# Build ESPN token keys
events_df = events_df.copy()
events_df["away_tok"] = events_df["away_abbrev"].apply(norm_team_token)
events_df["home_tok"] = events_df["home_abbrev"].apply(norm_team_token)

# Build mapping from (away_tok, home_tok) -> eventId (if duplicates exist, last wins; fine for this use)
pair_map = {(r.away_tok, r.home_tok): r.eventId for r in events_df.itertuples(index=False)}

# Ensure df has eventId column
if "eventId" not in df.columns:
    df["eventId"] = np.nan

# Create tokens from your GAME field
df["away_tok"], df["home_tok"] = zip(*df["GAME"].apply(parse_game_tokens))

# Apply alias to your tokens
df["away_tok2"] = df["away_tok"].apply(lambda x: norm_team_token(ALIAS.get(x, x)))
df["home_tok2"] = df["home_tok"].apply(lambda x: norm_team_token(ALIAS.get(x, x)))

# Fill missing eventId using exact token pair match
missing = df["eventId"].isna()
df.loc[missing, "eventId"] = df.loc[missing].apply(
    lambda r: pair_map.get((r["away_tok2"], r["home_tok2"])), axis=1
)

print("Missing eventId after token match:", df["eventId"].isna().sum())

# Show remaining missing unique games (if any)
still = df[df["eventId"].isna()][["GAME","away_tok2","home_tok2"]].drop_duplicates()
if len(still):
    print("\nStill missing (unique):")
    display(still)

Missing eventId after token match: 21

Still missing (unique):


,GAME,away_tok2,home_tok2
3,UCLA at MINN,UCLA,MINN
7,MIZZ at MSST,MIZ,MSST
10,MISS at AUB,MISS,AUB
15,NCST at ND,NCST,ND
33,TEX at TXAM,TEX,TAM
50,WIS at WASH,WIS,WASH
85,GTWN at XAV,GU,XAV


In [ ]:
import numpy as np
import re

def norm_team_token(s):
    # upper, remove non-alphanum (handles TA&M -> TAM)
    return re.sub(r"[^A-Z0-9]", "", str(s).upper().strip())

def parse_game_tokens(game_str):
    g = str(game_str).upper().strip()
    if " AT " in g:
        a, h = g.split(" AT ", 1)
        return norm_team_token(a), norm_team_token(h)
    return None, None

# Strong starter alias map for your sheet style -> ESPN style tokens
# You can extend this later, but these cover your missing list.
ALIAS = {
    "MIZZ": "MIZ",     # Missouri
    "TXAM": "TAM",     # Texas A&M (TA&M -> TAM)
    "GTWN": "GU",      # Georgetown
}

# Build ESPN token keys
events_df = events_df.copy()
events_df["away_tok"] = events_df["away_abbrev"].apply(norm_team_token)
events_df["home_tok"] = events_df["home_abbrev"].apply(norm_team_token)

# Build mapping from (away_tok, home_tok) -> eventId (if duplicates exist, last wins; fine for this use)
pair_map = {(r.away_tok, r.home_tok): r.eventId for r in events_df.itertuples(index=False)}

# Ensure df has eventId column
if "eventId" not in df.columns:
    df["eventId"] = np.nan

# Create tokens from your GAME field
df["away_tok"], df["home_tok"] = zip(*df["GAME"].apply(parse_game_tokens))

# Apply alias to your tokens
df["away_tok2"] = df["away_tok"].apply(lambda x: norm_team_token(ALIAS.get(x, x)))
df["home_tok2"] = df["home_tok"].apply(lambda x: norm_team_token(ALIAS.get(x, x)))

# Fill missing eventId using exact token pair match
missing = df["eventId"].isna()
df.loc[missing, "eventId"] = df.loc[missing].apply(
    lambda r: pair_map.get((r["away_tok2"], r["home_tok2"])), axis=1
)

print("Missing eventId after token match:", df["eventId"].isna().sum())

# Show remaining missing unique games (if any)
still = df[df["eventId"].isna()][["GAME","away_tok2","home_tok2"]].drop_duplicates()
if len(still):
    print("\nStill missing (unique):")
    display(still)

Missing eventId after token match: 21

Still missing (unique):


,GAME,away_tok2,home_tok2
3,UCLA at MINN,UCLA,MINN
7,MIZZ at MSST,MIZ,MSST
10,MISS at AUB,MISS,AUB
15,NCST at ND,NCST,ND
33,TEX at TXAM,TEX,TAM
50,WIS at WASH,WIS,WASH
85,GTWN at XAV,GU,XAV


In [ ]:
import pandas as pd
import requests

SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

# Put the slate date(s) from your prop sheet here
DATES = ["20260228"]  # YYYYMMDD (add multiple days if needed)

def pull_scoreboard_all_events(date_yyyymmdd, page_limit=200, max_pages=20):
    events = []
    offset = 0

    for _ in range(max_pages):
        sb = get_json(SCOREBOARD, params={"dates": date_yyyymmdd, "limit": page_limit, "offset": offset})
        page_events = sb.get("events", [])
        if not page_events:
            break

        for ev in page_events:
            comp = ev["competitions"][0]
            home = next(c for c in comp["competitors"] if c["homeAway"] == "home")
            away = next(c for c in comp["competitors"] if c["homeAway"] == "away")
            events.append({
                "date": date_yyyymmdd,
                "eventId": ev["id"],
                "home_abbrev": home["team"].get("abbreviation",""),
                "away_abbrev": away["team"].get("abbreviation",""),
                "home_name": home["team"].get("displayName",""),
                "away_name": away["team"].get("displayName",""),
            })

        # If fewer than limit returned, we're done
        if len(page_events) < page_limit:
            break

        offset += page_limit

    return events

all_events = []
for d in DATES:
    all_events.extend(pull_scoreboard_all_events(d))

events_df = pd.DataFrame(all_events).drop_duplicates(subset=["eventId"]).reset_index(drop=True)

print("ESPN events pulled (paginated):", len(events_df))

ESPN events pulled (paginated): 15


In [ ]:
import numpy as np
import re

def norm_team_token(s):
    return re.sub(r"[^A-Z0-9]", "", str(s).upper().strip())

def parse_game_tokens(game_str):
    g = str(game_str).upper().strip()
    if " AT " in g:
        a, h = g.split(" AT ", 1)
        return norm_team_token(a), norm_team_token(h)
    return None, None

# Alias map (keep what we already have)
ALIAS = {
    "MIZZ": "MIZ",
    "TXAM": "TAM",
    "GTWN": "GU",
}

events_df = events_df.copy()
events_df["away_tok"] = events_df["away_abbrev"].apply(norm_team_token)
events_df["home_tok"] = events_df["home_abbrev"].apply(norm_team_token)

pair_map = {(r.away_tok, r.home_tok): r.eventId for r in events_df.itertuples(index=False)}

if "eventId" not in df.columns:
    df["eventId"] = np.nan

df["away_tok"], df["home_tok"] = zip(*df["GAME"].apply(parse_game_tokens))
df["away_tok2"] = df["away_tok"].apply(lambda x: norm_team_token(ALIAS.get(x, x)))
df["home_tok2"] = df["home_tok"].apply(lambda x: norm_team_token(ALIAS.get(x, x)))

missing = df["eventId"].isna()
df.loc[missing, "eventId"] = df.loc[missing].apply(
    lambda r: pair_map.get((r["away_tok2"], r["home_tok2"])), axis=1
)

print("Missing eventId after paginated token match:", df["eventId"].isna().sum())

still = df[df["eventId"].isna()][["GAME","away_tok2","home_tok2"]].drop_duplicates()
if len(still):
    print("\nStill missing (unique):")
    display(still)

Missing eventId after paginated token match: 21

Still missing (unique):


,GAME,away_tok2,home_tok2
3,UCLA at MINN,UCLA,MINN
7,MIZZ at MSST,MIZ,MSST
10,MISS at AUB,MISS,AUB
15,NCST at ND,NCST,ND
33,TEX at TXAM,TEX,TAM
50,WIS at WASH,WIS,WASH
85,GTWN at XAV,GU,XAV


In [ ]:
def normtok(s):
    import re
    return re.sub(r"[^A-Z0-9]", "", str(s).upper().strip())

events_df["PAIR"] = events_df["away_abbrev"].apply(normtok) + " AT " + events_df["home_abbrev"].apply(normtok)

targets = [
    "UCLA AT MINN",
    "MIZ AT MSST",
    "MISS AT AUB",
    "NCST AT ND",
    "TEX AT TAM",
    "WIS AT WASH",
    "GU AT XAV",
]

for t in targets:
    hits = events_df[events_df["PAIR"] == t]
    print(t, "hits:", len(hits))

UCLA AT MINN hits: 0
MIZ AT MSST hits: 0
MISS AT AUB hits: 0
NCST AT ND hits: 0
TEX AT TAM hits: 0
WIS AT WASH hits: 0
GU AT XAV hits: 0


In [ ]:
import requests
from datetime import datetime, timedelta
import pandas as pd
import re

SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def normtok(s):
    return re.sub(r"[^A-Z0-9]", "", str(s).upper().strip())

def pull_events_for_date(d, limit=300, max_pages=10):
    events = []
    offset = 0
    for _ in range(max_pages):
        sb = get_json(SCOREBOARD, params={"dates": d, "limit": limit, "offset": offset})
        evs = sb.get("events", [])
        if not evs:
            break
        for ev in evs:
            comp = ev["competitions"][0]
            home = next(c for c in comp["competitors"] if c["homeAway"] == "home")
            away = next(c for c in comp["competitors"] if c["homeAway"] == "away")
            events.append({
                "date": d,
                "eventId": ev["id"],
                "away_tok": normtok(away["team"].get("abbreviation","")),
                "home_tok": normtok(home["team"].get("abbreviation","")),
            })
        if len(evs) < limit:
            break
        offset += limit
    return pd.DataFrame(events)

targets = {
    "UCLA AT MINN": ("UCLA","MINN"),
    "MIZ AT MSST": ("MIZ","MSST"),
    "MISS AT AUB": ("MISS","AUB"),
    "NCST AT ND": ("NCST","ND"),
    "TEX AT TAM": ("TEX","TAM"),
    "WIS AT WASH": ("WIS","WASH"),
    "GU AT XAV": ("GU","XAV"),
}

# Set a center date you think the slate is on:
CENTER_DATE = "20260228"  # change if needed

center = datetime.strptime(CENTER_DATE, "%Y%m%d")
date_list = [(center + timedelta(days=k)).strftime("%Y%m%d") for k in range(-10, 11)]

found = {}

for d in date_list:
    evdf = pull_events_for_date(d)
    if evdf.empty:
        continue
    for label, (a,h) in targets.items():
        if label in found:
            continue
        hit = evdf[(evdf["away_tok"] == a) & (evdf["home_tok"] == h)]
        if len(hit):
            found[label] = {"date": d, "eventId": hit.iloc[0]["eventId"]}

print("FOUND MATCHUPS:")
for k,v in found.items():
    print(k, "->", v)

missing = [k for k in targets.keys() if k not in found]
print("\nSTILL MISSING:", missing)

FOUND MATCHUPS:

STILL MISSING: ['UCLA AT MINN', 'MIZ AT MSST', 'MISS AT AUB', 'NCST AT ND', 'TEX AT TAM', 'WIS AT WASH', 'GU AT XAV']


In [ ]:
tokens = set()
for g in df["GAME"].dropna().unique():
    g = g.upper().strip()
    if " AT " in g:
        a,h = g.split(" AT ",1)
        tokens.add(a.strip())
        tokens.add(h.strip())

tokens = sorted(tokens)
print("Tokens:", tokens)
print("Count:", len(tokens))

Tokens: ['ALA', 'ARIZ', 'ARK', 'AUB', 'BYU', 'CLEM', 'COLO', 'DUKE', 'FLA', 'GONZ', 'GTWN', 'HOU', 'ISU', 'KU', 'LOU', 'MINN', 'MISS', 'MIZZ', 'MSST', 'NCST', 'ND', 'NEB', 'SJU', 'SMC', 'TENN', 'TEX', 'TTU', 'TXAM', 'UCLA', 'UK', 'UNC', 'USC', 'UVA', 'VAN', 'VILL', 'VT', 'WASH', 'WIS', 'WVU', 'XAV']
Count: 40


In [ ]:
import requests
import pandas as pd

TEAM_SEARCH = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams"

def search_team(q):
    r = requests.get(TEAM_SEARCH, params={"limit": 50, "search": q}, timeout=30)
    r.raise_for_status()
    j = r.json()
    out = []
    for it in j.get("sports",[{}])[0].get("leagues",[{}])[0].get("teams",[]):
        t = it.get("team", {})
        out.append({
            "query": q,
            "id": t.get("id"),
            "displayName": t.get("displayName"),
            "abbreviation": t.get("abbreviation"),
        })
    return out

rows = []
for t in tokens:
    rows.extend(search_team(t))

team_search_df = pd.DataFrame(rows)
display(team_search_df.head(50))
team_search_df.to_csv("team_search_results.csv", index=False)
print("Saved team_search_results.csv")

,query,id,displayName,abbreviation
0,ALA,44,American University Eagles,AMER
1,ALA,9,Arizona State Sun Devils,ASU
2,ALA,12,Arizona Wildcats,ARIZ
3,ALA,8,Arkansas Razorbacks,ARK
4,ALA,2,Auburn Tigers,AUB
5,ALA,91,Bellarmine Knights,BELL
6,ALA,68,Boise State Broncos,BOIS
7,ALA,71,Bradley Braves,BRAD
8,ALA,13,Cal Poly Mustangs,CP
9,ALA,25,California Golden Bears,CAL


Saved team_search_results.csv


In [ ]:
import requests, pandas as pd

TEAM_SEARCH = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams"

NEEDED = ["UCLA","MINN","MIZZ","MSST","MISS","AUB","NCST","ND","TEX","TXAM","WIS","WASH","GTWN","XAV"]

def search_team(q):
    r = requests.get(TEAM_SEARCH, params={"limit": 50, "search": q}, timeout=30)
    r.raise_for_status()
    j = r.json()
    out = []
    teams = j.get("sports",[{}])[0].get("leagues",[{}])[0].get("teams",[])
    for it in teams:
        t = it.get("team", {})
        out.append({
            "query": q,
            "id": t.get("id"),
            "displayName": t.get("displayName"),
            "abbreviation": t.get("abbreviation"),
        })
    return out

rows = []
for q in NEEDED:
    rows.extend(search_team(q))

team_search_df = pd.DataFrame(rows)
team_search_df.to_csv("team_search_needed.csv", index=False)

# Show only the best-looking candidates per query (abbrev contains the query token)
best = team_search_df[team_search_df.apply(lambda r: str(r["abbreviation"]).upper().find(r["query"]) >= 0, axis=1)]
print("All candidates saved: team_search_needed.csv")
display(best)

All candidates saved: team_search_needed.csv


,query,id,displayName,abbreviation
44,UCLA,26,UCLA Bruins,UCLA
254,AUB,2,Auburn Tigers,AUB
382,ND,87,Notre Dame Fighting Irish,ND
617,GTWN,46,Georgetown Hoyas,GTWN


In [ ]:
import pandas as pd

team_search_df = pd.read_csv("team_search_needed.csv")

def pick_team_id_for_token(token):
    token = token.upper().strip()

    sub = team_search_df[team_search_df["query"].str.upper() == token].copy()
    sub["abbr_u"] = sub["abbreviation"].astype(str).str.upper()

    # Priority 1: exact abbreviation match
    exact = sub[sub["abbr_u"] == token]
    if len(exact):
        return exact.iloc[0]["id"], exact.iloc[0]["abbreviation"], exact.iloc[0]["displayName"]

    # Priority 2: abbreviation contains token
    contains = sub[sub["abbr_u"].str.contains(token, na=False)]
    if len(contains):
        return contains.iloc[0]["id"], contains.iloc[0]["abbreviation"], contains.iloc[0]["displayName"]

    # Priority 3: fallback first result
    if len(sub):
        return sub.iloc[0]["id"], sub.iloc[0]["abbreviation"], sub.iloc[0]["displayName"]

    return None, None, None

NEEDED = ["UCLA","MINN","MIZZ","MSST","MISS","AUB","NCST","ND","TEX","TXAM","WIS","WASH","GTWN","XAV"]

picked = []
for t in NEEDED:
    tid, abbr, name = pick_team_id_for_token(t)
    picked.append({"token": t, "team_id": tid, "espn_abbr": abbr, "team_name": name})

token_map_df = pd.DataFrame(picked)
display(token_map_df)
token_map_df.to_csv("token_to_teamid_map.csv", index=False)
print("Saved: token_to_teamid_map.csv")

,token,team_id,espn_abbr,team_name
0,UCLA,26,UCLA,UCLA Bruins
1,MINN,44,AMER,American University Eagles
2,MIZZ,44,AMER,American University Eagles
3,MSST,44,AMER,American University Eagles
4,MISS,44,AMER,American University Eagles
5,AUB,2,AUB,Auburn Tigers
6,NCST,44,AMER,American University Eagles
7,ND,87,ND,Notre Dame Fighting Irish
8,TEX,44,AMER,American University Eagles
9,TXAM,44,AMER,American University Eagles


Saved: token_to_teamid_map.csv


In [ ]:
import pandas as pd

team_search_df = pd.read_csv("team_search_needed.csv")

def pick_team_id_for_token(token):
    token = token.upper().strip()

    sub = team_search_df[team_search_df["query"].str.upper() == token].copy()
    sub["abbr_u"] = sub["abbreviation"].astype(str).str.upper()

    # Priority 1: exact abbreviation match
    exact = sub[sub["abbr_u"] == token]
    if len(exact):
        return exact.iloc[0]["id"], exact.iloc[0]["abbreviation"], exact.iloc[0]["displayName"]

    # Priority 2: abbreviation contains token
    contains = sub[sub["abbr_u"].str.contains(token, na=False)]
    if len(contains):
        return contains.iloc[0]["id"], contains.iloc[0]["abbreviation"], contains.iloc[0]["displayName"]

    # Priority 3: fallback first result
    if len(sub):
        return sub.iloc[0]["id"], sub.iloc[0]["abbreviation"], sub.iloc[0]["displayName"]

    return None, None, None

NEEDED = ["UCLA","MINN","MIZZ","MSST","MISS","AUB","NCST","ND","TEX","TXAM","WIS","WASH","GTWN","XAV"]

picked = []
for t in NEEDED:
    tid, abbr, name = pick_team_id_for_token(t)
    picked.append({"token": t, "team_id": tid, "espn_abbr": abbr, "team_name": name})

token_map_df = pd.DataFrame(picked)
display(token_map_df)
token_map_df.to_csv("token_to_teamid_map.csv", index=False)
print("Saved: token_to_teamid_map.csv")

,token,team_id,espn_abbr,team_name
0,UCLA,26,UCLA,UCLA Bruins
1,MINN,44,AMER,American University Eagles
2,MIZZ,44,AMER,American University Eagles
3,MSST,44,AMER,American University Eagles
4,MISS,44,AMER,American University Eagles
5,AUB,2,AUB,Auburn Tigers
6,NCST,44,AMER,American University Eagles
7,ND,87,ND,Notre Dame Fighting Irish
8,TEX,44,AMER,American University Eagles
9,TXAM,44,AMER,American University Eagles


Saved: token_to_teamid_map.csv


In [ ]:
import pandas as pd, numpy as np, requests, re
from datetime import datetime, timedelta

SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def pull_events_ids_for_date(d, limit=300, max_pages=10):
    out = []
    offset = 0
    for _ in range(max_pages):
        sb = get_json(SCOREBOARD, params={"dates": d, "limit": limit, "offset": offset})
        evs = sb.get("events", [])
        if not evs:
            break
        for ev in evs:
            comp = ev["competitions"][0]
            home = next(c for c in comp["competitors"] if c["homeAway"] == "home")
            away = next(c for c in comp["competitors"] if c["homeAway"] == "away")
            out.append({
                "date": d,
                "eventId": ev["id"],
                "away_team_id": away["team"].get("id"),
                "home_team_id": home["team"].get("id"),
                "away_abbr": away["team"].get("abbreviation"),
                "home_abbr": home["team"].get("abbreviation"),
            })
        if len(evs) < limit:
            break
        offset += limit
    return out

# Build token->team_id dict
token_map_df = pd.read_csv("token_to_teamid_map.csv")
TOKEN_TO_ID = dict(zip(token_map_df["token"].astype(str).str.upper(), token_map_df["team_id"].astype(str)))

def parse_tokens(game_str):
    g = str(game_str).upper().strip()
    if " AT " in g:
        a,h = g.split(" AT ",1)
        return a.strip(), h.strip()
    return None, None

# Add token ids to df
df["away_token"], df["home_token"] = zip(*df["GAME"].apply(parse_tokens))
df["away_team_id"] = df["away_token"].map(TOKEN_TO_ID)
df["home_team_id"] = df["home_token"].map(TOKEN_TO_ID)

# Pull scoreboard across a wide range (adjust CENTER_DATE if you know it)
CENTER_DATE = "20260228"  # change if needed
center = datetime.strptime(CENTER_DATE, "%Y%m%d")
date_list = [(center + timedelta(days=k)).strftime("%Y%m%d") for k in range(-14, 15)]

events = []
for d in date_list:
    events.extend(pull_events_ids_for_date(d))

events_id_df = pd.DataFrame(events).drop_duplicates(subset=["eventId"])
pair_map = {(str(r.away_team_id), str(r.home_team_id)): r.eventId for r in events_id_df.itertuples(index=False)}

# Fill eventId by team ids (this is the robust match)
if "eventId" not in df.columns:
    df["eventId"] = np.nan

missing = df["eventId"].isna()
df.loc[missing, "eventId"] = df.loc[missing].apply(
    lambda r: pair_map.get((str(r["away_team_id"]), str(r["home_team_id"]))),
    axis=1
)

print("Missing eventId after TEAM-ID match:", df["eventId"].isna().sum())

still = df[df["eventId"].isna()][["GAME","away_token","home_token","away_team_id","home_team_id"]].drop_duplicates()
if len(still):
    print("\nStill missing (unique):")
    display(still)

Missing eventId after TEAM-ID match: 21

Still missing (unique):


,GAME,away_token,home_token,away_team_id,home_team_id
3,UCLA at MINN,UCLA,MINN,26,44
7,MIZZ at MSST,MIZZ,MSST,44,44
10,MISS at AUB,MISS,AUB,44,2
15,NCST at ND,NCST,ND,44,87
33,TEX at TXAM,TEX,TXAM,44,44
50,WIS at WASH,WIS,WASH,44,44
85,GTWN at XAV,GTWN,XAV,46,44


In [ ]:
import pandas as pd

team_search_df = pd.read_csv("team_search_needed.csv")
team_search_df["abbr_u"] = team_search_df["abbreviation"].astype(str).str.upper()

# Force the ESPN abbreviation we want for each token
TOKEN_TO_ESPN_ABBR = {
    "UCLA": "UCLA",
    "MINN": "MINN",
    "MIZZ": "MIZ",
    "MSST": "MSST",
    "MISS": "MISS",
    "AUB": "AUB",
    "NCST": "NCST",
    "ND": "ND",
    "TEX": "TEX",
    "TXAM": "TA&M",   # ESPN uses TA&M; we'll match normalized below
    "WIS": "WIS",
    "WASH": "WASH",
    "GTWN": "GTWN",
    "XAV": "XAV",
}

def norm_abbr(a):
    # normalize TA&M -> TAM for matching
    import re
    return re.sub(r"[^A-Z0-9]", "", str(a).upper())

rows = []
for token, want_abbr in TOKEN_TO_ESPN_ABBR.items():
    want_norm = norm_abbr(want_abbr)
    sub = team_search_df[team_search_df["query"].str.upper() == token].copy()
    sub["abbr_norm"] = sub["abbreviation"].apply(norm_abbr)

    pick = sub[sub["abbr_norm"] == want_norm]
    if len(pick) == 0:
        # fallback: show candidates so you can adjust
        print(f"NO EXACT MATCH for {token} -> want {want_abbr}. Candidates:")
        display(sub[["id","displayName","abbreviation"]].head(15))
        chosen = None
    else:
        chosen = pick.iloc[0]
        rows.append({
            "token": token,
            "team_id": str(chosen["id"]),
            "espn_abbr": chosen["abbreviation"],
            "team_name": chosen["displayName"]
        })

token_map_df = pd.DataFrame(rows)
display(token_map_df)
token_map_df.to_csv("token_to_teamid_map_fixed.csv", index=False)
print("Saved: token_to_teamid_map_fixed.csv")

NO EXACT MATCH for MINN -> want MINN. Candidates:


,id,displayName,abbreviation
50,44,American University Eagles,AMER
51,9,Arizona State Sun Devils,ASU
52,12,Arizona Wildcats,ARIZ
53,8,Arkansas Razorbacks,ARK
54,2,Auburn Tigers,AUB
55,91,Bellarmine Knights,BELL
56,68,Boise State Broncos,BOIS
57,71,Bradley Braves,BRAD
58,13,Cal Poly Mustangs,CP
59,25,California Golden Bears,CAL


NO EXACT MATCH for MIZZ -> want MIZ. Candidates:


,id,displayName,abbreviation
100,44,American University Eagles,AMER
101,9,Arizona State Sun Devils,ASU
102,12,Arizona Wildcats,ARIZ
103,8,Arkansas Razorbacks,ARK
104,2,Auburn Tigers,AUB
105,91,Bellarmine Knights,BELL
106,68,Boise State Broncos,BOIS
107,71,Bradley Braves,BRAD
108,13,Cal Poly Mustangs,CP
109,25,California Golden Bears,CAL


NO EXACT MATCH for MSST -> want MSST. Candidates:


...

,id,displayName,abbreviation
550,44,American University Eagles,AMER
551,9,Arizona State Sun Devils,ASU
552,12,Arizona Wildcats,ARIZ
553,8,Arkansas Razorbacks,ARK
554,2,Auburn Tigers,AUB
555,91,Bellarmine Knights,BELL
556,68,Boise State Broncos,BOIS
557,71,Bradley Braves,BRAD
558,13,Cal Poly Mustangs,CP
559,25,California Golden Bears,CAL


NO EXACT MATCH for XAV -> want XAV. Candidates:


,id,displayName,abbreviation
650,44,American University Eagles,AMER
651,9,Arizona State Sun Devils,ASU
652,12,Arizona Wildcats,ARIZ
653,8,Arkansas Razorbacks,ARK
654,2,Auburn Tigers,AUB
655,91,Bellarmine Knights,BELL
656,68,Boise State Broncos,BOIS
657,71,Bradley Braves,BRAD
658,13,Cal Poly Mustangs,CP
659,25,California Golden Bears,CAL


,token,team_id,espn_abbr,team_name
0,UCLA,26,UCLA,UCLA Bruins
1,AUB,2,AUB,Auburn Tigers
2,ND,87,ND,Notre Dame Fighting Irish
3,GTWN,46,GTWN,Georgetown Hoyas


Saved: token_to_teamid_map_fixed.csv


In [ ]:
import pandas as pd, numpy as np

token_map_df = pd.read_csv("token_to_teamid_map_fixed.csv")
TOKEN_TO_ID = dict(zip(token_map_df["token"].str.upper(), token_map_df["team_id"].astype(str)))

def parse_tokens(game_str):
    g = str(game_str).upper().strip()
    if " AT " in g:
        a,h = g.split(" AT ",1)
        return a.strip(), h.strip()
    return None, None

df["away_token"], df["home_token"] = zip(*df["GAME"].apply(parse_tokens))
df["away_team_id"] = df["away_token"].map(TOKEN_TO_ID)
df["home_team_id"] = df["home_token"].map(TOKEN_TO_ID)

pair_map = {(str(r.away_team_id), str(r.home_team_id)): r.eventId for r in events_id_df.itertuples(index=False)}

missing = df["eventId"].isna()
df.loc[missing, "eventId"] = df.loc[missing].apply(
    lambda r: pair_map.get((str(r["away_team_id"]), str(r["home_team_id"]))),
    axis=1
)

print("Missing eventId after TEAM-ID match (fixed map):", df["eventId"].isna().sum())

still = df[df["eventId"].isna()][["GAME","away_token","home_token","away_team_id","home_team_id"]].drop_duplicates()
if len(still):
    print("\nStill missing (unique):")
    display(still)

Missing eventId after TEAM-ID match (fixed map): 21

Still missing (unique):


,GAME,away_token,home_token,away_team_id,home_team_id
3,UCLA at MINN,UCLA,MINN,26,NaN
7,MIZZ at MSST,MIZZ,MSST,NaN,NaN
10,MISS at AUB,MISS,AUB,NaN,2
15,NCST at ND,NCST,ND,NaN,87
33,TEX at TXAM,TEX,TXAM,NaN,NaN
50,WIS at WASH,WIS,WASH,NaN,NaN
85,GTWN at XAV,GTWN,XAV,46,NaN


In [ ]:
import requests, pandas as pd, re, numpy as np

TEAM_SEARCH = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams"

def get_teams(search):
    r = requests.get(TEAM_SEARCH, params={"limit": 50, "search": search}, timeout=30)
    r.raise_for_status()
    j = r.json()
    out = []
    teams = j.get("sports",[{}])[0].get("leagues",[{}])[0].get("teams",[])
    for it in teams:
        t = it.get("team", {})
        out.append({
            "search": search,
            "id": str(t.get("id")),
            "displayName": t.get("displayName",""),
            "abbreviation": t.get("abbreviation",""),
        })
    return pd.DataFrame(out)

def norm_abbr(a):
    return re.sub(r"[^A-Z0-9]", "", str(a).upper())

# Better queries for tokens that fail direct abbreviation search
TOKEN_SEARCH_FALLBACK = {
    "MINN": "Minnesota",
    "MIZZ": "Missouri",
    "MSST": "Mississippi State",
    "MISS": "Ole Miss",
    "NCST": "NC State",
    "TEX": "Texas",
    "TXAM": "Texas A&M",
    "WIS": "Wisconsin",
    "WASH": "Washington",
    "XAV": "Xavier",
    # Keep these in case they ever fail:
    "UCLA": "UCLA",
    "AUB": "Auburn",
    "ND": "Notre Dame",
    "GTWN": "Georgetown",
}

# What ESPN abbreviation (normalized) we want to target, when possible
TOKEN_TO_ABBR_NORM = {
    "UCLA": "UCLA",
    "MINN": "MINN",
    "MIZZ": "MIZ",
    "MSST": "MSST",
    "MISS": "MISS",
    "AUB": "AUB",
    "NCST": "NCST",
    "ND": "ND",
    "TEX": "TEX",
    "TXAM": "TAM",   # TA&M -> TAM
    "WIS": "WIS",
    "WASH": "WASH",
    "GTWN": "GTWN",
    "XAV": "XAV",
}

NEEDED = ["UCLA","MINN","MIZZ","MSST","MISS","AUB","NCST","ND","TEX","TXAM","WIS","WASH","GTWN","XAV"]

resolved = []
for token in NEEDED:
    search_q = TOKEN_SEARCH_FALLBACK.get(token, token)
    cand = get_teams(search_q)
    if cand.empty:
        resolved.append({"token": token, "team_id": np.nan, "espn_abbr": None, "team_name": None, "search_used": search_q})
        continue

    cand["abbr_norm"] = cand["abbreviation"].apply(norm_abbr)
    want = TOKEN_TO_ABBR_NORM.get(token)

    pick = cand[cand["abbr_norm"] == want] if want else pd.DataFrame()

    if len(pick) == 0:
        # fallback: pick row whose displayName contains a key word from the search query
        key = search_q.split()[0].upper()
        pick = cand[cand["displayName"].str.upper().str.contains(key, na=False)]

    if len(pick) == 0:
        pick = cand.iloc[[0]]

    chosen = pick.iloc[0]
    resolved.append({
        "token": token,
        "team_id": chosen["id"],
        "espn_abbr": chosen["abbreviation"],
        "team_name": chosen["displayName"],
        "search_used": search_q
    })

token_map_df = pd.DataFrame(resolved)
display(token_map_df)
token_map_df.to_csv("token_to_teamid_map_fixed.csv", index=False)
print("Saved: token_to_teamid_map_fixed.csv")

,token,team_id,espn_abbr,team_name,search_used
0,UCLA,26,UCLA,UCLA Bruins,UCLA
1,MINN,44,AMER,American University Eagles,Minnesota
2,MIZZ,44,AMER,American University Eagles,Missouri
3,MSST,44,AMER,American University Eagles,Mississippi State
4,MISS,52,FSU,Florida State Seminoles,Ole Miss
5,AUB,2,AUB,Auburn Tigers,Auburn
6,NCST,68,BOIS,Boise State Broncos,NC State
7,ND,87,ND,Notre Dame Fighting Irish,Notre Dame
8,TEX,44,AMER,American University Eagles,Texas
9,TXAM,44,AMER,American University Eagles,Texas A&M


Saved: token_to_teamid_map_fixed.csv


In [ ]:
import pandas as pd, numpy as np

token_map_df = pd.read_csv("token_to_teamid_map_fixed.csv")
TOKEN_TO_ID = dict(zip(token_map_df["token"].str.upper(), token_map_df["team_id"].astype(str)))

def parse_tokens(game_str):
    g = str(game_str).upper().strip()
    if " AT " in g:
        a,h = g.split(" AT ",1)
        return a.strip(), h.strip()
    return None, None

df["away_token"], df["home_token"] = zip(*df["GAME"].apply(parse_tokens))
df["away_team_id"] = df["away_token"].map(TOKEN_TO_ID)
df["home_team_id"] = df["home_token"].map(TOKEN_TO_ID)

pair_map = {(str(r.away_team_id), str(r.home_team_id)): r.eventId for r in events_id_df.itertuples(index=False)}

# IMPORTANT: wipe any junk eventId_candidate columns; keep only eventId
if "eventId" not in df.columns:
    df["eventId"] = np.nan

missing = df["eventId"].isna()
df.loc[missing, "eventId"] = df.loc[missing].apply(
    lambda r: pair_map.get((str(r["away_team_id"]), str(r["home_team_id"]))),
    axis=1
)

print("Missing eventId after TEAM-ID match (fallback search):", df["eventId"].isna().sum())

still = df[df["eventId"].isna()][["GAME","away_token","home_token","away_team_id","home_team_id"]].drop_duplicates()
if len(still):
    print("\nStill missing (unique):")
    display(still)

Missing eventId after TEAM-ID match (fallback search): 21

Still missing (unique):


,GAME,away_token,home_token,away_team_id,home_team_id
3,UCLA at MINN,UCLA,MINN,26,44
7,MIZZ at MSST,MIZZ,MSST,44,44
10,MISS at AUB,MISS,AUB,52,2
15,NCST at ND,NCST,ND,68,87
33,TEX at TXAM,TEX,TXAM,44,44
50,WIS at WASH,WIS,WASH,44,45
85,GTWN at XAV,GTWN,XAV,46,44


In [ ]:
import requests, pandas as pd, re

TEAM_SEARCH = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def team_search(search):
    j = get_json(TEAM_SEARCH, params={"limit": 50, "search": search})
    out = []
    teams = j.get("sports",[{}])[0].get("leagues",[{}])[0].get("teams",[])
    for it in teams:
        t = it.get("team", {})
        out.append({
            "search": search,
            "id": str(t.get("id")),
            "displayName": t.get("displayName",""),
            "abbreviation": t.get("abbreviation",""),
        })
    return pd.DataFrame(out)

TARGETS = [
    ("UCLA",  "UCLA Bruins"),
    ("MINN",  "Minnesota Golden Gophers"),
    ("MIZZ",  "Missouri Tigers"),
    ("MSST",  "Mississippi State Bulldogs"),
    ("MISS",  "Ole Miss Rebels"),
    ("AUB",   "Auburn Tigers"),
    ("NCST",  "NC State Wolfpack"),
    ("ND",    "Notre Dame Fighting Irish"),
    ("TEX",   "Texas Longhorns"),
    ("TXAM",  "Texas A&M Aggies"),
    ("WIS",   "Wisconsin Badgers"),
    ("WASH",  "Washington Huskies"),
    ("GTWN",  "Georgetown Hoyas"),
    ("XAV",   "Xavier Musketeers"),
]

resolved = []

for token, exact_name in TARGETS:
    cand = team_search(exact_name)
    if cand.empty:
        resolved.append({"token": token, "team_id": None, "espn_abbr": None, "team_name": None, "search_used": exact_name})
        continue

    # exact match first
    hit = cand[cand["displayName"].str.upper() == exact_name.upper()]
    if len(hit) == 0:
        # contains fallback
        key = exact_name.split()[0].upper()
        hit = cand[cand["displayName"].str.upper().str.contains(key, na=False)]

    chosen = hit.iloc[0] if len(hit) else cand.iloc[0]
    resolved.append({
        "token": token,
        "team_id": chosen["id"],
        "espn_abbr": chosen["abbreviation"],
        "team_name": chosen["displayName"],
        "search_used": exact_name
    })

token_map_df = pd.DataFrame(resolved)
display(token_map_df)
token_map_df.to_csv("token_to_teamid_map_FIXED_EXACT.csv", index=False)
print("Saved: token_to_teamid_map_FIXED_EXACT.csv")

,token,team_id,espn_abbr,team_name,search_used
0,UCLA,26,UCLA,UCLA Bruins,UCLA Bruins
1,MINN,44,AMER,American University Eagles,Minnesota Golden Gophers
2,MIZZ,44,AMER,American University Eagles,Missouri Tigers
3,MSST,44,AMER,American University Eagles,Mississippi State Bulldogs
4,MISS,52,FSU,Florida State Seminoles,Ole Miss Rebels
5,AUB,2,AUB,Auburn Tigers,Auburn Tigers
6,NCST,68,BOIS,Boise State Broncos,NC State Wolfpack
7,ND,87,ND,Notre Dame Fighting Irish,Notre Dame Fighting Irish
8,TEX,44,AMER,American University Eagles,Texas Longhorns
9,TXAM,44,AMER,American University Eagles,Texas A&M Aggies


Saved: token_to_teamid_map_FIXED_EXACT.csv


In [ ]:
import pandas as pd, numpy as np

token_map_df = pd.read_csv("token_to_teamid_map_FIXED_EXACT.csv")
TOKEN_TO_ID = dict(zip(token_map_df["token"].str.upper(), token_map_df["team_id"].astype(str)))

def parse_tokens(game_str):
    g = str(game_str).upper().strip()
    if " AT " in g:
        a,h = g.split(" AT ",1)
        return a.strip(), h.strip()
    return None, None

df["away_token"], df["home_token"] = zip(*df["GAME"].apply(parse_tokens))
df["away_team_id"] = df["away_token"].map(TOKEN_TO_ID)
df["home_team_id"] = df["home_token"].map(TOKEN_TO_ID)

# Make sure events_id_df exists (from scoreboard pulls)
pair_map = {(str(r.away_team_id), str(r.home_team_id)): r.eventId for r in events_id_df.itertuples(index=False)}

if "eventId" not in df.columns:
    df["eventId"] = np.nan

missing = df["eventId"].isna()
df.loc[missing, "eventId"] = df.loc[missing].apply(
    lambda r: pair_map.get((str(r["away_team_id"]), str(r["home_team_id"]))),
    axis=1
)

print("Missing eventId after TEAM-ID match (EXACT NAMES):", df["eventId"].isna().sum())

still = df[df["eventId"].isna()][["GAME","away_team_id","home_team_id"]].drop_duplicates()
if len(still):
    print("\nStill missing (unique):")
    display(still)

Missing eventId after TEAM-ID match (EXACT NAMES): 21

Still missing (unique):


,GAME,away_team_id,home_team_id
3,UCLA at MINN,26,44
7,MIZZ at MSST,44,44
10,MISS at AUB,52,2
15,NCST at ND,68,87
33,TEX at TXAM,44,44
50,WIS at WASH,44,45
85,GTWN at XAV,46,44


In [ ]:
import requests, pandas as pd

ODDS_API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"

# Odds API endpoint (v4)
ODDS_GAMES_URL = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds"

params = {
    "apiKey": ODDS_API_KEY,
    "regions": "us",
    "markets": "h2h,spreads,totals",
    "oddsFormat": "american"
}

resp = requests.get(ODDS_GAMES_URL, params=params, timeout=30)
resp.raise_for_status()
odds_json = resp.json()

games = []
for g in odds_json:
    games.append({
        "odds_game_id": g.get("id"),
        "commence_time": g.get("commence_time"),
        "home_team": g.get("home_team"),
        "away_team": g.get("away_team"),
    })

odds_games_df = pd.DataFrame(games)
display(odds_games_df.head(20))
print("Odds games pulled:", len(odds_games_df))

,odds_game_id,commence_time,home_team,away_team
0,a7781091bccdf791848c91b9108a534e,2026-02-28T17:00:00Z,American Eagles,Boston Univ. Terriers
1,0bc16c6d6a309636768b10ad2a6f5705,2026-02-28T17:00:00Z,Holy Cross Crusaders,Loyola (MD) Greyhounds
2,ba279869081dfef0cbeceada1c4257cd,2026-02-28T17:00:17Z,Penn State Nittany Lions,Iowa Hawkeyes
3,7843a76531d124aacd8a825ecedcc899,2026-02-28T17:00:20Z,Notre Dame Fighting Irish,NC State Wolfpack
4,e7ecc1829d64c2ebfe7ee08e2d8c2553,2026-02-28T17:00:48Z,Georgia Tech Yellow Jackets,Florida St Seminoles
5,dc4a6e31728fec783a144751c0157571,2026-02-28T17:01:12Z,Duke Blue Devils,Virginia Cavaliers
6,ffdbf2262c4fff72db28df671a711c7c,2026-02-28T17:02:10Z,Houston Cougars,Colorado Buffaloes
7,884d429ad0bb6846449001883de1bd04,2026-02-28T17:02:41Z,Rhode Island Rams,Saint Joseph's Hawks
8,edbaabb57cf7d04c5a6a0c8cbba4823b,2026-02-28T17:02:57Z,UConn Huskies,Seton Hall Pirates
9,e2e5065bb7839ddeacce53b94caba1bb,2026-02-28T17:30:00Z,VCU Rams,Fordham Rams


Odds games pulled: 143


In [ ]:
import re
import pandas as pd

# df is your Hitrate28 working dataframe
df["GAME_UP"] = df["GAME"].astype(str).str.upper().str.strip()

def split_game(g):
    g = g.upper().strip()
    if " AT " in g:
        a,h = g.split(" AT ",1)
        return a.strip(), h.strip()
    return None, None

df["away_token"], df["home_token"] = zip(*df["GAME_UP"].apply(split_game))

# Build a small alias map from your tokens -> likely full names (expand as needed)
TOKEN_TO_SCHOOL = {
    "UCLA": "UCLA",
    "MINN": "MINNESOTA",
    "MIZZ": "MISSOURI",
    "MSST": "MISSISSIPPI STATE",
    "MISS": "OLE MISS",          # or "MISSISSIPPI" depending on your sheet meaning
    "AUB": "AUBURN",
    "NCST": "NC STATE",
    "ND": "NOTRE DAME",
    "TEX": "TEXAS",
    "TXAM": "TEXAS A&M",
    "WIS": "WISCONSIN",
    "WASH": "WASHINGTON",
    "GTWN": "GEORGETOWN",
    "XAV": "XAVIER",
    "KU": "KANSAS",
    "ARIZ": "ARIZONA",
    # add more as needed as you scale
}

def norm_name(s):
    return re.sub(r"[^A-Z ]","", str(s).upper()).strip()

odds_games_df["home_norm"] = odds_games_df["home_team"].apply(norm_name)
odds_games_df["away_norm"] = odds_games_df["away_team"].apply(norm_name)

df["away_guess"] = df["away_token"].map(TOKEN_TO_SCHOOL).fillna(df["away_token"]).apply(norm_name)
df["home_guess"] = df["home_token"].map(TOKEN_TO_SCHOOL).fillna(df["home_token"]).apply(norm_name)

# match by normalized names
odds_pair_map = {(r.away_norm, r.home_norm): r.odds_game_id for r in odds_games_df.itertuples(index=False)}

df["odds_game_id"] = df.apply(lambda r: odds_pair_map.get((r["away_guess"], r["home_guess"])), axis=1)

print("Rows missing odds_game_id:", df["odds_game_id"].isna().sum())
display(df[df["odds_game_id"].isna()][["GAME","away_guess","home_guess"]].drop_duplicates().head(25))

Rows missing odds_game_id: 100


,GAME,away_guess,home_guess
0,COLO at HOU,COLO,HOU
1,KU at ARIZ,KANSAS,ARIZONA
2,BYU at WVU,BYU,WVU
3,UCLA at MINN,UCLA,MINNESOTA
4,ARK at FLA,ARK,FLA
5,GONZ at SMC,GONZ,SMC
6,ALA at TENN,ALA,TENN
7,MIZZ at MSST,MISSOURI,MISSISSIPPI STATE
8,VILL at SJU,VILL,SJU
10,MISS at AUB,OLE MISS,AUBURN


In [ ]:
import requests
from datetime import datetime, timezone
import pandas as pd
import re
from difflib import SequenceMatcher

SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def norm(s):
    return re.sub(r"[^A-Z ]","", str(s).upper()).strip()

def ratio(a,b):
    return SequenceMatcher(None, a, b).ratio()

# Build a lookup from odds_game_id -> commence date (YYYYMMDD)
odds_games_df["commence_dt"] = pd.to_datetime(odds_games_df["commence_time"], utc=True, errors="coerce")
odds_games_df["commence_yyyymmdd"] = odds_games_df["commence_dt"].dt.strftime("%Y%m%d")

odds_date_map = dict(zip(odds_games_df["odds_game_id"], odds_games_df["commence_yyyymmdd"]))

# Pull ESPN events for the needed dates (and +/-1 day buffer)
needed_dates = sorted(set(df["odds_game_id"].dropna().map(odds_date_map)))
buffer_dates = sorted(set(needed_dates + [
    (datetime.strptime(d,"%Y%m%d") - pd.Timedelta(days=1)).strftime("%Y%m%d") for d in needed_dates
] + [
    (datetime.strptime(d,"%Y%m%d") + pd.Timedelta(days=1)).strftime("%Y%m%d") for d in needed_dates
]))

events = []
for d in buffer_dates:
    sb = get_json(SCOREBOARD, params={"dates": d, "limit": 500, "offset": 0})
    for ev in sb.get("events", []):
        comp = ev["competitions"][0]
        home = next(c for c in comp["competitors"] if c["homeAway"]=="home")
        away = next(c for c in comp["competitors"] if c["homeAway"]=="away")
        events.append({
            "date": d,
            "eventId": ev["id"],
            "home_name": home["team"].get("displayName",""),
            "away_name": away["team"].get("displayName","")
        })

events_df = pd.DataFrame(events).drop_duplicates(subset=["eventId"])

# For each df row, match to best ESPN event on/near that date using team name similarity
def find_event_id(row):
    ogid = row["odds_game_id"]
    if pd.isna(ogid):
        return None
    d = odds_date_map.get(ogid)
    if not d:
        return None

    # candidate ESPN events within buffer dates (already pulled)
    cand = events_df[events_df["date"].isin([d,
                                            (datetime.strptime(d,"%Y%m%d")-pd.Timedelta(days=1)).strftime("%Y%m%d"),
                                            (datetime.strptime(d,"%Y%m%d")+pd.Timedelta(days=1)).strftime("%Y%m%d")])]

    if cand.empty:
        return None

    away = norm(odds_games_df.loc[odds_games_df["odds_game_id"]==ogid, "away_team"].values[0])
    home = norm(odds_games_df.loc[odds_games_df["odds_game_id"]==ogid, "home_team"].values[0])

    best_id, best_score = None, -1
    for _, e in cand.iterrows():
        s = 0.5*ratio(away, norm(e["away_name"])) + 0.5*ratio(home, norm(e["home_name"]))
        if s > best_score:
            best_score = s
            best_id = e["eventId"]

    return best_id if best_score >= 0.78 else None  # threshold

df["eventId"] = df.apply(find_event_id, axis=1)
print("Rows missing eventId after Odds->ESPN match:", df["eventId"].isna().sum())
display(df[df["eventId"].isna()][["GAME","odds_game_id","away_guess","home_guess"]].drop_duplicates().head(25))

Rows missing eventId after Odds->ESPN match: 100


,GAME,odds_game_id,away_guess,home_guess
0,COLO at HOU,None,COLO,HOU
1,KU at ARIZ,None,KANSAS,ARIZONA
2,BYU at WVU,None,BYU,WVU
3,UCLA at MINN,None,UCLA,MINNESOTA
4,ARK at FLA,None,ARK,FLA
5,GONZ at SMC,None,GONZ,SMC
6,ALA at TENN,None,ALA,TENN
7,MIZZ at MSST,None,MISSOURI,MISSISSIPPI STATE
8,VILL at SJU,None,VILL,SJU
10,MISS at AUB,None,OLE MISS,AUBURN


In [ ]:
import requests, pandas as pd

ODDS_API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"

EVENTS_URL = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/events"

resp = requests.get(EVENTS_URL, params={"apiKey": ODDS_API_KEY}, timeout=30)
resp.raise_for_status()
events_json = resp.json()

odds_events_df = pd.DataFrame([{
    "odds_event_id": e.get("id"),
    "commence_time": e.get("commence_time"),
    "home_team": e.get("home_team"),
    "away_team": e.get("away_team"),
} for e in events_json])

print("Odds fixtures pulled:", len(odds_events_df))
display(odds_events_df.head(20))

Odds fixtures pulled: 143


,odds_event_id,commence_time,home_team,away_team
0,a7781091bccdf791848c91b9108a534e,2026-02-28T17:00:00Z,American Eagles,Boston Univ. Terriers
1,0bc16c6d6a309636768b10ad2a6f5705,2026-02-28T17:00:00Z,Holy Cross Crusaders,Loyola (MD) Greyhounds
2,ba279869081dfef0cbeceada1c4257cd,2026-02-28T17:00:17Z,Penn State Nittany Lions,Iowa Hawkeyes
3,7843a76531d124aacd8a825ecedcc899,2026-02-28T17:00:20Z,Notre Dame Fighting Irish,NC State Wolfpack
4,e7ecc1829d64c2ebfe7ee08e2d8c2553,2026-02-28T17:00:48Z,Georgia Tech Yellow Jackets,Florida St Seminoles
5,dc4a6e31728fec783a144751c0157571,2026-02-28T17:01:12Z,Duke Blue Devils,Virginia Cavaliers
6,ffdbf2262c4fff72db28df671a711c7c,2026-02-28T17:02:10Z,Houston Cougars,Colorado Buffaloes
7,884d429ad0bb6846449001883de1bd04,2026-02-28T17:02:41Z,Rhode Island Rams,Saint Joseph's Hawks
8,edbaabb57cf7d04c5a6a0c8cbba4823b,2026-02-28T17:02:57Z,UConn Huskies,Seton Hall Pirates
9,e2e5065bb7839ddeacce53b94caba1bb,2026-02-28T17:30:00Z,VCU Rams,Fordham Rams


In [ ]:
import re
from difflib import SequenceMatcher
import pandas as pd
import numpy as np

def norm(s):
    return re.sub(r"[^A-Z0-9 ]","", str(s).upper()).strip()

def ratio(a,b):
    return SequenceMatcher(None, norm(a), norm(b)).ratio()

# Your sheet tokens -> school keywords (expand anytime)
TOKEN_TO_SCHOOL = {
    "COLO": "COLORADO",
    "HOU": "HOUSTON",
    "KU": "KANSAS",
    "ARIZ": "ARIZONA",
    "BYU": "BYU",
    "WVU": "WEST VIRGINIA",
    "UCLA": "UCLA",
    "MINN": "MINNESOTA",
    "ARK": "ARKANSAS",
    "FLA": "FLORIDA",
    "GONZ": "GONZAGA",
    "SMC": "SAINT MARYS",     # might appear as "Saint Mary's (CA)"
    "ALA": "ALABAMA",
    "TENN": "TENNESSEE",
    "MIZZ": "MISSOURI",
    "MSST": "MISSISSIPPI STATE",
    "VILL": "VILLANOVA",
    "SJU": "ST JOHNS",        # might appear as "St. John's"
    "MISS": "MISSISSIPPI",    # Ole Miss often shows as "Mississippi"
    "AUB": "AUBURN",
    "VAN": "VANDERBILT",
    "UK": "KENTUCKY",
    "NEB": "NEBRASKA",
    "USC": "USC",
    "NCST": "NC STATE",
    "ND": "NOTRE DAME",
    "LOU": "LOUISVILLE",
    "CLEM": "CLEMSON",
    "VT": "VIRGINIA TECH",
    "UNC": "NORTH CAROLINA",
    "TTU": "TEXAS TECH",
    "ISU": "IOWA STATE",
    "TEX": "TEXAS",
    "TXAM": "TEXAS AM",
    "UVA": "VIRGINIA",
    "DUKE": "DUKE",
    "WIS": "WISCONSIN",
    "WASH": "WASHINGTON",
    "GTWN": "GEORGETOWN",
    "XAV": "XAVIER",
}

def split_game(g):
    g = str(g).upper().strip()
    if " AT " in g:
        a,h = g.split(" AT ",1)
        return a.strip(), h.strip()
    return None, None

df["GAME_UP"] = df["GAME"].astype(str).str.upper().str.strip()
df["away_token"], df["home_token"] = zip(*df["GAME_UP"].apply(split_game))

df["away_school"] = df["away_token"].map(TOKEN_TO_SCHOOL).fillna(df["away_token"])
df["home_school"] = df["home_token"].map(TOKEN_TO_SCHOOL).fillna(df["home_token"])

# Pre-normalize odds teams
odds_events_df["away_norm"] = odds_events_df["away_team"].apply(norm)
odds_events_df["home_norm"] = odds_events_df["home_team"].apply(norm)

def best_odds_event(away_school, home_school):
    best_id, best_score = None, -1
    for r in odds_events_df.itertuples(index=False):
        s = 0.5*ratio(away_school, r.away_norm) + 0.5*ratio(home_school, r.home_norm)
        if s > best_score:
            best_score = s
            best_id = r.odds_event_id
    return best_id, best_score

# Match each unique game once, then map back (faster)
game_map = {}
for g, a_s, h_s in df[["GAME_UP","away_school","home_school"]].drop_duplicates().itertuples(index=False):
    oid, score = best_odds_event(a_s, h_s)
    game_map[g] = (oid, score)

df["odds_event_id"] = df["GAME_UP"].map(lambda x: game_map.get(x, (None,None))[0])
df["odds_match_score"] = df["GAME_UP"].map(lambda x: game_map.get(x, (None,None))[1])

print("Rows missing odds_event_id:", df["odds_event_id"].isna().sum())
print("Low-score matches (check these):")
display(df[df["odds_match_score"].fillna(0) < 0.78][["GAME","away_school","home_school","odds_match_score"]].drop_duplicates().head(25))

Rows missing odds_event_id: 0
Low-score matches (check these):


,GAME,away_school,home_school,odds_match_score
0,COLO at HOU,COLORADO,HOUSTON,0.625874
1,KU at ARIZ,KANSAS,ARIZONA,0.590062
2,BYU at WVU,BYU,WEST VIRGINIA,0.547619
3,UCLA at MINN,UCLA,MINNESOTA,0.539394
4,ARK at FLA,ARKANSAS,FLORIDA,0.629630
5,GONZ at SMC,GONZAGA,SAINT MARYS,0.697205
6,ALA at TENN,ALABAMA,TENNESSEE,0.569604
7,MIZZ at MSST,MISSOURI,MISSISSIPPI STATE,0.697826
8,VILL at SJU,VILLANOVA,ST JOHNS,0.641026
10,MISS at AUB,MISSISSIPPI,AUBURN,0.508097


In [ ]:
import pandas as pd
import requests
from datetime import datetime, timedelta
import re
from difflib import SequenceMatcher
import numpy as np

SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def norm(s):
    return re.sub(r"[^A-Z0-9 ]","", str(s).upper()).strip()

def ratio(a,b):
    return SequenceMatcher(None, norm(a), norm(b)).ratio()

# Build odds_event_id -> commence date
tmp = odds_events_df.copy()
tmp["commence_dt"] = pd.to_datetime(tmp["commence_time"], utc=True, errors="coerce")
tmp["commence_yyyymmdd"] = tmp["commence_dt"].dt.strftime("%Y%m%d")
odds_date_map = dict(zip(tmp["odds_event_id"], tmp["commence_yyyymmdd"]))

needed_dates = sorted(set(df["odds_event_id"].dropna().map(odds_date_map)))
buffer_dates = sorted(set(needed_dates + [
    (datetime.strptime(d,"%Y%m%d") - timedelta(days=1)).strftime("%Y%m%d") for d in needed_dates
] + [
    (datetime.strptime(d,"%Y%m%d") + timedelta(days=1)).strftime("%Y%m%d") for d in needed_dates
]))

print("ESPN dates we will pull:", buffer_dates)

# Pull ESPN events for all buffer dates
events = []
for d in buffer_dates:
    sb = get_json(SCOREBOARD, params={"dates": d, "limit": 500, "offset": 0})
    for ev in sb.get("events", []):
        comp = ev["competitions"][0]
        home = next(c for c in comp["competitors"] if c["homeAway"]=="home")
        away = next(c for c in comp["competitors"] if c["homeAway"]=="away")
        events.append({
            "date": d,
            "eventId": ev["id"],
            "home_name": home["team"].get("displayName",""),
            "away_name": away["team"].get("displayName",""),
        })

events_df = pd.DataFrame(events).drop_duplicates(subset=["eventId"])

# Match each row to ESPN event by name similarity on those dates
def find_event_id(row):
    oid = row["odds_event_id"]
    if pd.isna(oid):
        return None
    d = odds_date_map.get(oid)
    if not d:
        return None

    cand = events_df[events_df["date"].isin([
        d,
        (datetime.strptime(d,"%Y%m%d")-timedelta(days=1)).strftime("%Y%m%d"),
        (datetime.strptime(d,"%Y%m%d")+timedelta(days=1)).strftime("%Y%m%d"),
    ])]
    if cand.empty:
        return None

    # Use odds full names (not tokens)
    odds_row = tmp[tmp["odds_event_id"] == oid].iloc[0]
    away = odds_row["away_team"]
    home = odds_row["home_team"]

    best_id, best_score = None, -1
    for _, e in cand.iterrows():
        s = 0.5*ratio(away, e["away_name"]) + 0.5*ratio(home, e["home_name"])
        if s > best_score:
            best_score = s
            best_id = e["eventId"]

    return best_id if best_score >= 0.78 else None

df["eventId"] = df.apply(find_event_id, axis=1)
print("Rows missing eventId after Odds(fixtures)->ESPN match:", df["eventId"].isna().sum())
display(df[df["eventId"].isna()][["GAME","away_school","home_school","odds_match_score"]].drop_duplicates().head(25))

ESPN dates we will pull: ['20260227', '20260228', '20260301', '20260302']
Rows missing eventId after Odds(fixtures)->ESPN match: 21


,GAME,away_school,home_school,odds_match_score
3,UCLA at MINN,UCLA,MINNESOTA,0.539394
7,MIZZ at MSST,MISSOURI,MISSISSIPPI STATE,0.697826
10,MISS at AUB,MISSISSIPPI,AUBURN,0.508097
15,NCST at ND,NC STATE,NOTRE DAME,0.605714
33,TEX at TXAM,TEXAS,TEXAS AM,0.597826
50,WIS at WASH,WISCONSIN,WASHINGTON,0.703297
85,GTWN at XAV,GEORGETOWN,XAVIER,0.645485


In [ ]:
import re
import numpy as np
from datetime import datetime, timedelta

def norm_simple(s):
    return re.sub(r"[^A-Z0-9 ]","", str(s).upper())

# Pre-normalize ESPN event names
events_df["away_norm"] = events_df["away_name"].apply(norm_simple)
events_df["home_norm"] = events_df["home_name"].apply(norm_simple)

# Pre-normalize Odds team names
tmp["away_norm"] = tmp["away_team"].apply(norm_simple)
tmp["home_norm"] = tmp["home_team"].apply(norm_simple)

def strong_match(odds_away, odds_home, espn_away, espn_home):
    # direct containment instead of fuzzy ratio
    return (
        odds_away in espn_away and odds_home in espn_home
    ) or (
        odds_away in espn_home and odds_home in espn_away
    )

def find_event_id_strict(row):
    oid = row["odds_event_id"]
    if pd.isna(oid):
        return None

    d = odds_date_map.get(oid)
    if not d:
        return None

    cand = events_df[events_df["date"].isin([
        d,
        (datetime.strptime(d,"%Y%m%d")-timedelta(days=1)).strftime("%Y%m%d"),
        (datetime.strptime(d,"%Y%m%d")+timedelta(days=1)).strftime("%Y%m%d"),
    ])]

    if cand.empty:
        return None

    odds_row = tmp[tmp["odds_event_id"] == oid].iloc[0]
    away = odds_row["away_norm"]
    home = odds_row["home_norm"]

    for _, e in cand.iterrows():
        if strong_match(away, home, e["away_norm"], e["home_norm"]):
            return e["eventId"]

    return None

df["eventId"] = df.apply(find_event_id_strict, axis=1)

print("Rows missing eventId after STRICT match:", df["eventId"].isna().sum())
display(df[df["eventId"].isna()][["GAME","away_school","home_school"]].drop_duplicates())

Rows missing eventId after STRICT match: 21


,GAME,away_school,home_school
3,UCLA at MINN,UCLA,MINNESOTA
7,MIZZ at MSST,MISSOURI,MISSISSIPPI STATE
10,MISS at AUB,MISSISSIPPI,AUBURN
15,NCST at ND,NC STATE,NOTRE DAME
33,TEX at TXAM,TEXAS,TEXAS AM
50,WIS at WASH,WISCONSIN,WASHINGTON
85,GTWN at XAV,GEORGETOWN,XAVIER


In [ ]:
import pandas as pd
import numpy as np
import re
from difflib import SequenceMatcher

# expects: df with column GAME
# expects: odds_events_df with columns odds_event_id, home_team, away_team, commence_time

def norm(s):
    return re.sub(r"[^A-Z0-9 ]","", str(s).upper()).strip()

def ratio(a, b):
    return SequenceMatcher(None, norm(a), norm(b)).ratio()

def split_game(g):
    g = str(g).upper().strip()
    if " AT " in g:
        a, h = g.split(" AT ", 1)
        return a.strip(), h.strip()
    return None, None

# Your current token -> school keyword map (keep / extend as needed)
TOKEN_TO_SCHOOL = {
    "COLO": "COLORADO",
    "HOU": "HOUSTON",
    "KU": "KANSAS",
    "ARIZ": "ARIZONA",
    "BYU": "BYU",
    "WVU": "WEST VIRGINIA",
    "UCLA": "UCLA",
    "MINN": "MINNESOTA",
    "ARK": "ARKANSAS",
    "FLA": "FLORIDA",
    "GONZ": "GONZAGA",
    "SMC": "SAINT MARYS",
    "ALA": "ALABAMA",
    "TENN": "TENNESSEE",
    "MIZZ": "MISSOURI",
    "MSST": "MISSISSIPPI STATE",
    "VILL": "VILLANOVA",
    "SJU": "ST JOHNS",
    "MISS": "MISSISSIPPI",      # could be OLE MISS; we'll test both below
    "AUB": "AUBURN",
    "VAN": "VANDERBILT",
    "UK": "KENTUCKY",
    "NEB": "NEBRASKA",
    "USC": "USC",
    "NCST": "NC STATE",
    "ND": "NOTRE DAME",
    "LOU": "LOUISVILLE",
    "CLEM": "CLEMSON",
    "VT": "VIRGINIA TECH",
    "UNC": "NORTH CAROLINA",
    "TTU": "TEXAS TECH",
    "ISU": "IOWA STATE",
    "TEX": "TEXAS",
    "TXAM": "TEXAS A M",
    "UVA": "VIRGINIA",
    "DUKE": "DUKE",
    "WIS": "WISCONSIN",
    "WASH": "WASHINGTON",
    "GTWN": "GEORGETOWN",
    "XAV": "XAVIER",
}

# Pre-normalize odds fixture names
od = odds_events_df.copy()
od["away_norm"] = od["away_team"].apply(norm)
od["home_norm"] = od["home_team"].apply(norm)

# Build unique games from sheet
games = df["GAME"].dropna().astype(str).str.upper().str.strip().drop_duplicates().tolist()

rows = []
TOPK = 5

for g in games:
    away_tok, home_tok = split_game(g)
    if away_tok is None:
        continue

    away_guess = TOKEN_TO_SCHOOL.get(away_tok, away_tok)
    home_guess = TOKEN_TO_SCHOOL.get(home_tok, home_tok)

    # score every odds fixture
    scores = []
    for r in od.itertuples(index=False):
        s1 = 0.5*ratio(away_guess, r.away_norm) + 0.5*ratio(home_guess, r.home_norm)
        s2 = 0.5*ratio(away_guess, r.home_norm) + 0.5*ratio(home_guess, r.away_norm)  # handle swapped
        s = max(s1, s2)
        scores.append((r.odds_event_id, r.commence_time, r.away_team, r.home_team, s))

    scores.sort(key=lambda x: x[4], reverse=True)
    top = scores[:TOPK]

    for rank, (eid, ct, away_team, home_team, s) in enumerate(top, start=1):
        rows.append({
            "GAME_SHEET": g,
            "AWAY_TOKEN": away_tok,
            "HOME_TOKEN": home_tok,
            "AWAY_GUESS": away_guess,
            "HOME_GUESS": home_guess,
            "CAND_RANK": rank,
            "ODDS_EVENT_ID": eid,
            "COMMENCE_TIME": ct,
            "ODDS_AWAY": away_team,
            "ODDS_HOME": home_team,
            "MATCH_SCORE": round(s, 4)
        })

diag = pd.DataFrame(rows)

# Summarize the best match per game
best = diag.sort_values(["GAME_SHEET","MATCH_SCORE"], ascending=[True, False]).groupby("GAME_SHEET").head(1)
best = best.sort_values("MATCH_SCORE")

print("Best-match distribution (lowest first):")
display(best[["GAME_SHEET","AWAY_GUESS","HOME_GUESS","ODDS_AWAY","ODDS_HOME","COMMENCE_TIME","MATCH_SCORE"]])

print("\nFull Top-5 candidates per game saved.")
diag.to_csv("odds_matchup_diagnostics_top5.csv", index=False)
print("Saved: odds_matchup_diagnostics_top5.csv")

Best-match distribution (lowest first):


,GAME_SHEET,AWAY_GUESS,HOME_GUESS,ODDS_AWAY,ODDS_HOME,COMMENCE_TIME,MATCH_SCORE
55,NEB AT USC,NEBRASKA,USC,Nebraska Cornhuskers,USC Trojans,2026-02-28T21:00:00Z,0.5000
85,UVA AT DUKE,VIRGINIA,DUKE,Virginia Cavaliers,Duke Blue Devils,2026-02-28T17:01:12Z,0.5077
45,MISS AT AUB,MISSISSIPPI,AUBURN,Ole Miss Rebels,Auburn Tigers,2026-03-01T01:30:41Z,0.5081
15,UCLA AT MINN,UCLA,MINNESOTA,UCLA Bruins,Minnesota Golden Gophers,2026-02-28T19:00:39Z,0.5394
10,BYU AT WVU,BYU,WEST VIRGINIA,BYU Cougars,West Virginia Mountaineers,2026-02-28T22:30:00Z,0.5476
30,ALA AT TENN,ALABAMA,TENNESSEE,Alabama Crimson Tide,Tennessee Volunteers,2026-02-28T23:00:11Z,0.5696
80,TEX AT TXAM,TEXAS,TEXAS A M,Texas Longhorns,Texas A&M Aggies,2026-02-28T21:00:36Z,0.5833
5,KU AT ARIZ,KANSAS,ARIZONA,Kansas Jayhawks,Arizona Wildcats,2026-02-28T21:00:00Z,0.5901
60,NCST AT ND,NC STATE,NOTRE DAME,NC State Wolfpack,Notre Dame Fighting Irish,2026-02-28T17:00:20Z,0.6057
0,COLO AT HOU,COLORADO,HOUSTON,Colorado Buffaloes,Houston Cougars,2026-02-28T17:02:10Z,0.6259



Full Top-5 candidates per game saved.
Saved: odds_matchup_diagnostics_top5.csv


In [ ]:
import re
import numpy as np
from datetime import datetime, timedelta

def norm_simple(s):
    return re.sub(r"[^A-Z0-9 ]","", str(s).upper())

# normalize once
odds_events_df["away_norm"] = odds_events_df["away_team"].apply(norm_simple)
odds_events_df["home_norm"] = odds_events_df["home_team"].apply(norm_simple)

events = []

for d in buffer_dates:
    sb = get_json(SCOREBOARD, params={"dates": d, "limit": 500})
    for ev in sb.get("events", []):
        comp = ev["competitions"][0]
        home = next(c for c in comp["competitors"] if c["homeAway"]=="home")
        away = next(c for c in comp["competitors"] if c["homeAway"]=="away")
        events.append({
            "date": d,
            "eventId": ev["id"],
            "home_norm": norm_simple(home["team"].get("displayName","")),
            "away_norm": norm_simple(away["team"].get("displayName","")),
        })

events_df = pd.DataFrame(events).drop_duplicates("eventId")

# map odds_event_id -> full school names
odds_events_df["away_school"] = odds_events_df["away_team"].apply(norm_simple)
odds_events_df["home_school"] = odds_events_df["home_team"].apply(norm_simple)

odds_map = dict(zip(odds_events_df["odds_event_id"],
                    zip(odds_events_df["away_school"], odds_events_df["home_school"])))

def strict_containment_match(row):
    oid = row["odds_event_id"]
    if pd.isna(oid):
        return None

    if oid not in odds_map:
        return None

    away_school, home_school = odds_map[oid]

    for _, e in events_df.iterrows():
        if (
            away_school in e["away_norm"] and home_school in e["home_norm"]
        ) or (
            away_school in e["home_norm"] and home_school in e["away_norm"]
        ):
            return e["eventId"]

    return None

df["eventId"] = df.apply(strict_containment_match, axis=1)

print("Rows missing eventId after containment match:", df["eventId"].isna().sum())
display(df[df["eventId"].isna()][["GAME","away_school","home_school"]].drop_duplicates())

Rows missing eventId after containment match: 21


,GAME,away_school,home_school
3,UCLA at MINN,UCLA,MINNESOTA
7,MIZZ at MSST,MISSOURI,MISSISSIPPI STATE
10,MISS at AUB,MISSISSIPPI,AUBURN
15,NCST at ND,NC STATE,NOTRE DAME
33,TEX at TXAM,TEXAS,TEXAS AM
50,WIS at WASH,WISCONSIN,WASHINGTON
85,GTWN at XAV,GEORGETOWN,XAVIER


In [ ]:
import pandas as pd
import numpy as np

# df = your working dataframe (already has your cleaned columns + odds + trend model pieces)
# eventId column exists but some are NaN

df["eventId"] = df["eventId"].astype(str)
df.loc[df["eventId"].isin(["nan","None","NaN",""]), "eventId"] = np.nan

df_ok = df[df["eventId"].notna()].copy()
df_bad = df[df["eventId"].isna()].copy()

print("Rows kept (ESPN match):", len(df_ok))
print("Rows dropped (no ESPN match):", len(df_bad))
print("\nDropped games (unique):")
print(df_bad["GAME"].dropna().drop_duplicates().to_list())

Rows kept (ESPN match): 79
Rows dropped (no ESPN match): 21

Dropped games (unique):
['UCLA at MINN', 'MIZZ at MSST', 'MISS at AUB', 'NCST at ND', 'TEX at TXAM', 'WIS at WASH', 'GTWN at XAV']


In [ ]:
import requests, re
from functools import lru_cache

ESPN_SUMMARY = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

@lru_cache(maxsize=2048)
def get_summary(event_id: str):
    return get_json(ESPN_SUMMARY, params={"event": str(event_id)})

def norm_name(s):
    s = str(s).upper().strip()
    s = re.sub(r"\b(JR|SR|II|III|IV)\b", "", s)
    s = re.sub(r"[^A-Z ]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def parse_minutes(mmss):
    # "34" or "34:12" or None
    if mmss is None or (isinstance(mmss,float) and np.isnan(mmss)):
        return np.nan
    s = str(mmss).strip()
    if ":" in s:
        m, sec = s.split(":", 1)
        try:
            return float(m) + float(sec)/60.0
        except:
            return np.nan
    try:
        return float(s)
    except:
        return np.nan

def extract_event_player_minutes(event_id: str):
    j = get_summary(event_id)
    box = j.get("boxscore", {})
    players = box.get("players", [])

    rows = []
    for team_block in players:
        team = team_block.get("team", {})
        team_id = team.get("id")
        team_name = team.get("displayName")

        # The "statistics" list usually contains multiple stat groups; minutes show under a group that includes "MIN"
        for stat_group in team_block.get("statistics", []):
            labels = stat_group.get("labels", [])
            if "MIN" not in labels:
                continue
            min_idx = labels.index("MIN")

            for a in stat_group.get("athletes", []):
                ath = a.get("athlete", {}) or {}
                disp = ath.get("displayName", "")
                pid = ath.get("id")
                stats = a.get("stats", [])

                mm = stats[min_idx] if min_idx < len(stats) else None
                minutes = parse_minutes(mm)

                # Status sometimes appears here; often missing
                status = None
                if "status" in ath and isinstance(ath["status"], dict):
                    status = ath["status"].get("type") or ath["status"].get("name")

                rows.append({
                    "eventId": str(event_id),
                    "team_id": team_id,
                    "team_name": team_name,
                    "player_id": pid,
                    "player_name": disp,
                    "player_name_norm": norm_name(disp),
                    "minutes": minutes,
                    "status": status
                })
    dfm = pd.DataFrame(rows)
    if dfm.empty:
        return dfm

    # Starter proxy: top 5 minutes on team
    dfm["starter_proxy"] = 0
    dfm = dfm.sort_values(["team_id","minutes"], ascending=[True, False])
    dfm["starter_proxy"] = dfm.groupby("team_id").cumcount().lt(5).astype(int)
    return dfm

# Build a per-event minutes table
event_ids = sorted(df_ok["eventId"].dropna().astype(str).unique().tolist())

minutes_tables = []
for eid in event_ids:
    try:
        minutes_tables.append(extract_event_player_minutes(eid))
    except Exception as e:
        print("Failed minutes for event", eid, ":", e)

minutes_df = pd.concat(minutes_tables, ignore_index=True) if len(minutes_tables) else pd.DataFrame()
print("Minutes rows:", len(minutes_df))
display(minutes_df.head(10))

Minutes rows: 57


,eventId,team_id,team_name,player_id,player_name,player_name_norm,minutes,status,starter_proxy
0,401820771,150,Duke Blue Devils,5041935,Cameron Boozer,CAMERON BOOZER,18.0,None,1
1,401820771,150,Duke Blue Devils,5287474,Dame Sarr,DAME SARR,16.0,None,1
2,401820771,150,Duke Blue Devils,5061585,Isaiah Evans,ISAIAH EVANS,13.0,None,1
3,401820771,150,Duke Blue Devils,4873209,Patrick Ngongba II,PATRICK NGONGBA,12.0,None,1
4,401820771,150,Duke Blue Devils,4711256,Caleb Foster,CALEB FOSTER,11.0,None,1
5,401820771,150,Duke Blue Devils,5105337,Maliq Brown,MALIQ BROWN,8.0,None,0
6,401820771,150,Duke Blue Devils,5041937,Cayden Boozer,CAYDEN BOOZER,8.0,None,0
7,401820771,150,Duke Blue Devils,4873107,Darren Harris,DARREN HARRIS,5.0,None,0
8,401820771,150,Duke Blue Devils,5144124,Nikolas Khamenia,NIKOLAS KHAMENIA,3.0,None,0
9,401820771,150,Duke Blue Devils,5107141,Ifeanyi Ufochukwu,IFEANYI UFOCHUKWU,NaN,None,0


In [ ]:
from difflib import SequenceMatcher

def best_player_match(player_name, event_id):
    if minutes_df.empty:
        return (np.nan, np.nan, np.nan, 0, np.nan)
    pool = minutes_df[minutes_df["eventId"] == str(event_id)]
    if pool.empty:
        return (np.nan, np.nan, np.nan, 0, np.nan)

    target = norm_name(player_name)
    # exact match first
    exact = pool[pool["player_name_norm"] == target]
    if len(exact):
        r = exact.iloc[0]
        return (r["player_id"], r["player_name"], r["minutes"], r["starter_proxy"], r["status"])

    # fuzzy within-event
    best_i, best_s = None, -1
    for i, r in pool.iterrows():
        s = SequenceMatcher(None, target, r["player_name_norm"]).ratio()
        if s > best_s:
            best_s = s
            best_i = i
    r = pool.loc[best_i]
    return (r["player_id"], r["player_name"], r["minutes"], r["starter_proxy"], r["status"]) if best_s >= 0.86 else (np.nan, np.nan, np.nan, 0, np.nan)

df_ok[["ESPN_PLAYER_ID","ESPN_PLAYER_NAME","ESPN_MIN_LAST","ESPN_STARTER_PROXY","ESPN_STATUS"]] = df_ok.apply(
    lambda r: pd.Series(best_player_match(r["PLAYER"], r["eventId"])),
    axis=1
)

print("Rows missing ESPN player match:", df_ok["ESPN_PLAYER_ID"].isna().sum())
display(df_ok[df_ok["ESPN_PLAYER_ID"].isna()][["PLAYER","GAME","eventId"]].head(20))

Rows missing ESPN player match: 71


,PLAYER,GAME,eventId
1,Melvin Council Jr.,KU at ARIZ,401827707
2,Brenen Lorient,BYU at WVU,401827714
4,Xaivian Lee,ARK at FLA,401808266
5,Graham Ike,GONZ at SMC,401829265
6,Aden Holloway,ALA at TENN,401808271
8,Tyler Perkins,VILL at SJU,401822962
9,Oziyah Sellers,VILL at SJU,401822962
11,Devin McGlockton,VAN at UK,401808268
13,Brenen Lorient,BYU at WVU,401827714
14,Rienk Mast,NEB at USC,401825548


In [ ]:
def status_penalty(status):
    if status is None or (isinstance(status,float) and np.isnan(status)):
        return 0.0
    s = str(status).upper()
    if "OUT" in s:
        return -0.20
    if "DOUBT" in s:
        return -0.10
    if "QUES" in s:
        return -0.06
    return 0.0

def minutes_adjust(mins):
    # Conservative: cap impact. Treat <18 mins as higher volatility.
    if pd.isna(mins):
        return 0.0
    if mins >= 30:
        return +0.03
    if mins >= 24:
        return +0.015
    if mins >= 18:
        return 0.0
    return -0.04

def starter_adjust(starter_proxy):
    return 0.015 if int(starter_proxy) == 1 else 0.0

df_ok["ADJ_MIN"] = df_ok["ESPN_MIN_LAST"].apply(minutes_adjust)
df_ok["ADJ_START"] = df_ok["ESPN_STARTER_PROXY"].apply(starter_adjust)
df_ok["ADJ_STATUS"] = df_ok["ESPN_STATUS"].apply(status_penalty)

# If you already have MODEL_PROB from earlier, use it.
# Otherwise fallback to P_TREND (+ ADJ_DVP if you built that).
base_col = "MODEL_PROB" if "MODEL_PROB" in df_ok.columns else ("P_TREND" if "P_TREND" in df_ok.columns else None)
if base_col is None:
    raise ValueError("Need base probability column: MODEL_PROB or P_TREND")

df_ok["MODEL_PROB_ESPN"] = (
    df_ok[base_col]
    + df_ok["ADJ_MIN"]
    + df_ok["ADJ_START"]
    + df_ok["ADJ_STATUS"]
).clip(0.01, 0.99)

# Rank Top 10 Most Likely
cols_show = ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",base_col,"MODEL_PROB_ESPN","ESPN_MIN_LAST","ESPN_STARTER_PROXY","ESPN_STATUS"]
top10 = df_ok.sort_values("MODEL_PROB_ESPN", ascending=False).head(10)

display(top10[cols_show])

top10.to_csv("prop_engine_v1_top10_most_likely_ESPN.csv", index=False)
df_ok.to_csv("prop_engine_v1_full_ESPN_enriched.csv", index=False)
print("Saved: prop_engine_v1_top10_most_likely_ESPN.csv")
print("Saved: prop_engine_v1_full_ESPN_enriched.csv")

ValueError: Need base probability column: MODEL_PROB or P_TREND

In [ ]:
import numpy as np

# --------- 1) Normalize % columns safely ---------

PCT_COLS = ["L5","L10","2025","HM_AW","H2H","DVP_L5","DVP_L10","DVP_L10_2","DVP_HM_AW"]

def pct_to_float(x):
    if pd.isna(x): return np.nan
    s = str(x).strip()
    if s in ["--","-","—",""]: return np.nan
    s = s.replace("%","")
    try:
        return float(s)/100.0
    except:
        return np.nan

for c in PCT_COLS:
    if c in df_ok.columns:
        df_ok[c] = df_ok[c].apply(pct_to_float)

# --------- 2) Weighted Trend Model (Anti-overfit) ---------

W = {
    "L5": 0.22,
    "L10": 0.28,
    "2025": 0.28,
    "HM_AW": 0.12,
    "H2H": 0.10
}

def weighted_prob(row):
    num, den = 0.0, 0.0
    for k,w in W.items():
        v = row.get(k, np.nan)
        if pd.notna(v):
            num += w * v
            den += w
    return num/den if den > 0 else np.nan

df_ok["P_TREND"] = df_ok.apply(weighted_prob, axis=1)

# --------- 3) DVP Matchup Proxy (Capped Influence) ---------

DVP_COLS = ["DVP_L5","DVP_L10","DVP_L10_2","DVP_HM_AW"]

df_ok["P_DVP_RAW"] = df_ok[DVP_COLS].mean(axis=1, skipna=True)

def dvp_adjust(p_dvp):
    if pd.isna(p_dvp):
        return 0.0
    adj = (p_dvp - 0.50)
    return float(np.clip(adj, -0.07, 0.07))   # cap impact

df_ok["ADJ_DVP"] = df_ok["P_DVP_RAW"].apply(dvp_adjust)

# --------- 4) Base Model Probability ---------

df_ok["MODEL_PROB"] = (df_ok["P_TREND"] + df_ok["ADJ_DVP"]).clip(0.01, 0.99)

print("Base model rebuilt successfully.")
display(df_ok[["PLAYER","GAME","P_TREND","ADJ_DVP","MODEL_PROB"]].head())

KeyError: "None of [Index(['DVP_L5', 'DVP_L10', 'DVP_L10_2', 'DVP_HM_AW'], dtype='object')] are in the [columns]"

In [ ]:
import re

print("Columns containing 'DVP':")
print([c for c in df_ok.columns if "DVP" in c.upper()])

# Standardize common DVP variants -> our standard names
rename_map = {}

for c in df_ok.columns:
    cu = c.upper().strip()
    # normalize spaces and punctuation
    cu_clean = re.sub(r"[^A-Z0-9]+", "_", cu)

    if cu_clean == "DVP_L5":
        rename_map[c] = "DVP_L5"
    elif cu_clean == "DVP_L10":
        rename_map[c] = "DVP_L10"
    elif cu_clean in ["DVP_L10_1","DVP_L10_2","DVP_L10_1_","DVP_L10_2_"]:
        rename_map[c] = "DVP_L10_2"
    elif cu_clean in ["DVP_HM_AW","DVP_HM_AW_","DVP_HOME_AWAY","DVP_HOME_AWAY_"]:
        rename_map[c] = "DVP_HM_AW"
    elif cu_clean in ["DVP_L10_2", "DVP_L10_02"]:
        rename_map[c] = "DVP_L10_2"

df_ok.rename(columns=rename_map, inplace=True)

print("\nAfter rename, DVP columns present:")
print([c for c in df_ok.columns if c.startswith("DVP")])

Columns containing 'DVP':
['DVP L5', 'DVP L10', 'DVP L10.1', 'DVP HM/AW']

After rename, DVP columns present:
['DVP_L5', 'DVP_L10', 'DVP_L10_2', 'DVP_HM_AW']


In [ ]:
import numpy as np
import pandas as pd

# 1) Normalize % columns (again safe if already floats)
PCT_COLS = ["L5","L10","2025","HM_AW","H2H"] + [c for c in df_ok.columns if c.startswith("DVP")]

def pct_to_float(x):
    if pd.isna(x): return np.nan
    if isinstance(x, (int,float,np.floating)):  # already numeric
        return float(x) if 0 <= float(x) <= 1 else np.nan
    s = str(x).strip()
    if s in ["--","-","—",""]: return np.nan
    s = s.replace("%","")
    try:
        v = float(s)/100.0
        return v if 0 <= v <= 1 else np.nan
    except:
        return np.nan

for c in PCT_COLS:
    if c in df_ok.columns:
        df_ok[c] = df_ok[c].apply(pct_to_float)

# 2) Weighted Trend (anti-overfit)
W = {"L5":0.22, "L10":0.28, "2025":0.28, "HM_AW":0.12, "H2H":0.10}

def weighted_prob(row):
    num, den = 0.0, 0.0
    for k,w in W.items():
        v = row.get(k, np.nan)
        if pd.notna(v):
            num += w*v
            den += w
    return num/den if den else np.nan

df_ok["P_TREND"] = df_ok.apply(weighted_prob, axis=1)

# 3) DVP proxy using whatever DVP columns exist
dvp_cols = [c for c in ["DVP_L5","DVP_L10","DVP_L10_2","DVP_HM_AW"] if c in df_ok.columns]

if len(dvp_cols) == 0:
    df_ok["P_DVP_RAW"] = np.nan
    df_ok["ADJ_DVP"] = 0.0
else:
    df_ok["P_DVP_RAW"] = df_ok[dvp_cols].mean(axis=1, skipna=True)

    def dvp_adjust(p_dvp):
        if pd.isna(p_dvp):
            return 0.0
        adj = (p_dvp - 0.50)
        return float(np.clip(adj, -0.07, 0.07))

    df_ok["ADJ_DVP"] = df_ok["P_DVP_RAW"].apply(dvp_adjust)

# 4) Base model probability
df_ok["MODEL_PROB"] = (df_ok["P_TREND"] + df_ok["ADJ_DVP"]).clip(0.01, 0.99)

print("Base model rebuilt.")
print("DVP columns used:", dvp_cols)
display(df_ok[["PLAYER","GAME","P_TREND","ADJ_DVP","MODEL_PROB"]].head())

Base model rebuilt.
DVP columns used: ['DVP_L5', 'DVP_L10', 'DVP_L10_2', 'DVP_HM_AW']


,PLAYER,GAME,P_TREND,ADJ_DVP,MODEL_PROB
0,Barrington Hargress,COLO at HOU,0.843436,-0.01175,0.831686
1,Melvin Council Jr.,KU at ARIZ,0.816364,-0.07000,0.746364
2,Brenen Lorient,BYU at WVU,0.831282,0.07000,0.901282
4,Xaivian Lee,ARK at FLA,0.810103,0.07000,0.880103
5,Graham Ike,GONZ at SMC,0.719773,-0.07000,0.649773


In [ ]:
import numpy as np
import pandas as pd

def status_penalty(status):
    if status is None or (isinstance(status,float) and np.isnan(status)):
        return 0.0
    s = str(status).upper()
    if "OUT" in s:
        return -0.20
    if "DOUBT" in s:
        return -0.10
    if "QUES" in s:
        return -0.06
    return 0.0

def minutes_adjust(mins):
    # Conservative: cap impact. Treat low mins as more volatile.
    if pd.isna(mins):
        return 0.0
    if mins >= 30:
        return +0.03
    if mins >= 24:
        return +0.015
    if mins >= 18:
        return 0.0
    return -0.04

def starter_adjust(starter_proxy):
    try:
        return 0.015 if int(starter_proxy) == 1 else 0.0
    except:
        return 0.0

df_ok["ADJ_MIN"] = df_ok["ESPN_MIN_LAST"].apply(minutes_adjust)
df_ok["ADJ_START"] = df_ok["ESPN_STARTER_PROXY"].apply(starter_adjust)
df_ok["ADJ_STATUS"] = df_ok["ESPN_STATUS"].apply(status_penalty)

df_ok["MODEL_PROB_ESPN"] = (
    df_ok["MODEL_PROB"]
    + df_ok["ADJ_MIN"]
    + df_ok["ADJ_START"]
    + df_ok["ADJ_STATUS"]
).clip(0.01, 0.99)

# Rank Top 10 Most Likely
cols_show = [
    "PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "MODEL_PROB","MODEL_PROB_ESPN",
    "ESPN_MIN_LAST","ESPN_STARTER_PROXY","ESPN_STATUS",
    "ADJ_MIN","ADJ_START","ADJ_STATUS"
]

top10 = df_ok.dropna(subset=["MODEL_PROB_ESPN"]).sort_values("MODEL_PROB_ESPN", ascending=False).head(10)
display(top10[cols_show])

# Save outputs
top10.to_csv("prop_engine_v1_top10_most_likely_ESPN.csv", index=False)
df_ok.to_csv("prop_engine_v1_full_ESPN_enriched.csv", index=False)

print("Saved: prop_engine_v1_top10_most_likely_ESPN.csv")
print("Saved: prop_engine_v1_full_ESPN_enriched.csv")
print("Rows missing ESPN player match:", df_ok["ESPN_PLAYER_ID"].isna().sum())

KeyError: "['AMERICAN_ODDS'] not in index"

In [ ]:
import pandas as pd
import numpy as np

# Ensure AMERICAN_ODDS exists
if "AMERICAN_ODDS" not in df_ok.columns:
    if "ODDS" in df_ok.columns:
        df_ok["AMERICAN_ODDS"] = pd.to_numeric(df_ok["ODDS"], errors="coerce")
    else:
        df_ok["AMERICAN_ODDS"] = np.nan  # safe fallback

# Recompute BOOK_PROB if you want it displayed later (optional)
def american_to_prob(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    if o < 0:
        return (-o) / ((-o) + 100.0)
    else:
        return 100.0 / (o + 100.0)

if "BOOK_PROB" not in df_ok.columns:
    df_ok["BOOK_PROB"] = df_ok["AMERICAN_ODDS"].apply(american_to_prob)

# Build display columns safely (only those that exist)
cols_show = [
    "PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "MODEL_PROB","MODEL_PROB_ESPN",
    "ESPN_MIN_LAST","ESPN_STARTER_PROXY","ESPN_STATUS",
    "ADJ_MIN","ADJ_START","ADJ_STATUS"
]
cols_show = [c for c in cols_show if c in df_ok.columns]

top10 = df_ok.dropna(subset=["MODEL_PROB_ESPN"]).sort_values("MODEL_PROB_ESPN", ascending=False).head(10)
display(top10[cols_show])

top10.to_csv("prop_engine_v1_top10_most_likely_ESPN.csv", index=False)
df_ok.to_csv("prop_engine_v1_full_ESPN_enriched.csv", index=False)

print("Saved: prop_engine_v1_top10_most_likely_ESPN.csv")
print("Saved: prop_engine_v1_full_ESPN_enriched.csv")
print("Rows missing ESPN player match:", df_ok["ESPN_PLAYER_ID"].isna().sum() if "ESPN_PLAYER_ID" in df_ok.columns else "N/A")

,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,MODEL_PROB,MODEL_PROB_ESPN,ESPN_MIN_LAST,ESPN_STARTER_PROXY,ESPN_STATUS,ADJ_MIN,ADJ_START,ADJ_STATUS
9,Oziyah Sellers,VILL at SJU,Rebounds,u3.5,NaN,0.904545,0.904545,NaN,0.0,NaN,0.0,0.000,0.0
2,Brenen Lorient,BYU at WVU,Points,o9.5,NaN,0.901282,0.901282,NaN,0.0,NaN,0.0,0.000,0.0
39,Brayden Burries,KU at ARIZ,Rebounds,u5.5,NaN,0.882273,0.882273,NaN,0.0,NaN,0.0,0.000,0.0
4,Xaivian Lee,ARK at FLA,Rebounds,u3.5,NaN,0.880103,0.880103,NaN,0.0,NaN,0.0,0.000,0.0
13,Brenen Lorient,BYU at WVU,Three Pointers Made,u0.5,NaN,0.879744,0.879744,NaN,0.0,NaN,0.0,0.000,0.0
21,Jamarques Lawrence,NEB at USC,Rebounds,u3.5,NaN,0.866410,0.866410,NaN,0.0,NaN,0.0,0.000,0.0
0,Barrington Hargress,COLO at HOU,Points,o9.5,NaN,0.831686,0.846686,18.0,1.0,NaN,0.0,0.015,0.0
17,Amari Allen,ALA at TENN,Points,o9.5,NaN,0.841282,0.841282,NaN,0.0,NaN,0.0,0.000,0.0
47,Zuby Ejiofor,VILL at SJU,Rebounds,u7.5,NaN,0.837273,0.837273,NaN,0.0,NaN,0.0,0.000,0.0
14,Rienk Mast,NEB at USC,Three Pointers Made,u1.5,NaN,0.835538,0.835538,NaN,0.0,NaN,0.0,0.000,0.0


Saved: prop_engine_v1_top10_most_likely_ESPN.csv
Saved: prop_engine_v1_full_ESPN_enriched.csv
Rows missing ESPN player match: 71


In [ ]:
import pandas as pd
import numpy as np

print("Columns that look like odds:")
odds_like = [c for c in df_ok.columns if "ODD" in c.upper() or "PRICE" in c.upper() or "LINE" in c.upper()]
print(odds_like)

# If your original file uses "ODDS", make sure it's numeric and copied over
if "AMERICAN_ODDS" not in df_ok.columns or df_ok["AMERICAN_ODDS"].isna().all():
    if "ODDS" in df_ok.columns:
        df_ok["AMERICAN_ODDS"] = pd.to_numeric(df_ok["ODDS"], errors="coerce")
    elif "AMERICAN ODDS" in df_ok.columns:
        df_ok["AMERICAN_ODDS"] = pd.to_numeric(df_ok["AMERICAN ODDS"], errors="coerce")
    else:
        # Try the first odds-like column if it exists
        if len(odds_like) > 0:
            df_ok["AMERICAN_ODDS"] = pd.to_numeric(df_ok[odds_like[0]], errors="coerce")

print("AMERICAN_ODDS null rate:", df_ok["AMERICAN_ODDS"].isna().mean())
display(df_ok[["PLAYER","GAME","ODDS","AMERICAN_ODDS"]].head(10) if "ODDS" in df_ok.columns else df_ok[["PLAYER","GAME","AMERICAN_ODDS"]].head(10))

Columns that look like odds:
['ODDS', 'odds_game_id', 'odds_event_id', 'odds_match_score', 'AMERICAN_ODDS']
AMERICAN_ODDS null rate: 1.0


,PLAYER,GAME,ODDS,AMERICAN_ODDS
0,Barrington Hargress,COLO at HOU,BET\n -176,NaN
1,Melvin Council Jr.,KU at ARIZ,BET\n -130,NaN
2,Brenen Lorient,BYU at WVU,BET\n -164,NaN
4,Xaivian Lee,ARK at FLA,BET\n -170,NaN
5,Graham Ike,GONZ at SMC,BET\n -120,NaN
6,Aden Holloway,ALA at TENN,BET\n -155,NaN
8,Tyler Perkins,VILL at SJU,BET\n -124,NaN
9,Oziyah Sellers,VILL at SJU,BET\n -170,NaN
11,Devin McGlockton,VAN at UK,BET\n -152,NaN
13,Brenen Lorient,BYU at WVU,BET\n -190,NaN


In [ ]:
import pandas as pd
import numpy as np
import requests, re
from functools import lru_cache
from difflib import SequenceMatcher

ESPN_SUMMARY = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

@lru_cache(maxsize=2048)
def get_summary(event_id: str):
    return get_json(ESPN_SUMMARY, params={"event": str(event_id)})

def norm_name(s):
    s = str(s).upper().strip()
    s = re.sub(r"\b(JR|SR|II|III|IV)\b", "", s)
    s = re.sub(r"[^A-Z ]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def parse_minutes(mmss):
    if mmss is None or (isinstance(mmss,float) and np.isnan(mmss)):
        return np.nan
    s = str(mmss).strip()
    if ":" in s:
        m, sec = s.split(":", 1)
        try:
            return float(m) + float(sec)/60.0
        except:
            return np.nan
    try:
        return float(s)
    except:
        return np.nan

def extract_event_minutes(event_id: str):
    j = get_summary(event_id)
    box = j.get("boxscore", {})
    players = box.get("players", [])
    rows = []

    for team_block in players:
        team = (team_block.get("team") or {})
        team_id = team.get("id")
        team_name = team.get("displayName")

        for stat_group in team_block.get("statistics", []):
            labels = stat_group.get("labels", [])
            if "MIN" not in labels:
                continue
            min_idx = labels.index("MIN")

            for a in stat_group.get("athletes", []):
                ath = a.get("athlete", {}) or {}
                disp = ath.get("displayName", "")
                pid = ath.get("id")
                stats = a.get("stats", [])

                mm = stats[min_idx] if min_idx < len(stats) else None
                minutes = parse_minutes(mm)

                rows.append({
                    "eventId": str(event_id),
                    "team_id": team_id,
                    "team_name": team_name,
                    "player_id": pid,
                    "player_name": disp,
                    "player_name_norm": norm_name(disp),
                    "minutes": minutes
                })

    dfm = pd.DataFrame(rows)
    if dfm.empty:
        return dfm

    # Keep the max minutes per player (in case multiple groups produce entries)
    dfm = dfm.sort_values("minutes", ascending=False).drop_duplicates(["eventId","player_id"], keep="first")

    # Starter proxy: top 5 minutes per team
    dfm["starter_proxy"] = 0
    dfm = dfm.sort_values(["team_id","minutes"], ascending=[True, False])
    dfm["starter_proxy"] = dfm.groupby("team_id").cumcount().lt(5).astype(int)

    return dfm

event_ids = sorted(df_ok["eventId"].dropna().astype(str).unique())
minutes_tables = []
for eid in event_ids:
    try:
        minutes_tables.append(extract_event_minutes(eid))
    except Exception as e:
        print("Failed event", eid, e)

minutes_df = pd.concat(minutes_tables, ignore_index=True) if minutes_tables else pd.DataFrame()
print("Minutes rows:", len(minutes_df))
display(minutes_df.head(10))

def best_player_match(player_name, event_id):
    pool = minutes_df[minutes_df["eventId"] == str(event_id)]
    if pool.empty:
        return (np.nan, np.nan, np.nan, 0)
    target = norm_name(player_name)

    # exact normalized
    exact = pool[pool["player_name_norm"] == target]
    if len(exact):
        r = exact.iloc[0]
        return (r["player_id"], r["player_name"], r["minutes"], r["starter_proxy"])

    # fuzzy within event
    best_i, best_s = None, -1
    for i, r in pool.iterrows():
        s = SequenceMatcher(None, target, r["player_name_norm"]).ratio()
        if s > best_s:
            best_s = s
            best_i = i
    if best_s < 0.86:
        return (np.nan, np.nan, np.nan, 0)
    r = pool.loc[best_i]
    return (r["player_id"], r["player_name"], r["minutes"], r["starter_proxy"])

df_ok[["ESPN_PLAYER_ID","ESPN_PLAYER_NAME","ESPN_MIN_LAST","ESPN_STARTER_PROXY"]] = df_ok.apply(
    lambda r: pd.Series(best_player_match(r["PLAYER"], r["eventId"])),
    axis=1
)

print("Rows missing ESPN player match (after rebuild):", df_ok["ESPN_PLAYER_ID"].isna().sum())

Minutes rows: 57


,eventId,team_id,team_name,player_id,player_name,player_name_norm,minutes,starter_proxy
0,401820771,150,Duke Blue Devils,5041935,Cameron Boozer,CAMERON BOOZER,19.0,1
1,401820771,150,Duke Blue Devils,5287474,Dame Sarr,DAME SARR,17.0,1
2,401820771,150,Duke Blue Devils,5061585,Isaiah Evans,ISAIAH EVANS,14.0,1
3,401820771,150,Duke Blue Devils,4873209,Patrick Ngongba II,PATRICK NGONGBA,13.0,1
4,401820771,150,Duke Blue Devils,4711256,Caleb Foster,CALEB FOSTER,11.0,1
5,401820771,150,Duke Blue Devils,5041937,Cayden Boozer,CAYDEN BOOZER,9.0,0
6,401820771,150,Duke Blue Devils,5105337,Maliq Brown,MALIQ BROWN,8.0,0
7,401820771,150,Duke Blue Devils,4873107,Darren Harris,DARREN HARRIS,5.0,0
8,401820771,150,Duke Blue Devils,5144124,Nikolas Khamenia,NIKOLAS KHAMENIA,3.0,0
9,401820771,150,Duke Blue Devils,5107141,Ifeanyi Ufochukwu,IFEANYI UFOCHUKWU,NaN,0


Rows missing ESPN player match (after rebuild): 71


In [ ]:
def minutes_adjust(mins):
    if pd.isna(mins): return 0.0
    if mins >= 30: return +0.03
    if mins >= 24: return +0.015
    if mins >= 18: return 0.0
    return -0.04

def starter_adjust(x):
    try:
        return 0.015 if int(x)==1 else 0.0
    except:
        return 0.0

df_ok["ADJ_MIN"] = df_ok["ESPN_MIN_LAST"].apply(minutes_adjust)
df_ok["ADJ_START"] = df_ok["ESPN_STARTER_PROXY"].apply(starter_adjust)

df_ok["MODEL_PROB_ESPN"] = (df_ok["MODEL_PROB"] + df_ok["ADJ_MIN"] + df_ok["ADJ_START"]).clip(0.01, 0.99)

top10 = df_ok.sort_values("MODEL_PROB_ESPN", ascending=False).head(10)
display(top10[["PLAYER","GAME","PROP","OUTCOME","MODEL_PROB","MODEL_PROB_ESPN","ESPN_MIN_LAST","ESPN_STARTER_PROXY","ADJ_MIN","ADJ_START"]])

,PLAYER,GAME,PROP,OUTCOME,MODEL_PROB,MODEL_PROB_ESPN,ESPN_MIN_LAST,ESPN_STARTER_PROXY,ADJ_MIN,ADJ_START
9,Oziyah Sellers,VILL at SJU,Rebounds,u3.5,0.904545,0.904545,NaN,0.0,0.0,0.000
2,Brenen Lorient,BYU at WVU,Points,o9.5,0.901282,0.901282,NaN,0.0,0.0,0.000
39,Brayden Burries,KU at ARIZ,Rebounds,u5.5,0.882273,0.882273,NaN,0.0,0.0,0.000
4,Xaivian Lee,ARK at FLA,Rebounds,u3.5,0.880103,0.880103,NaN,0.0,0.0,0.000
13,Brenen Lorient,BYU at WVU,Three Pointers Made,u0.5,0.879744,0.879744,NaN,0.0,0.0,0.000
21,Jamarques Lawrence,NEB at USC,Rebounds,u3.5,0.866410,0.866410,NaN,0.0,0.0,0.000
0,Barrington Hargress,COLO at HOU,Points,o9.5,0.831686,0.846686,18.0,1.0,0.0,0.015
17,Amari Allen,ALA at TENN,Points,o9.5,0.841282,0.841282,NaN,0.0,0.0,0.000
47,Zuby Ejiofor,VILL at SJU,Rebounds,u7.5,0.837273,0.837273,NaN,0.0,0.0,0.000
14,Rienk Mast,NEB at USC,Three Pointers Made,u1.5,0.835538,0.835538,NaN,0.0,0.0,0.000


In [ ]:
import pandas as pd, numpy as np, requests, re
from functools import lru_cache

ESPN_SUMMARY = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

@lru_cache(maxsize=4096)
def get_summary(event_id: str):
    return get_json(ESPN_SUMMARY, params={"event": str(event_id)})

def norm_name(s):
    s = str(s).upper().strip()
    s = re.sub(r"\b(JR|SR|II|III|IV)\b", "", s)
    s = re.sub(r"[^A-Z ]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# df_ok = rows we are keeping (eventId not null)
df_ok["eventId"] = df_ok["eventId"].astype(str)
df_ok.loc[df_ok["eventId"].isin(["nan","None","NaN",""]), "eventId"] = np.nan
df_ok = df_ok[df_ok["eventId"].notna()].copy()

# Build event -> team_id map
event_team_rows = []
for eid in sorted(df_ok["eventId"].unique()):
    j = get_summary(eid)
    comp = (j.get("header", {}).get("competitions", [{}])[0])
    for c in comp.get("competitors", []):
        team = c.get("team", {}) or {}
        event_team_rows.append({
            "eventId": str(eid),
            "homeAway": c.get("homeAway"),
            "team_id": str(team.get("id")),
            "team_name": team.get("displayName"),
            "team_abbr": team.get("abbreviation"),
        })

event_teams_df = pd.DataFrame(event_team_rows).drop_duplicates()
display(event_teams_df.head(10))
print("Unique events:", event_teams_df["eventId"].nunique(), "Unique teams:", event_teams_df["team_id"].nunique())

,eventId,homeAway,team_id,team_name,team_abbr
0,401808266,home,57,Florida Gators,FLA
1,401808266,away,8,Arkansas Razorbacks,ARK
2,401808268,home,96,Kentucky Wildcats,UK
3,401808268,away,238,Vanderbilt Commodores,VAN
4,401808271,home,2633,Tennessee Volunteers,TENN
5,401808271,away,333,Alabama Crimson Tide,ALA
6,401820770,home,228,Clemson Tigers,CLEM
7,401820770,away,97,Louisville Cardinals,LOU
8,401820771,home,150,Duke Blue Devils,DUKE
9,401820771,away,258,Virginia Cavaliers,UVA


Unique events: 13 Unique teams: 26


In [ ]:
ESPN_ROSTER = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/roster"

@lru_cache(maxsize=4096)
def get_roster(team_id: str):
    return get_json(ESPN_ROSTER.format(team_id=str(team_id)))

def roster_to_df(team_id: str):
    j = get_roster(team_id)
    athletes = []
    # roster structure can vary; handle common layouts
    for grp in j.get("athletes", []) or []:
        # sometimes already flat athlete objects
        if isinstance(grp, dict) and "id" in grp and "displayName" in grp:
            athletes.append(grp)
    if not athletes:
        # common: j["athletes"] is list of groups with ["items"]
        for grp in j.get("athletes", []) or []:
            for a in (grp.get("items", []) if isinstance(grp, dict) else []):
                athletes.append(a)

    rows = []
    for a in athletes:
        rows.append({
            "team_id": str(team_id),
            "athlete_id": str(a.get("id")),
            "athlete_name": a.get("displayName") or a.get("fullName") or "",
            "athlete_name_norm": norm_name(a.get("displayName") or a.get("fullName") or ""),
        })
    return pd.DataFrame(rows)

team_ids = sorted(event_teams_df["team_id"].dropna().unique().tolist())
roster_frames = []
for tid in team_ids:
    try:
        roster_frames.append(roster_to_df(tid))
    except Exception as e:
        print("Roster fail team_id", tid, e)

rosters_df = pd.concat(roster_frames, ignore_index=True) if roster_frames else pd.DataFrame()
print("Roster rows:", len(rosters_df))
display(rosters_df.head(10))

# Match each row's PLAYER to either team's roster for that event
from difflib import SequenceMatcher

def best_roster_match(player_name, event_id):
    teams = event_teams_df[event_teams_df["eventId"] == str(event_id)]["team_id"].tolist()
    pool = rosters_df[rosters_df["team_id"].isin(teams)]
    if pool.empty:
        return (np.nan, np.nan, np.nan)

    target = norm_name(player_name)

    # exact
    exact = pool[pool["athlete_name_norm"] == target]
    if len(exact):
        r = exact.iloc[0]
        return (r["athlete_id"], r["athlete_name"], 1.0)

    # fuzzy
    best_i, best_s = None, -1
    for i, r in pool.iterrows():
        s = SequenceMatcher(None, target, r["athlete_name_norm"]).ratio()
        if s > best_s:
            best_s = s
            best_i = i

    if best_s < 0.88:
        return (np.nan, np.nan, best_s)

    r = pool.loc[best_i]
    return (r["athlete_id"], r["athlete_name"], best_s)

df_ok[["ESPN_ATHLETE_ID","ESPN_ATHLETE_NAME","ESPN_NAME_SCORE"]] = df_ok.apply(
    lambda r: pd.Series(best_roster_match(r["PLAYER"], r["eventId"])),
    axis=1
)

print("Missing athleteId after roster-match:", df_ok["ESPN_ATHLETE_ID"].isna().sum())
display(df_ok[df_ok["ESPN_ATHLETE_ID"].isna()][["PLAYER","GAME","eventId"]].head(25))

Roster rows: 396


,team_id,athlete_id,athlete_name,athlete_name_norm
0,12,5186456,Dwayne Aristode,DWAYNE ARISTODE
1,12,4896359,Addison Arnold,ADDISON ARNOLD
2,12,5105555,Tobe Awaka,TOBE AWAKA
3,12,4432737,Jaden Bradley,JADEN BRADLEY
4,12,5082206,Brayden Burries,BRAYDEN BURRIES
5,12,5174956,Jackson Cook,JACKSON COOK
6,12,5108141,Anthony Dell'Orso,ANTHONY DELLORSO
7,12,5239487,Sven Djopmo,SVEN DJOPMO
8,12,5105542,Jackson Francois,JACKSON FRANCOIS
9,12,5311897,Sidi Gueye,SIDI GUEYE


Missing athleteId after roster-match: 0


,PLAYER,GAME,eventId


In [ ]:
ESPN_GAMELOG = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/athletes/{athlete_id}/gamelog"

@lru_cache(maxsize=20000)
def get_gamelog(athlete_id: str, season: int = 2026, seasontype: int = 2):
    # Try with season params; fallback to no params if ESPN ignores them
    try:
        return get_json(ESPN_GAMELOG.format(athlete_id=str(athlete_id)),
                        params={"season": season, "seasonType": seasontype})
    except:
        return get_json(ESPN_GAMELOG.format(athlete_id=str(athlete_id)))

def extract_minutes_from_gamelog(j):
    """
    ESPN gamelog formats vary. We'll search tables for a column named MIN.
    Return list of minutes as floats (most recent first where possible).
    """
    mins = []

    # common: j["events"] with "stats" arrays not always labeled; more common: j["categories"]/tables
    tables = []
    if "categories" in j:
        for cat in j.get("categories", []):
            for t in cat.get("events", []) or []:
                pass  # ignore
            for t in cat.get("tables", []) or []:
                tables.append(t)
    if "tables" in j:
        tables += j.get("tables", []) or []

    for t in tables:
        cols = t.get("columns", []) or []
        # locate MIN column
        min_idx = None
        for idx, c in enumerate(cols):
            lab = (c.get("label") if isinstance(c, dict) else c) or ""
            if str(lab).upper() == "MIN":
                min_idx = idx
                break
        if min_idx is None:
            continue

        for r in t.get("rows", []) or []:
            cells = r.get("cells", []) or r.get("entries", []) or []
            if min_idx < len(cells):
                v = cells[min_idx].get("value") if isinstance(cells[min_idx], dict) else cells[min_idx]
                try:
                    s = str(v)
                    if ":" in s:
                        m, sec = s.split(":", 1)
                        mins.append(float(m) + float(sec)/60.0)
                    else:
                        mins.append(float(s))
                except:
                    continue

    # Deduplicate weird repeats while preserving order
    out = []
    for m in mins:
        if m not in out:
            out.append(m)
    return out

# Build minutes features per athlete
athlete_ids = df_ok["ESPN_ATHLETE_ID"].dropna().astype(str).unique().tolist()

feat_rows = []
for aid in athlete_ids:
    try:
        j = get_gamelog(aid, season=2026, seasontype=2)
        mins = extract_minutes_from_gamelog(j)
        if len(mins) == 0:
            feat_rows.append({"ESPN_ATHLETE_ID": aid, "MIN_L5": np.nan, "MIN_L10": np.nan, "MIN_SEASON": np.nan, "MIN_SD_L10": np.nan, "GAMES_FOUND": 0})
            continue

        arr = np.array(mins, dtype=float)
        l5 = np.nanmean(arr[:5]) if len(arr) >= 1 else np.nan
        l10 = np.nanmean(arr[:10]) if len(arr) >= 1 else np.nan
        sd10 = np.nanstd(arr[:10]) if len(arr) >= 2 else np.nan
        seas = np.nanmean(arr) if len(arr) >= 1 else np.nan

        feat_rows.append({
            "ESPN_ATHLETE_ID": aid,
            "MIN_L5": l5,
            "MIN_L10": l10,
            "MIN_SEASON": seas,
            "MIN_SD_L10": sd10,
            "GAMES_FOUND": int(len(arr))
        })
    except Exception as e:
        feat_rows.append({"ESPN_ATHLETE_ID": aid, "MIN_L5": np.nan, "MIN_L10": np.nan, "MIN_SEASON": np.nan, "MIN_SD_L10": np.nan, "GAMES_FOUND": 0})
        print("Gamelog fail athlete", aid, e)

mins_feat_df = pd.DataFrame(feat_rows)
display(mins_feat_df.head(10))

df_ok = df_ok.merge(mins_feat_df, on="ESPN_ATHLETE_ID", how="left")

print("Rows missing MIN_L10:", df_ok["MIN_L10"].isna().sum())

Gamelog fail athlete 4712832 404 Client Error: Not Found for url: https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/athletes/4712832/gamelog
Gamelog fail athlete 5177301 404 Client Error: Not Found for url: https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/athletes/5177301/gamelog
Gamelog fail athlete 5106864 404 Client Error: Not Found for url: https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/athletes/5106864/gamelog
Gamelog fail athlete 5107169 404 Client Error: Not Found for url: https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/athletes/5107169/gamelog
Gamelog fail athlete 4703396 404 Client Error: Not Found for url: https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/athletes/4703396/gamelog
Gamelog fail athlete 4684276 404 Client Error: Not Found for url: https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-bask

,ESPN_ATHLETE_ID,MIN_L5,MIN_L10,MIN_SEASON,MIN_SD_L10,GAMES_FOUND
0,4712832,NaN,NaN,NaN,NaN,0
1,5177301,NaN,NaN,NaN,NaN,0
2,5106864,NaN,NaN,NaN,NaN,0
3,5107169,NaN,NaN,NaN,NaN,0
4,4703396,NaN,NaN,NaN,NaN,0
5,4684276,NaN,NaN,NaN,NaN,0
6,5182412,NaN,NaN,NaN,NaN,0
7,5105642,NaN,NaN,NaN,NaN,0
8,4791227,NaN,NaN,NaN,NaN,0
9,4592677,NaN,NaN,NaN,NaN,0


Rows missing MIN_L10: 79


In [ ]:
def role_adj(min_l10, sd_l10):
    # Minutes signal (role)
    if pd.isna(min_l10):
        base = 0.0
    elif min_l10 >= 30:
        base = +0.035
    elif min_l10 >= 26:
        base = +0.020
    elif min_l10 >= 20:
        base = +0.005
    elif min_l10 >= 14:
        base = -0.020
    else:
        base = -0.045

    # Stability penalty (volatile minutes)
    if pd.isna(sd_l10):
        stab = 0.0
    elif sd_l10 >= 8:
        stab = -0.020
    elif sd_l10 >= 5:
        stab = -0.010
    else:
        stab = 0.0

    return base + stab

df_ok["ADJ_ROLE"] = df_ok.apply(lambda r: role_adj(r["MIN_L10"], r["MIN_SD_L10"]), axis=1)

# Final ESPN-aware probability
df_ok["MODEL_PROB_ESPN"] = (df_ok["MODEL_PROB"] + df_ok["ADJ_ROLE"]).clip(0.01, 0.99)

top10 = df_ok.sort_values("MODEL_PROB_ESPN", ascending=False).head(10)

display(top10[[
    "PLAYER","GAME","PROP","OUTCOME",
    "MODEL_PROB","ADJ_ROLE","MODEL_PROB_ESPN",
    "MIN_L10","MIN_SD_L10","GAMES_FOUND","ESPN_NAME_SCORE"
]])

top10.to_csv("prop_engine_v1_top10_most_likely_ESPN_ROLE.csv", index=False)
df_ok.to_csv("prop_engine_v1_full_ESPN_role_enriched.csv", index=False)
print("Saved: prop_engine_v1_top10_most_likely_ESPN_ROLE.csv")
print("Saved: prop_engine_v1_full_ESPN_role_enriched.csv")

,PLAYER,GAME,PROP,OUTCOME,MODEL_PROB,ADJ_ROLE,MODEL_PROB_ESPN,MIN_L10,MIN_SD_L10,GAMES_FOUND,ESPN_NAME_SCORE
7,Oziyah Sellers,VILL at SJU,Rebounds,u3.5,0.904545,0.0,0.904545,NaN,NaN,0,1.0
2,Brenen Lorient,BYU at WVU,Points,o9.5,0.901282,0.0,0.901282,NaN,NaN,0,1.0
30,Brayden Burries,KU at ARIZ,Rebounds,u5.5,0.882273,0.0,0.882273,NaN,NaN,0,1.0
3,Xaivian Lee,ARK at FLA,Rebounds,u3.5,0.880103,0.0,0.880103,NaN,NaN,0,1.0
9,Brenen Lorient,BYU at WVU,Three Pointers Made,u0.5,0.879744,0.0,0.879744,NaN,NaN,0,1.0
16,Jamarques Lawrence,NEB at USC,Rebounds,u3.5,0.866410,0.0,0.866410,NaN,NaN,0,1.0
12,Amari Allen,ALA at TENN,Points,o9.5,0.841282,0.0,0.841282,NaN,NaN,0,1.0
36,Zuby Ejiofor,VILL at SJU,Rebounds,u7.5,0.837273,0.0,0.837273,NaN,NaN,0,1.0
10,Rienk Mast,NEB at USC,Three Pointers Made,u1.5,0.835538,0.0,0.835538,NaN,NaN,0,1.0
0,Barrington Hargress,COLO at HOU,Points,o9.5,0.831686,0.0,0.831686,NaN,NaN,0,1.0


Saved: prop_engine_v1_top10_most_likely_ESPN_ROLE.csv
Saved: prop_engine_v1_full_ESPN_role_enriched.csv


In [ ]:
# pick a sample athlete id from your merged data
sample_id = df_ok["ESPN_ATHLETE_ID"].dropna().astype(str).iloc[0]
print("Sample athlete id:", sample_id)

j = get_gamelog(sample_id, season=2026, seasontype=2)

# Print top-level keys to see schema
print("Top keys:", list(j.keys())[:50])

# Show a compact preview of the JSON (first ~2000 chars)
import json
preview = json.dumps(j)[:2000]
print(preview)

Sample athlete id: 4712832


HTTPError: 404 Client Error: Not Found for url: https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/athletes/4712832/gamelog

In [ ]:
import requests

athlete_id = "4712832"

CANDIDATE_URLS = [
    # alternate host (very common)
    f"https://site.web.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/athletes/{athlete_id}/gamelog",
    f"https://site.web.api.espn.com/apis/site/v2/sports/basketball/ncaab/athletes/{athlete_id}/gamelog",

    # original host but alternate league key
    f"https://site.api.espn.com/apis/site/v2/sports/basketball/ncaab/athletes/{athlete_id}/gamelog",

    # core API variants (sometimes used)
    f"https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/athletes/{athlete_id}/gamelog",
    f"https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/athletes/{athlete_id}/gamelogs",
]

params_list = [
    {"season": 2026, "seasonType": 2},
    {"season": 2025, "seasonType": 2},
    None
]

for url in CANDIDATE_URLS:
    for params in params_list:
        try:
            r = requests.get(url, params=params, timeout=20)
            print(r.status_code, url, "params=", params)
            if r.status_code == 200:
                print("SUCCESS URL:", r.url)
                print("Top keys:", list(r.json().keys())[:30])
                raise SystemExit
        except Exception as e:
            print("ERR", url, "params=", params, "->", e)

404 https://site.web.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/athletes/4712832/gamelog params= {'season': 2026, 'seasonType': 2}
404 https://site.web.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/athletes/4712832/gamelog params= {'season': 2025, 'seasonType': 2}
404 https://site.web.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/athletes/4712832/gamelog params= None
404 https://site.web.api.espn.com/apis/site/v2/sports/basketball/ncaab/athletes/4712832/gamelog params= {'season': 2026, 'seasonType': 2}
404 https://site.web.api.espn.com/apis/site/v2/sports/basketball/ncaab/athletes/4712832/gamelog params= {'season': 2025, 'seasonType': 2}
404 https://site.web.api.espn.com/apis/site/v2/sports/basketball/ncaab/athletes/4712832/gamelog params= None
404 https://site.api.espn.com/apis/site/v2/sports/basketball/ncaab/athletes/4712832/gamelog params= {'season': 2026, 'seasonType': 2}
404 https://site.api.espn.com/apis/site/v2

In [ ]:
import requests
from functools import lru_cache

# Replace/keep candidates; the first working one will be used.
GAMELOG_BASES = [
    "https://site.web.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/athletes/{athlete_id}/gamelog",
    "https://site.web.api.espn.com/apis/site/v2/sports/basketball/ncaab/athletes/{athlete_id}/gamelog",
    "https://site.api.espn.com/apis/site/v2/sports/basketball/ncaab/athletes/{athlete_id}/gamelog",
    "https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/athletes/{athlete_id}/gamelog",
    "https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/athletes/{athlete_id}/gamelogs",
]

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

@lru_cache(maxsize=20000)
def get_gamelog(athlete_id: str, season: int = 2026, seasontype: int = 2):
    last_err = None
    for base in GAMELOG_BASES:
        url = base.format(athlete_id=str(athlete_id))
        for params in ({"season": season, "seasonType": seasontype},
                       {"season": season-1, "seasonType": seasontype},
                       None):
            try:
                return get_json(url, params=params)
            except Exception as e:
                last_err = e
                continue
    raise last_err

In [ ]:
import pandas as pd, numpy as np, requests, re
from functools import lru_cache
from datetime import datetime

ESPN_SUMMARY = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary"
ESPN_SCHEDULE = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/schedule"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

@lru_cache(maxsize=4096)
def get_summary(event_id: str):
    return get_json(ESPN_SUMMARY, params={"event": str(event_id)})

@lru_cache(maxsize=4096)
def get_schedule(team_id: str):
    return get_json(ESPN_SCHEDULE.format(team_id=str(team_id)))

def norm_name(s):
    s = str(s).upper().strip()
    s = re.sub(r"\b(JR|SR|II|III|IV)\b", "", s)
    s = re.sub(r"[^A-Z ]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def parse_minutes(mmss):
    if mmss is None or (isinstance(mmss,float) and np.isnan(mmss)):
        return np.nan
    s = str(mmss).strip()
    if ":" in s:
        m, sec = s.split(":", 1)
        try:
            return float(m) + float(sec)/60.0
        except:
            return np.nan
    try:
        return float(s)
    except:
        return np.nan

# event_teams_df should already exist from earlier (eventId -> team_id mapping)
team_ids = sorted(event_teams_df["team_id"].dropna().astype(str).unique().tolist())

def completed_event_ids_from_schedule(team_id, max_games=15):
    j = get_schedule(team_id)
    evs = []

    # schedule structure varies; try common containers
    blocks = []
    if isinstance(j.get("events"), list):
        blocks = j["events"]
    elif isinstance(j.get("schedule"), list):
        blocks = j["schedule"]

    for e in blocks:
        # ESPN often stores competitions under "competitions" or event fields
        eid = e.get("id") or e.get("uid") or None
        # try nested
        if eid is None and isinstance(e.get("competitions"), list) and len(e["competitions"]):
            eid = e["competitions"][0].get("id")

        status = None
        if isinstance(e.get("status"), dict):
            status = (e["status"].get("type") or {}).get("name") or (e["status"].get("type") or {}).get("state")
        # sometimes: e["competitions"][0]["status"]["type"]["name"]
        if status is None and isinstance(e.get("competitions"), list) and len(e["competitions"]):
            st = (e["competitions"][0].get("status") or {}).get("type") or {}
            status = st.get("name") or st.get("state")

        # completed names vary: "STATUS_FINAL", "post", "FINAL"
        status_up = str(status).upper() if status else ""
        if "FINAL" not in status_up and "POST" not in status_up:
            continue

        # event date
        dt = e.get("date")
        if dt is None and isinstance(e.get("competitions"), list) and len(e["competitions"]):
            dt = e["competitions"][0].get("date")

        evs.append({"eventId": str(eid), "date": dt})

    # sort most recent first
    def dt_key(x):
        try:
            return pd.to_datetime(x["date"], utc=True)
        except:
            return pd.Timestamp.min.tz_localize("UTC")

    evs = sorted(evs, key=dt_key, reverse=True)
    evs = [x for x in evs if x["eventId"] not in ["None","nan",""]]

    return [x["eventId"] for x in evs[:max_games]]

team_recent_events = {}
for tid in team_ids:
    try:
        team_recent_events[tid] = completed_event_ids_from_schedule(tid, max_games=15)
    except Exception as e:
        team_recent_events[tid] = []
        print("Schedule fail team", tid, e)

print("Example team recent events counts:", {k: len(v) for k,v in list(team_recent_events.items())[:5]})

Example team recent events counts: {'12': 15, '150': 15, '153': 15, '158': 15, '222': 15}


In [ ]:
def extract_minutes_for_player_in_event(event_id: str, player_name_norm: str):
    """
    Returns minutes (float) for this player from the event boxscore, or NaN if not found.
    """
    j = get_summary(event_id)
    box = j.get("boxscore", {})
    players = box.get("players", [])
    if not players:
        return np.nan

    for team_block in players:
        for stat_group in team_block.get("statistics", []):
            labels = stat_group.get("labels", [])
            if "MIN" not in labels:
                continue
            min_idx = labels.index("MIN")

            for a in stat_group.get("athletes", []):
                ath = a.get("athlete", {}) or {}
                nm = norm_name(ath.get("displayName",""))
                if nm != player_name_norm:
                    continue
                stats = a.get("stats", [])
                mm = stats[min_idx] if min_idx < len(stats) else None
                return parse_minutes(mm)

    return np.nan

# Map each row to its two team_ids (from the eventId mapping)
event_to_teams = event_teams_df.groupby("eventId")["team_id"].apply(list).to_dict()

# Build minutes history per unique (eventId, player)
unique_keys = df_ok[["eventId","PLAYER","ESPN_ATHLETE_NAME"]].drop_duplicates().copy()
unique_keys["player_norm"] = unique_keys["PLAYER"].apply(norm_name)

hist_rows = []
N_BACK = 10  # last 10 completed games

for _, r in unique_keys.iterrows():
    eid = str(r["eventId"])
    pnorm = r["player_norm"]

    teams = event_to_teams.get(eid, [])
    if not teams:
        continue

    # pull last N completed games for BOTH teams (player belongs to one of them)
    candidate_events = []
    for tid in teams:
        candidate_events += team_recent_events.get(str(tid), [])
    # dedupe while preserving order
    seen = set()
    candidate_events = [x for x in candidate_events if not (x in seen or seen.add(x))]
    candidate_events = candidate_events[:N_BACK]

    mins_list = []
    for peid in candidate_events:
        try:
            m = extract_minutes_for_player_in_event(peid, pnorm)
            if pd.notna(m):
                mins_list.append(m)
        except:
            continue

    # store
    if len(mins_list) == 0:
        hist_rows.append({"eventId": eid, "player_norm": pnorm, "MIN_L5": np.nan, "MIN_L10": np.nan, "MIN_SD_L10": np.nan, "GAMES_FOUND": 0})
    else:
        arr = np.array(mins_list, dtype=float)
        hist_rows.append({
            "eventId": eid,
            "player_norm": pnorm,
            "MIN_L5": float(np.nanmean(arr[:5])),
            "MIN_L10": float(np.nanmean(arr[:10])),
            "MIN_SD_L10": float(np.nanstd(arr[:10])) if len(arr[:10]) >= 2 else np.nan,
            "GAMES_FOUND": int(len(arr))
        })

mins_hist_df = pd.DataFrame(hist_rows)
print("Minutes history rows:", len(mins_hist_df))
display(mins_hist_df.head(15))

# Merge back to df_ok
df_ok["player_norm"] = df_ok["PLAYER"].apply(norm_name)
df_ok = df_ok.merge(mins_hist_df, on=["eventId","player_norm"], how="left")

print("Rows with GAMES_FOUND>0:", int((df_ok["GAMES_FOUND"]>0).sum()), "of", len(df_ok))

Minutes history rows: 47


,eventId,player_norm,MIN_L5,MIN_L10,MIN_SD_L10,GAMES_FOUND
0,401827711,BARRINGTON HARGRESS,NaN,NaN,NaN,0
1,401827707,MELVIN COUNCIL,40.0,40.0,NaN,1
2,401827714,BRENEN LORIENT,31.8,30.2,3.370460,10
3,401808266,XAIVIAN LEE,29.6,25.7,5.797413,10
4,401829265,GRAHAM IKE,37.0,37.0,NaN,1
5,401808271,ADEN HOLLOWAY,NaN,NaN,NaN,0
6,401822962,TYLER PERKINS,NaN,NaN,NaN,0
7,401822962,OZIYAH SELLERS,28.6,27.9,6.040695,10
8,401808268,DEVIN MCGLOCKTON,32.0,32.0,NaN,1
9,401825548,RIENK MAST,NaN,NaN,NaN,0


KeyError: 'GAMES_FOUND'

In [ ]:
# If you already merged once and it collided, re-merge cleanly:
# 1) drop any existing minutes history cols
for c in ["MIN_L5","MIN_L10","MIN_SD_L10","GAMES_FOUND"]:
    if c in df_ok.columns:
        df_ok.drop(columns=[c], inplace=True)

# 2) merge with explicit suffixes (safe)
df_ok = df_ok.merge(mins_hist_df, on=["eventId","player_norm"], how="left", suffixes=("", "_mh"))

# 3) if collision columns exist from earlier runs, consolidate
# (handles cases where you already have MIN_L10_x/MIN_L10_y etc.)
def coalesce_cols(df, base):
    candidates = [base, f"{base}_mh", f"{base}_x", f"{base}_y"]
    existing = [c for c in candidates if c in df.columns]
    if not existing:
        df[base] = np.nan
        return
    df[base] = df[existing].bfill(axis=1).iloc[:,0]
    # drop extras
    for c in existing:
        if c != base and c in df.columns:
            df.drop(columns=[c], inplace=True)

for base in ["MIN_L5","MIN_L10","MIN_SD_L10","GAMES_FOUND"]:
    coalesce_cols(df_ok, base)

print("Columns now present:", [c for c in ["MIN_L5","MIN_L10","MIN_SD_L10","GAMES_FOUND"] if c in df_ok.columns])
print("Rows with GAMES_FOUND>0:", int((df_ok["GAMES_FOUND"].fillna(0)>0).sum()), "of", len(df_ok))
display(df_ok[["PLAYER","GAME","MIN_L10","MIN_SD_L10","GAMES_FOUND"]].head(15))

Columns now present: ['MIN_L5', 'MIN_L10', 'MIN_SD_L10', 'GAMES_FOUND']
Rows with GAMES_FOUND>0: 45 of 79


,PLAYER,GAME,MIN_L10,MIN_SD_L10,GAMES_FOUND
0,Barrington Hargress,COLO at HOU,NaN,NaN,0
1,Melvin Council Jr.,KU at ARIZ,40.0,NaN,1
2,Brenen Lorient,BYU at WVU,30.2,3.370460,10
3,Xaivian Lee,ARK at FLA,25.7,5.797413,10
4,Graham Ike,GONZ at SMC,37.0,NaN,1
5,Aden Holloway,ALA at TENN,NaN,NaN,0
6,Tyler Perkins,VILL at SJU,NaN,NaN,0
7,Oziyah Sellers,VILL at SJU,27.9,6.040695,10
8,Devin McGlockton,VAN at UK,32.0,NaN,1
9,Brenen Lorient,BYU at WVU,30.2,3.370460,10


In [ ]:
def role_adj(min_l10, sd_l10):
    if pd.isna(min_l10):
        base = 0.0
    elif min_l10 >= 30:
        base = +0.035
    elif min_l10 >= 26:
        base = +0.020
    elif min_l10 >= 20:
        base = +0.005
    elif min_l10 >= 14:
        base = -0.020
    else:
        base = -0.045

    if pd.isna(sd_l10):
        stab = 0.0
    elif sd_l10 >= 8:
        stab = -0.020
    elif sd_l10 >= 5:
        stab = -0.010
    else:
        stab = 0.0

    return base + stab

df_ok["ADJ_ROLE"] = df_ok.apply(lambda r: role_adj(r["MIN_L10"], r["MIN_SD_L10"]), axis=1)
df_ok["MODEL_PROB_ESPN"] = (df_ok["MODEL_PROB"] + df_ok["ADJ_ROLE"]).clip(0.01, 0.99)

top10 = df_ok.sort_values("MODEL_PROB_ESPN", ascending=False).head(10)
display(top10[["PLAYER","GAME","PROP","OUTCOME","MODEL_PROB","ADJ_ROLE","MODEL_PROB_ESPN","MIN_L10","MIN_SD_L10","GAMES_FOUND"]])

top10.to_csv("prop_engine_v1_top10_most_likely_ESPN_ROLE.csv", index=False)
df_ok.to_csv("prop_engine_v1_full_ESPN_role_enriched.csv", index=False)
print("Saved: prop_engine_v1_top10_most_likely_ESPN_ROLE.csv")
print("Saved: prop_engine_v1_full_ESPN_role_enriched.csv")

,PLAYER,GAME,PROP,OUTCOME,MODEL_PROB,ADJ_ROLE,MODEL_PROB_ESPN,MIN_L10,MIN_SD_L10,GAMES_FOUND
2,Brenen Lorient,BYU at WVU,Points,o9.5,0.901282,0.035,0.936282,30.2,3.370460,10
30,Brayden Burries,KU at ARIZ,Rebounds,u5.5,0.882273,0.035,0.917273,32.5,3.853570,10
9,Brenen Lorient,BYU at WVU,Three Pointers Made,u0.5,0.879744,0.035,0.914744,30.2,3.370460,10
7,Oziyah Sellers,VILL at SJU,Rebounds,u3.5,0.904545,0.010,0.914545,27.9,6.040695,10
3,Xaivian Lee,ARK at FLA,Rebounds,u3.5,0.880103,-0.005,0.875103,25.7,5.797413,10
36,Zuby Ejiofor,VILL at SJU,Rebounds,u7.5,0.837273,0.035,0.872273,30.9,4.784349,10
16,Jamarques Lawrence,NEB at USC,Rebounds,u3.5,0.866410,0.000,0.866410,NaN,NaN,0
12,Amari Allen,ALA at TENN,Points,o9.5,0.841282,0.000,0.841282,NaN,NaN,0
33,Brenen Lorient,BYU at WVU,Points,u12.5,0.806205,0.035,0.841205,30.2,3.370460,10
10,Rienk Mast,NEB at USC,Three Pointers Made,u1.5,0.835538,0.000,0.835538,NaN,NaN,0


Saved: prop_engine_v1_top10_most_likely_ESPN_ROLE.csv
Saved: prop_engine_v1_full_ESPN_role_enriched.csv


In [ ]:
# 1) Require role evidence: at least 3 games found (you can set to 5 later)
df_rank = df_ok[df_ok["GAMES_FOUND"].fillna(0) >= 3].copy()

# 2) Deduplicate players: keep the single best prop per player (highest MODEL_PROB_ESPN)
df_rank = df_rank.sort_values("MODEL_PROB_ESPN", ascending=False)
df_rank = df_rank.drop_duplicates(subset=["PLAYER"], keep="first")

top10_verified = df_rank.head(10)

display(top10_verified[[
    "PLAYER","GAME","PROP","OUTCOME",
    "MODEL_PROB","ADJ_ROLE","MODEL_PROB_ESPN",
    "MIN_L10","MIN_SD_L10","GAMES_FOUND"
]])

top10_verified.to_csv("prop_engine_v1_top10_most_likely_ESPN_ROLE_verified.csv", index=False)
print("Saved: prop_engine_v1_top10_most_likely_ESPN_ROLE_verified.csv")

,PLAYER,GAME,PROP,OUTCOME,MODEL_PROB,ADJ_ROLE,MODEL_PROB_ESPN,MIN_L10,MIN_SD_L10,GAMES_FOUND
2,Brenen Lorient,BYU at WVU,Points,o9.5,0.901282,0.035,0.936282,30.2,3.370460,10
30,Brayden Burries,KU at ARIZ,Rebounds,u5.5,0.882273,0.035,0.917273,32.5,3.853570,10
7,Oziyah Sellers,VILL at SJU,Rebounds,u3.5,0.904545,0.010,0.914545,27.9,6.040695,10
3,Xaivian Lee,ARK at FLA,Rebounds,u3.5,0.880103,-0.005,0.875103,25.7,5.797413,10
36,Zuby Ejiofor,VILL at SJU,Rebounds,u7.5,0.837273,0.035,0.872273,30.9,4.784349,10
26,Ja'Kobi Gillespie,ALA at TENN,Points,u20.5,0.790591,0.035,0.825591,36.0,2.932576,10
47,Kingston Flemings,COLO at HOU,Points,u19.5,0.776667,0.035,0.811667,33.2,2.959730,10
54,Alijah Arenas,NEB at USC,Three Pointers Made,u1.5,0.798205,0.010,0.808205,27.4,7.472617,10
52,Boogie Fland,ARK at FLA,Rebounds,u3.5,0.759744,0.035,0.794744,30.0,4.098780,10
43,Jaden Bradley,KU at ARIZ,Assists,u4.5,0.749500,0.035,0.784500,33.6,4.800000,10


Saved: prop_engine_v1_top10_most_likely_ESPN_ROLE_verified.csv


In [ ]:
def bench_risk(min_l10, games_found):
    if games_found is None or pd.isna(games_found) or games_found == 0:
        return "UNKNOWN"
    if pd.isna(min_l10):
        return "UNKNOWN"
    if min_l10 < 18:
        return "HIGH"
    if min_l10 < 22:
        return "MED"
    return "LOW"

df_ok["BENCH_RISK"] = df_ok.apply(lambda r: bench_risk(r["MIN_L10"], r["GAMES_FOUND"]), axis=1)

In [ ]:
top10_verified = df_rank.head(10)
print(top10_verified[[
    "PLAYER","GAME","PROP","OUTCOME",
    "MODEL_PROB","ADJ_ROLE","MODEL_PROB_ESPN",
    "MIN_L10","MIN_SD_L10","GAMES_FOUND"
]].to_string(index=False))

top10_verified.to_csv("prop_engine_v1_top10_most_likely_ESPN_ROLE_verified.csv", index=False)
print("\nSaved: prop_engine_v1_top10_most_likely_ESPN_ROLE_verified.csv")

           PLAYER        GAME                PROP OUTCOME  MODEL_PROB  ADJ_ROLE  MODEL_PROB_ESPN  MIN_L10  MIN_SD_L10  GAMES_FOUND
   Brenen Lorient  BYU at WVU              Points    o9.5    0.901282     0.035         0.936282     30.2    3.370460           10
  Brayden Burries  KU at ARIZ            Rebounds    u5.5    0.882273     0.035         0.917273     32.5    3.853570           10
   Oziyah Sellers VILL at SJU            Rebounds    u3.5    0.904545     0.010         0.914545     27.9    6.040695           10
      Xaivian Lee  ARK at FLA            Rebounds    u3.5    0.880103    -0.005         0.875103     25.7    5.797413           10
     Zuby Ejiofor VILL at SJU            Rebounds    u7.5    0.837273     0.035         0.872273     30.9    4.784349           10
Ja'Kobi Gillespie ALA at TENN              Points   u20.5    0.790591     0.035         0.825591     36.0    2.932576           10
Kingston Flemings COLO at HOU              Points   u19.5    0.776667     0.035    

In [ ]:
# Build per-player minutes list already exists implicitly in mins_list step,
# but we only stored aggregates. We'll approximate DNP with volatility + low mins.
# Better: re-store mins_count_found vs N_BACK (=10).

# We saved GAMES_FOUND as count of games where player appeared with minutes in those 10.
df_ok["DNP_RATE_L10"] = (10 - df_ok["GAMES_FOUND"].fillna(0)).clip(0,10) / 10.0

# Minutes trend proxy: use MIN_L5 and MIN_L10 (already computed)
df_ok["MIN_TREND_L5_L10"] = df_ok["MIN_L5"] - df_ok["MIN_L10"]

def injury_role_penalty(dnp_rate, min_trend):
    pen = 0.0
    if pd.notna(dnp_rate) and dnp_rate >= 0.3:
        pen -= 0.02
    if pd.notna(dnp_rate) and dnp_rate >= 0.5:
        pen -= 0.04
    if pd.notna(min_trend) and min_trend <= -4:
        pen -= 0.02
    if pd.notna(min_trend) and min_trend <= -7:
        pen -= 0.04
    return pen

df_ok["ADJ_INJ_PROXY"] = df_ok.apply(lambda r: injury_role_penalty(r["DNP_RATE_L10"], r["MIN_TREND_L5_L10"]), axis=1)

# Updated model prob including injury proxy
df_ok["MODEL_PROB_ESPN_V1"] = (df_ok["MODEL_PROB_ESPN"] + df_ok["ADJ_INJ_PROXY"]).clip(0.01, 0.99)

# Rebuild verified + deduped Top 10
df_rank = df_ok[df_ok["GAMES_FOUND"].fillna(0) >= 3].copy()
df_rank = df_rank.sort_values("MODEL_PROB_ESPN_V1", ascending=False)
df_rank = df_rank.drop_duplicates(subset=["PLAYER"], keep="first")

top10_v1 = df_rank.head(10)

print(top10_v1[[
    "PLAYER","GAME","PROP","OUTCOME",
    "MODEL_PROB_ESPN","ADJ_INJ_PROXY","MODEL_PROB_ESPN_V1",
    "MIN_L10","MIN_L5","MIN_TREND_L5_L10","DNP_RATE_L10"
]].to_string(index=False))

top10_v1.to_csv("prop_engine_v1_top10_most_likely_ESPN_ROLE_INJPROXY.csv", index=False)
print("\nSaved: prop_engine_v1_top10_most_likely_ESPN_ROLE_INJPROXY.csv")

           PLAYER        GAME                PROP OUTCOME  MODEL_PROB_ESPN  ADJ_INJ_PROXY  MODEL_PROB_ESPN_V1  MIN_L10  MIN_L5  MIN_TREND_L5_L10  DNP_RATE_L10
   Brenen Lorient  BYU at WVU              Points    o9.5         0.936282            0.0            0.936282     30.2    31.8               1.6           0.0
  Brayden Burries  KU at ARIZ            Rebounds    u5.5         0.917273            0.0            0.917273     32.5    34.4               1.9           0.0
   Oziyah Sellers VILL at SJU            Rebounds    u3.5         0.914545            0.0            0.914545     27.9    28.6               0.7           0.0
      Xaivian Lee  ARK at FLA            Rebounds    u3.5         0.875103            0.0            0.875103     25.7    29.6               3.9           0.0
     Zuby Ejiofor VILL at SJU            Rebounds    u7.5         0.872273            0.0            0.872273     30.9    30.4              -0.5           0.0
Ja'Kobi Gillespie ALA at TENN              Poi

In [ ]:
import numpy as np
import pandas as pd

# Ensure AMERICAN_ODDS exists
if "AMERICAN_ODDS" not in df_ok.columns:
    if "ODDS" in df_ok.columns:
        df_ok["AMERICAN_ODDS"] = pd.to_numeric(df_ok["ODDS"], errors="coerce")
    else:
        df_ok["AMERICAN_ODDS"] = np.nan

def american_to_prob(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    if o < 0:
        return (-o) / ((-o) + 100.0)
    return 100.0 / (o + 100.0)

df_ok["BOOK_PROB"] = df_ok["AMERICAN_ODDS"].apply(american_to_prob)

# Edge vs book (informational, not ranking)
df_ok["EDGE_VS_BOOK"] = df_ok["MODEL_PROB_ESPN_V1"] - df_ok["BOOK_PROB"]

# Build Top 10 (already computed) but re-derive to guarantee current columns
df_rank = df_ok[df_ok["GAMES_FOUND"].fillna(0) >= 3].copy()
df_rank = df_rank.sort_values("MODEL_PROB_ESPN_V1", ascending=False)
df_rank = df_rank.drop_duplicates(subset=["PLAYER"], keep="first")

top10_hit_sheet = df_rank.head(10).copy()
top10_hit_sheet.insert(0, "RANK", range(1, len(top10_hit_sheet)+1))

# Clean column order for the hit sheet
cols = [
    "RANK",
    "PLAYER","GAME","PROP","OUTCOME",
    "AMERICAN_ODDS","BOOK_PROB",
    "MODEL_PROB_ESPN_V1","EDGE_VS_BOOK",
    "MIN_L10","MIN_L5","MIN_TREND_L5_L10","MIN_SD_L10","DNP_RATE_L10",
    "MODEL_PROB","ADJ_DVP","ADJ_ROLE","ADJ_INJ_PROXY"
]
cols = [c for c in cols if c in top10_hit_sheet.columns]

print(top10_hit_sheet[cols].to_string(index=False))

top10_hit_sheet[cols].to_csv("hit_sheet_top10.csv", index=False)
print("\nSaved: hit_sheet_top10.csv")

 RANK            PLAYER        GAME                PROP OUTCOME  AMERICAN_ODDS  BOOK_PROB  MODEL_PROB_ESPN_V1  EDGE_VS_BOOK  MIN_L10  MIN_L5  MIN_TREND_L5_L10  MIN_SD_L10  DNP_RATE_L10  MODEL_PROB  ADJ_DVP  ADJ_ROLE  ADJ_INJ_PROXY
    1    Brenen Lorient  BYU at WVU              Points    o9.5            NaN        NaN            0.936282           NaN     30.2    31.8               1.6    3.370460           0.0    0.901282     0.07     0.035            0.0
    2   Brayden Burries  KU at ARIZ            Rebounds    u5.5            NaN        NaN            0.917273           NaN     32.5    34.4               1.9    3.853570           0.0    0.882273     0.07     0.035            0.0
    3    Oziyah Sellers VILL at SJU            Rebounds    u3.5            NaN        NaN            0.914545           NaN     27.9    28.6               0.7    6.040695           0.0    0.904545     0.07     0.010            0.0
    4       Xaivian Lee  ARK at FLA            Rebounds    u3.5            N

In [ ]:
import pandas as pd
import numpy as np
import re

CSV_PATH = "NCAA night v2.csv"  # <-- updated file

df = pd.read_csv(CSV_PATH)

# Drop unnamed index column if present
df = df.loc[:, ~df.columns.str.contains(r"^Unnamed", case=False)]
df.columns = [c.strip() for c in df.columns]

rename_map = {
    "IM PROB %": "IM_PROB_PCT",
    "HM/AW": "HM_AW",
    "DVP L5": "DVP_L5",
    "DVP L10": "DVP_L10",
    "DVP 2025": "DVP_2025",
    "DVP HM/AW": "DVP_HM_AW",
}
df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns}, inplace=True)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())
df.head(5)

FileNotFoundError: [Errno 2] No such file or directory: 'NCAA night v2.csv'

In [ ]:
import os
print(os.listdir("."))

['.config', 'sample_data']


In [ ]:
from google.colab import files
files.upload()

Saving NCAA night v2.csv to NCAA night v2.csv


{'NCAA night v2.csv': b',,,,,,,,,,,,,,,,\r\n,Emmanuel Innocenti,GONZ at SMC,Points,o7.5,"BET\n-130",-1314,56.5%,100%,60%,30%,23.1%,25%,60%,70%,63.3%,57.9%\r\n,Graham Ike,GONZ at SMC,Points,o19.5,"BET\n-166",-1826,62.4%,80%,90%,57.7%,66.7%,42.9%,0%,0%,10%,5.3%\r\n,Graham Ike,GONZ at SMC,Points,u21.5,"BET\n-115",-1173,53.5%,80%,40%,65.4%,58.3%,57.1%,100%,100%,93.3%,100%\r\n,Graham Ike,GONZ at SMC,Points + Rebounds + Assists,u31.5,"BET\n-110",-1122,52.4%,80%,40%,57.7%,50%,57.1%,100%,100%,100%,100%\r\n,Emmanuel Innocenti,GONZ at SMC,Points + Rebounds,o12.5,"BET\n-110",-1101,52.4%,80%,50%,26.7%,23.1%,25%,40%,60%,53.3%,47.4%\r\n,Emmanuel Innocenti,GONZ at SMC,Rebounds,u4.5,"BET\n-150",-1452,60.0%,60%,50%,70%,61.5%,75%,80%,70%,70%,73.7%\r\n,Adam Miller,GONZ at SMC,Points,o7.5,"BET\n-105",-1104,51.2%,60%,40%,53.3%,53.8%,50%,100%,70%,70%,68.4%\r\n,Graham Ike,GONZ at SMC,Assists,o2.5,"BET\n+115",1117,46.5%,60%,50%,57.7%,66.7%,0%,0%,10%,3.3%,0%\r\n,Graham Ike,GONZ at SMC,Three Pointers Made,o0.5,

In [ ]:
import pandas as pd
import numpy as np
import re

CSV_PATH = "NCAA night v2.csv"

df_raw = pd.read_csv(CSV_PATH, header=0)

# Drop unnamed/empty leading columns
df_raw = df_raw.loc[:, ~df_raw.columns.str.contains(r"^Unnamed", case=False)]
df_raw.columns = [c.strip() for c in df_raw.columns]

# If the first column is blank header, pandas may name it '' -> drop if mostly empty
if "" in df_raw.columns:
    if df_raw[""].isna().mean() > 0.9:
        df_raw = df_raw.drop(columns=[""])

# If PLAYER column is missing, the data is shifted right by 1 (common with leading comma)
# Detect and shift if needed
if "PLAYER" not in df_raw.columns:
    # Try to rename the first real column to PLAYER
    cols = df_raw.columns.tolist()
    # Often first col becomes something like 'Emmanuel Innocenti' column header issue; fallback:
    # We will force expected columns if count matches 16 or 17
    pass

# Remove rows that are clearly page/footer noise
noise_patterns = [
    r"FIRST PAGE", r"PREVIOUS PAGE", r"NEXT PAGE", r"LAST PAGE",
    r"^\s*LIVE\s*$", r"^\s*STARTER\s*$", r"BEST LINES", r"\d+\s*-\s*\d+\s+OF\s+\d+"
]

def is_noise_row(row):
    # If PLAYER missing and row contains navigation-like text anywhere
    joined = " ".join([str(x) for x in row.values if pd.notna(x)]).upper()
    return any(re.search(p, joined) for p in noise_patterns)

df = df_raw.copy()
df = df[~df.apply(is_noise_row, axis=1)].copy()

# Drop rows with no player/game/prop/outcome (the real rows always have these)
for col in ["PLAYER","GAME","PROP","OUTCOME"]:
    if col not in df.columns:
        raise ValueError(f"Missing column {col}. Columns found: {df.columns.tolist()}")

df = df.dropna(subset=["PLAYER","GAME","PROP","OUTCOME"]).copy()

# Strip whitespace
for c in ["PLAYER","GAME","PROP","OUTCOME"]:
    df[c] = df[c].astype(str).str.strip()

print("Loaded clean rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head(10))

ValueError: Missing column PLAYER. Columns found: []

In [ ]:
import pandas as pd
import numpy as np
import re

CSV_PATH = "NCAA night v2.csv"

# Load with no header (file starts with commas / blank header)
raw = pd.read_csv(CSV_PATH, header=None, dtype=str)

# Drop rows that are completely empty (all NaN)
raw = raw.dropna(how="all").copy()

# Drop rows that are only commas / blanks (all empty strings after strip)
def row_is_blank(r):
    vals = ["" if pd.isna(x) else str(x).strip() for x in r.values]
    return all(v == "" for v in vals)

raw = raw[~raw.apply(row_is_blank, axis=1)].copy()

print("Raw shape:", raw.shape)
display(raw.head(8))

# Your file rows are: [blank_leading, PLAYER, GAME, PROP, OUTCOME, ODDS, AVG, IM_PROB, L5, L10, 2025, HM/AW, H2H, DVP_L5, DVP_L10, DVP_2025, DVP_HM/AW]
# That is 17 columns after the leading blank.
# We'll keep the first 17 columns and name them accordingly.
EXPECTED_COLS = [
    "_LEAD",
    "PLAYER","GAME","PROP","OUTCOME","ODDS","AVG","IM_PROB_PCT",
    "L5","L10","2025","HM_AW","H2H","DVP_L5","DVP_L10","DVP_2025","DVP_HM_AW"
]

# If the file has more cols, truncate; if fewer, pad with NaN
n = raw.shape[1]
need = len(EXPECTED_COLS)

if n < need:
    for _ in range(need - n):
        raw[n] = np.nan
    n = raw.shape[1]

df = raw.iloc[:, :need].copy()
df.columns = EXPECTED_COLS

# Drop leading blank column
df = df.drop(columns=["_LEAD"])

# Remove obvious footer / navigation noise rows
noise_patterns = [
    r"FIRST PAGE", r"PREVIOUS PAGE", r"NEXT PAGE", r"LAST PAGE",
    r"^\s*LIVE\s*$", r"^\s*STARTER\s*$", r"BEST LINES", r"\d+\s*-\s*\d+\s+OF\s+\d+"
]

def is_noise_row(r):
    joined = " ".join([str(x) for x in r.values if pd.notna(x)]).upper()
    return any(re.search(p, joined) for p in noise_patterns)

df = df[~df.apply(is_noise_row, axis=1)].copy()

# Strip whitespace
for c in ["PLAYER","GAME","PROP","OUTCOME"]:
    df[c] = df[c].astype(str).str.strip()

# Drop rows with missing core fields
df = df.replace({"": np.nan}).dropna(subset=["PLAYER","GAME","PROP","OUTCOME"]).copy()

print("Clean rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head(10))

Raw shape: (101, 17)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
1,NaN,Emmanuel Innocenti,GONZ at SMC,Points,o7.5,BET\n-130,-1314,56.5%,100%,60%,30%,23.1%,25%,60%,70%,63.3%,57.9%
2,NaN,Graham Ike,GONZ at SMC,Points,o19.5,BET\n-166,-1826,62.4%,80%,90%,57.7%,66.7%,42.9%,0%,0%,10%,5.3%
3,NaN,Graham Ike,GONZ at SMC,Points,u21.5,BET\n-115,-1173,53.5%,80%,40%,65.4%,58.3%,57.1%,100%,100%,93.3%,100%
4,NaN,Graham Ike,GONZ at SMC,Points + Rebounds + Assists,u31.5,BET\n-110,-1122,52.4%,80%,40%,57.7%,50%,57.1%,100%,100%,100%,100%
5,NaN,Emmanuel Innocenti,GONZ at SMC,Points + Rebounds,o12.5,BET\n-110,-1101,52.4%,80%,50%,26.7%,23.1%,25%,40%,60%,53.3%,47.4%
6,NaN,Emmanuel Innocenti,GONZ at SMC,Rebounds,u4.5,BET\n-150,-1452,60.0%,60%,50%,70%,61.5%,75%,80%,70%,70%,73.7%
7,NaN,Adam Miller,GONZ at SMC,Points,o7.5,BET\n-105,-1104,51.2%,60%,40%,53.3%,53.8%,50%,100%,70%,70%,68.4%
8,NaN,Graham Ike,GONZ at SMC,Assists,o2.5,BET\n+115,1117,46.5%,60%,50%,57.7%,66.7%,0%,0%,10%,3.3%,0%


Clean rows: 96
Columns: ['PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS', 'AVG', 'IM_PROB_PCT', 'L5', 'L10', '2025', 'HM_AW', 'H2H', 'DVP_L5', 'DVP_L10', 'DVP_2025', 'DVP_HM_AW']


,PLAYER,GAME,PROP,OUTCOME,ODDS,AVG,IM_PROB_PCT,L5,L10,2025,HM_AW,H2H,DVP_L5,DVP_L10,DVP_2025,DVP_HM_AW
1,Emmanuel Innocenti,GONZ at SMC,Points,o7.5,BET\n-130,-1314,56.5%,100%,60%,30%,23.1%,25%,60%,70%,63.3%,57.9%
2,Graham Ike,GONZ at SMC,Points,o19.5,BET\n-166,-1826,62.4%,80%,90%,57.7%,66.7%,42.9%,0%,0%,10%,5.3%
3,Graham Ike,GONZ at SMC,Points,u21.5,BET\n-115,-1173,53.5%,80%,40%,65.4%,58.3%,57.1%,100%,100%,93.3%,100%
4,Graham Ike,GONZ at SMC,Points + Rebounds + Assists,u31.5,BET\n-110,-1122,52.4%,80%,40%,57.7%,50%,57.1%,100%,100%,100%,100%
5,Emmanuel Innocenti,GONZ at SMC,Points + Rebounds,o12.5,BET\n-110,-1101,52.4%,80%,50%,26.7%,23.1%,25%,40%,60%,53.3%,47.4%
6,Emmanuel Innocenti,GONZ at SMC,Rebounds,u4.5,BET\n-150,-1452,60.0%,60%,50%,70%,61.5%,75%,80%,70%,70%,73.7%
7,Adam Miller,GONZ at SMC,Points,o7.5,BET\n-105,-1104,51.2%,60%,40%,53.3%,53.8%,50%,100%,70%,70%,68.4%
8,Graham Ike,GONZ at SMC,Assists,o2.5,BET\n+115,1117,46.5%,60%,50%,57.7%,66.7%,0%,0%,10%,3.3%,0%
9,Graham Ike,GONZ at SMC,Three Pointers Made,o0.5,BET\n-179,-1942,64.2%,60%,70%,46.2%,41.7%,14.3%,60%,30%,46.7%,36.8%
10,Graham Ike,GONZ at SMC,Rebounds,u7.5,BET\n+105,-1084,48.8%,60%,50%,38.5%,25%,57.1%,100%,100%,96.7%,94.7%


In [ ]:
import numpy as np
import pandas as pd
import re

# --- American odds from ODDS like "BET\n-130" ---
def extract_american(x):
    if pd.isna(x):
        return np.nan
    m = re.search(r"([+-]\d{2,5})", str(x))
    return float(m.group(1)) if m else np.nan

df["AMERICAN_ODDS"] = df["ODDS"].apply(extract_american)

# --- Percent parsing: "56.5%" -> 0.565 ; "--" -> NaN ; already-decimal stays ---
def pct_to_float(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in ["--","-","", "None", "nan"]:
        return np.nan
    s = s.replace("%","").strip()
    try:
        v = float(s)
    except:
        return np.nan
    if v > 1:
        v = v / 100.0
    return max(0.0, min(1.0, v))

pct_cols = ["IM_PROB_PCT","L5","L10","2025","HM_AW","H2H","DVP_L5","DVP_L10","DVP_2025","DVP_HM_AW"]
for c in pct_cols:
    if c in df.columns:
        df[c] = df[c].apply(pct_to_float)

# --- Odds -> book prob + decimal ---
def american_to_prob(o):
    if pd.isna(o):
        return np.nan
    o = float(o)
    return (-o)/((-o)+100) if o < 0 else 100/(o+100)

def american_to_decimal(o):
    if pd.isna(o):
        return np.nan
    o = float(o)
    return 1 + (100/(-o)) if o < 0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

print("Rows:", len(df))
print("AMERICAN_ODDS missing:", int(df["AMERICAN_ODDS"].isna().sum()))
display(df[["PLAYER","GAME","PROP","OUTCOME","ODDS","AMERICAN_ODDS","BOOK_PROB"]].head(10))

Rows: 96
AMERICAN_ODDS missing: 0


,PLAYER,GAME,PROP,OUTCOME,ODDS,AMERICAN_ODDS,BOOK_PROB
1,Emmanuel Innocenti,GONZ at SMC,Points,o7.5,BET\n-130,-130.0,0.565217
2,Graham Ike,GONZ at SMC,Points,o19.5,BET\n-166,-166.0,0.624060
3,Graham Ike,GONZ at SMC,Points,u21.5,BET\n-115,-115.0,0.534884
4,Graham Ike,GONZ at SMC,Points + Rebounds + Assists,u31.5,BET\n-110,-110.0,0.523810
5,Emmanuel Innocenti,GONZ at SMC,Points + Rebounds,o12.5,BET\n-110,-110.0,0.523810
6,Emmanuel Innocenti,GONZ at SMC,Rebounds,u4.5,BET\n-150,-150.0,0.600000
7,Adam Miller,GONZ at SMC,Points,o7.5,BET\n-105,-105.0,0.512195
8,Graham Ike,GONZ at SMC,Assists,o2.5,BET\n+115,115.0,0.465116
9,Graham Ike,GONZ at SMC,Three Pointers Made,o0.5,BET\n-179,-179.0,0.641577
10,Graham Ike,GONZ at SMC,Rebounds,u7.5,BET\n+105,105.0,0.487805


In [ ]:
# Weights (avoid L5 overfit)
W = {
    "L5":   0.18,
    "L10":  0.24,
    "2025": 0.24,
    "HM_AW":0.14,
    "H2H":  0.06
}

DVP_CAP = 0.07  # cap matchup impact

# Ensure columns exist (if any are missing, create NaN)
for c in ["L5","L10","2025","HM_AW","H2H"]:
    if c not in df.columns:
        df[c] = np.nan

# Trend core
df["P_TREND"] = (
    W["L5"]   * df["L5"] +
    W["L10"]  * df["L10"] +
    W["2025"] * df["2025"] +
    W["HM_AW"]* df["HM_AW"] +
    W["H2H"]  * df["H2H"].fillna(df["L10"])   # fallback when H2H missing
)

# DVP layer
dvp_cols = [c for c in ["DVP_L5","DVP_L10","DVP_2025","DVP_HM_AW"] if c in df.columns]
df["P_DVP_RAW"] = df[dvp_cols].mean(axis=1, skipna=True) if dvp_cols else np.nan
df["ADJ_DVP"] = (df["P_DVP_RAW"] - 0.50).clip(-DVP_CAP, DVP_CAP)

# Final model probability
df["MODEL_PROB"] = (df["P_TREND"] + df["ADJ_DVP"]).clip(0.01, 0.99)

# Edge + EV (if odds present)
df["EDGE_VS_BOOK"] = df["MODEL_PROB"] - df["BOOK_PROB"]
df["EV_1U"] = df["MODEL_PROB"] * (df["DEC_ODDS"] - 1) - (1 - df["MODEL_PROB"])

In [ ]:
# Weights (avoid L5 overfit)
W = {
    "L5":   0.18,
    "L10":  0.24,
    "2025": 0.24,
    "HM_AW":0.14,
    "H2H":  0.06
}

DVP_CAP = 0.07  # cap matchup impact

# Ensure columns exist (if any are missing, create NaN)
for c in ["L5","L10","2025","HM_AW","H2H"]:
    if c not in df.columns:
        df[c] = np.nan

# Trend core
df["P_TREND"] = (
    W["L5"]   * df["L5"] +
    W["L10"]  * df["L10"] +
    W["2025"] * df["2025"] +
    W["HM_AW"]* df["HM_AW"] +
    W["H2H"]  * df["H2H"].fillna(df["L10"])   # fallback when H2H missing
)

# DVP layer
dvp_cols = [c for c in ["DVP_L5","DVP_L10","DVP_2025","DVP_HM_AW"] if c in df.columns]
df["P_DVP_RAW"] = df[dvp_cols].mean(axis=1, skipna=True) if dvp_cols else np.nan
df["ADJ_DVP"] = (df["P_DVP_RAW"] - 0.50).clip(-DVP_CAP, DVP_CAP)

# Final model probability
df["MODEL_PROB"] = (df["P_TREND"] + df["ADJ_DVP"]).clip(0.01, 0.99)

# Edge + EV (if odds present)
df["EDGE_VS_BOOK"] = df["MODEL_PROB"] - df["BOOK_PROB"]
df["EV_1U"] = df["MODEL_PROB"] * (df["DEC_ODDS"] - 1) - (1 - df["MODEL_PROB"])

In [ ]:
top10 = df.sort_values("MODEL_PROB", ascending=False).head(10).copy()
top10.insert(0, "RANK", range(1, len(top10)+1))

display(top10[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "AMERICAN_ODDS","BOOK_PROB","MODEL_PROB","EDGE_VS_BOOK"
]])

top10.to_csv("hit_sheet_top10_ncaam.csv", index=False)
df.to_csv("prop_engine_full_ncaam.csv", index=False)

print("Saved: hit_sheet_top10_ncaam.csv")
print("Saved: prop_engine_full_ncaam.csv")

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,BOOK_PROB,MODEL_PROB,EDGE_VS_BOOK
38,1,Tahaad Pettiford,MISS at AUB,Rebounds,o2.5,-200.0,0.666667,0.62954,-0.037127
37,2,KeShawn Murphy,MISS at AUB,Points,u12.5,-117.0,0.539171,0.59748,0.058309
6,3,Emmanuel Innocenti,GONZ at SMC,Rebounds,u4.5,-150.0,0.600000,0.59710,-0.002900
55,4,Seth Trimble,VT at UNC,Assists,u2.5,-190.0,0.655172,0.59624,-0.058932
42,5,Amani Hansberry,VT at UNC,Points + Rebounds + Assists,u21.5,-120.0,0.545455,0.59064,0.045185
49,6,Jailen Bedford,VT at UNC,Points,o10.5,-120.0,0.545455,0.59040,0.044945
3,7,Graham Ike,GONZ at SMC,Points,u21.5,-115.0,0.534884,0.58284,0.047956
44,8,Graham Ike,GONZ at SMC,Points,u21.5,-115.0,0.534884,0.58284,0.047956
45,9,Xaivian Lee,ARK at FLA,Points,o9.5,-168.0,0.626866,0.57510,-0.051766
54,10,Meleek Thomas,ARK at FLA,Rebounds,u4.5,-108.0,0.519231,0.57432,0.055089


Saved: hit_sheet_top10_ncaam.csv
Saved: prop_engine_full_ncaam.csv


In [ ]:
def discord_hit_sheet(df_block):
    lines = ["NCAA PROP ENGINE — TOP 10 MOST LIKELY", "—"]
    for _, r in df_block.iterrows():
        odds_txt = "" if pd.isna(r.AMERICAN_ODDS) else f" ({int(r.AMERICAN_ODDS)})"
        edge_txt = "NA" if pd.isna(r.EDGE_VS_BOOK) else f"{r.EDGE_VS_BOOK*100:+.1f} pts"
        lines.append(
            f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n"
            f"{r.PROP} {r.OUTCOME}{odds_txt}\n"
            f"Model: {r.MODEL_PROB*100:.1f}% | Edge: {edge_txt}"
        )
    return "\n".join(lines)

print(discord_hit_sheet(top10))

NCAA PROP ENGINE — TOP 10 MOST LIKELY
—
1) Tahaad Pettiford — MISS at AUB
Rebounds o2.5 (-200)
Model: 63.0% | Edge: -3.7 pts
2) KeShawn Murphy — MISS at AUB
Points u12.5 (-117)
Model: 59.7% | Edge: +5.8 pts
3) Emmanuel Innocenti — GONZ at SMC
Rebounds u4.5 (-150)
Model: 59.7% | Edge: -0.3 pts
4) Seth Trimble — VT at UNC
Assists u2.5 (-190)
Model: 59.6% | Edge: -5.9 pts
5) Amani Hansberry — VT at UNC
Points + Rebounds + Assists u21.5 (-120)
Model: 59.1% | Edge: +4.5 pts
6) Jailen Bedford — VT at UNC
Points o10.5 (-120)
Model: 59.0% | Edge: +4.5 pts
7) Graham Ike — GONZ at SMC
Points u21.5 (-115)
Model: 58.3% | Edge: +4.8 pts
8) Graham Ike — GONZ at SMC
Points u21.5 (-115)
Model: 58.3% | Edge: +4.8 pts
9) Xaivian Lee — ARK at FLA
Points o9.5 (-168)
Model: 57.5% | Edge: -5.2 pts
10) Meleek Thomas — ARK at FLA
Rebounds u4.5 (-108)
Model: 57.4% | Edge: +5.5 pts


In [ ]:
import numpy as np
import pandas as pd
import re

# ---- helpers ----
def extract_american(x):
    if pd.isna(x):
        return np.nan
    m = re.search(r"([+-]\d{2,5})", str(x))
    return float(m.group(1)) if m else np.nan

def pct_to_float(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in ["--","-","", "None", "nan"]:
        return np.nan
    s = s.replace("%","").strip()
    try:
        v = float(s)
    except:
        return np.nan
    if v > 1:
        v = v/100.0
    return max(0.0, min(1.0, v))

def american_to_prob(o):
    if pd.isna(o): return np.nan
    o = float(o)
    return (-o)/((-o)+100) if o < 0 else 100/(o+100)

def american_to_decimal(o):
    if pd.isna(o): return np.nan
    o = float(o)
    return 1 + (100/(-o)) if o < 0 else 1 + (o/100)

# ---- ensure cols exist + normalize ----
# (assumes df already created by your Step 1 loader)
df["AMERICAN_ODDS"] = df["ODDS"].apply(extract_american)

pct_cols = ["IM_PROB_PCT","L5","L10","2025","HM_AW","H2H","DVP_L5","DVP_L10","DVP_2025","DVP_HM_AW"]
for c in pct_cols:
    if c in df.columns:
        df[c] = df[c].apply(pct_to_float)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

# ---- de-dupe key (fixes Graham Ike duplicate etc.) ----
df["DEDUP_KEY"] = (
    df["PLAYER"].astype(str).str.upper().str.strip() + "|" +
    df["GAME"].astype(str).str.upper().str.strip() + "|" +
    df["PROP"].astype(str).str.upper().str.strip() + "|" +
    df["OUTCOME"].astype(str).str.upper().str.strip()
)

# Keep the first instance per unique market
df = df.sort_values(["DEDUP_KEY"]).drop_duplicates("DEDUP_KEY", keep="first").copy()

print("Rows after dedupe:", len(df))
display(df[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","BOOK_PROB"]].head(10))

Rows after dedupe: 86


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,BOOK_PROB
7,Adam Miller,GONZ at SMC,Points,o7.5,-105.0,0.512195
17,Adam Miller,GONZ at SMC,Points,u7.5,-121.0,0.547511
27,Adam Miller,GONZ at SMC,Three Pointers Made,o1.5,155.0,0.392157
70,Alex Condon,ARK at FLA,Assists,u3.5,-145.0,0.591837
41,Alex Condon,ARK at FLA,Rebounds,o7.5,-105.0,0.512195
42,Amani Hansberry,VT at UNC,Points + Rebounds + Assists,u21.5,-120.0,0.545455
92,Amani Hansberry,VT at UNC,Points + Rebounds,o19.5,-115.0,0.534884
61,Amani Hansberry,VT at UNC,Points,o12.5,-115.0,0.534884
99,Amani Hansberry,VT at UNC,Points,o14.5,150.0,0.400000
71,Amani Hansberry,VT at UNC,Rebounds,o6.5,-125.0,0.555556


In [ ]:
# Weights (avoid L5 overfit)
W = {"L5":0.18, "L10":0.24, "2025":0.24, "HM_AW":0.14, "H2H":0.06}
DVP_CAP = 0.07  # cap matchup layer

for c in ["L5","L10","2025","HM_AW","H2H"]:
    if c not in df.columns:
        df[c] = np.nan

df["P_TREND"] = (
    W["L5"]   * df["L5"] +
    W["L10"]  * df["L10"] +
    W["2025"] * df["2025"] +
    W["HM_AW"]* df["HM_AW"] +
    W["H2H"]  * df["H2H"].fillna(df["L10"])  # graceful fallback
)

dvp_cols = [c for c in ["DVP_L5","DVP_L10","DVP_2025","DVP_HM_AW"] if c in df.columns]
df["P_DVP_RAW"] = df[dvp_cols].mean(axis=1, skipna=True) if dvp_cols else np.nan
df["ADJ_DVP"]   = (df["P_DVP_RAW"] - 0.50).clip(-DVP_CAP, DVP_CAP)

df["MODEL_PROB_BASE"] = (df["P_TREND"] + df["ADJ_DVP"]).clip(0.01, 0.99)


In [ ]:
import requests
from datetime import datetime, timezone

# ESPN endpoints
SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"
TEAM_STATS = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

# --- Build a minimal team token -> ESPN team_id map from today's scoreboard window ---
# (We only need teams that appear in your sheet)
sheet_games = df["GAME"].astype(str).str.upper().str.strip().unique().tolist()

def parse_game_tokens(g):
    # "ARK at FLA" -> ("ARK","FLA")
    parts = [p.strip() for p in g.replace("AT", "at").split("at")]
    if len(parts) != 2:
        return (None, None)
    return (parts[0].strip(), parts[1].strip())

need_tokens = set()
for g in sheet_games:
    a,h = parse_game_tokens(g)
    if a: need_tokens.add(a)
    if h: need_tokens.add(h)

# Pull scoreboard for today +/-1 day (covers late games)
today = datetime.now(timezone.utc).strftime("%Y%m%d")
j = get_json(SCOREBOARD, params={"dates": today})

team_map = {}  # token -> team_id
abbr_map = {}  # token -> full abbreviation found

for ev in j.get("events", []):
    comp = (ev.get("competitions") or [{}])[0]
    for c in comp.get("competitors", []):
        t = c.get("team", {})
        abbr = (t.get("abbreviation") or "").upper()
        tid = t.get("id")
        if abbr and tid:
            team_map[abbr] = tid
            abbr_map[abbr] = abbr

# Basic aliasing for common sheet tokens
alias = {
    "GONZ":"GONZ", "SMC":"SMC",
    "ARK":"ARK", "FLA":"FLA",
    "VT":"VT", "UNC":"UNC",
    "MISS":"MISS", "AUB":"AUB"
}
# If your sheet uses these short tokens, they should match ESPN abbreviations; if not found, we just skip.
# (No hard fail)

# ---- Team context proxies: use Opponent defensive strength proxy from ESPN team stats ----
# We’ll compute opponent "defense proxy" = Opponent Points Allowed Per Game (lower = tougher)
# Then small adjustment: tougher defense -> reduce over props a bit / boost unders a bit (capped)

def find_team_id(tok):
    tok = tok.upper().strip()
    tok = alias.get(tok, tok)
    return team_map.get(tok)

def get_team_def_proxy(team_id):
    # returns opp points per game (float) or NaN
    try:
        tj = get_json(TEAM_STATS.format(team_id=str(team_id)))
    except:
        return np.nan
    # ESPN schema varies; search stats for opponent PPG
    stats = tj.get("statistics", [])
    flat = {}
    for grp in stats:
        for item in grp.get("stats", []):
            name = item.get("name")
            val = item.get("value")
            if name and val is not None:
                flat[name] = val
    # common keys: "oppPointsPerGame" or similar; fallback tries
    for k in ["oppPointsPerGame","opponentPointsPerGame","pointsAllowedPerGame","oppPointsAvg"]:
        if k in flat:
            try: return float(flat[k])
            except: pass
    return np.nan

# Build opponent defense proxy per row
opp_def = []
for _, r in df.iterrows():
    away_tok, home_tok = parse_game_tokens(r["GAME"].upper())
    # opponent for a prop is always the opposing team; we need which side player is on (we don't know from sheet)
    # So we apply a neutral proxy: average of both teams' oppPPG if available.
    a_id = find_team_id(away_tok) if away_tok else None
    h_id = find_team_id(home_tok) if home_tok else None
    a_def = get_team_def_proxy(a_id) if a_id else np.nan
    h_def = get_team_def_proxy(h_id) if h_id else np.nan
    opp_def.append(np.nanmean([a_def, h_def]) if (not np.isnan(a_def) or not np.isnan(h_def)) else np.nan)

df["ESPN_OPP_DEF_PPG_PROXY"] = opp_def

# Adjustment: compare proxy to NCAA-ish baseline (70). Tougher defense (lower) hurts overs, helps unders.
BASELINE_DEF = 70.0
CTX_CAP = 0.03  # small cap for context layer

def ctx_adjust(def_ppg, outcome):
    if pd.isna(def_ppg):
        return 0.0
    # lower allowed -> tougher -> negative for overs, positive for unders
    delta = (BASELINE_DEF - def_ppg) / 100.0  # typically small
    delta = float(np.clip(delta, -CTX_CAP, CTX_CAP))
    is_over = str(outcome).lower().startswith("o")
    is_under = str(outcome).lower().startswith("u")
    if is_over:  return -delta
    if is_under: return +delta
    return 0.0

df["ADJ_CTX"] = [ctx_adjust(d, o) for d, o in zip(df["ESPN_OPP_DEF_PPG_PROXY"], df["OUTCOME"])]

# Placeholder injury/status layer (only applied if you later merge a status column)
df["ADJ_INJ_PROXY"] = 0.0

df["MODEL_PROB"] = (df["MODEL_PROB_BASE"] + df["ADJ_CTX"] + df["ADJ_INJ_PROXY"]).clip(0.01, 0.99)

df["EDGE_VS_BOOK"] = df["MODEL_PROB"] - df["BOOK_PROB"]

In [ ]:
top10 = df.sort_values("MODEL_PROB", ascending=False).head(10).copy()
top10.insert(0, "RANK", range(1, len(top10)+1))

display(top10[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "AMERICAN_ODDS","BOOK_PROB","MODEL_PROB","EDGE_VS_BOOK",
    "ESPN_OPP_DEF_PPG_PROXY","ADJ_CTX"
]])

top10.to_csv("hit_sheet_top10_ncaam_uniform.csv", index=False)
df.to_csv("prop_engine_full_ncaam_uniform.csv", index=False)

print("Saved: hit_sheet_top10_ncaam_uniform.csv")
print("Saved: prop_engine_full_ncaam_uniform.csv")

def discord_hit_sheet(df_block):
    lines = ["NCAA PROP ENGINE — TOP 10 MOST LIKELY (UNIFORM)", "—"]
    for _, r in df_block.iterrows():
        odds_txt = "" if pd.isna(r.AMERICAN_ODDS) else f" ({int(r.AMERICAN_ODDS)})"
        edge_txt = "NA" if pd.isna(r.EDGE_VS_BOOK) else f"{r.EDGE_VS_BOOK*100:+.1f} pts"
        lines.append(
            f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n"
            f"{r.PROP} {r.OUTCOME}{odds_txt}\n"
            f"Model: {r.MODEL_PROB*100:.1f}% | Edge: {edge_txt}"
        )
    return "\n".join(lines)

print(discord_hit_sheet(top10))

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,BOOK_PROB,MODEL_PROB,EDGE_VS_BOOK,ESPN_OPP_DEF_PPG_PROXY,ADJ_CTX
38,1,Tahaad Pettiford,MISS at AUB,Rebounds,o2.5,-200.0,0.666667,0.62954,-0.037127,NaN,0.0
37,2,KeShawn Murphy,MISS at AUB,Points,u12.5,-117.0,0.539171,0.59748,0.058309,NaN,0.0
6,3,Emmanuel Innocenti,GONZ at SMC,Rebounds,u4.5,-150.0,0.600000,0.59710,-0.002900,NaN,0.0
55,4,Seth Trimble,VT at UNC,Assists,u2.5,-190.0,0.655172,0.59624,-0.058932,NaN,0.0
42,5,Amani Hansberry,VT at UNC,Points + Rebounds + Assists,u21.5,-120.0,0.545455,0.59064,0.045185,NaN,0.0
49,6,Jailen Bedford,VT at UNC,Points,o10.5,-120.0,0.545455,0.59040,0.044945,NaN,0.0
44,7,Graham Ike,GONZ at SMC,Points,u21.5,-115.0,0.534884,0.58284,0.047956,NaN,0.0
45,8,Xaivian Lee,ARK at FLA,Points,o9.5,-168.0,0.626866,0.57510,-0.051766,NaN,0.0
54,9,Meleek Thomas,ARK at FLA,Rebounds,u4.5,-108.0,0.519231,0.57432,0.055089,NaN,0.0
46,10,Luka Bogavac,VT at UNC,Points,u12.5,-112.0,0.528302,0.57275,0.044448,NaN,0.0


Saved: hit_sheet_top10_ncaam_uniform.csv
Saved: prop_engine_full_ncaam_uniform.csv
NCAA PROP ENGINE — TOP 10 MOST LIKELY (UNIFORM)
—
1) Tahaad Pettiford — MISS at AUB
Rebounds o2.5 (-200)
Model: 63.0% | Edge: -3.7 pts
2) KeShawn Murphy — MISS at AUB
Points u12.5 (-117)
Model: 59.7% | Edge: +5.8 pts
3) Emmanuel Innocenti — GONZ at SMC
Rebounds u4.5 (-150)
Model: 59.7% | Edge: -0.3 pts
4) Seth Trimble — VT at UNC
Assists u2.5 (-190)
Model: 59.6% | Edge: -5.9 pts
5) Amani Hansberry — VT at UNC
Points + Rebounds + Assists u21.5 (-120)
Model: 59.1% | Edge: +4.5 pts
6) Jailen Bedford — VT at UNC
Points o10.5 (-120)
Model: 59.0% | Edge: +4.5 pts
7) Graham Ike — GONZ at SMC
Points u21.5 (-115)
Model: 58.3% | Edge: +4.8 pts
8) Xaivian Lee — ARK at FLA
Points o9.5 (-168)
Model: 57.5% | Edge: -5.2 pts
9) Meleek Thomas — ARK at FLA
Rebounds u4.5 (-108)
Model: 57.4% | Edge: +5.5 pts
10) Luka Bogavac — VT at UNC
Points u12.5 (-112)
Model: 57.3% | Edge: +4.4 pts


In [ ]:
import requests, numpy as np, pandas as pd, re
from datetime import datetime, timezone, timedelta

SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"
TEAM_STATS = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

# --- parse tokens like "MISS at AUB" ---
def parse_game_tokens(g):
    g = str(g).upper().strip()
    g = g.replace(" AT ", " at ")
    parts = [p.strip() for p in g.split(" at ")]
    if len(parts) != 2:
        return (None, None)
    return (parts[0], parts[1])

# --- build date list (UTC) for ESPN pull ---
utc_now = datetime.now(timezone.utc)
dates = [(utc_now + timedelta(days=i)).strftime("%Y%m%d") for i in range(0, 5)]  # today..+4

# --- build team_map from multiple scoreboards ---
team_map = {}   # abbreviation -> team_id
name_map = {}   # abbreviation -> display name (debug)

events_count = 0
for d in dates:
    j = get_json(SCOREBOARD, params={"dates": d})
    events = j.get("events", []) or []
    events_count += len(events)
    for ev in events:
        comp = (ev.get("competitions") or [{}])[0]
        for c in comp.get("competitors", []):
            t = c.get("team", {}) or {}
            abbr = (t.get("abbreviation") or "").upper()
            tid = t.get("id")
            disp = t.get("displayName")
            if abbr and tid:
                team_map[abbr] = str(tid)
                name_map[abbr] = disp

print("ESPN scoreboards pulled:", dates, "| total events:", events_count)
print("Team abbreviations captured:", len(team_map))

# --- light aliasing for common short tokens ---
ALIASES = {
    "GONZ":"GONZ", "SMC":"SMC",
    "ARK":"ARK", "FLA":"FLA",
    "VT":"VT", "UNC":"UNC",
    "MISS":"MISS", "AUB":"AUB",
    "TENN":"TENN", "ALA":"ALA",
    "NEB":"NEB", "USC":"USC",
    "KU":"KU", "ARIZ":"ARIZ",
    "MIZZ":"MIZZ", "MSST":"MSST",
    "VILL":"VILL", "SJU":"SJU",
}

def resolve_team_id(tok):
    if tok is None:
        return None
    tok = tok.upper().strip()
    tok = ALIASES.get(tok, tok)
    return team_map.get(tok)

def get_team_def_proxy(team_id):
    if not team_id:
        return np.nan
    try:
        tj = get_json(TEAM_STATS.format(team_id=str(team_id)))
    except:
        return np.nan

    # flatten stats into dict
    flat = {}
    for grp in tj.get("statistics", []) or []:
        for item in grp.get("stats", []) or []:
            nm = item.get("name")
            val = item.get("value")
            if nm and val is not None:
                flat[nm] = val

    # try common keys (ESPN varies)
    for k in ["oppPointsPerGame","opponentPointsPerGame","pointsAllowedPerGame","oppPointsAvg"]:
        if k in flat:
            try:
                return float(flat[k])
            except:
                pass

    # last resort: search any key containing both "opp" and "points" and "game"
    for k, v in flat.items():
        ks = k.lower()
        if ("opp" in ks or "opponent" in ks) and "points" in ks and ("game" in ks or "per" in ks):
            try:
                return float(v)
            except:
                continue
    return np.nan

BASELINE_DEF = 70.0
CTX_CAP = 0.03

def ctx_adjust(def_ppg, outcome):
    if pd.isna(def_ppg):
        return 0.0
    delta = (BASELINE_DEF - def_ppg) / 100.0
    delta = float(np.clip(delta, -CTX_CAP, CTX_CAP))
    o = str(outcome).lower().strip()
    if o.startswith("o"):  # over
        return -delta
    if o.startswith("u"):  # under
        return +delta
    return 0.0

# --- compute proxy per row (avg of both teams' defense proxy) ---
opp_def = []
for _, r in df.iterrows():
    away_tok, home_tok = parse_game_tokens(r["GAME"])
    a_id = resolve_team_id(away_tok)
    h_id = resolve_team_id(home_tok)
    a_def = get_team_def_proxy(a_id)
    h_def = get_team_def_proxy(h_id)
    opp_def.append(np.nanmean([a_def, h_def]) if (not np.isnan(a_def) or not np.isnan(h_def)) else np.nan)

df["ESPN_OPP_DEF_PPG_PROXY"] = opp_def
df["ADJ_CTX"] = [ctx_adjust(d, o) for d, o in zip(df["ESPN_OPP_DEF_PPG_PROXY"], df["OUTCOME"])]

# keep injury proxy placeholder unless you add it later
if "ADJ_INJ_PROXY" not in df.columns:
    df["ADJ_INJ_PROXY"] = 0.0

# final model prob
df["MODEL_PROB"] = (df["MODEL_PROB_BASE"] + df["ADJ_CTX"] + df["ADJ_INJ_PROXY"]).clip(0.01, 0.99)
df["EDGE_VS_BOOK"] = df["MODEL_PROB"] - df["BOOK_PROB"]

print("Rows with ESPN context applied (non-NaN proxy):", int(df["ESPN_OPP_DEF_PPG_PROXY"].notna().sum()), "of", len(df))
display(df[["PLAYER","GAME","OUTCOME","ESPN_OPP_DEF_PPG_PROXY","ADJ_CTX","MODEL_PROB"]].head(10))

ESPN scoreboards pulled: ['20260301', '20260302', '20260303', '20260304', '20260305'] | total events: 24
Team abbreviations captured: 46
Rows with ESPN context applied (non-NaN proxy): 0 of 86


,PLAYER,GAME,OUTCOME,ESPN_OPP_DEF_PPG_PROXY,ADJ_CTX,MODEL_PROB
7,Adam Miller,GONZ at SMC,o7.5,NaN,0.0,0.50724
17,Adam Miller,GONZ at SMC,u7.5,NaN,0.0,0.35276
27,Adam Miller,GONZ at SMC,o1.5,NaN,0.0,0.12382
70,Alex Condon,ARK at FLA,u3.5,NaN,0.0,0.54656
41,Alex Condon,ARK at FLA,o7.5,NaN,0.0,0.38776
42,Amani Hansberry,VT at UNC,u21.5,NaN,0.0,0.59064
92,Amani Hansberry,VT at UNC,o19.5,NaN,0.0,0.35992
61,Amani Hansberry,VT at UNC,o12.5,NaN,0.0,0.41704
99,Amani Hansberry,VT at UNC,o14.5,NaN,0.0,0.34216
71,Amani Hansberry,VT at UNC,o6.5,NaN,0.0,0.37324


In [ ]:
import requests, numpy as np, pandas as pd
from datetime import datetime, timezone, timedelta

SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

utc_now = datetime.now(timezone.utc)

# include yesterday..+4 (covers 2/28 and 3/1 slates safely)
dates = [(utc_now + timedelta(days=i)).strftime("%Y%m%d") for i in range(-1, 5)]

team_map = {}
name_map = {}
events_count = 0

for d in dates:
    # groups=50 forces the full D1 slate instead of featured games
    j = get_json(SCOREBOARD, params={"dates": d, "groups": "50"})
    events = j.get("events", []) or []
    events_count += len(events)

    for ev in events:
        comp = (ev.get("competitions") or [{}])[0]
        for c in comp.get("competitors", []):
            t = c.get("team", {}) or {}
            abbr = (t.get("abbreviation") or "").upper().strip()
            tid  = t.get("id")
            disp = t.get("displayName")
            if abbr and tid:
                team_map[abbr] = str(tid)
                name_map[abbr] = disp

print("ESPN scoreboards pulled:", dates, "| total events:", events_count)
print("Team abbreviations captured:", len(team_map))
print("Sample abbreviations:", sorted(list(team_map.keys()))[:30])

ESPN scoreboards pulled: ['20260228', '20260301', '20260302', '20260303', '20260304', '20260305'] | total events: 323
Team abbreviations captured: 355
Sample abbreviations: ['AAMU', 'ACU', 'AF', 'AKR', 'ALA', 'ALCN', 'ALST', 'AMCC', 'AMER', 'APSU', 'ARIZ', 'ARK', 'ARMY', 'ARST', 'ASU', 'AUB', 'BALL', 'BAY', 'BC', 'BCU', 'BEL', 'BELL', 'BGSU', 'BING', 'BOIS', 'BRAD', 'BRWN', 'BRY', 'BU', 'BUCK']


In [ ]:
need = set()
for g in df["GAME"].astype(str).str.upper().unique():
    if " AT " in g:
        a,h = [x.strip() for x in g.split(" AT ")]
        need.add(a); need.add(h)

missing = sorted([t for t in need if t not in team_map])
present = sorted([t for t in need if t in team_map])

print("Tokens needed:", sorted(list(need)))
print("Present in ESPN map:", present)
print("Missing from ESPN map:", missing)

Tokens needed: ['ARK', 'AUB', 'FLA', 'GONZ', 'MISS', 'SMC', 'UNC', 'VT']
Present in ESPN map: ['ARK', 'AUB', 'FLA', 'GONZ', 'MISS', 'SMC', 'UNC', 'VT']
Missing from ESPN map: []


In [ ]:
# --- compute ESPN opponent defense proxy + context adjustment (now that team_map is correct) ---

TEAM_STATS = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics"

def parse_game_tokens(g):
    g = str(g).upper().strip()
    g = g.replace(" AT ", " at ")
    parts = [p.strip() for p in g.split(" at ")]
    if len(parts) != 2:
        return (None, None)
    return (parts[0], parts[1])

def get_team_def_proxy(team_id):
    if not team_id:
        return np.nan
    try:
        tj = get_json(TEAM_STATS.format(team_id=str(team_id)))
    except Exception:
        return np.nan

    flat = {}
    for grp in tj.get("statistics", []) or []:
        for item in grp.get("stats", []) or []:
            nm = item.get("name")
            val = item.get("value")
            if nm and val is not None:
                flat[nm] = val

    for k in ["oppPointsPerGame","opponentPointsPerGame","pointsAllowedPerGame","oppPointsAvg"]:
        if k in flat:
            try:
                return float(flat[k])
            except:
                pass

    for k, v in flat.items():
        ks = k.lower()
        if ("opp" in ks or "opponent" in ks) and "points" in ks and ("game" in ks or "per" in ks):
            try:
                return float(v)
            except:
                continue
    return np.nan

BASELINE_DEF = 70.0
CTX_CAP = 0.03

def ctx_adjust(def_ppg, outcome):
    if pd.isna(def_ppg):
        return 0.0
    delta = (BASELINE_DEF - def_ppg) / 100.0
    delta = float(np.clip(delta, -CTX_CAP, CTX_CAP))
    o = str(outcome).lower().strip()
    if o.startswith("o"):  # over
        return -delta
    if o.startswith("u"):  # under
        return +delta
    return 0.0

# compute proxy for each row
opp_def = []
for _, r in df.iterrows():
    away_tok, home_tok = parse_game_tokens(r["GAME"])
    a_id = resolve_team_id(away_tok)
    h_id = resolve_team_id(home_tok)

    a_def = get_team_def_proxy(a_id)
    h_def = get_team_def_proxy(h_id)

    opp_def.append(np.nanmean([a_def, h_def]) if (not np.isnan(a_def) or not np.isnan(h_def)) else np.nan)

df["ESPN_OPP_DEF_PPG_PROXY"] = opp_def
df["ADJ_CTX"] = [ctx_adjust(d, o) for d, o in zip(df["ESPN_OPP_DEF_PPG_PROXY"], df["OUTCOME"])]

# if injury proxy not present, keep at 0
if "ADJ_INJ_PROXY" not in df.columns:
    df["ADJ_INJ_PROXY"] = 0.0

# rebuild final probability + edge
df["MODEL_PROB"] = (df["MODEL_PROB_BASE"] + df["ADJ_CTX"] + df["ADJ_INJ_PROXY"]).clip(0.01, 0.99)
df["EDGE_VS_BOOK"] = df["MODEL_PROB"] - df["BOOK_PROB"]

print("Rows with ESPN context applied (non-NaN proxy):", int(df["ESPN_OPP_DEF_PPG_PROXY"].notna().sum()), "of", len(df))
display(df[["PLAYER","GAME","OUTCOME","ESPN_OPP_DEF_PPG_PROXY","ADJ_CTX","MODEL_PROB","BOOK_PROB","EDGE_VS_BOOK"]].head(15))

Rows with ESPN context applied (non-NaN proxy): 0 of 86


,PLAYER,GAME,OUTCOME,ESPN_OPP_DEF_PPG_PROXY,ADJ_CTX,MODEL_PROB,BOOK_PROB,EDGE_VS_BOOK
7,Adam Miller,GONZ at SMC,o7.5,NaN,0.0,0.50724,0.512195,-0.004955
17,Adam Miller,GONZ at SMC,u7.5,NaN,0.0,0.35276,0.547511,-0.194751
27,Adam Miller,GONZ at SMC,o1.5,NaN,0.0,0.12382,0.392157,-0.268337
70,Alex Condon,ARK at FLA,u3.5,NaN,0.0,0.54656,0.591837,-0.045277
41,Alex Condon,ARK at FLA,o7.5,NaN,0.0,0.38776,0.512195,-0.124435
42,Amani Hansberry,VT at UNC,u21.5,NaN,0.0,0.59064,0.545455,0.045185
92,Amani Hansberry,VT at UNC,o19.5,NaN,0.0,0.35992,0.534884,-0.174964
61,Amani Hansberry,VT at UNC,o12.5,NaN,0.0,0.41704,0.534884,-0.117844
99,Amani Hansberry,VT at UNC,o14.5,NaN,0.0,0.34216,0.400000,-0.057840
71,Amani Hansberry,VT at UNC,o6.5,NaN,0.0,0.37324,0.555556,-0.182316


In [ ]:
import requests, pandas as pd, numpy as np

TEAM_STATS_TRY = [
    "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics",
    "https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/teams/{team_id}/statistics",
]

def debug_team_stats(team_abbr):
    tid = team_map.get(team_abbr)
    print("TEAM:", team_abbr, "| team_id:", tid, "| display:", name_map.get(team_abbr))
    if not tid:
        print("No team_id resolved.")
        return

    for url_tmpl in TEAM_STATS_TRY:
        url = url_tmpl.format(team_id=str(tid))
        try:
            r = requests.get(url, timeout=30)
            print("\nURL:", url)
            print("Status:", r.status_code)
            print("Content-Type:", r.headers.get("content-type"))
            txt = r.text[:300]
            print("Body head:", txt.replace("\n"," ")[:300])
            if r.status_code == 200:
                j = r.json()
                # print top-level keys to understand schema
                if isinstance(j, dict):
                    print("JSON keys:", list(j.keys())[:25])
                else:
                    print("JSON type:", type(j))
        except Exception as e:
            print("\nURL:", url)
            print("ERROR:", repr(e))

# try a few from your slate
for abbr in ["GONZ","SMC","ARK","FLA","VT","UNC","MISS","AUB"]:
    debug_team_stats(abbr)
    print("\n" + "-"*80)

TEAM: GONZ | team_id: 2250 | display: Gonzaga Bulldogs

URL: https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/2250/statistics
Status: 200
Content-Type: application/json;charset=UTF-8
Body head: {"status":"success","results":{"stats":{"id":"0","name":"Season","abbreviation":"Season","categories":[{"name":"general","displayName":"General","abbreviation":"gen","stats":[{"name":"avgRebounds","displayName":"Rebounds Per Game","shortDisplayName":"RPG","description":"The average rebounds per game
JSON keys: ['status', 'results', 'season', 'requestedSeason', 'team']

URL: https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/teams/2250/statistics
Status: 404
Content-Type: None
Body head: {"error":{"message":"application error","code":404}}

--------------------------------------------------------------------------------
TEAM: SMC | team_id: 2608 | display: Saint Mary's Gaels

URL: https://site.api.espn.com/apis/site/v2/sport

In [ ]:
import re, functools

TEAM_STATS_FALLBACKS = [
    "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics",
    "https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/teams/{team_id}/statistics",
    # season-scoped core endpoint (often works when the generic one doesn’t)
    "https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/seasons/{season}/types/2/teams/{team_id}/statistics",
]

SEASON_GUESS = int(datetime.now(timezone.utc).year)  # 2026 in your case

def flatten_stats_any_schema(j):
    """
    ESPN schemas vary. This tries to flatten anything that looks like stats.
    Returns dict(name->value) where possible.
    """
    flat = {}

    if not isinstance(j, dict):
        return flat

    # schema A: {"statistics":[{"stats":[{"name":..,"value":..},...]},...]}
    if "statistics" in j and isinstance(j["statistics"], list):
        for grp in j["statistics"]:
            for item in (grp.get("stats") or []):
                nm = item.get("name")
                val = item.get("value")
                if nm and val is not None:
                    flat[nm] = val

    # schema B: core api sometimes uses {"splits":{"categories":[{"stats":[...]},...]}}
    splits = j.get("splits")
    if isinstance(splits, dict):
        cats = splits.get("categories")
        if isinstance(cats, list):
            for cat in cats:
                for item in (cat.get("stats") or []):
                    nm = item.get("name")
                    val = item.get("value")
                    if nm and val is not None:
                        flat[nm] = val

    return flat

@functools.lru_cache(maxsize=256)
def get_team_def_proxy_cached(team_id):
    if not team_id:
        return np.nan

    last_err = None

    for tmpl in TEAM_STATS_FALLBACKS:
        url = tmpl.format(team_id=str(team_id), season=str(SEASON_GUESS))
        try:
            r = requests.get(url, timeout=30)
            if r.status_code != 200:
                last_err = f"{r.status_code} @ {url}"
                continue

            j = r.json()
            flat = flatten_stats_any_schema(j)

            # direct common keys
            for k in ["oppPointsPerGame","opponentPointsPerGame","pointsAllowedPerGame","oppPointsAvg"]:
                if k in flat:
                    try:
                        return float(flat[k])
                    except:
                        pass

            # heuristic keys
            for k, v in flat.items():
                ks = k.lower()
                if ("opp" in ks or "opponent" in ks) and "points" in ks and ("game" in ks or "per" in ks):
                    try:
                        return float(v)
                    except:
                        continue

            # if 200 but no useful keys, record
            last_err = f"200 but no opp points keys @ {url} (flat_keys={len(flat)})"

        except Exception as e:
            last_err = f"EXC {repr(e)} @ {url}"
            continue

    # if nothing worked, print one example once (optional)
    return np.nan

# recompute proxy now that function works/caches
def parse_game_tokens(g):
    g = str(g).upper().strip()
    if " AT " in g:
        a,h = [x.strip() for x in g.split(" AT ")]
        return a,h
    return None,None

BASELINE_DEF = 70.0
CTX_CAP = 0.03

def ctx_adjust(def_ppg, outcome):
    if pd.isna(def_ppg):
        return 0.0
    delta = (BASELINE_DEF - def_ppg) / 100.0
    delta = float(np.clip(delta, -CTX_CAP, CTX_CAP))
    o = str(outcome).lower().strip()
    if o.startswith("o"):
        return -delta
    if o.startswith("u"):
        return +delta
    return 0.0

opp_def = []
for _, r in df.iterrows():
    away_tok, home_tok = parse_game_tokens(r["GAME"])
    a_id = resolve_team_id(away_tok)
    h_id = resolve_team_id(home_tok)

    a_def = get_team_def_proxy_cached(a_id)
    h_def = get_team_def_proxy_cached(h_id)

    opp_def.append(np.nanmean([a_def, h_def]) if (not np.isnan(a_def) or not np.isnan(h_def)) else np.nan)

df["ESPN_OPP_DEF_PPG_PROXY"] = opp_def
df["ADJ_CTX"] = [ctx_adjust(d, o) for d, o in zip(df["ESPN_OPP_DEF_PPG_PROXY"], df["OUTCOME"])]

if "ADJ_INJ_PROXY" not in df.columns:
    df["ADJ_INJ_PROXY"] = 0.0

df["MODEL_PROB"] = (df["MODEL_PROB_BASE"] + df["ADJ_CTX"] + df["ADJ_INJ_PROXY"]).clip(0.01, 0.99)
df["EDGE_VS_BOOK"] = df["MODEL_PROB"] - df["BOOK_PROB"]

print("Rows with ESPN context applied (non-NaN proxy):", int(df["ESPN_OPP_DEF_PPG_PROXY"].notna().sum()), "of", len(df))
display(df[["PLAYER","GAME","OUTCOME","ESPN_OPP_DEF_PPG_PROXY","ADJ_CTX","MODEL_PROB","BOOK_PROB","EDGE_VS_BOOK"]].head(15))

Rows with ESPN context applied (non-NaN proxy): 0 of 86


,PLAYER,GAME,OUTCOME,ESPN_OPP_DEF_PPG_PROXY,ADJ_CTX,MODEL_PROB,BOOK_PROB,EDGE_VS_BOOK
7,Adam Miller,GONZ at SMC,o7.5,NaN,0.0,0.50724,0.512195,-0.004955
17,Adam Miller,GONZ at SMC,u7.5,NaN,0.0,0.35276,0.547511,-0.194751
27,Adam Miller,GONZ at SMC,o1.5,NaN,0.0,0.12382,0.392157,-0.268337
70,Alex Condon,ARK at FLA,u3.5,NaN,0.0,0.54656,0.591837,-0.045277
41,Alex Condon,ARK at FLA,o7.5,NaN,0.0,0.38776,0.512195,-0.124435
42,Amani Hansberry,VT at UNC,u21.5,NaN,0.0,0.59064,0.545455,0.045185
92,Amani Hansberry,VT at UNC,o19.5,NaN,0.0,0.35992,0.534884,-0.174964
61,Amani Hansberry,VT at UNC,o12.5,NaN,0.0,0.41704,0.534884,-0.117844
99,Amani Hansberry,VT at UNC,o14.5,NaN,0.0,0.34216,0.400000,-0.057840
71,Amani Hansberry,VT at UNC,o6.5,NaN,0.0,0.37324,0.555556,-0.182316


In [ ]:
import requests, pandas as pd, numpy as np

TEAM_STATS_TRY = [
    "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics",
    "https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/teams/{team_id}/statistics",
]

def debug_team_stats(team_abbr):
    tid = team_map.get(team_abbr)
    print("TEAM:", team_abbr, "| team_id:", tid, "| display:", name_map.get(team_abbr))
    if not tid:
        print("No team_id resolved.")
        return

    for url_tmpl in TEAM_STATS_TRY:
        url = url_tmpl.format(team_id=str(tid))
        try:
            r = requests.get(url, timeout=30)
            print("\nURL:", url)
            print("Status:", r.status_code)
            print("Content-Type:", r.headers.get("content-type"))
            txt = r.text[:300]
            print("Body head:", txt.replace("\n"," ")[:300])
            if r.status_code == 200:
                j = r.json()
                # print top-level keys to understand schema
                if isinstance(j, dict):
                    print("JSON keys:", list(j.keys())[:25])
                else:
                    print("JSON type:", type(j))
        except Exception as e:
            print("\nURL:", url)
            print("ERROR:", repr(e))

# try a few from your slate
for abbr in ["GONZ","SMC","ARK","FLA","VT","UNC","MISS","AUB"]:
    debug_team_stats(abbr)
    print("\n" + "-"*80)

TEAM: GONZ | team_id: 2250 | display: Gonzaga Bulldogs

URL: https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/2250/statistics
Status: 200
Content-Type: application/json;charset=UTF-8
Body head: {"status":"success","results":{"stats":{"id":"0","name":"Season","abbreviation":"Season","categories":[{"name":"general","displayName":"General","abbreviation":"gen","stats":[{"name":"avgRebounds","displayName":"Rebounds Per Game","shortDisplayName":"RPG","description":"The average rebounds per game
JSON keys: ['status', 'results', 'season', 'requestedSeason', 'team']

URL: https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/teams/2250/statistics
Status: 404
Content-Type: None
Body head: {"error":{"message":"application error","code":404}}

--------------------------------------------------------------------------------
TEAM: SMC | team_id: 2608 | display: Saint Mary's Gaels

URL: https://site.api.espn.com/apis/site/v2/sport

In [ ]:
import re, functools

TEAM_STATS_FALLBACKS = [
    "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics",
    "https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/teams/{team_id}/statistics",
    # season-scoped core endpoint (often works when the generic one doesn’t)
    "https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/seasons/{season}/types/2/teams/{team_id}/statistics",
]

SEASON_GUESS = int(datetime.now(timezone.utc).year)  # 2026 in your case

def flatten_stats_any_schema(j):
    """
    ESPN schemas vary. This tries to flatten anything that looks like stats.
    Returns dict(name->value) where possible.
    """
    flat = {}

    if not isinstance(j, dict):
        return flat

    # schema A: {"statistics":[{"stats":[{"name":..,"value":..},...]},...]}
    if "statistics" in j and isinstance(j["statistics"], list):
        for grp in j["statistics"]:
            for item in (grp.get("stats") or []):
                nm = item.get("name")
                val = item.get("value")
                if nm and val is not None:
                    flat[nm] = val

    # schema B: core api sometimes uses {"splits":{"categories":[{"stats":[...]},...]}}
    splits = j.get("splits")
    if isinstance(splits, dict):
        cats = splits.get("categories")
        if isinstance(cats, list):
            for cat in cats:
                for item in (cat.get("stats") or []):
                    nm = item.get("name")
                    val = item.get("value")
                    if nm and val is not None:
                        flat[nm] = val

    return flat

@functools.lru_cache(maxsize=256)
def get_team_def_proxy_cached(team_id):
    if not team_id:
        return np.nan

    last_err = None

    for tmpl in TEAM_STATS_FALLBACKS:
        url = tmpl.format(team_id=str(team_id), season=str(SEASON_GUESS))
        try:
            r = requests.get(url, timeout=30)
            if r.status_code != 200:
                last_err = f"{r.status_code} @ {url}"
                continue

            j = r.json()
            flat = flatten_stats_any_schema(j)

            # direct common keys
            for k in ["oppPointsPerGame","opponentPointsPerGame","pointsAllowedPerGame","oppPointsAvg"]:
                if k in flat:
                    try:
                        return float(flat[k])
                    except:
                        pass

            # heuristic keys
            for k, v in flat.items():
                ks = k.lower()
                if ("opp" in ks or "opponent" in ks) and "points" in ks and ("game" in ks or "per" in ks):
                    try:
                        return float(v)
                    except:
                        continue

            # if 200 but no useful keys, record
            last_err = f"200 but no opp points keys @ {url} (flat_keys={len(flat)})"

        except Exception as e:
            last_err = f"EXC {repr(e)} @ {url}"
            continue

    # if nothing worked, print one example once (optional)
    return np.nan

# recompute proxy now that function works/caches
def parse_game_tokens(g):
    g = str(g).upper().strip()
    if " AT " in g:
        a,h = [x.strip() for x in g.split(" AT ")]
        return a,h
    return None,None

BASELINE_DEF = 70.0
CTX_CAP = 0.03

def ctx_adjust(def_ppg, outcome):
    if pd.isna(def_ppg):
        return 0.0
    delta = (BASELINE_DEF - def_ppg) / 100.0
    delta = float(np.clip(delta, -CTX_CAP, CTX_CAP))
    o = str(outcome).lower().strip()
    if o.startswith("o"):
        return -delta
    if o.startswith("u"):
        return +delta
    return 0.0

opp_def = []
for _, r in df.iterrows():
    away_tok, home_tok = parse_game_tokens(r["GAME"])
    a_id = resolve_team_id(away_tok)
    h_id = resolve_team_id(home_tok)

    a_def = get_team_def_proxy_cached(a_id)
    h_def = get_team_def_proxy_cached(h_id)

    opp_def.append(np.nanmean([a_def, h_def]) if (not np.isnan(a_def) or not np.isnan(h_def)) else np.nan)

df["ESPN_OPP_DEF_PPG_PROXY"] = opp_def
df["ADJ_CTX"] = [ctx_adjust(d, o) for d, o in zip(df["ESPN_OPP_DEF_PPG_PROXY"], df["OUTCOME"])]

if "ADJ_INJ_PROXY" not in df.columns:
    df["ADJ_INJ_PROXY"] = 0.0

df["MODEL_PROB"] = (df["MODEL_PROB_BASE"] + df["ADJ_CTX"] + df["ADJ_INJ_PROXY"]).clip(0.01, 0.99)
df["EDGE_VS_BOOK"] = df["MODEL_PROB"] - df["BOOK_PROB"]

print("Rows with ESPN context applied (non-NaN proxy):", int(df["ESPN_OPP_DEF_PPG_PROXY"].notna().sum()), "of", len(df))
display(df[["PLAYER","GAME","OUTCOME","ESPN_OPP_DEF_PPG_PROXY","ADJ_CTX","MODEL_PROB","BOOK_PROB","EDGE_VS_BOOK"]].head(15))

Rows with ESPN context applied (non-NaN proxy): 0 of 86


,PLAYER,GAME,OUTCOME,ESPN_OPP_DEF_PPG_PROXY,ADJ_CTX,MODEL_PROB,BOOK_PROB,EDGE_VS_BOOK
7,Adam Miller,GONZ at SMC,o7.5,NaN,0.0,0.50724,0.512195,-0.004955
17,Adam Miller,GONZ at SMC,u7.5,NaN,0.0,0.35276,0.547511,-0.194751
27,Adam Miller,GONZ at SMC,o1.5,NaN,0.0,0.12382,0.392157,-0.268337
70,Alex Condon,ARK at FLA,u3.5,NaN,0.0,0.54656,0.591837,-0.045277
41,Alex Condon,ARK at FLA,o7.5,NaN,0.0,0.38776,0.512195,-0.124435
42,Amani Hansberry,VT at UNC,u21.5,NaN,0.0,0.59064,0.545455,0.045185
92,Amani Hansberry,VT at UNC,o19.5,NaN,0.0,0.35992,0.534884,-0.174964
61,Amani Hansberry,VT at UNC,o12.5,NaN,0.0,0.41704,0.534884,-0.117844
99,Amani Hansberry,VT at UNC,o14.5,NaN,0.0,0.34216,0.400000,-0.057840
71,Amani Hansberry,VT at UNC,o6.5,NaN,0.0,0.37324,0.555556,-0.182316


In [ ]:
import requests, json, pandas as pd, numpy as np

def probe(url):
    r = requests.get(url, timeout=30)
    print("URL:", url)
    print("STATUS:", r.status_code)
    print("CONTENT-TYPE:", r.headers.get("content-type"))
    print("HEAD:", r.text[:200].replace("\n"," "))
    print("-"*80)

# confirm IDs we will use
for abbr in ["GONZ","SMC","ARK","FLA","VT","UNC","MISS","AUB"]:
    tid = team_map.get(abbr)
    print(abbr, "-> team_id:", tid, "|", name_map.get(abbr))
print("="*80)

# probe ESPN team statistics endpoints for one matchup
gon_id = team_map["GONZ"]
smc_id  = team_map["SMC"]

urls = [
    f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{gon_id}/statistics",
    f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{smc_id}/statistics",
    f"https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/teams/{gon_id}/statistics",
    f"https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/teams/{smc_id}/statistics",
    f"https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/seasons/2026/types/2/teams/{gon_id}/statistics",
    f"https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/seasons/2026/types/2/teams/{smc_id}/statistics",
]

for u in urls:
    probe(u)

GONZ -> team_id: 2250 | Gonzaga Bulldogs
SMC -> team_id: 2608 | Saint Mary's Gaels
ARK -> team_id: 8 | Arkansas Razorbacks
FLA -> team_id: 57 | Florida Gators
VT -> team_id: 259 | Virginia Tech Hokies
UNC -> team_id: 153 | North Carolina Tar Heels
MISS -> team_id: 145 | Ole Miss Rebels
AUB -> team_id: 2 | Auburn Tigers
URL: https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/2250/statistics
STATUS: 200
CONTENT-TYPE: application/json;charset=UTF-8
HEAD: {"status":"success","results":{"stats":{"id":"0","name":"Season","abbreviation":"Season","categories":[{"name":"general","displayName":"General","abbreviation":"gen","stats":[{"name":"avgRebounds","di
--------------------------------------------------------------------------------
URL: https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/2608/statistics
STATUS: 200
CONTENT-TYPE: application/json;charset=UTF-8
HEAD: {"status":"success","results":{"stats":{"id":"0","name"

In [ ]:
import functools, numpy as np, pandas as pd, requests

TEAM_STATS_SITE = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics"

def flatten_team_stats_site(j):
    flat = {}
    if not isinstance(j, dict):
        return flat

    # schema you are actually getting:
    # {"status":"success","results":{"stats":{"categories":[{"stats":[{"name":..,"value":..}, ...]}, ...]}}}
    stats_root = (((j.get("results") or {}).get("stats")) or {})
    cats = stats_root.get("categories") or []
    if isinstance(cats, list):
        for cat in cats:
            for item in (cat.get("stats") or []):
                nm = item.get("name")
                val = item.get("value")
                if nm and val is not None:
                    flat[nm] = val
    return flat

@functools.lru_cache(maxsize=256)
def get_team_def_proxy(team_id):
    if not team_id:
        return np.nan
    url = TEAM_STATS_SITE.format(team_id=str(team_id))
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        return np.nan

    j = r.json()
    flat = flatten_team_stats_site(j)

    # common keys to try first
    for k in ["oppPointsPerGame","opponentPointsPerGame","pointsAllowedPerGame","oppPointsAvg"]:
        if k in flat:
            try:
                return float(flat[k])
            except:
                pass

    # heuristic fallback
    for k, v in flat.items():
        ks = k.lower()
        if ("opp" in ks or "opponent" in ks) and "points" in ks and ("game" in ks or "per" in ks):
            try:
                return float(v)
            except:
                continue

    return np.nan

def parse_game_tokens(g):
    g = str(g).upper().strip()
    if " AT " in g:
        a,h = [x.strip() for x in g.split(" AT ")]
        return a,h
    return None,None

BASELINE_DEF = 70.0
CTX_CAP = 0.03

def ctx_adjust(def_ppg, outcome):
    if pd.isna(def_ppg):
        return 0.0
    delta = (BASELINE_DEF - def_ppg) / 100.0
    delta = float(np.clip(delta, -CTX_CAP, CTX_CAP))
    o = str(outcome).lower().strip()
    if o.startswith("o"):  # over harder vs good defense
        return -delta
    if o.startswith("u"):  # under easier vs good defense
        return +delta
    return 0.0

# --- recompute opponent defense proxy using away+home team IDs ---
opp_def = []
for _, r in df.iterrows():
    away_tok, home_tok = parse_game_tokens(r["GAME"])
    a_id = resolve_team_id(away_tok)
    h_id = resolve_team_id(home_tok)

    a_def = get_team_def_proxy(a_id)
    h_def = get_team_def_proxy(h_id)

    opp_def.append(np.nanmean([a_def, h_def]) if (not np.isnan(a_def) or not np.isnan(h_def)) else np.nan)

df["ESPN_OPP_DEF_PPG_PROXY"] = opp_def
df["ADJ_CTX"] = [ctx_adjust(d, o) for d, o in zip(df["ESPN_OPP_DEF_PPG_PROXY"], df["OUTCOME"])]

# injury proxy safety
if "ADJ_INJ_PROXY" not in df.columns:
    df["ADJ_INJ_PROXY"] = 0.0

# rebuild final prob + edge
df["MODEL_PROB"] = (df["MODEL_PROB_BASE"] + df["ADJ_CTX"] + df["ADJ_INJ_PROXY"]).clip(0.01, 0.99)
df["EDGE_VS_BOOK"] = df["MODEL_PROB"] - df["BOOK_PROB"]

print("Rows with ESPN context applied (non-NaN proxy):", int(df["ESPN_OPP_DEF_PPG_PROXY"].notna().sum()), "of", len(df))
display(df[["PLAYER","GAME","OUTCOME","ESPN_OPP_DEF_PPG_PROXY","ADJ_CTX","MODEL_PROB","BOOK_PROB","EDGE_VS_BOOK"]].head(15))

Rows with ESPN context applied (non-NaN proxy): 0 of 86


,PLAYER,GAME,OUTCOME,ESPN_OPP_DEF_PPG_PROXY,ADJ_CTX,MODEL_PROB,BOOK_PROB,EDGE_VS_BOOK
7,Adam Miller,GONZ at SMC,o7.5,NaN,0.0,0.50724,0.512195,-0.004955
17,Adam Miller,GONZ at SMC,u7.5,NaN,0.0,0.35276,0.547511,-0.194751
27,Adam Miller,GONZ at SMC,o1.5,NaN,0.0,0.12382,0.392157,-0.268337
70,Alex Condon,ARK at FLA,u3.5,NaN,0.0,0.54656,0.591837,-0.045277
41,Alex Condon,ARK at FLA,o7.5,NaN,0.0,0.38776,0.512195,-0.124435
42,Amani Hansberry,VT at UNC,u21.5,NaN,0.0,0.59064,0.545455,0.045185
92,Amani Hansberry,VT at UNC,o19.5,NaN,0.0,0.35992,0.534884,-0.174964
61,Amani Hansberry,VT at UNC,o12.5,NaN,0.0,0.41704,0.534884,-0.117844
99,Amani Hansberry,VT at UNC,o14.5,NaN,0.0,0.34216,0.400000,-0.057840
71,Amani Hansberry,VT at UNC,o6.5,NaN,0.0,0.37324,0.555556,-0.182316


In [ ]:
import numpy as np, pandas as pd, requests, functools

TEAM_STATS_SITE = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics"

def flatten_team_stats_site(j):
    flat = {}
    stats_root = (((j.get("results") or {}).get("stats")) or {})
    cats = stats_root.get("categories") or []
    for cat in cats:
        for item in (cat.get("stats") or []):
            nm = item.get("name")
            val = item.get("value")
            if nm and val is not None:
                flat[nm] = val
    return flat

@functools.lru_cache(maxsize=256)
def get_team_def_ppg(team_id):
    if team_id is None or (isinstance(team_id, float) and np.isnan(team_id)):
        return np.nan
    url = TEAM_STATS_SITE.format(team_id=str(int(team_id)))
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        return np.nan
    flat = flatten_team_stats_site(r.json())

    # try common keys
    for k in ["oppPointsPerGame","opponentPointsPerGame","pointsAllowedPerGame"]:
        if k in flat:
            try:
                return float(flat[k])
            except:
                pass

    # fallback: scan any key that looks like opponent points per game
    for k, v in flat.items():
        ks = k.lower()
        if ("opp" in ks or "opponent" in ks) and "points" in ks and ("per" in ks or "game" in ks):
            try:
                return float(v)
            except:
                continue

    return np.nan

# ---- GAME TOKEN PARSER ----
def parse_game_tokens(g):
    g = str(g).upper().strip()
    if " AT " in g:
        a, h = [x.strip() for x in g.split(" AT ")]
        return a, h
    return None, None

# ---- TEAM ID RESOLUTION (BYPASS resolve_team_id entirely) ----
ALIAS = {
    "TXAM": "TAM",
    "A&M": "TAM",
    "MISSST": "MSST",
    "STJOHNS": "SJU",
}

def team_id_from_token(tok):
    if tok is None:
        return None
    t = str(tok).upper().replace(".", "").replace("'", "").replace("-", "").strip()
    t = ALIAS.get(t, t)
    # direct map
    if t in team_map:
        return team_map[t]
    # try containment (rare)
    for k in team_map.keys():
        if t == k:
            return team_map[k]
    return None

# ---- CONTEXT ADJUSTMENT ----
BASELINE_DEF = 70.0
CTX_CAP = 0.03

def ctx_adjust(def_ppg, outcome):
    if pd.isna(def_ppg):
        return 0.0
    delta = (BASELINE_DEF - def_ppg) / 100.0
    delta = float(np.clip(delta, -CTX_CAP, CTX_CAP))
    o = str(outcome).lower().strip()
    if o.startswith("o"):
        return -delta
    if o.startswith("u"):
        return +delta
    return 0.0

# ---- DIAGNOSTIC: show first 10 resolved team ids ----
tmp = df.head(10).copy()
tmp["away_tok"], tmp["home_tok"] = zip(*tmp["GAME"].apply(parse_game_tokens))
tmp["away_id"] = tmp["away_tok"].apply(team_id_from_token)
tmp["home_id"] = tmp["home_tok"].apply(team_id_from_token)
print("TEAM-ID RESOLUTION SAMPLE (first 10):")
display(tmp[["GAME","away_tok","home_tok","away_id","home_id"]])

# ---- BUILD OPP DEF PROXY ----
opp_def = []
for _, r in df.iterrows():
    away_tok, home_tok = parse_game_tokens(r["GAME"])
    a_id = team_id_from_token(away_tok)
    h_id = team_id_from_token(home_tok)

    a_def = get_team_def_ppg(a_id)
    h_def = get_team_def_ppg(h_id)

    opp_def.append(np.nanmean([a_def, h_def]) if (not np.isnan(a_def) or not np.isnan(h_def)) else np.nan)

df["ESPN_OPP_DEF_PPG_PROXY"] = opp_def
df["ADJ_CTX"] = [ctx_adjust(d, o) for d, o in zip(df["ESPN_OPP_DEF_PPG_PROXY"], df["OUTCOME"])]

# injury proxy safety
if "ADJ_INJ_PROXY" not in df.columns:
    df["ADJ_INJ_PROXY"] = 0.0

# IMPORTANT: base probability column name may differ in your notebook
base_col = "MODEL_PROB_BASE" if "MODEL_PROB_BASE" in df.columns else ("MODEL_PROB" if "MODEL_PROB" in df.columns else None)
if base_col is None:
    raise ValueError("Need base prob column (MODEL_PROB_BASE or MODEL_PROB)")

df["MODEL_PROB"] = (df[base_col] + df["ADJ_CTX"] + df["ADJ_INJ_PROXY"]).clip(0.01, 0.99)
df["EDGE_VS_BOOK"] = df["MODEL_PROB"] - df["BOOK_PROB"]

print("Rows with ESPN context applied (non-NaN proxy):", int(df["ESPN_OPP_DEF_PPG_PROXY"].notna().sum()), "of", len(df))
display(df[["PLAYER","GAME","OUTCOME","ESPN_OPP_DEF_PPG_PROXY","ADJ_CTX","MODEL_PROB","BOOK_PROB","EDGE_VS_BOOK"]].head(15))

TEAM-ID RESOLUTION SAMPLE (first 10):


,GAME,away_tok,home_tok,away_id,home_id
7,GONZ at SMC,GONZ,SMC,2250,2608
17,GONZ at SMC,GONZ,SMC,2250,2608
27,GONZ at SMC,GONZ,SMC,2250,2608
70,ARK at FLA,ARK,FLA,8,57
41,ARK at FLA,ARK,FLA,8,57
42,VT at UNC,VT,UNC,259,153
92,VT at UNC,VT,UNC,259,153
61,VT at UNC,VT,UNC,259,153
99,VT at UNC,VT,UNC,259,153
71,VT at UNC,VT,UNC,259,153


Rows with ESPN context applied (non-NaN proxy): 0 of 86


,PLAYER,GAME,OUTCOME,ESPN_OPP_DEF_PPG_PROXY,ADJ_CTX,MODEL_PROB,BOOK_PROB,EDGE_VS_BOOK
7,Adam Miller,GONZ at SMC,o7.5,NaN,0.0,0.50724,0.512195,-0.004955
17,Adam Miller,GONZ at SMC,u7.5,NaN,0.0,0.35276,0.547511,-0.194751
27,Adam Miller,GONZ at SMC,o1.5,NaN,0.0,0.12382,0.392157,-0.268337
70,Alex Condon,ARK at FLA,u3.5,NaN,0.0,0.54656,0.591837,-0.045277
41,Alex Condon,ARK at FLA,o7.5,NaN,0.0,0.38776,0.512195,-0.124435
42,Amani Hansberry,VT at UNC,u21.5,NaN,0.0,0.59064,0.545455,0.045185
92,Amani Hansberry,VT at UNC,o19.5,NaN,0.0,0.35992,0.534884,-0.174964
61,Amani Hansberry,VT at UNC,o12.5,NaN,0.0,0.41704,0.534884,-0.117844
99,Amani Hansberry,VT at UNC,o14.5,NaN,0.0,0.34216,0.400000,-0.057840
71,Amani Hansberry,VT at UNC,o6.5,NaN,0.0,0.37324,0.555556,-0.182316


In [ ]:
import requests, pandas as pd, numpy as np

TEAM_STATS_SITE = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics"

def inspect_team_stats(team_id):
    url = TEAM_STATS_SITE.format(team_id=str(int(team_id)))
    r = requests.get(url, timeout=30)
    print("URL:", url)
    print("STATUS:", r.status_code)
    j = r.json()

    stats_root = (((j.get("results") or {}).get("stats")) or {})
    cats = stats_root.get("categories") or []
    rows = []
    for cat in cats:
        cat_name = cat.get("name")
        for item in (cat.get("stats") or []):
            rows.append({
                "category": cat_name,
                "name": item.get("name"),
                "displayName": item.get("displayName"),
                "value": item.get("value"),
            })

    dfk = pd.DataFrame(rows)
    print("Total stats rows:", len(dfk))

    # show likely candidates for opponent points allowed
    mask = (
        dfk["name"].fillna("").str.contains("opp|opponent|against|allowed", case=False, regex=True) |
        dfk["displayName"].fillna("").str.contains("opp|opponent|against|allowed", case=False, regex=True)
    ) & (
        dfk["name"].fillna("").str.contains("point", case=False, regex=True) |
        dfk["displayName"].fillna("").str.contains("point", case=False, regex=True)
    )

    cand = dfk[mask].copy()
    print("\nCANDIDATES (opp/allowed/against + points):")
    display(cand.sort_values(["category","displayName"]).head(50))

    print("\nFIRST 50 stats (for schema sanity):")
    display(dfk.head(50))

    return dfk

# test one team from your mapping that we know works
_ = inspect_team_stats(2250)  # Gonzaga

URL: https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/2250/statistics
STATUS: 200
Total stats rows: 45

CANDIDATES (opp/allowed/against + points):


,category,name,displayName,value



FIRST 50 stats (for schema sanity):


,category,name,displayName,value
0,general,avgRebounds,Rebounds Per Game,40.333332
1,general,assistTurnoverRatio,Assist To Turnover Ratio,1.900000
2,general,avgFouls,Fouls Per Game,17.200000
3,general,gamesPlayed,Games Played,30.000000
4,general,gamesStarted,Games Started,0.000000
5,general,minutes,Minutes,6025.000000
6,general,avgMinutes,Minutes Per Game,200.833330
7,general,totalRebounds,Rebounds,1210.000000
8,general,rebounds,Rebounds,1210.000000
9,offensive,freeThrowPct,Free Throw Percentage,69.651741


In [ ]:
import requests, pandas as pd, numpy as np, re

CORE_TEAM_STATS = "https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/seasons/{season}/types/{stype}/teams/{team_id}/statistics"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def deref_ref(j):
    # Some core endpoints return {"$ref": "..."} first
    if isinstance(j, dict) and "$ref" in j:
        return get_json(j["$ref"])
    return j

def flatten_stats(obj, prefix=""):
    """
    Recursively flatten dict/list into (path, value) rows.
    Useful because ESPN core schemas vary.
    """
    rows = []
    if isinstance(obj, dict):
        for k,v in obj.items():
            p = f"{prefix}.{k}" if prefix else str(k)
            rows += flatten_stats(v, p)
    elif isinstance(obj, list):
        for i,v in enumerate(obj):
            p = f"{prefix}[{i}]"
            rows += flatten_stats(v, p)
    else:
        rows.append((prefix, obj))
    return rows

def extract_points_against_per_game(core_stats_json):
    """
    Try multiple heuristics to find a 'points allowed / points against per game' number.
    Returns float or np.nan
    """
    flat = flatten_stats(core_stats_json)
    df_flat = pd.DataFrame(flat, columns=["path","value"])

    # Only numeric-ish values
    def to_float(x):
        try:
            return float(x)
        except:
            return np.nan

    df_flat["val"] = df_flat["value"].apply(to_float)

    # Heuristic match patterns
    patterns = [
        r"pointsagainst", r"pointagainst", r"oppoints", r"opppoints", r"opp.*points",
        r"pointsallowed", r"allowedpoints", r"against.*points", r"ptsagainst", r"opppts"
    ]
    pat = re.compile("|".join(patterns), re.IGNORECASE)

    cand = df_flat[df_flat["path"].str.contains(pat, na=False)].copy()
    cand = cand.dropna(subset=["val"])

    # Prefer per-game style fields
    # (paths that include avg/pergame/perGame etc)
    if len(cand):
        cand["score"] = 0
        cand.loc[cand["path"].str.contains(r"avg|pergame|perGame|per_game", case=False, regex=True), "score"] += 2
        cand.loc[cand["path"].str.contains(r"displayName|shortDisplayName", case=False, regex=True), "score"] -= 1
        # choose highest score then first
        cand = cand.sort_values(["score"], ascending=False)
        return float(cand.iloc[0]["val"])

    return np.nan

def team_points_allowed_ppg(team_id, season=2026, stype=2):
    try:
        j = get_json(CORE_TEAM_STATS.format(season=season, stype=stype, team_id=int(team_id)))
        j = deref_ref(j)
        return extract_points_against_per_game(j)
    except Exception:
        return np.nan

# ----------------------------
# BUILD DEF PPG MAP
# You need a DataFrame that has home_id/away_id already resolved.
# If yours is called team_map_df or df_team_ids, set it here.
# Expected columns: away_id, home_id, away_tok, home_tok
# ----------------------------

# Example: if you already have a df called team_id_df from your earlier resolution:
# team_id_df = df_with_ids[["away_tok","home_tok","away_id","home_id"]].drop_duplicates()

team_id_df = team_id_df.drop_duplicates().copy()  # <-- CHANGE this variable name if yours differs

all_team_ids = pd.unique(pd.concat([team_id_df["away_id"], team_id_df["home_id"]], ignore_index=True))
all_team_ids = [int(x) for x in all_team_ids if pd.notna(x)]

def_map = {}
for tid in all_team_ids:
    def_map[tid] = team_points_allowed_ppg(tid, season=2026, stype=2)

def_ppg_df = pd.DataFrame({"team_id": list(def_map.keys()), "points_allowed_ppg": list(def_map.values())})
print("Team points_allowed_ppg non-NaN:", def_ppg_df["points_allowed_ppg"].notna().sum(), "of", len(def_ppg_df))
display(def_ppg_df.head(10))

NameError: name 'team_id_df' is not defined

In [ ]:
import pandas as pd
import numpy as np

# ---- Confirm your df has away_id & home_id ----
if "away_id" not in df.columns or "home_id" not in df.columns:
    raise ValueError("away_id / home_id not found in df. Run the team-id resolution cell first.")

# ---- Build unique team list directly from df ----
all_team_ids = pd.unique(
    pd.concat([df["away_id"], df["home_id"]], ignore_index=True)
)

all_team_ids = [int(x) for x in all_team_ids if pd.notna(x)]

print("Unique team IDs detected:", all_team_ids)

ValueError: away_id / home_id not found in df. Run the team-id resolution cell first.

In [ ]:
import pandas as pd
import numpy as np
import re, requests

CSV_PATH = "NCAA night v2.csv"  # <- exact uploaded filename

# ---------- 0) Load CSV robustly ----------
# Some of your exports have a blank first row / unnamed columns.
raw = pd.read_csv(CSV_PATH, header=None)

# Find the first row that looks like real data: has " at " in col2 or col3 and prop-like strings
def looks_like_data(row):
    s = " ".join([str(x) for x in row.values if pd.notna(x)])
    return (" at " in s) and any(k in s for k in ["Points", "Rebounds", "Assists", "Three Pointers", "Points +", "Double Double"])

start_idx = None
for i in range(min(50, len(raw))):
    if looks_like_data(raw.iloc[i]):
        start_idx = i
        break

if start_idx is None:
    raise ValueError("Could not detect start of data rows in CSV. Upload a clean CSV or share first 5 lines.")

df = raw.iloc[start_idx:].copy()

# Your file structure appears like:
# col0 blank, col1 PLAYER, col2 GAME, col3 PROP, col4 OUTCOME, col5 ODDS, col6 AVG, col7 IM PROB %, ...
# We'll force the column names accordingly (truncate/extend safely)
cols = [
    "IDX", "PLAYER", "GAME", "PROP", "OUTCOME", "ODDS", "AVG", "IM_PROB_PCT",
    "L5", "L10", "SEASON_2025", "HM_AW", "H2H",
    "DVP_L5", "DVP_L10", "DVP_2025", "DVP_HM_AW"
]
df = df.iloc[:, :len(cols)]
df.columns = cols

# Clean junk rows
df["PLAYER"] = df["PLAYER"].astype(str).str.strip()
df = df[~df["PLAYER"].str.contains("First page|Previous page|Next page|Last page|Live|STARTER|Best lines", case=False, na=False)]
df = df[df["GAME"].astype(str).str.contains(" at ", na=False)].copy()

# ---------- 1) Parse away/home tokens from GAME ----------
def parse_game_tokens(g):
    g = str(g).strip().upper()
    if " AT " not in g:
        return (np.nan, np.nan)
    away, home = [x.strip() for x in g.split(" AT ", 1)]
    return away, home

df[["away_tok","home_tok"]] = df["GAME"].apply(lambda x: pd.Series(parse_game_tokens(x)))

# ---------- 2) ESPN team lookup ----------
TEAM_SEARCH = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams"

def espn_team_search(q):
    r = requests.get(TEAM_SEARCH, params={"search": q}, timeout=30)
    r.raise_for_status()
    j = r.json()
    out = []
    for it in j.get("sports",[{}])[0].get("leagues",[{}])[0].get("teams",[]):
        t = it.get("team", {})
        out.append({
            "id": t.get("id"),
            "displayName": t.get("displayName"),
            "abbreviation": t.get("abbreviation"),
            "shortDisplayName": t.get("shortDisplayName"),
        })
    return pd.DataFrame(out)

# Common alias map (expand any time we see misses)
ALIAS = {
    "GONZ":"GONZAGA",
    "SMC":"SAINT MARYS",
    "MISS":"OLE MISS",
    "TXAM":"TEXAS A&M",
    "NCST":"NC STATE",
    "GTWN":"GEORGETOWN",
    "WVU":"WEST VIRGINIA",
}

def resolve_team_id(tok):
    if pd.isna(tok):
        return np.nan
    tok = str(tok).strip().upper()
    q = ALIAS.get(tok, tok)

    # Search ESPN
    try:
        d = espn_team_search(q)
        if d.empty:
            return np.nan

        # Prefer exact abbreviation match first
        abbr = d[d["abbreviation"].astype(str).str.upper().eq(tok)]
        if len(abbr) > 0:
            return int(abbr.iloc[0]["id"])

        # Otherwise take first result
        return int(d.iloc[0]["id"])
    except:
        return np.nan

# Resolve ids
df["away_id"] = df["away_tok"].apply(resolve_team_id)
df["home_id"] = df["home_tok"].apply(resolve_team_id)

print("Rows:", len(df))
print("Missing away_id:", int(df["away_id"].isna().sum()))
print("Missing home_id:", int(df["home_id"].isna().sum()))

display(df[["GAME","away_tok","home_tok","away_id","home_id"]].drop_duplicates().head(20))

Rows: 96
Missing away_id: 0
Missing home_id: 0


,GAME,away_tok,home_tok,away_id,home_id
1,GONZ at SMC,GONZ,SMC,44,44
34,ARK at FLA,ARK,FLA,8,57
35,VT at UNC,VT,UNC,44,44
37,MISS at AUB,MISS,AUB,44,2


In [ ]:
import pandas as pd
import numpy as np
import re, requests

TEAM_SEARCH = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams"

def espn_team_search(q):
    r = requests.get(TEAM_SEARCH, params={"search": q}, timeout=30)
    r.raise_for_status()
    j = r.json()
    out = []
    for it in j.get("sports",[{}])[0].get("leagues",[{}])[0].get("teams",[]):
        t = it.get("team", {})
        out.append({
            "id": int(t.get("id")) if t.get("id") is not None else None,
            "displayName": t.get("displayName",""),
            "shortDisplayName": t.get("shortDisplayName",""),
            "abbreviation": t.get("abbreviation",""),
        })
    return pd.DataFrame(out)

def norm(s):
    return re.sub(r"[^A-Z0-9]+","", str(s).upper())

# Hard overrides for common sheet tokens -> correct ESPN team id
HARD_TEAM_ID = {
    "GONZ": 2250,   # Gonzaga
    "SMC": 2608,    # Saint Mary's
    "VT": 259,      # Virginia Tech
    "UNC": 153,     # North Carolina
    "MISS": 145,    # Ole Miss
    "AUB": 2,       # Auburn
    "ARK": 8,       # Arkansas
    "FLA": 57,      # Florida
    "UCLA": 26,     # UCLA
    "MINN": 135,    # Minnesota (if it shows up later)
    "WIS": 275,     # Wisconsin
    "WASH": 264,    # Washington
    "ND": 87,       # Notre Dame
    "KU": 2305,     # Kansas
    "ARIZ": 12,     # Arizona
    "TXAM": 245,    # Texas A&M
    "TEX": 251,     # Texas
    "GTWN": 46,     # Georgetown
    "XAV": 2752,    # Xavier
}

# Alias expansions help search when token isn't a real abbr
ALIAS = {
    "GONZ":"GONZAGA",
    "SMC":"SAINT MARYS",
    "MISS":"OLE MISS",
    "TXAM":"TEXAS A&M",
    "GTWN":"GEORGETOWN",
    "UVA":"VIRGINIA",
    "NEB":"NEBRASKA",
}

def best_team_id_for_token(tok):
    tok = str(tok).strip().upper()
    if tok in HARD_TEAM_ID:
        return HARD_TEAM_ID[tok], 999  # score

    q = ALIAS.get(tok, tok)
    d = espn_team_search(q)
    if d.empty:
        return np.nan, -1

    tokN = norm(tok)
    qN = norm(q)

    best = (np.nan, -1)
    for _, r in d.iterrows():
        ab = norm(r["abbreviation"])
        dn = norm(r["displayName"])
        sd = norm(r["shortDisplayName"])

        score = 0
        # Strong signals
        if ab == tokN: score += 10
        if sd == tokN: score += 8
        if tokN and tokN in dn: score += 6
        if qN and qN in dn: score += 6

        # Weaker signals
        if dn.startswith(tokN): score += 3
        if dn.startswith(qN): score += 3

        # If token is short (abbr-like) but doesn't match abbreviation at all, penalize
        if len(tok) <= 4 and tokN not in ab:
            score -= 2

        if score > best[1]:
            best = (int(r["id"]), score)

    # Require minimum confidence
    if best[1] < 5:
        return np.nan, best[1]
    return best

# --- rebuild ids + diagnostics ---
tokens = pd.unique(pd.concat([df["away_tok"], df["home_tok"]], ignore_index=True))
rows = []
for t in tokens:
    tid, sc = best_team_id_for_token(t)
    rows.append({"token": t, "team_id": tid, "score": sc})
token_map = pd.DataFrame(rows)

df = df.drop(columns=[c for c in ["away_id","home_id"] if c in df.columns]).copy()
df = df.merge(token_map.rename(columns={"token":"away_tok","team_id":"away_id","score":"away_score"}), on="away_tok", how="left")
df = df.merge(token_map.rename(columns={"token":"home_tok","team_id":"home_id","score":"home_score"}), on="home_tok", how="left")

print("Missing away_id:", int(df["away_id"].isna().sum()))
print("Missing home_id:", int(df["home_id"].isna().sum()))

# show the matchups (unique) + scores
diag = df[["GAME","away_tok","home_tok","away_id","home_id","away_score","home_score"]].drop_duplicates().sort_values("GAME")
display(diag.head(30))

# if anything is still wrong-looking, surface it
sus = diag[(diag["away_id"]==44) | (diag["home_id"]==44) | diag["away_id"].isna() | diag["home_id"].isna()]
print("Suspicious mappings (id==44 or NaN):", len(sus))
if len(sus):
    display(sus)

Missing away_id: 0
Missing home_id: 0


,GAME,away_tok,home_tok,away_id,home_id,away_score,home_score
28,ARK at FLA,ARK,FLA,8,57,999,999
0,GONZ at SMC,GONZ,SMC,2250,2608,999,999
31,MISS at AUB,MISS,AUB,145,2,999,999
29,VT at UNC,VT,UNC,259,153,999,999


Suspicious mappings (id==44 or NaN): 0


In [ ]:
import pandas as pd
import numpy as np
import requests
import re

TEAM_STATS = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics"

def safe_get(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def flatten_team_stats(j):
    """
    Returns a DataFrame with columns: category, name, displayName, value
    """
    cats = j.get("results", {}).get("stats", {}).get("categories", [])
    rows = []
    for c in cats:
        cat_name = c.get("name")
        for s in c.get("stats", []):
            rows.append({
                "category": cat_name,
                "name": s.get("name"),
                "displayName": s.get("displayName"),
                "value": s.get("value"),
            })
    return pd.DataFrame(rows)

def pick_def_ppg_allowed(stats_df):
    """
    ESPN schema varies. We try multiple patterns for opponent points / points allowed.
    We return np.nan if not found.
    """
    if stats_df is None or stats_df.empty:
        return np.nan

    # Candidate filters by name/displayName
    df = stats_df.copy()
    df["name_u"] = df["name"].astype(str).str.upper()
    df["disp_u"] = df["displayName"].astype(str).str.upper()

    # Common patterns across ESPN team stats:
    # - "oppPointsPerGame" / "opponentPointsPerGame"
    # - displayName includes "Opponent Points Per Game"
    # - "pointsAllowed" with "Per Game" in displayName
    candidates = df[
        df["name_u"].str.contains("OPP") & df["name_u"].str.contains("POINT") & df["name_u"].str.contains("AVG|PERGAME|PER_GAME", regex=True)
        | df["disp_u"].str.contains("OPPONENT") & df["disp_u"].str.contains("POINT") & df["disp_u"].str.contains("PER GAME")
        | df["disp_u"].str.contains("POINTS ALLOWED") & df["disp_u"].str.contains("PER GAME")
        | df["name_u"].str.contains("POINT") & df["name_u"].str.contains("ALLOWED") & df["name_u"].str.contains("AVG|PERGAME|PER_GAME", regex=True)
    ]

    # If nothing matched, try a looser search
    if candidates.empty:
        candidates = df[
            (df["disp_u"].str.contains("OPP") | df["disp_u"].str.contains("OPPONENT") | df["disp_u"].str.contains("ALLOWED"))
            & df["disp_u"].str.contains("POINT")
            & (df["disp_u"].str.contains("PER GAME") | df["disp_u"].str.contains("AVG"))
        ]

    if candidates.empty:
        return np.nan

    # Choose first non-null numeric value
    for v in candidates["value"].tolist():
        try:
            vv = float(v)
            if np.isfinite(vv):
                return vv
        except:
            continue

    return np.nan

# --- Pull stats for each unique team_id we need (OPPONENT teams = home_id for away players, away_id for home players)
# In your sheet, GAME is formatted "AWAY at HOME", and PLAYER belongs to the away team listed? Not guaranteed.
# So we take a simple proxy: opponent = home_id when token says "X at Y" (away plays at home team).
# If that assumption is wrong for some rows, we can add a TEAM column later. For now: opponent=home_id.
unique_team_ids = pd.unique(df["home_id"])
team_def = []

for tid in unique_team_ids:
    j = safe_get(TEAM_STATS.format(team_id=int(tid)))
    s = flatten_team_stats(j)
    opp_def_ppg = pick_def_ppg_allowed(s)
    team_def.append({"team_id": int(tid), "OPP_DEF_PPG_ALLOWED": opp_def_ppg})

team_def_df = pd.DataFrame(team_def)
print("Team defense rows:", len(team_def_df))
display(team_def_df.head(10))

# --- Join opponent defense proxy onto df ---
df = df.merge(team_def_df, left_on="home_id", right_on="team_id", how="left").drop(columns=["team_id"])

# --- Build context adjustment ---
# Interpretation:
# - Lower opponent points allowed => tougher defense => reduce model probability slightly for overs, increase for unders.
# - Higher points allowed => softer defense => increase for overs, reduce for unders.
#
# We cap adjustment so it can't overpower your prop trend model.
#
# Define baseline for points allowed in NCAAB (rough central tendency). Tune later if you want.
BASE_DEF = 72.0

# Scale: 1 point away from baseline = 0.25% probability shift (0.0025)
SCALE = 0.0025

# Hard cap: +/- 2.5 percentage points
CAP = 0.025

def is_under(outcome):
    return str(outcome).strip().lower().startswith("u")

def ctx_adjust(row):
    ppg = row.get("OPP_DEF_PPG_ALLOWED", np.nan)
    if pd.isna(ppg):
        return 0.0
    delta = (ppg - BASE_DEF) * SCALE  # softer defense -> positive
    delta = float(np.clip(delta, -CAP, CAP))
    # For unders: invert sign
    if is_under(row.get("OUTCOME","")):
        delta *= -1
    return delta

df["ESPN_OPP_DEF_PPG_PROXY"] = df["OPP_DEF_PPG_ALLOWED"]
df["ADJ_CTX"] = df.apply(ctx_adjust, axis=1)

# --- Apply to model prob (use your existing MODEL_PROB if present) ---
if "MODEL_PROB" not in df.columns:
    raise ValueError("MODEL_PROB not found in df. Run your base model probability cell first.")

df["MODEL_PROB_CTX"] = np.clip(df["MODEL_PROB"] + df["ADJ_CTX"], 0.01, 0.99)

# --- Book prob + Edge (if AMERICAN_ODDS exists) ---
def american_to_prob(o):
    if pd.isna(o): return np.nan
    o = float(o)
    if o < 0:
        return (-o) / ((-o) + 100.0)
    return 100.0 / (o + 100.0)

if "AMERICAN_ODDS" in df.columns:
    df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
    df["EDGE_VS_BOOK"] = df["MODEL_PROB_CTX"] - df["BOOK_PROB"]
else:
    df["BOOK_PROB"] = np.nan
    df["EDGE_VS_BOOK"] = np.nan

# --- Output Top 10 Most Likely (uniform) ---
top10 = (df.dropna(subset=["MODEL_PROB_CTX"])
           .sort_values("MODEL_PROB_CTX", ascending=False)
           .head(10)
           .copy())
top10.insert(0, "RANK", range(1, len(top10)+1))

cols = ["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","BOOK_PROB","MODEL_PROB","ESPN_OPP_DEF_PPG_PROXY","ADJ_CTX","MODEL_PROB_CTX","EDGE_VS_BOOK"]
cols = [c for c in cols if c in top10.columns]

display(top10[cols])

# Save full + top10
top10.to_csv("hit_sheet_top10_ncaam_ctx.csv", index=False)
df.to_csv("prop_engine_full_ncaam_ctx.csv", index=False)
print("Saved: hit_sheet_top10_ncaam_ctx.csv")
print("Saved: prop_engine_full_ncaam_ctx.csv")

Team defense rows: 4


,team_id,OPP_DEF_PPG_ALLOWED
0,2608,NaN
1,57,NaN
2,153,NaN
3,2,NaN


ValueError: MODEL_PROB not found in df. Run your base model probability cell first.

In [ ]:
import pandas as pd
import numpy as np
import re

# --- helpers ---
def to_prob(x):
    """Accepts '56.5%', '0.565', 56.5, '--', NaN -> float in [0,1] or NaN"""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in ["--", "-", "—", "–", "None", "nan", ""]:
        return np.nan
    # remove commas
    s = s.replace(",", "")
    # percent
    if s.endswith("%"):
        try:
            return float(s[:-1]) / 100.0
        except:
            return np.nan
    # numeric
    try:
        v = float(s)
        # if looks like 0-1, keep; if 1-100, treat as percent
        if v > 1.0 and v <= 100.0:
            return v / 100.0
        return v
    except:
        return np.nan

def clamp01(x):
    if pd.isna(x): return np.nan
    return float(np.clip(x, 0.01, 0.99))

# --- normalize key columns ---
# If your df is already loaded, this uses it; otherwise load it:
# df = pd.read_csv("NCAA night v2.csv")

df.columns = [str(c).strip() for c in df.columns]

# Common rename map (safe)
rename_map = {
    "IM PROB %": "IM_PROB_PCT",
    "HM/AW": "HM_AW",
    "DVP L5": "DVP_L5",
    "DVP L10": "DVP_L10",
    "DVP L10.1": "DVP_L10_2",
    "DVP L10_2": "DVP_L10_2",
    "DVP HM/AW": "DVP_HM_AW",
    "DVP 2025": "DVP_2025",
    "2025": "SEASON",
    "L5": "L5",
    "L10": "L10",
    "H2H": "H2H",
}
df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns}, inplace=True)

# Ensure required columns exist
REQ = ["PLAYER","GAME","PROP","OUTCOME"]
for c in REQ:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}. Found: {df.columns.tolist()}")

# Convert trend columns to probabilities
for c in ["L5","L10","SEASON","HM_AW","H2H","IM_PROB_PCT","DVP_L5","DVP_L10","DVP_L10_2","DVP_HM_AW","DVP_2025"]:
    if c in df.columns:
        df[c] = df[c].apply(to_prob)

# Build P_TREND (avoid overfitting L5)
# weights: L10 0.35, SEASON 0.30, HM/AW 0.15, L5 0.10, H2H 0.10 (only if available)
w = {"L10":0.35, "SEASON":0.30, "HM_AW":0.15, "L5":0.10, "H2H":0.10}

def weighted_mean(row, cols_w):
    num = 0.0
    den = 0.0
    for col, wt in cols_w.items():
        if col in row.index:
            v = row[col]
            if pd.notna(v):
                num += wt * float(v)
                den += wt
    return num/den if den > 0 else np.nan

df["P_TREND"] = df.apply(lambda r: weighted_mean(r, w), axis=1)

# DVP blend (cap it so it can't overpower)
DVP_COLS = [c for c in ["DVP_L5","DVP_L10","DVP_L10_2","DVP_HM_AW","DVP_2025"] if c in df.columns]
df["P_DVP_RAW"] = df[DVP_COLS].mean(axis=1, skipna=True) if len(DVP_COLS) else np.nan

# Convert DVP probability to adjustment around 0.5, capped
DVP_CAP = 0.07  # max +/- 7 percentage points
df["ADJ_DVP"] = (df["P_DVP_RAW"] - 0.5).clip(-DVP_CAP, DVP_CAP).fillna(0.0)

# Base model probability
df["MODEL_PROB"] = (df["P_TREND"] + df["ADJ_DVP"]).apply(clamp01)

print("Built MODEL_PROB. Preview:")
display(df[["PLAYER","GAME","PROP","OUTCOME","P_TREND","ADJ_DVP","MODEL_PROB"]].head(10))

Built MODEL_PROB. Preview:


,PLAYER,GAME,PROP,OUTCOME,P_TREND,ADJ_DVP,MODEL_PROB
0,Emmanuel Innocenti,GONZ at SMC,Points,o7.5,0.528071,0.07000,0.598071
1,Graham Ike,GONZ at SMC,Points,o19.5,0.768500,-0.07000,0.698500
2,Graham Ike,GONZ at SMC,Points,u21.5,0.520786,0.07000,0.590786
3,Graham Ike,GONZ at SMC,Points + Rebounds + Assists,u31.5,0.503000,0.07000,0.573000
4,Emmanuel Innocenti,GONZ at SMC,Points + Rebounds,o12.5,0.449500,0.00175,0.451250
5,Emmanuel Innocenti,GONZ at SMC,Rebounds,u4.5,0.574643,0.07000,0.644643
6,Adam Miller,GONZ at SMC,Points,o7.5,0.472429,0.07000,0.542429
7,Graham Ike,GONZ at SMC,Assists,o2.5,0.478643,-0.07000,0.408643
8,Graham Ike,GONZ at SMC,Three Pointers Made,o0.5,0.545500,-0.06625,0.479250
9,Graham Ike,GONZ at SMC,Rebounds,u7.5,0.470857,0.07000,0.540857


In [ ]:
import requests
import numpy as np
import pandas as pd

CORE_TEAM_STATS = "https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/seasons/2026/types/2/teams/{team_id}/statistics"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def follow_ref(j):
    if isinstance(j, dict) and "$ref" in j:
        return get_json(j["$ref"])
    return j

def extract_all_stats_from_core(j):
    """
    Returns rows: category, name, displayName, value
    Core stats object has categories[].stats[] similar to site endpoint, but includes more keys.
    """
    j = follow_ref(j)
    cats = j.get("splits", {}).get("categories", []) or j.get("categories", [])
    rows = []
    for c in cats:
        cat = c.get("name") or c.get("displayName")
        for s in c.get("stats", []):
            rows.append({
                "category": cat,
                "name": s.get("name"),
                "displayName": s.get("displayName"),
                "value": s.get("value"),
            })
    return pd.DataFrame(rows)

def pick_points_allowed_per_game(stats_df):
    """
    Try common variants for points allowed / points against per game.
    """
    if stats_df is None or stats_df.empty:
        return np.nan

    df = stats_df.copy()
    df["name_u"] = df["name"].astype(str).str.upper()
    df["disp_u"] = df["displayName"].astype(str).str.upper()

    # Targets (broad):
    # - AVG POINTS AGAINST
    # - POINTS AGAINST PER GAME
    # - POINTS ALLOWED PER GAME
    # - OPPONENT POINTS PER GAME
    mask = (
        (df["disp_u"].str.contains("POINT") & (df["disp_u"].str.contains("AGAINST") | df["disp_u"].str.contains("ALLOWED")) & df["disp_u"].str.contains("PER GAME"))
        | (df["disp_u"].str.contains("OPPONENT") & df["disp_u"].str.contains("POINT") & df["disp_u"].str.contains("PER GAME"))
        | (df["name_u"].str.contains("POINT") & (df["name_u"].str.contains("AGAINST") | df["name_u"].str.contains("ALLOWED") | df["name_u"].str.contains("OPP")))
    )
    cand = df[mask].copy()

    # Prefer rows that clearly indicate per game average
    if not cand.empty:
        cand["score"] = 0
        cand.loc[cand["disp_u"].str.contains("PER GAME"), "score"] += 2
        cand.loc[cand["disp_u"].str.contains("AVG"), "score"] += 1
        cand.loc[cand["name_u"].str.contains("AVG|PERGAME|PER_GAME", regex=True), "score"] += 1
        cand = cand.sort_values("score", ascending=False)

        for v in cand["value"].tolist():
            try:
                vv = float(v)
                if np.isfinite(vv) and vv > 0:
                    return vv
            except:
                continue

    return np.nan

# --- Make sure you have team ids on df (home_id used as opponent proxy for "AWAY at HOME") ---
for c in ["home_id"]:
    if c not in df.columns:
        raise ValueError("home_id not found in df. Run your TEAM-ID resolution cell first.")

unique_home_ids = sorted(pd.unique(df["home_id"].dropna()).astype(int).tolist())
team_def = []

for tid in unique_home_ids:
    core = get_json(CORE_TEAM_STATS.format(team_id=tid))
    stats_df = extract_all_stats_from_core(core)
    ppg_allowed = pick_points_allowed_per_game(stats_df)
    team_def.append({"team_id": tid, "DEF_PPG_ALLOWED": ppg_allowed})

team_def_df = pd.DataFrame(team_def)
print("Team defense rows:", len(team_def_df))
display(team_def_df)

# --- Join + apply ADJ_CTX ---
df = df.merge(team_def_df, left_on="home_id", right_on="team_id", how="left").drop(columns=["team_id"])
df["ESPN_OPP_DEF_PPG_PROXY"] = df["DEF_PPG_ALLOWED"]

BASE_DEF = 72.0
SCALE = 0.0025
CAP = 0.025

def is_under(outcome):
    return str(outcome).strip().lower().startswith("u")

def ctx_adjust(ppg_allowed, outcome):
    if pd.isna(ppg_allowed):
        return 0.0
    delta = (float(ppg_allowed) - BASE_DEF) * SCALE
    delta = float(np.clip(delta, -CAP, CAP))
    if is_under(outcome):
        delta *= -1
    return delta

df["ADJ_CTX"] = df.apply(lambda r: ctx_adjust(r["DEF_PPG_ALLOWED"], r["OUTCOME"]), axis=1)
df["MODEL_PROB_CTX"] = np.clip(df["MODEL_PROB"] + df["ADJ_CTX"], 0.01, 0.99)

# Edge vs book (if american odds exists)
def american_to_prob(o):
    if pd.isna(o): return np.nan
    o = float(o)
    if o < 0:
        return (-o) / ((-o) + 100.0)
    return 100.0 / (o + 100.0)

if "AMERICAN_ODDS" in df.columns:
    df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
    df["EDGE_VS_BOOK"] = df["MODEL_PROB_CTX"] - df["BOOK_PROB"]
else:
    df["BOOK_PROB"] = np.nan
    df["EDGE_VS_BOOK"] = np.nan

# --- Top 10 Most Likely (uniform, now with context when available) ---
top10 = df.sort_values("MODEL_PROB_CTX", ascending=False).head(10).copy()
top10.insert(0, "RANK", range(1, len(top10)+1))

cols = ["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","BOOK_PROB","MODEL_PROB","ESPN_OPP_DEF_PPG_PROXY","ADJ_CTX","MODEL_PROB_CTX","EDGE_VS_BOOK"]
cols = [c for c in cols if c in top10.columns]
display(top10[cols])

top10.to_csv("hit_sheet_top10_ncaam_ctx.csv", index=False)
df.to_csv("prop_engine_full_ncaam_ctx.csv", index=False)
print("Saved: hit_sheet_top10_ncaam_ctx.csv")
print("Saved: prop_engine_full_ncaam_ctx.csv")

Team defense rows: 4


,team_id,DEF_PPG_ALLOWED
0,2,NaN
1,57,NaN
2,153,NaN
3,2608,NaN


,RANK,PLAYER,GAME,PROP,OUTCOME,BOOK_PROB,MODEL_PROB,ESPN_OPP_DEF_PPG_PROXY,ADJ_CTX,MODEL_PROB_CTX,EDGE_VS_BOOK
36,1,Amani Hansberry,VT at UNC,Points + Rebounds + Assists,u21.5,NaN,0.750667,NaN,0.0,0.750667,NaN
49,2,Seth Trimble,VT at UNC,Assists,u2.5,NaN,0.734286,NaN,0.0,0.734286,NaN
43,3,Jailen Bedford,VT at UNC,Points,o10.5,NaN,0.717333,NaN,0.0,0.717333,NaN
32,4,Tahaad Pettiford,MISS at AUB,Rebounds,o2.5,NaN,0.715786,NaN,0.0,0.715786,NaN
1,5,Graham Ike,GONZ at SMC,Points,o19.5,NaN,0.698500,NaN,0.0,0.698500,NaN
28,6,Rueben Chinyelu,ARK at FLA,Rebounds,u11.5,NaN,0.661929,NaN,0.0,0.661929,NaN
48,7,Meleek Thomas,ARK at FLA,Rebounds,u4.5,NaN,0.661667,NaN,0.0,0.661667,NaN
52,8,Henri Veesaar,VT at UNC,Points,u16.5,NaN,0.660750,NaN,0.0,0.660750,NaN
34,9,Keyshawn Hall,MISS at AUB,Three Pointers Made,u1.5,NaN,0.655714,NaN,0.0,0.655714,NaN
40,10,Luka Bogavac,VT at UNC,Points,u12.5,NaN,0.653750,NaN,0.0,0.653750,NaN


Saved: hit_sheet_top10_ncaam_ctx.csv
Saved: prop_engine_full_ncaam_ctx.csv


In [ ]:
import pandas as pd
import numpy as np
import re

def parse_american_odds(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    # common patterns: "BET\n-150", "-150", "+115"
    m = re.search(r'([+-]\d{2,5})', s.replace(",", ""))
    if m:
        try:
            return float(m.group(1))
        except:
            return np.nan
    return np.nan

def american_to_prob(o):
    if pd.isna(o): return np.nan
    o = float(o)
    if o < 0:
        return (-o) / ((-o) + 100.0)
    return 100.0 / (o + 100.0)

# build AMERICAN_ODDS if missing/empty
if "AMERICAN_ODDS" not in df.columns or df["AMERICAN_ODDS"].isna().all():
    src = "ODDS" if "ODDS" in df.columns else None
    if src is None:
        raise ValueError("Need either AMERICAN_ODDS or ODDS column to derive odds.")
    df["AMERICAN_ODDS"] = df[src].apply(parse_american_odds)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
print("AMERICAN_ODDS non-null:", int(df["AMERICAN_ODDS"].notna().sum()), "of", len(df))
display(df[["PLAYER","GAME","PROP","OUTCOME","ODDS","AMERICAN_ODDS","BOOK_PROB"]].head(12))

AMERICAN_ODDS non-null: 96 of 96


,PLAYER,GAME,PROP,OUTCOME,ODDS,AMERICAN_ODDS,BOOK_PROB
0,Emmanuel Innocenti,GONZ at SMC,Points,o7.5,BET\n-130,-130.0,0.565217
1,Graham Ike,GONZ at SMC,Points,o19.5,BET\n-166,-166.0,0.624060
2,Graham Ike,GONZ at SMC,Points,u21.5,BET\n-115,-115.0,0.534884
3,Graham Ike,GONZ at SMC,Points + Rebounds + Assists,u31.5,BET\n-110,-110.0,0.523810
4,Emmanuel Innocenti,GONZ at SMC,Points + Rebounds,o12.5,BET\n-110,-110.0,0.523810
5,Emmanuel Innocenti,GONZ at SMC,Rebounds,u4.5,BET\n-150,-150.0,0.600000
6,Adam Miller,GONZ at SMC,Points,o7.5,BET\n-105,-105.0,0.512195
7,Graham Ike,GONZ at SMC,Assists,o2.5,BET\n+115,115.0,0.465116
8,Graham Ike,GONZ at SMC,Three Pointers Made,o0.5,BET\n-179,-179.0,0.641577
9,Graham Ike,GONZ at SMC,Rebounds,u7.5,BET\n+105,105.0,0.487805


In [ ]:
import requests
import pandas as pd
import numpy as np

CORE_TEAM_STATS = "https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/seasons/2026/types/2/teams/{team_id}/statistics"

def get_json(url):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

def follow_ref(j):
    # ESPN core often returns {"$ref": ".../statistics/0?..."}
    if isinstance(j, dict) and "$ref" in j:
        return get_json(j["$ref"])
    return j

def flatten_stat_objects(obj, out):
    """
    Walks any dict/list tree and collects objects that look like a stat:
    keys include (name/displayName/value) OR (abbreviation/value) etc.
    """
    if isinstance(obj, dict):
        # stat-like object
        if ("value" in obj) and (("name" in obj) or ("displayName" in obj) or ("abbreviation" in obj)):
            out.append({
                "name": obj.get("name"),
                "displayName": obj.get("displayName"),
                "abbreviation": obj.get("abbreviation"),
                "value": obj.get("value"),
            })
        for v in obj.values():
            flatten_stat_objects(v, out)
    elif isinstance(obj, list):
        for it in obj:
            flatten_stat_objects(it, out)

def extract_points_allowed_per_game(flat_df):
    if flat_df is None or flat_df.empty:
        return np.nan
    d = flat_df.copy()
    for c in ["name","displayName","abbreviation"]:
        d[c] = d[c].astype(str).str.upper()

    # score candidates: prefer explicit per game + against/allowed
    d["score"] = 0
    d.loc[d["displayName"].str.contains("PER GAME"), "score"] += 2
    d.loc[d["displayName"].str.contains("AGAINST|ALLOWED|OPP", regex=True), "score"] += 2
    d.loc[d["displayName"].str.contains("POINT"), "score"] += 1
    d.loc[d["name"].str.contains("AGAINST|ALLOWED|OPP|PA", regex=True), "score"] += 1
    d.loc[d["name"].str.contains("POINT", regex=True), "score"] += 1

    cand = d[d["score"] >= 4].sort_values("score", ascending=False).copy()

    # print top candidates for sanity
    print("Top candidate stats (first 25):")
    display(cand[["name","displayName","abbreviation","value","score"]].head(25))

    # pick first numeric value that makes sense
    for v in cand["value"].tolist():
        try:
            vv = float(v)
            if np.isfinite(vv) and 40 <= vv <= 110:
                return vv
        except:
            continue
    return np.nan

# ---- Build unique team list from df home_id (opponent) ----
if "home_id" not in df.columns:
    raise ValueError("home_id not in df. Run your TEAM-ID resolution cell first.")

team_ids = sorted(pd.unique(df["home_id"].dropna()).astype(int).tolist())
print("Teams to pull:", team_ids)

team_def = []
for tid in team_ids:
    raw = get_json(CORE_TEAM_STATS.format(team_id=tid))
    j = follow_ref(raw)

    flat = []
    flatten_stat_objects(j, flat)
    flat_df = pd.DataFrame(flat).drop_duplicates()

    print("\n==============================")
    print("TEAM_ID:", tid, "| flat_stat_rows:", len(flat_df))
    def_ppg = extract_points_allowed_per_game(flat_df)

    team_def.append({"team_id": tid, "DEF_PPG_ALLOWED": def_ppg})

team_def_df = pd.DataFrame(team_def)
print("\nFINAL TEAM DEF TABLE:")
display(team_def_df)

Teams to pull: [2, 57, 153, 2608]

TEAM_ID: 2 | flat_stat_rows: 77
Top candidate stats (first 25):


,name,displayName,abbreviation,value,score
59,AVGPOINTS,POINTS PER GAME,PTS,83.500000,4
72,AVGTWOPOINTFIELDGOALSMADE,2-POINT FIELD GOALS MADE PER GAME,2PM,19.714285,4
73,AVGTWOPOINTFIELDGOALSATTEMPTED,2-POINT FIELD GOALS ATTEMPTED PER GAME,2PA,37.142857,4



TEAM_ID: 57 | flat_stat_rows: 77
Top candidate stats (first 25):


,name,displayName,abbreviation,value,score
59,AVGPOINTS,POINTS PER GAME,PTS,86.285710,4
72,AVGTWOPOINTFIELDGOALSMADE,2-POINT FIELD GOALS MADE PER GAME,2PM,23.035715,4
73,AVGTWOPOINTFIELDGOALSATTEMPTED,2-POINT FIELD GOALS ATTEMPTED PER GAME,2PA,39.428570,4



TEAM_ID: 153 | flat_stat_rows: 77
Top candidate stats (first 25):


,name,displayName,abbreviation,value,score
59,AVGPOINTS,POINTS PER GAME,PTS,80.607140,4
72,AVGTWOPOINTFIELDGOALSMADE,2-POINT FIELD GOALS MADE PER GAME,2PM,19.857143,4
73,AVGTWOPOINTFIELDGOALSATTEMPTED,2-POINT FIELD GOALS ATTEMPTED PER GAME,2PA,34.678570,4



TEAM_ID: 2608 | flat_stat_rows: 77
Top candidate stats (first 25):


,name,displayName,abbreviation,value,score
59,AVGPOINTS,POINTS PER GAME,PTS,78.700000,4
72,AVGTWOPOINTFIELDGOALSMADE,2-POINT FIELD GOALS MADE PER GAME,2PM,18.900000,4
73,AVGTWOPOINTFIELDGOALSATTEMPTED,2-POINT FIELD GOALS ATTEMPTED PER GAME,2PA,37.033333,4



FINAL TEAM DEF TABLE:


,team_id,DEF_PPG_ALLOWED
0,2,83.50000
1,57,86.28571
2,153,80.60714
3,2608,78.70000


In [ ]:
import numpy as np
import pandas as pd

# --- Sanity checks ---
if "team_def_df" not in globals():
    raise ValueError("team_def_df not found. Run Cell B first.")

print("team_def_df columns:", team_def_df.columns.tolist())
display(team_def_df.head())

# --- Find the defense column automatically ---
def_col_candidates = [c for c in team_def_df.columns if c.upper() in [
    "DEF_PPG_ALLOWED", "OPP_PPG_ALLOWED", "POINTS_ALLOWED", "PPG_ALLOWED", "DEF_PPG"
]]

# fallback: any numeric column besides team_id
if not def_col_candidates:
    numeric_cols = [c for c in team_def_df.columns
                    if c != "team_id" and pd.api.types.is_numeric_dtype(team_def_df[c])]
    if len(numeric_cols) == 0:
        raise ValueError("No numeric defense column found in team_def_df.")
    def_col = numeric_cols[0]
else:
    def_col = def_col_candidates[0]

print("Using defense column:", def_col)

# --- Merge into main df ---
if "home_id" not in df.columns:
    raise ValueError("home_id not in df. Run the TEAM-ID resolution step first.")

df = df.merge(team_def_df[["team_id", def_col]], left_on="home_id", right_on="team_id", how="left")
df.drop(columns=["team_id"], inplace=True)

df["ESPN_OPP_DEF_PPG_PROXY"] = df[def_col]

# --- Context adjustment ---
BASE_DEF = 72.0
SCALE = 0.0025
CAP = 0.025

def is_under(outcome):
    return str(outcome).strip().lower().startswith("u")

def ctx_adjust(ppg_allowed, outcome):
    if pd.isna(ppg_allowed):
        return 0.0
    delta = (float(ppg_allowed) - BASE_DEF) * SCALE
    delta = float(np.clip(delta, -CAP, CAP))
    if is_under(outcome):
        delta *= -1
    return delta

df["ADJ_CTX"] = df.apply(lambda r: ctx_adjust(r["ESPN_OPP_DEF_PPG_PROXY"], r["OUTCOME"]), axis=1)

# --- Apply to model prob ---
if "MODEL_PROB" not in df.columns:
    raise ValueError("MODEL_PROB not found. Run your base model probability cell first.")

df["MODEL_PROB_CTX"] = np.clip(df["MODEL_PROB"] + df["ADJ_CTX"], 0.01, 0.99)

# --- Edge vs book ---
if "BOOK_PROB" not in df.columns:
    raise ValueError("BOOK_PROB not found. Run the odds parsing cell first.")

df["EDGE_VS_BOOK"] = df["MODEL_PROB_CTX"] - df["BOOK_PROB"]

# --- Top 10 ---
top10 = df.sort_values("MODEL_PROB_CTX", ascending=False).head(10).copy()
top10.insert(0, "RANK", range(1, len(top10)+1))

display(top10[[
    "RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "BOOK_PROB","MODEL_PROB","ESPN_OPP_DEF_PPG_PROXY","ADJ_CTX",
    "MODEL_PROB_CTX","EDGE_VS_BOOK"
]])

top10.to_csv("hit_sheet_top10_ncaam_ctx.csv", index=False)
df.to_csv("prop_engine_full_ncaam_ctx.csv", index=False)

print("Saved: hit_sheet_top10_ncaam_ctx.csv")
print("Saved: prop_engine_full_ncaam_ctx.csv")

team_def_df columns: ['team_id', 'DEF_PPG_ALLOWED']


,team_id,DEF_PPG_ALLOWED
0,2,83.50000
1,57,86.28571
2,153,80.60714
3,2608,78.70000


Using defense column: DEF_PPG_ALLOWED


,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,BOOK_PROB,MODEL_PROB,ESPN_OPP_DEF_PPG_PROXY,ADJ_CTX,MODEL_PROB_CTX,EDGE_VS_BOOK
32,1,Tahaad Pettiford,MISS at AUB,Rebounds,o2.5,-200.0,0.666667,0.715786,83.50000,0.025000,0.740786,0.074119
43,2,Jailen Bedford,VT at UNC,Points,o10.5,-120.0,0.545455,0.717333,80.60714,0.021518,0.738851,0.193397
36,3,Amani Hansberry,VT at UNC,Points + Rebounds + Assists,u21.5,-120.0,0.545455,0.750667,80.60714,-0.021518,0.729149,0.183694
1,4,Graham Ike,GONZ at SMC,Points,o19.5,-166.0,0.624060,0.698500,78.70000,0.016750,0.715250,0.091190
49,5,Seth Trimble,VT at UNC,Assists,u2.5,-190.0,0.655172,0.734286,80.60714,-0.021518,0.712768,0.057595
39,6,Xaivian Lee,ARK at FLA,Points,o9.5,-168.0,0.626866,0.627250,86.28571,0.025000,0.652250,0.025384
52,7,Henri Veesaar,VT at UNC,Points,u16.5,102.0,0.495050,0.660750,80.60714,-0.021518,0.639232,0.144183
28,8,Rueben Chinyelu,ARK at FLA,Rebounds,u11.5,-115.0,0.534884,0.661929,86.28571,-0.025000,0.636929,0.102045
48,9,Meleek Thomas,ARK at FLA,Rebounds,u4.5,-108.0,0.519231,0.661667,86.28571,-0.025000,0.636667,0.117436
44,10,Jailen Bedford,VT at UNC,Points,o11.5,115.0,0.465116,0.612083,80.60714,0.021518,0.633601,0.168485


Saved: hit_sheet_top10_ncaam_ctx.csv
Saved: prop_engine_full_ncaam_ctx.csv


In [ ]:
def discord_hit_sheet_top10(df_top10):
    lines = []
    lines.append("NCAA PROP ENGINE — TOP 10 MOST LIKELY (CTX)")
    lines.append("—")
    for _, r in df_top10.iterrows():
        odds = f"({int(r.AMERICAN_ODDS)})" if pd.notna(r.AMERICAN_ODDS) else ""
        lines.append(
            f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n"
            f"{r.PROP} {r.OUTCOME} {odds}\n"
            f"Model: {r.MODEL_PROB_CTX*100:.1f}% | Edge: {r.EDGE_VS_BOOK*100:+.1f} pts | Opp DEF PPG: {r.ESPN_OPP_DEF_PPG_PROXY:.1f}"
        )
    return "\n".join(lines)

print(discord_hit_sheet_top10(top10))

NCAA PROP ENGINE — TOP 10 MOST LIKELY (CTX)
—
1) Tahaad Pettiford — MISS at AUB
Rebounds o2.5 (-200)
Model: 74.1% | Edge: +7.4 pts | Opp DEF PPG: 83.5
2) Jailen Bedford — VT at UNC
Points o10.5 (-120)
Model: 73.9% | Edge: +19.3 pts | Opp DEF PPG: 80.6
3) Amani Hansberry — VT at UNC
Points + Rebounds + Assists u21.5 (-120)
Model: 72.9% | Edge: +18.4 pts | Opp DEF PPG: 80.6
4) Graham Ike — GONZ at SMC
Points o19.5 (-166)
Model: 71.5% | Edge: +9.1 pts | Opp DEF PPG: 78.7
5) Seth Trimble — VT at UNC
Assists u2.5 (-190)
Model: 71.3% | Edge: +5.8 pts | Opp DEF PPG: 80.6
6) Xaivian Lee — ARK at FLA
Points o9.5 (-168)
Model: 65.2% | Edge: +2.5 pts | Opp DEF PPG: 86.3
7) Henri Veesaar — VT at UNC
Points u16.5 (102)
Model: 63.9% | Edge: +14.4 pts | Opp DEF PPG: 80.6
8) Rueben Chinyelu — ARK at FLA
Rebounds u11.5 (-115)
Model: 63.7% | Edge: +10.2 pts | Opp DEF PPG: 86.3
9) Meleek Thomas — ARK at FLA
Rebounds u4.5 (-108)
Model: 63.7% | Edge: +11.7 pts | Opp DEF PPG: 86.3
10) Jailen Bedford — VT at

In [ ]:
import requests, numpy as np, pandas as pd, re
from functools import lru_cache

ESPN_TEAM_STATS = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics"

def safe_get(d, path, default=None):
    cur = d
    for p in path:
        if isinstance(cur, dict) and p in cur:
            cur = cur[p]
        else:
            return default
    return cur

@lru_cache(maxsize=512)
def fetch_team_stats(team_id: int):
    url = ESPN_TEAM_STATS.format(team_id=int(team_id))
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    j = r.json()
    stats = []
    cats = safe_get(j, ["results","stats","categories"], []) or []
    for cat in cats:
        cat_name = cat.get("name")
        for s in cat.get("stats", []) or []:
            stats.append({
                "category": cat_name,
                "name": s.get("name"),
                "displayName": s.get("displayName"),
                "value": s.get("value")
            })
    return pd.DataFrame(stats)

def pick_stat(stats_df, candidates):
    """Return first non-null value matching any candidate stat name."""
    if stats_df is None or len(stats_df)==0:
        return np.nan
    for c in candidates:
        m = stats_df.loc[stats_df["name"].str.lower() == c.lower(), "value"]
        if len(m) and pd.notna(m.iloc[0]):
            return float(m.iloc[0])
    return np.nan

def compute_pace_proxy(stats_df):
    """
    Possessions proxy per game:
    poss ≈ FGA + 0.44*FTA - ORB + TO
    Uses team offensive per-game averages if available.
    """
    fga = pick_stat(stats_df, ["avgFieldGoalsAttempted"])
    fta = pick_stat(stats_df, ["avgFreeThrowsAttempted"])
    orb = pick_stat(stats_df, ["avgOffensiveRebounds"])
    to  = pick_stat(stats_df, ["avgTurnovers"])
    if any(pd.isna(x) for x in [fga, fta, orb, to]):
        return np.nan
    return float(fga + 0.44*fta - orb + to)

def get_def_context(team_id: int):
    """
    Tries multiple ESPN stat-name candidates because ESPN naming can vary.
    Returns:
      DEF_PPG_ALLOWED (points allowed per game)
      OPP_FG_PCT_ALLOWED
      OPP_3P_PCT_ALLOWED
      PACE_PROXY
    """
    s = fetch_team_stats(team_id)

    # Points allowed (common variants)
    def_ppg = pick_stat(s, [
        "avgPointsAgainst", "avgPointsAllowed", "avgOpponentPoints",
        "pointsAgainstPerGame", "avgPointsAgainstPerGame"
    ])

    # Opponent shooting allowed (common variants)
    opp_fg = pick_stat(s, [
        "opponentFieldGoalPct", "oppFieldGoalPct", "opponentFGPct", "oppFGPct"
    ])
    opp_3p = pick_stat(s, [
        "opponentThreePointFieldGoalPct", "oppThreePointFieldGoalPct",
        "opponent3PointFieldGoalPct", "opp3PointFieldGoalPct", "opponent3PFieldGoalPct"
    ])

    pace = compute_pace_proxy(s)

    return {
        "team_id": int(team_id),
        "DEF_PPG_ALLOWED": def_ppg,
        "OPP_FG_PCT_ALLOWED": opp_fg,
        "OPP_3P_PCT_ALLOWED": opp_3p,
        "PACE_PROXY": pace
    }

# ---------- Build opponent-context table from HOME team id (opponent) ----------
if "home_id" not in df.columns:
    raise ValueError("Need df['home_id'] as opponent team_id (AWAY at HOME).")

opp_ids = sorted(df["home_id"].dropna().astype(int).unique().tolist())

rows = []
for tid in opp_ids:
    try:
        rows.append(get_def_context(tid))
    except Exception as e:
        rows.append({"team_id": tid, "DEF_PPG_ALLOWED": np.nan, "OPP_FG_PCT_ALLOWED": np.nan, "OPP_3P_PCT_ALLOWED": np.nan, "PACE_PROXY": np.nan})

ctx_df = pd.DataFrame(rows)

# ---------- Merge into df ----------
df = df.merge(ctx_df, left_on="home_id", right_on="team_id", how="left").drop(columns=["team_id"])

# ---------- Convert context -> capped adjustment ----------
# Baselines (tweakable; keep stable + capped)
BASE_DEF_PPG = 72.0        # rough NCAAB baseline
BASE_PACE    = 70.0        # rough possessions proxy baseline (proxy units)
BASE_OPP_FG  = 43.0        # pct
BASE_OPP_3P  = 33.0        # pct

def context_adjust(row):
    prop = str(row.get("PROP","")).lower()
    out  = str(row.get("OUTCOME","")).lower()

    is_over = out.startswith("o")
    is_under = out.startswith("u")

    # If outcome not recognized, no context push
    if not (is_over or is_under):
        return 0.0

    # Pull values
    def_ppg = row.get("DEF_PPG_ALLOWED", np.nan)
    pace    = row.get("PACE_PROXY", np.nan)
    opp_fg  = row.get("OPP_FG_PCT_ALLOWED", np.nan)
    opp_3p  = row.get("OPP_3P_PCT_ALLOWED", np.nan)

    adj = 0.0

    # 1) Points allowed: high allowed -> helps overs; hurts unders
    if pd.notna(def_ppg):
        delta = (def_ppg - BASE_DEF_PPG) / 20.0   # scale
        delta = float(np.clip(delta, -0.02, 0.02))
        adj += delta if is_over else -delta

    # 2) Pace proxy: faster -> helps overs; hurts unders (counts more for points/ra/pra)
    if pd.notna(pace):
        delta = (pace - BASE_PACE) / 40.0
        delta = float(np.clip(delta, -0.015, 0.015))
        # boost impact for multi-cats
        mult = 1.2 if ("points" in prop and ("rebound" in prop or "assist" in prop)) else 1.0
        adj += mult*delta if is_over else -mult*delta

    # 3) Opp shooting allowed: matters most for points + threes
    if pd.notna(opp_fg):
        delta = (opp_fg - BASE_OPP_FG) / 50.0
        delta = float(np.clip(delta, -0.012, 0.012))
        if "point" in prop:
            adj += delta if is_over else -delta

    if pd.notna(opp_3p):
        delta = (opp_3p - BASE_OPP_3P) / 50.0
        delta = float(np.clip(delta, -0.012, 0.012))
        if "three" in prop:
            adj += delta if is_over else -delta

    # Final cap so context never dominates
    return float(np.clip(adj, -0.03, 0.03))

if "MODEL_PROB" not in df.columns:
    raise ValueError("MODEL_PROB not found. Run base model first.")

df["ADJ_CTX_V2"] = df.apply(context_adjust, axis=1)
df["MODEL_PROB_CTX_V2"] = np.clip(df["MODEL_PROB"] + df["ADJ_CTX_V2"], 0.01, 0.99)

# Recompute edge using context v2
if "BOOK_PROB" in df.columns:
    df["EDGE_VS_BOOK_V2"] = df["MODEL_PROB_CTX_V2"] - df["BOOK_PROB"]

print("Context v2 applied (non-null DEF_PPG):", int(df["DEF_PPG_ALLOWED"].notna().sum()), "of", len(df))
print("Context v2 applied (non-null PACE):", int(df["PACE_PROXY"].notna().sum()), "of", len(df))
print("Context v2 applied (non-null opp FG/3P):",
      int(df["OPP_FG_PCT_ALLOWED"].notna().sum()), "/", int(df["OPP_3P_PCT_ALLOWED"].notna().sum()))

MergeError: Passing 'suffixes' which cause duplicate columns {'DEF_PPG_ALLOWED_x'} is not allowed.

In [ ]:
# ---------- FIX: avoid duplicate ctx columns on merge ----------
# If you already ran a context pull once, these columns may exist and cause merge collisions.
CTX_COLS = [
    "team_id",
    "DEF_PPG_ALLOWED",
    "OPP_FG_PCT_ALLOWED",
    "OPP_3P_PCT_ALLOWED",
    "PACE_PROXY",
    "ADJ_CTX_V2",
    "MODEL_PROB_CTX_V2",
    "EDGE_VS_BOOK_V2"
]

# Drop any existing versions in df (except home_id)
for c in CTX_COLS:
    if c in df.columns:
        df = df.drop(columns=[c])

# Also remove any lingering _x/_y columns from prior merges
df = df.drop(columns=[c for c in df.columns if re.search(r"_(x|y)$", c)], errors="ignore")

# Now safe to merge ctx_df
df = df.merge(ctx_df, left_on="home_id", right_on="team_id", how="left").drop(columns=["team_id"])
print("Merged ctx_df successfully. Current columns added:",
      [c for c in ["DEF_PPG_ALLOWED","OPP_FG_PCT_ALLOWED","OPP_3P_PCT_ALLOWED","PACE_PROXY"] if c in df.columns])

Merged ctx_df successfully. Current columns added: ['DEF_PPG_ALLOWED', 'OPP_FG_PCT_ALLOWED', 'OPP_3P_PCT_ALLOWED', 'PACE_PROXY']


In [ ]:
import numpy as np
import pandas as pd

# ---------- guardrails ----------
need_cols = ["MODEL_PROB","AMERICAN_ODDS","BOOK_PROB","DEF_PPG_ALLOWED","PACE_PROXY"]
missing = [c for c in need_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}. You likely need to run Step 2–5 base first.")

# ---------- Context adjustment (capped) ----------
# Baselines (tweakable)
BASE_DEF_PPG = 72.0   # "average" defensive points allowed proxy baseline
BASE_PACE    = 70.0   # "average" pace proxy baseline

# Caps (so context never overpowers player trend)
CAP_DEF  = 0.025      # max +/- 2.5% from opponent defense
CAP_PACE = 0.015      # max +/- 1.5% from pace

def clip01(x):
    return float(np.clip(x, 0.01, 0.99))

# Scale defense: higher DEF_PPG_ALLOWED => softer defense => help OVERS, hurt UNDERS
def def_adj(def_ppg, is_over):
    if pd.isna(def_ppg):
        return 0.0
    delta = (def_ppg - BASE_DEF_PPG) / 20.0   # 20 pts span ~ full effect
    adj = np.clip(delta, -1, 1) * CAP_DEF
    return adj if is_over else -adj

# Scale pace: higher pace => more possessions => help OVERS, hurt UNDERS
def pace_adj(pace, is_over):
    if pd.isna(pace):
        return 0.0
    delta = (pace - BASE_PACE) / 15.0         # 15 poss span ~ full effect
    adj = np.clip(delta, -1, 1) * CAP_PACE
    return adj if is_over else -adj

# Determine over/under from OUTCOME like "o12.5" / "u7.5" / "Yes" etc.
def is_over_outcome(x):
    s = str(x).strip().lower()
    if s.startswith("o"): return True
    if s.startswith("u"): return False
    # for props like "Yes/No", treat "Yes" as over-ish positive outcome (no ctx sign flip)
    return True

df["IS_OVER"] = df["OUTCOME"].apply(is_over_outcome)

df["ADJ_DEF"]  = df.apply(lambda r: def_adj(r["DEF_PPG_ALLOWED"], r["IS_OVER"]), axis=1)
df["ADJ_PACE"] = df.apply(lambda r: pace_adj(r["PACE_PROXY"], r["IS_OVER"]), axis=1)

# Total context adjustment, capped again for safety
df["ADJ_CTX_V2"] = np.clip(df["ADJ_DEF"] + df["ADJ_PACE"], -0.035, 0.035)

df["MODEL_PROB_CTX_V2"] = (df["MODEL_PROB"] + df["ADJ_CTX_V2"]).apply(clip01)
df["EDGE_VS_BOOK_V2"]   = df["MODEL_PROB_CTX_V2"] - df["BOOK_PROB"]

# ---------- Top 10 Most Likely ----------
top10 = (
    df.dropna(subset=["MODEL_PROB_CTX_V2"])
      .sort_values("MODEL_PROB_CTX_V2", ascending=False)
      .head(10)
      .copy()
)

top10.insert(0, "RANK", range(1, len(top10)+1))

cols_show = [
    "RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "BOOK_PROB","MODEL_PROB","DEF_PPG_ALLOWED","PACE_PROXY",
    "ADJ_CTX_V2","MODEL_PROB_CTX_V2","EDGE_VS_BOOK_V2"
]
display(top10[cols_show])

# ---------- Discord-ready hit sheet (no units) ----------
def fmt_odds(o):
    try:
        o = float(o)
        return f"{int(o)}" if o.is_integer() else f"{o:.0f}"
    except:
        return "NA"

lines = ["NCAA PROP ENGINE — TOP 10 MOST LIKELY (CTX v2)", "—"]
for _, r in top10.iterrows():
    lines.append(
        f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n"
        f"{r.PROP} {r.OUTCOME} ({fmt_odds(r.AMERICAN_ODDS)})\n"
        f"Model: {r.MODEL_PROB_CTX_V2*100:.1f}% | Edge: {r.EDGE_VS_BOOK_V2*100:+.1f} pts | "
        f"Opp DEF PPG: {r.DEF_PPG_ALLOWED:.1f} | Pace: {r.PACE_PROXY:.1f}"
    )

print("\n".join(lines))

# ---------- Save outputs ----------
top10.to_csv("hit_sheet_top10_ncaam_ctx_v2.csv", index=False)
df.to_csv("prop_engine_full_ncaam_ctx_v2.csv", index=False)
print("\nSaved: hit_sheet_top10_ncaam_ctx_v2.csv")
print("Saved: prop_engine_full_ncaam_ctx_v2.csv")

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,BOOK_PROB,MODEL_PROB,DEF_PPG_ALLOWED,PACE_PROXY,ADJ_CTX_V2,MODEL_PROB_CTX_V2,EDGE_VS_BOOK_V2
36,1,Amani Hansberry,VT at UNC,Points + Rebounds + Assists,u21.5,-120.0,0.545455,0.750667,NaN,68.139999,0.001860,0.752527,0.207072
49,2,Seth Trimble,VT at UNC,Assists,u2.5,-190.0,0.655172,0.734286,NaN,68.139999,0.001860,0.736146,0.080973
43,3,Jailen Bedford,VT at UNC,Points,o10.5,-120.0,0.545455,0.717333,NaN,68.139999,-0.001860,0.715473,0.170019
32,4,Tahaad Pettiford,MISS at AUB,Rebounds,o2.5,-200.0,0.666667,0.715786,NaN,68.821430,-0.001179,0.714607,0.047940
1,5,Graham Ike,GONZ at SMC,Points,o19.5,-166.0,0.624060,0.698500,NaN,65.557332,-0.004443,0.694057,0.069997
52,6,Henri Veesaar,VT at UNC,Points,u16.5,102.0,0.495050,0.660750,NaN,68.139999,0.001860,0.662610,0.167560
28,7,Rueben Chinyelu,ARK at FLA,Rebounds,u11.5,-115.0,0.534884,0.661929,NaN,71.247142,-0.001247,0.660681,0.125798
48,8,Meleek Thomas,ARK at FLA,Rebounds,u4.5,-108.0,0.519231,0.661667,NaN,71.247142,-0.001247,0.660420,0.141189
34,9,Keyshawn Hall,MISS at AUB,Three Pointers Made,u1.5,130.0,0.434783,0.655714,NaN,68.821430,0.001179,0.656893,0.222110
40,10,Luka Bogavac,VT at UNC,Points,u12.5,-112.0,0.528302,0.653750,NaN,68.139999,0.001860,0.655610,0.127308


NCAA PROP ENGINE — TOP 10 MOST LIKELY (CTX v2)
—
1) Amani Hansberry — VT at UNC
Points + Rebounds + Assists u21.5 (-120)
Model: 75.3% | Edge: +20.7 pts | Opp DEF PPG: nan | Pace: 68.1
2) Seth Trimble — VT at UNC
Assists u2.5 (-190)
Model: 73.6% | Edge: +8.1 pts | Opp DEF PPG: nan | Pace: 68.1
3) Jailen Bedford — VT at UNC
Points o10.5 (-120)
Model: 71.5% | Edge: +17.0 pts | Opp DEF PPG: nan | Pace: 68.1
4) Tahaad Pettiford — MISS at AUB
Rebounds o2.5 (-200)
Model: 71.5% | Edge: +4.8 pts | Opp DEF PPG: nan | Pace: 68.8
5) Graham Ike — GONZ at SMC
Points o19.5 (-166)
Model: 69.4% | Edge: +7.0 pts | Opp DEF PPG: nan | Pace: 65.6
6) Henri Veesaar — VT at UNC
Points u16.5 (102)
Model: 66.3% | Edge: +16.8 pts | Opp DEF PPG: nan | Pace: 68.1
7) Rueben Chinyelu — ARK at FLA
Rebounds u11.5 (-115)
Model: 66.1% | Edge: +12.6 pts | Opp DEF PPG: nan | Pace: 71.2
8) Meleek Thomas — ARK at FLA
Rebounds u4.5 (-108)
Model: 66.0% | Edge: +14.1 pts | Opp DEF PPG: nan | Pace: 71.2
9) Keyshawn Hall — MISS 

In [ ]:
import pandas as pd
import numpy as np

# 1) Make sure we DON'T have an old/duplicated defense column messing merges up
for c in ["DEF_PPG_ALLOWED", "DEF_PPG_ALLOWED_x", "DEF_PPG_ALLOWED_y", "OPP_DEF_PPG_ALLOWED", "ESPN_OPP_DEF_PPG_PROXY"]:
    if c in df.columns:
        df.drop(columns=[c], inplace=True)

# 2) Rebuild/confirm ctx_df exists and has team_id + DEF_PPG_ALLOWED
# If your object is team_def_df instead, swap ctx_df -> team_def_df below.
if "ctx_df" not in globals():
    raise ValueError("ctx_df not found. Use the cell that builds ctx_df (team defense + pace) first.")

def_col = None
for cand in ["DEF_PPG_ALLOWED", "OPP_DEF_PPG_ALLOWED"]:
    if cand in ctx_df.columns:
        def_col = cand
        break
if def_col is None:
    raise ValueError(f"ctx_df missing defense column. Found: {ctx_df.columns.tolist()}")

# 3) Merge opponent defense into df using opponent = HOME team (since sheet is 'AWAY at HOME')
# Expect df has home_id already
if "home_id" not in df.columns:
    raise ValueError("home_id not found in df. Run team-id resolution first.")

ctx_tmp = ctx_df[["team_id", def_col, "PACE_PROXY"]].drop_duplicates().copy()
ctx_tmp = ctx_tmp.rename(columns={def_col: "DEF_PPG_ALLOWED"})

df = df.merge(ctx_tmp, left_on="home_id", right_on="team_id", how="left").drop(columns=["team_id"])

print("Defense merge check:")
print("Rows with DEF_PPG_ALLOWED non-null:", int(df["DEF_PPG_ALLOWED"].notna().sum()), "of", len(df))
display(df[["GAME","home_id","DEF_PPG_ALLOWED","PACE_PROXY"]].drop_duplicates().head(10))

Defense merge check:
Rows with DEF_PPG_ALLOWED non-null: 0 of 96


KeyError: "['PACE_PROXY'] not in index"

In [ ]:
import pandas as pd
import numpy as np
import re

# -----------------------
# 0) Sanity: make sure ctx_df exists
# -----------------------
if "ctx_df" not in globals():
    raise ValueError("ctx_df not found. Run the cell that builds ctx_df first.")

print("df columns (preview):", df.columns.tolist()[:25])
print("ctx_df columns:", ctx_df.columns.tolist())

# -----------------------
# 1) Find the DEF and PACE columns inside ctx_df
# -----------------------
def_col = None
for cand in ["DEF_PPG_ALLOWED", "OPP_DEF_PPG_ALLOWED", "DEF_PPG", "DEF_PTS_ALLOWED", "PTS_ALLOWED"]:
    if cand in ctx_df.columns:
        def_col = cand
        break

pace_col = None
pace_candidates = [c for c in ctx_df.columns if re.search("pace|tempo", c, re.IGNORECASE)]
if len(pace_candidates) > 0:
    pace_col = pace_candidates[0]  # take first match

print("Detected def_col:", def_col)
print("Detected pace_col:", pace_col)

if def_col is None:
    raise ValueError(f"Could not find a defense PPG allowed column in ctx_df. Found: {ctx_df.columns.tolist()}")
if pace_col is None:
    print("WARNING: No pace/tempo column found in ctx_df. We'll merge defense only.")

# -----------------------
# 2) Normalize join keys (this is usually the bug)
# -----------------------
if "home_id" not in df.columns:
    raise ValueError("home_id not found in df. Run team-id resolution first.")

print("\n--- BEFORE COERCE ---")
print("df.home_id dtype:", df["home_id"].dtype, "sample:", df["home_id"].dropna().head(5).tolist())
print("ctx_df.team_id dtype:", ctx_df["team_id"].dtype, "sample:", ctx_df["team_id"].dropna().head(5).tolist())

df["home_id"] = pd.to_numeric(df["home_id"], errors="coerce").astype("Int64")
ctx_df["team_id"] = pd.to_numeric(ctx_df["team_id"], errors="coerce").astype("Int64")

print("\n--- AFTER COERCE ---")
print("df.home_id dtype:", df["home_id"].dtype, "unique:", sorted(df["home_id"].dropna().unique().tolist())[:20])
print("ctx_df.team_id dtype:", ctx_df["team_id"].dtype, "unique:", sorted(ctx_df["team_id"].dropna().unique().tolist())[:20])

# -----------------------
# 3) Clean old merged cols so we don't get suffix collisions
# -----------------------
for c in ["DEF_PPG_ALLOWED", "PACE_PROXY", "PACE", "tempo", "DEF_PPG_ALLOWED_x", "DEF_PPG_ALLOWED_y"]:
    if c in df.columns:
        df.drop(columns=[c], inplace=True)

# -----------------------
# 4) Merge
# -----------------------
keep_cols = ["team_id", def_col] + ([pace_col] if pace_col is not None else [])
ctx_tmp = ctx_df[keep_cols].drop_duplicates().copy()
ctx_tmp = ctx_tmp.rename(columns={def_col: "DEF_PPG_ALLOWED"})
if pace_col is not None:
    ctx_tmp = ctx_tmp.rename(columns={pace_col: "PACE_PROXY"})

df = df.merge(ctx_tmp, left_on="home_id", right_on="team_id", how="left").drop(columns=["team_id"])

# -----------------------
# 5) Verify
# -----------------------
print("\nDefense merge check:")
print("Rows with DEF_PPG_ALLOWED non-null:", int(df["DEF_PPG_ALLOWED"].notna().sum()), "of", len(df))

if "PACE_PROXY" in df.columns:
    print("Rows with PACE_PROXY non-null:", int(df["PACE_PROXY"].notna().sum()), "of", len(df))
    display(df[["GAME","home_id","DEF_PPG_ALLOWED","PACE_PROXY"]].drop_duplicates().head(10))
else:
    display(df[["GAME","home_id","DEF_PPG_ALLOWED"]].drop_duplicates().head(10))

df columns (preview): ['IDX', 'PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS', 'AVG', 'IM_PROB_PCT', 'L5', 'L10', 'SEASON_2025', 'HM_AW', 'H2H', 'DVP_L5', 'DVP_L10', 'DVP_2025', 'DVP_HM_AW', 'away_tok', 'home_tok', 'away_id', 'away_score', 'home_id', 'home_score', 'ADJ_CTX', 'P_TREND']
ctx_df columns: ['team_id', 'DEF_PPG_ALLOWED', 'OPP_FG_PCT_ALLOWED', 'OPP_3P_PCT_ALLOWED', 'PACE_PROXY']
Detected def_col: DEF_PPG_ALLOWED
Detected pace_col: PACE_PROXY

--- BEFORE COERCE ---
df.home_id dtype: int64 sample: [2608, 2608, 2608, 2608, 2608]
ctx_df.team_id dtype: int64 sample: [2, 57, 153, 2608]

--- AFTER COERCE ---
df.home_id dtype: Int64 unique: [2, 57, 153, 2608]
ctx_df.team_id dtype: Int64 unique: [2, 57, 153, 2608]

Defense merge check:
Rows with DEF_PPG_ALLOWED non-null: 0 of 96
Rows with PACE_PROXY non-null: 96 of 96


,GAME,home_id,DEF_PPG_ALLOWED,PACE_PROXY
0,GONZ at SMC,2608,NaN,65.557332
28,ARK at FLA,57,NaN,71.247142
29,VT at UNC,153,NaN,68.139999
31,MISS at AUB,2,NaN,68.821430


In [ ]:
print("ctx_df DEF_PPG_ALLOWED non-null:", int(ctx_df["DEF_PPG_ALLOWED"].notna().sum()), "of", len(ctx_df))
display(ctx_df[["team_id","DEF_PPG_ALLOWED","PACE_PROXY"]])

ctx_df DEF_PPG_ALLOWED non-null: 0 of 4


,team_id,DEF_PPG_ALLOWED,PACE_PROXY
0,2,NaN,68.821430
1,57,NaN,71.247142
2,153,NaN,68.139999
3,2608,NaN,65.557332


In [ ]:
import requests
import pandas as pd
import numpy as np
import re

LEAGUE = "basketball/mens-college-basketball"
TEAM_STATS_URL = "https://site.api.espn.com/apis/site/v2/sports/{LEAGUE}/teams/{team_id}/statistics"

def fetch_team_stats(team_id: int):
    url = TEAM_STATS_URL.format(LEAGUE=LEAGUE, team_id=int(team_id))
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

def flatten_stats(j):
    rows = []
    cats = j.get("results", {}).get("stats", {}).get("categories", []) or []
    for cat in cats:
        cat_name = cat.get("name")
        for s in cat.get("stats", []) or []:
            rows.append({
                "category": cat_name,
                "name": s.get("name"),
                "displayName": s.get("displayName"),
                "value": s.get("value"),
            })
    return pd.DataFrame(rows)

def extract_points_allowed(stats_df: pd.DataFrame):
    # Try displayName matches first (most stable)
    disp = stats_df["displayName"].fillna("").str.lower()
    name = stats_df["name"].fillna("").str.lower()

    # Common ESPN labels across sports/feeds
    candidates = stats_df[
        disp.str.contains("points allowed|points against|opp points|opponent points", regex=True)
        | name.str.contains("pointsagainst|avgpointsagainst|oppoints|opppoints", regex=True)
    ].copy()

    # If we found candidates, pick the first numeric value
    if len(candidates) > 0:
        candidates["value_num"] = pd.to_numeric(candidates["value"], errors="coerce")
        candidates = candidates.dropna(subset=["value_num"])
        if len(candidates) > 0:
            return float(candidates.iloc[0]["value_num"]), candidates.iloc[0][["category","name","displayName"]].to_dict()

    # If not found, return NaN
    return np.nan, None

def extract_pace_proxy(stats_df: pd.DataFrame):
    # Pace may not exist in this endpoint; if you already compute it elsewhere, keep it.
    # We'll attempt to detect any "possessions" / "tempo" style field, else NaN.
    disp = stats_df["displayName"].fillna("").str.lower()
    name = stats_df["name"].fillna("").str.lower()
    pace_rows = stats_df[
        disp.str.contains("possessions|tempo|pace", regex=True)
        | name.str.contains("possessions|tempo|pace", regex=True)
    ].copy()
    if len(pace_rows) > 0:
        pace_rows["value_num"] = pd.to_numeric(pace_rows["value"], errors="coerce")
        pace_rows = pace_rows.dropna(subset=["value_num"])
        if len(pace_rows) > 0:
            return float(pace_rows.iloc[0]["value_num"])
    return np.nan

# --- rebuild for teams present in your sheet ---
team_ids = sorted(pd.unique(df["home_id"].dropna().astype(int)))
rows = []

for tid in team_ids:
    j = fetch_team_stats(tid)
    sdfs = flatten_stats(j)

    def_ppg, def_src = extract_points_allowed(sdfs)

    # keep your existing pace if you already had it earlier (from ctx_df)
    # otherwise attempt to extract
    if "ctx_df" in globals() and "PACE_PROXY" in ctx_df.columns:
        old = ctx_df.loc[ctx_df["team_id"].astype(int) == int(tid), "PACE_PROXY"]
        pace = float(old.iloc[0]) if len(old) else extract_pace_proxy(sdfs)
    else:
        pace = extract_pace_proxy(sdfs)

    rows.append({
        "team_id": int(tid),
        "DEF_PPG_ALLOWED": def_ppg,
        "PACE_PROXY": pace,
        "DEF_SRC": def_src
    })

ctx_df = pd.DataFrame(rows)
print("Rebuilt ctx_df:")
print("DEF_PPG_ALLOWED non-null:", int(ctx_df["DEF_PPG_ALLOWED"].notna().sum()), "of", len(ctx_df))
display(ctx_df)

Rebuilt ctx_df:
DEF_PPG_ALLOWED non-null: 0 of 4


,team_id,DEF_PPG_ALLOWED,PACE_PROXY,DEF_SRC
0,2,NaN,68.821430,None
1,57,NaN,71.247142,None
2,153,NaN,68.139999,None
3,2608,NaN,65.557332,None


In [ ]:
# Drop any old columns to avoid suffix issues
for c in ["DEF_PPG_ALLOWED", "PACE_PROXY"]:
    if c in df.columns:
        df.drop(columns=[c], inplace=True)

df["home_id"] = pd.to_numeric(df["home_id"], errors="coerce").astype("Int64")
ctx_df["team_id"] = pd.to_numeric(ctx_df["team_id"], errors="coerce").astype("Int64")

df = df.merge(ctx_df[["team_id","DEF_PPG_ALLOWED","PACE_PROXY"]], left_on="home_id", right_on="team_id", how="left").drop(columns=["team_id"])

print("Post-merge check:")
print("Rows with DEF_PPG_ALLOWED non-null:", int(df["DEF_PPG_ALLOWED"].notna().sum()), "of", len(df))
print("Rows with PACE_PROXY non-null:", int(df["PACE_PROXY"].notna().sum()), "of", len(df))
display(df[["GAME","home_id","DEF_PPG_ALLOWED","PACE_PROXY"]].drop_duplicates())

Post-merge check:
Rows with DEF_PPG_ALLOWED non-null: 0 of 96
Rows with PACE_PROXY non-null: 96 of 96


,GAME,home_id,DEF_PPG_ALLOWED,PACE_PROXY
0,GONZ at SMC,2608,NaN,65.557332
28,ARK at FLA,57,NaN,71.247142
29,VT at UNC,153,NaN,68.139999
31,MISS at AUB,2,NaN,68.821430


In [ ]:
import requests
import pandas as pd
import numpy as np

LEAGUE = "basketball/mens-college-basketball"
SCHEDULE_URL = "https://site.api.espn.com/apis/site/v2/sports/{LEAGUE}/teams/{team_id}/schedule"

def fetch_team_schedule(team_id: int):
    url = SCHEDULE_URL.format(LEAGUE=LEAGUE, team_id=int(team_id))
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

def score_to_float(x):
    # ESPN sometimes returns "score" as a string, number, or a dict like {"value": 78}
    if x is None:
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    if isinstance(x, str):
        return pd.to_numeric(x, errors="coerce")
    if isinstance(x, dict):
        # common shapes: {"value": 78} or {"displayValue":"78"}
        if "value" in x:
            return pd.to_numeric(x.get("value"), errors="coerce")
        if "displayValue" in x:
            return pd.to_numeric(x.get("displayValue"), errors="coerce")
        return np.nan
    return np.nan

def compute_points_allowed_from_schedule(team_id: int, max_games=30):
    j = fetch_team_schedule(team_id)
    events = j.get("events", []) or []

    rows = []
    for ev in events:
        comps = (ev.get("competitions") or [])
        if not comps:
            continue
        comp = comps[0]

        status = (comp.get("status") or {}).get("type") or {}
        if not status.get("completed"):
            continue

        competitors = comp.get("competitors", []) or []
        if len(competitors) < 2:
            continue

        me = None
        opp = None
        for c in competitors:
            tid = c.get("team", {}).get("id")
            if str(tid) == str(team_id):
                me = c
            else:
                opp = c
        if me is None or opp is None:
            continue

        me_score  = score_to_float(me.get("score"))
        opp_score = score_to_float(opp.get("score"))
        if pd.isna(me_score) or pd.isna(opp_score):
            continue

        rows.append({
            "event_id": ev.get("id"),
            "date": ev.get("date"),
            "pts_for": float(me_score),
            "pts_allowed": float(opp_score),
            "home_away": me.get("homeAway")
        })

    if len(rows) == 0:
        return np.nan, 0, pd.DataFrame()

    gdf = pd.DataFrame(rows).sort_values("date", ascending=False).head(max_games)
    return float(gdf["pts_allowed"].mean()), int(len(gdf)), gdf

# ---- Build ctx_df from teams in your sheet ----
team_ids = sorted(pd.unique(df["home_id"].dropna().astype(int)))

ctx_rows = []
debug_samples = {}

for tid in team_ids:
    def_ppg, n_games, gdf = compute_points_allowed_from_schedule(tid, max_games=30)

    pace = np.nan
    if "ctx_df" in globals() and isinstance(ctx_df, pd.DataFrame) and "PACE_PROXY" in ctx_df.columns:
        old = ctx_df.loc[ctx_df["team_id"].astype(int) == int(tid), "PACE_PROXY"]
        if len(old):
            pace = float(old.iloc[0])

    ctx_rows.append({
        "team_id": int(tid),
        "DEF_PPG_ALLOWED": def_ppg,
        "DEF_GAMES_USED": n_games,
        "PACE_PROXY": pace
    })
    debug_samples[int(tid)] = gdf.head(5)

ctx_df = pd.DataFrame(ctx_rows)

print("Rebuilt ctx_df from schedules:")
print("DEF_PPG_ALLOWED non-null:", int(ctx_df["DEF_PPG_ALLOWED"].notna().sum()), "of", len(ctx_df))
display(ctx_df)

print("\nDEBUG: last-5 completed games used per team_id")
for tid, sample in debug_samples.items():
    print("\nteam_id:", tid)
    display(sample)

Rebuilt ctx_df from schedules:
DEF_PPG_ALLOWED non-null: 4 of 4


,team_id,DEF_PPG_ALLOWED,DEF_GAMES_USED,PACE_PROXY
0,2,79.714286,28,68.821430
1,57,71.214286,28,71.247142
2,153,70.750000,28,68.139999
3,2608,64.366667,30,65.557332



DEBUG: last-5 completed games used per team_id

team_id: 2


,event_id,date,pts_for,pts_allowed,home_away
27,401808262,2026-02-25T02:00Z,79.0,91.0,away
26,401808250,2026-02-22T01:38Z,75.0,74.0,home
25,401808242,2026-02-19T02:00Z,85.0,91.0,away
24,401808233,2026-02-15T01:40Z,75.0,88.0,away
23,401808229,2026-02-11T00:00Z,76.0,84.0,home



team_id: 57


,event_id,date,pts_for,pts_allowed,home_away
27,401808264,2026-02-26T00:00Z,84.0,71.0,away
26,401808253,2026-02-21T17:00Z,94.0,75.0,away
25,401808241,2026-02-18T00:00Z,76.0,62.0,home
24,401808234,2026-02-14T20:00Z,92.0,83.0,home
23,401808230,2026-02-12T00:00Z,86.0,66.0,away



team_id: 153


,event_id,date,pts_for,pts_allowed,home_away
27,401820762,2026-02-24T00:00Z,77.0,74.0,home
26,401820759,2026-02-21T18:00Z,77.0,64.0,away
25,401820752,2026-02-18T00:00Z,58.0,82.0,away
24,401820742,2026-02-14T19:00Z,79.0,65.0,home
23,401820731,2026-02-11T00:00Z,66.0,75.0,away



team_id: 2608


,event_id,date,pts_for,pts_allowed,home_away
29,401829262,2026-02-26T04:00Z,86.0,67.0,home
28,401829258,2026-02-22T03:07Z,83.0,67.0,away
27,401829253,2026-02-19T02:00Z,72.0,70.0,away
26,401829248,2026-02-15T03:35Z,72.0,61.0,away
25,401829241,2026-02-12T03:00Z,88.0,60.0,home


In [ ]:
for c in ["DEF_PPG_ALLOWED", "DEF_GAMES_USED", "PACE_PROXY"]:
    if c in df.columns:
        df.drop(columns=[c], inplace=True)

df["home_id"] = pd.to_numeric(df["home_id"], errors="coerce").astype("Int64")
ctx_df["team_id"] = pd.to_numeric(ctx_df["team_id"], errors="coerce").astype("Int64")

df = df.merge(
    ctx_df[["team_id","DEF_PPG_ALLOWED","DEF_GAMES_USED","PACE_PROXY"]],
    left_on="home_id", right_on="team_id", how="left"
).drop(columns=["team_id"])

print("Post-merge check:")
print("Rows with DEF_PPG_ALLOWED non-null:", int(df["DEF_PPG_ALLOWED"].notna().sum()), "of", len(df))
display(df[["GAME","home_id","DEF_PPG_ALLOWED","DEF_GAMES_USED","PACE_PROXY"]].drop_duplicates())

Post-merge check:
Rows with DEF_PPG_ALLOWED non-null: 96 of 96


,GAME,home_id,DEF_PPG_ALLOWED,DEF_GAMES_USED,PACE_PROXY
0,GONZ at SMC,2608,64.366667,30,65.557332
28,ARK at FLA,57,71.214286,28,71.247142
29,VT at UNC,153,70.750000,28,68.139999
31,MISS at AUB,2,79.714286,28,68.821430


In [ ]:
import numpy as np

# ---- choose which probability column is your final ----
FINAL_COL = "MODEL_PROB_CTX_V2" if "MODEL_PROB_CTX_V2" in df.columns else "MODEL_PROB"

# ---- create rank table ----
need_cols = ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","BOOK_PROB",FINAL_COL,
             "DEF_PPG_ALLOWED","DEF_GAMES_USED","PACE_PROXY"]

for c in need_cols:
    if c not in df.columns:
        df[c] = np.nan

out = df.copy()
out["MODEL_FINAL"] = out[FINAL_COL]
out["EDGE_FINAL"]  = out["MODEL_FINAL"] - out["BOOK_PROB"]

top10 = (
    out.dropna(subset=["PLAYER","GAME","PROP","OUTCOME","MODEL_FINAL"])
      .sort_values("MODEL_FINAL", ascending=False)
      .drop_duplicates(subset=["PLAYER","GAME","PROP","OUTCOME"], keep="first")
      .head(10)
      .copy()
)

top10.insert(0, "RANK", range(1, len(top10)+1))

# ---- save hit sheet ----
top10.to_csv("hit_sheet_top10_ncaam_ctx_v2_fixed.csv", index=False)
print("Saved: hit_sheet_top10_ncaam_ctx_v2_fixed.csv")

# ---- discord-ready (no units) ----
def fmt_odds(x):
    if pd.isna(x): return ""
    x = int(round(float(x)))
    return f"({x})" if x < 0 else f"({x})"

lines = ["NCAA PROP ENGINE — TOP 10 MOST LIKELY (CTX v2)", "—"]
for _, r in top10.iterrows():
    model_pct = r["MODEL_FINAL"]*100
    edge_pts  = r["EDGE_FINAL"]*100 if not pd.isna(r["EDGE_FINAL"]) else np.nan
    defppg    = r["DEF_PPG_ALLOWED"]
    pace      = r["PACE_PROXY"]

    lines.append(
        f'{int(r["RANK"])}) {r["PLAYER"]} — {r["GAME"]}\n'
        f'{r["PROP"]} {r["OUTCOME"]} {fmt_odds(r["AMERICAN_ODDS"])}\n'
        f'Model: {model_pct:.1f}% | Edge: {edge_pts:+.1f} pts | Opp DEF PPG: {defppg:.1f} | Pace: {pace:.1f}'
    )

print("\n" + "\n".join(lines))

Saved: hit_sheet_top10_ncaam_ctx_v2_fixed.csv

NCAA PROP ENGINE — TOP 10 MOST LIKELY (CTX v2)
—
1) Amani Hansberry — VT at UNC
Points + Rebounds + Assists u21.5 (-120)
Model: 75.3% | Edge: +20.7 pts | Opp DEF PPG: 70.8 | Pace: 68.1
2) Seth Trimble — VT at UNC
Assists u2.5 (-190)
Model: 73.6% | Edge: +8.1 pts | Opp DEF PPG: 70.8 | Pace: 68.1
3) Jailen Bedford — VT at UNC
Points o10.5 (-120)
Model: 71.5% | Edge: +17.0 pts | Opp DEF PPG: 70.8 | Pace: 68.1
4) Tahaad Pettiford — MISS at AUB
Rebounds o2.5 (-200)
Model: 71.5% | Edge: +4.8 pts | Opp DEF PPG: 79.7 | Pace: 68.8
5) Graham Ike — GONZ at SMC
Points o19.5 (-166)
Model: 69.4% | Edge: +7.0 pts | Opp DEF PPG: 64.4 | Pace: 65.6
6) Henri Veesaar — VT at UNC
Points u16.5 (102)
Model: 66.3% | Edge: +16.8 pts | Opp DEF PPG: 70.8 | Pace: 68.1
7) Rueben Chinyelu — ARK at FLA
Rebounds u11.5 (-115)
Model: 66.1% | Edge: +12.6 pts | Opp DEF PPG: 71.2 | Pace: 71.2
8) Meleek Thomas — ARK at FLA
Rebounds u4.5 (-108)
Model: 66.0% | Edge: +14.1 pts | 

In [ ]:
# ==============================
# TODAY RUN — INPUT
# ==============================

CSV_PATH = "NCAA prop 3:1.csv"   # <-- your uploaded file name in Colab

import pandas as pd, numpy as np, re
from datetime import datetime

df = pd.read_csv(CSV_PATH)

# Normalize columns exactly like your notebook style
df.columns = (
    df.columns.astype(str)
      .str.strip()
      .str.replace(" ", "_")
      .str.replace("/", "_")
      .str.replace("%", "PCT")
)

# Force PLAYER column detection (your notebook does this)
for col in df.columns:
    if col.lower() in ["layer", "player"]:
        df.rename(columns={col: "PLAYER"}, inplace=True)

print("Loaded:", df.shape)
print("Columns:", df.columns.tolist()[:30])


FileNotFoundError: [Errno 2] No such file or directory: 'NCAA prop 3:1.csv'

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving New ncaa prop.csv to New ncaa prop.csv


In [ ]:
import os
print(os.listdir())

['.config', 'New ncaa prop.csv', 'NCAA prop 3:1.csv', 'sample_data']


In [ ]:
import os
import pandas as pd

# Automatically detect the uploaded CSV
csv_files = [f for f in os.listdir() if f.lower().endswith(".csv")]
print("CSV files found:", csv_files)

CSV_PATH = csv_files[0]  # uses the first uploaded CSV

df = pd.read_csv(CSV_PATH)

print("Loaded:", CSV_PATH)
print("Shape:", df.shape)

CSV files found: ['New ncaa prop.csv', 'NCAA prop 3:1.csv']
Loaded: New ncaa prop.csv
Shape: (101, 17)


In [ ]:
import pandas as pd, numpy as np, re

# normalize columns to match our model expectations
df.columns = (
    df.columns.astype(str)
      .str.strip()
      .str.replace(" ", "_")
      .str.replace("/", "_")
      .str.replace("%", "PCT")
)

# PLAYER guard
for col in df.columns:
    if col.lower() in ["layer", "player"]:
        df.rename(columns={col: "PLAYER"}, inplace=True)

# ODDS -> AMERICAN_ODDS
if "AMERICAN_ODDS" not in df.columns:
    if "ODDS" in df.columns:
        df["AMERICAN_ODDS"] = pd.to_numeric(df["ODDS"].astype(str).str.extract(r'(-?\d+)')[0], errors="coerce")
    else:
        df["AMERICAN_ODDS"] = np.nan

print("Cols:", df.columns.tolist())
print("AMERICAN_ODDS null:", df["AMERICAN_ODDS"].isna().sum())

Cols: ['Unnamed:_0', 'Unnamed:_1', 'Unnamed:_2', 'Unnamed:_3', 'Unnamed:_4', 'Unnamed:_5', 'Unnamed:_6', 'Unnamed:_7', 'Unnamed:_8', 'Unnamed:_9', 'Unnamed:_10', 'Unnamed:_11', 'Unnamed:_12', 'Unnamed:_13', 'Unnamed:_14', 'Unnamed:_15', 'Unnamed:_16', 'AMERICAN_ODDS']
AMERICAN_ODDS null: 101


In [ ]:
BASE_WEIGHTS = {"L10":0.25,"L5":0.15,"2025":0.25,"HM_AW":0.15,"H2H":0.05,"DVP":0.15}
L5_SHRINK=0.75
L10_SHRINK=0.90
DVP_DELTA_CAP=0.15
PROB_CLAMP=(0.02,0.98)
KELLY_CAP=0.05

def pct_to_float(x):
    if pd.isna(x): return np.nan
    if isinstance(x, (int,float,np.floating)):
        v=float(x); return v if v<=1 else v/100
    s=str(x).strip().replace("%","")
    if s in ["--","-","–","—",""]: return np.nan
    try:
        v=float(s); return v/100 if v>1 else v
    except: return np.nan

def american_to_prob(odds):
    if pd.isna(odds): return np.nan
    o=float(odds)
    return 100/(o+100) if o>0 else (-o)/((-o)+100)

def american_to_decimal(odds):
    if pd.isna(odds): return np.nan
    o=float(odds)
    return 1 + o/100 if o>0 else 1 + 100/(-o)

def shrink(p, factor):
    if pd.isna(p): return np.nan
    return 0.5 + (float(p)-0.5)*factor

def clamp(p):
    return max(PROB_CLAMP[0], min(PROB_CLAMP[1], float(p)))

def kelly_frac(p, dec_odds, cap=KELLY_CAP):
    if pd.isna(p) or pd.isna(dec_odds) or dec_odds<=1:
        return 0.0
    b=dec_odds-1
    q=1-p
    f=(b*p - q)/b
    if pd.isna(f): return 0.0
    return float(max(0.0, min(f, cap)))

# Convert % cols if present
pct_cols = ["IM_PROB_PCT","L5","L10","2025","HM_AW","H2H","DVP_L5","DVP_L10","DVP_HM_AW","DVP_2025"]
for c in pct_cols:
    if c in df.columns:
        df[c] = df[c].apply(pct_to_float)

# Book prob + decimal odds
df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

# DVP composite + cap
dvp_cols = [c for c in ["DVP_L5","DVP_L10","DVP_HM_AW","DVP_2025"] if c in df.columns]
df["DVP_RAW"] = df[dvp_cols].mean(axis=1) if dvp_cols else np.nan
df["DVP"] = df["DVP_RAW"].apply(lambda x: 0.5 + np.clip(x-0.5, -DVP_DELTA_CAP, DVP_DELTA_CAP) if not pd.isna(x) else np.nan)

# Shrink L5/L10
if "L5" in df.columns:  df["L5"]  = df["L5"].apply(lambda x: shrink(x, L5_SHRINK))
if "L10" in df.columns: df["L10"] = df["L10"].apply(lambda x: shrink(x, L10_SHRINK))

# MODEL_PROB (weighted avg of available)
model_probs=[]
for _,r in df.iterrows():
    feats = {k: r[k] if k in r.index else np.nan for k in ["L10","L5","2025","HM_AW","H2H","DVP"]}
    valid={k:v for k,v in feats.items() if not pd.isna(v)}
    if not valid:
        model_probs.append(np.nan); continue
    wsum=sum(BASE_WEIGHTS[k] for k in valid)
    prob=sum((BASE_WEIGHTS[k]/wsum)*valid[k] for k in valid)
    model_probs.append(clamp(prob))
df["MODEL_PROB"] = model_probs

# EV + Kelly
df["EDGE_VS_BOOK"] = df["MODEL_PROB"] - df["BOOK_PROB"]
df["EV_1U"] = df["MODEL_PROB"]*(df["DEC_ODDS"]-1) - (1-df["MODEL_PROB"])
df["KELLY_FULL_FRAC"] = [kelly_frac(p,o) for p,o in zip(df["MODEL_PROB"], df["DEC_ODDS"])]

print("Base model done. Missing MODEL_PROB:", df["MODEL_PROB"].isna().sum())

Base model done. Missing MODEL_PROB: 101


In [ ]:
import requests
from datetime import datetime, timezone, timedelta

SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def parse_game_tokens(g):
    g = str(g).upper().strip().replace(" AT ", " at ")
    parts = [p.strip() for p in g.split(" at ")]
    if len(parts) != 2:
        return (None, None)
    return (parts[0], parts[1])

utc_now = datetime.now(timezone.utc)
dates = [(utc_now + timedelta(days=i)).strftime("%Y%m%d") for i in range(0, 4)]  # today + next 3

need_tokens=set()
for g in df["GAME"].astype(str).unique():
    a,h = parse_game_tokens(g)
    if a: need_tokens.add(a)
    if h: need_tokens.add(h)

team_map = {}   # token -> team_id
for d in dates:
    sb = get_json(SCOREBOARD, params={"dates": d, "limit": 300})
    for ev in sb.get("events", []) or []:
        comp = (ev.get("competitions") or [{}])[0]
        for c in comp.get("competitors", []) or []:
            t = c.get("team") or {}
            abbr = str(t.get("abbreviation","")).upper().strip()
            tid  = t.get("id")
            if abbr and tid and abbr in need_tokens and abbr not in team_map:
                team_map[abbr] = int(tid)

df["away_tok"], df["home_tok"] = zip(*df["GAME"].astype(str).map(parse_game_tokens))
df["away_id"] = df["away_tok"].map(team_map)
df["home_id"] = df["home_tok"].map(team_map)

print("Needed tokens:", len(need_tokens))
print("Mapped tokens:", len(team_map))
print("Missing home_id:", df["home_id"].isna().sum(), "| Missing away_id:", df["away_id"].isna().sum())

KeyError: 'GAME'

In [ ]:
# ==============================
# FRESH START — TODAY RUN
# ==============================

import pandas as pd
import numpy as np
import os
import re

# Auto-detect latest CSV
csv_files = [f for f in os.listdir() if f.lower().endswith(".csv")]
print("CSV files found:", csv_files)

CSV_PATH = csv_files[-1]  # use most recent uploaded
print("Using:", CSV_PATH)

df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

CSV files found: ['New ncaa prop.csv', 'NCAA prop 3:1.csv']
Using: NCAA prop 3:1.csv
Shape: (101, 17)
Columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16']


In [ ]:
# ==============================
# HARD RESET — TODAY FILE
# ==============================

import pandas as pd
import numpy as np

CSV_PATH = "New ncaa prop.csv"

df = pd.read_csv(CSV_PATH)

print("Loaded:", CSV_PATH)
print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Loaded: New ncaa prop.csv
Shape: (101, 17)
Columns:
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16']


In [ ]:
# ==============================
# FIX HEADERS (take row 0 as header)
# ==============================

import pandas as pd
import numpy as np

# If first row looks like real headers, use it
first_row = df.iloc[0].astype(str).tolist()
print("First row preview (candidate headers):")
print(first_row)

df2 = df.copy()
df2.columns = first_row
df2 = df2.iloc[1:].reset_index(drop=True)

print("\nAfter header fix:")
print("Shape:", df2.shape)
print("Columns:", df2.columns.tolist())

First row preview (candidate headers):
['nan', 'PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS', 'AVG', 'IM PROB %', 'L5', 'L10', '2025', 'HM/AW', 'H2H', 'DVP L5', 'DVP L10', 'DVP 2025', 'DVP HM/AW']

After header fix:
Shape: (100, 17)
Columns: ['nan', 'PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS', 'AVG', 'IM PROB %', 'L5', 'L10', '2025', 'HM/AW', 'H2H', 'DVP L5', 'DVP L10', 'DVP 2025', 'DVP HM/AW']


In [ ]:
# ==============================
# NORMALIZE COLUMN NAMES
# ==============================

df = df2.copy()

df.columns = (
    df.columns.astype(str)
      .str.strip()
      .str.replace(" ", "_")
      .str.replace("/", "_")
      .str.replace("%", "PCT")
)

# Kill totally empty columns
df = df.dropna(axis=1, how="all")

print("Normalized columns:", df.columns.tolist())
print("Shape:", df.shape)

Normalized columns: ['PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS', 'AVG', 'IM_PROB_PCT', 'L5', 'L10', '2025', 'HM_AW', 'H2H', 'DVP_L5', 'DVP_L10', 'DVP_2025', 'DVP_HM_AW']
Shape: (100, 16)


In [ ]:
# ==============================
# FORCE GAME COLUMN
# ==============================

# rename any "game" variant -> GAME
for c in df.columns:
    if c.strip().lower() == "game":
        df = df.rename(columns={c: "GAME"})
        break

# if still missing, try common alternatives
if "GAME" not in df.columns:
    for c in df.columns:
        if c.strip().lower() in ["matchup","game_matchup","event","fixture"]:
            df = df.rename(columns={c: "GAME"})
            break

print("Has GAME?", "GAME" in df.columns)
if "GAME" in df.columns:
    print("GAME sample:", df["GAME"].head(10).tolist())

Has GAME? True
GAME sample: ['MSU at IND', 'PUR at OSU', 'MSU at IND', 'MSU at IND', 'PUR at OSU', 'MSU at IND', 'PUR at OSU', 'PUR at OSU', 'PUR at OSU', 'PUR at OSU']


In [ ]:
# ==============================
# STANDARDIZE KEY COLUMNS
# ==============================

# PLAYER
for c in df.columns:
    if c.strip().lower() in ["player","layer","name"]:
        df = df.rename(columns={c:"PLAYER"})
        break

# PROP
for c in df.columns:
    if c.strip().lower() in ["prop","market","stat","bet_type"]:
        df = df.rename(columns={c:"PROP"})
        break

# OUTCOME (o/u)
for c in df.columns:
    if c.strip().lower() in ["outcome","side","ou","pick"]:
        df = df.rename(columns={c:"OUTCOME"})
        break

# ODDS
for c in df.columns:
    if c.strip().lower() in ["odds","american_odds","price"]:
        df = df.rename(columns={c:"ODDS"})
        break

print("Columns now:", df.columns.tolist())
print("Missing required:",
      [c for c in ["PLAYER","GAME","PROP","OUTCOME","ODDS"] if c not in df.columns])

print(df[["PLAYER","GAME","PROP","OUTCOME","ODDS"]].head(10))

Columns now: ['PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS', 'AVG', 'IM_PROB_PCT', 'L5', 'L10', '2025', 'HM_AW', 'H2H', 'DVP_L5', 'DVP_L10', 'DVP_2025', 'DVP_HM_AW']
Missing required: []
              PLAYER        GAME                         PROP OUTCOME  \
0   Jeremy Fears Jr.  MSU at IND                      Assists    o3.5   
1           C.J. Cox  PUR at OSU                     Rebounds    o1.5   
2       Jaxon Kohler  MSU at IND          Three Pointers Made    u1.5   
3          Nick Dorn  MSU at IND                     Rebounds    u2.5   
4       Braden Smith  PUR at OSU  Points + Rebounds + Assists   u28.5   
5    Lamar Wilkerson  MSU at IND                     Rebounds    u4.5   
6       Braden Smith  PUR at OSU                       Points   u15.5   
7  Trey Kaufman-Renn  PUR at OSU                       Points   u14.5   
8        Devin Royal  PUR at OSU          Three Pointers Made    u0.5   
9     Bruce Thornton  PUR at OSU                     Rebounds    u5.5   

        ODD

In [ ]:
# ==============================
# BASE HIT RATER — FULL MODEL CORE
# ==============================

import numpy as np
import pandas as pd

BASE_WEIGHTS = {"L10":0.25,"L5":0.15,"2025":0.25,"HM_AW":0.15,"H2H":0.05,"DVP":0.15}
L5_SHRINK=0.75
L10_SHRINK=0.90
DVP_DELTA_CAP=0.15
PROB_CLAMP=(0.02,0.98)
KELLY_CAP=0.05

def pct_to_float(x):
    if pd.isna(x): return np.nan
    if isinstance(x,(int,float,np.floating)):
        v=float(x); return v if v<=1 else v/100
    s=str(x).strip().replace("%","")
    if s in ["--","-","–","—",""]: return np.nan
    try:
        v=float(s); return v/100 if v>1 else v
    except:
        return np.nan

def american_to_prob(odds):
    if pd.isna(odds): return np.nan
    o=float(odds)
    return 100/(o+100) if o>0 else (-o)/((-o)+100)

def american_to_decimal(odds):
    if pd.isna(odds): return np.nan
    o=float(odds)
    return 1 + o/100 if o>0 else 1 + 100/(-o)

def shrink(p,f):
    if pd.isna(p): return np.nan
    return 0.5 + (float(p)-0.5)*f

def clamp(p):
    return max(PROB_CLAMP[0], min(PROB_CLAMP[1], float(p)))

def kelly_frac(p, dec_odds, cap=KELLY_CAP):
    if pd.isna(p) or pd.isna(dec_odds) or dec_odds<=1:
        return 0.0
    b=dec_odds-1
    q=1-p
    f=(b*p - q)/b
    if pd.isna(f): return 0.0
    return float(max(0.0, min(f, cap)))

# Odds parsing
df["AMERICAN_ODDS"] = pd.to_numeric(df["ODDS"].astype(str).str.extract(r'(-?\d+)')[0], errors="coerce")
df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

# Convert percent columns if present
pct_cols = ["IM_PROB_PCT","L5","L10","2025","HM_AW","H2H","DVP_L5","DVP_L10","DVP_HM_AW","DVP_2025"]
for c in pct_cols:
    if c in df.columns:
        df[c] = df[c].apply(pct_to_float)

# DVP composite + cap
dvp_cols = [c for c in ["DVP_L5","DVP_L10","DVP_HM_AW","DVP_2025"] if c in df.columns]
df["DVP_RAW"] = df[dvp_cols].mean(axis=1) if dvp_cols else np.nan
df["DVP"] = df["DVP_RAW"].apply(lambda x: 0.5 + np.clip(x-0.5, -DVP_DELTA_CAP, DVP_DELTA_CAP) if not pd.isna(x) else np.nan)

# shrink L5/L10
if "L5" in df.columns: df["L5"] = df["L5"].apply(lambda x: shrink(x, L5_SHRINK))
if "L10" in df.columns: df["L10"] = df["L10"].apply(lambda x: shrink(x, L10_SHRINK))

# Model prob
model=[]
for _, r in df.iterrows():
    feats = {k:r[k] for k in ["L10","L5","2025","HM_AW","H2H","DVP"] if k in df.columns and not pd.isna(r.get(k))}
    if not feats:
        model.append(np.nan); continue
    w = sum(BASE_WEIGHTS[k] for k in feats)
    prob = sum((BASE_WEIGHTS[k]/w)*feats[k] for k in feats)
    model.append(clamp(prob))

df["MODEL_PROB"] = model
df["EDGE_VS_BOOK"] = df["MODEL_PROB"] - df["BOOK_PROB"]
df["EV_1U"] = df["MODEL_PROB"]*(df["DEC_ODDS"]-1) - (1-df["MODEL_PROB"])
df["KELLY_FULL_FRAC"] = [kelly_frac(p,o) for p,o in zip(df["MODEL_PROB"], df["DEC_ODDS"])]

print("Base model complete. Missing MODEL_PROB:", df["MODEL_PROB"].isna().sum())

Base model complete. Missing MODEL_PROB: 0


In [ ]:
# ==============================
# ESPN ENRICHMENT — TEAM ID MAP
# ==============================

import requests
from datetime import datetime, timezone, timedelta

SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def parse_game_tokens(g):
    g = str(g).upper().strip().replace(" AT ", " at ")
    parts = [p.strip() for p in g.split(" at ")]
    if len(parts) != 2:
        return (None, None)
    return (parts[0], parts[1])

utc_now = datetime.now(timezone.utc)
dates = [(utc_now + timedelta(days=i)).strftime("%Y%m%d") for i in range(0, 4)]

need=set()
for g in df["GAME"].astype(str).unique():
    a,h = parse_game_tokens(g)
    if a: need.add(a)
    if h: need.add(h)

team_map={}
for d in dates:
    sb = get_json(SCOREBOARD, params={"dates": d, "limit": 300})
    for ev in sb.get("events", []) or []:
        comp = (ev.get("competitions") or [{}])[0]
        for c in comp.get("competitors", []) or []:
            t = c.get("team") or {}
            abbr = str(t.get("abbreviation","")).upper().strip()
            tid  = t.get("id")
            if abbr and tid and abbr in need and abbr not in team_map:
                team_map[abbr] = int(tid)

df["away_tok"], df["home_tok"] = zip(*df["GAME"].astype(str).map(parse_game_tokens))
df["away_id"] = df["away_tok"].map(team_map)
df["home_id"] = df["home_tok"].map(team_map)

print("Needed tokens:", len(need))
print("Mapped tokens:", len(team_map))
print("Missing home_id:", df["home_id"].isna().sum(), "| Missing away_id:", df["away_id"].isna().sum())

Needed tokens: 4
Mapped tokens: 3
Missing home_id: 51 | Missing away_id: 0


In [ ]:
# ==============================
# ESPN ENRICHMENT — TEAM STATS
# ==============================

from functools import lru_cache

ESPN_TEAM_STATS = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics"

def safe_get(d, path, default=None):
    cur = d
    for p in path:
        if isinstance(cur, dict) and p in cur:
            cur = cur[p]
        else:
            return default
    return cur

@lru_cache(maxsize=512)
def fetch_team_stats(team_id: int):
    url = ESPN_TEAM_STATS.format(team_id=int(team_id))
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    j = r.json()
    stats=[]
    cats = safe_get(j, ["results","stats","categories"], []) or []
    for cat in cats:
        for s in (cat.get("stats") or []):
            stats.append({"name": s.get("name"), "value": s.get("value")})
    return pd.DataFrame(stats)

def stat_value(team_id, names):
    try:
        tdf = fetch_team_stats(int(team_id))
    except Exception:
        return np.nan
    for nm in names:
        m = tdf.loc[tdf["name"].astype(str).str.lower() == nm.lower(), "value"]
        if len(m)>0:
            try:
                return float(m.iloc[0])
            except:
                return np.nan
    return np.nan

needed_team_ids = sorted({int(x) for x in pd.concat([df["home_id"], df["away_id"]]).dropna().unique()})
rows=[]
for tid in needed_team_ids:
    def_ppg_allowed = stat_value(tid, ["pointsAllowedPerGame", "oppPointsPerGame", "pointsAllowed"])
    pace_proxy      = stat_value(tid, ["possessionsPerGame", "pace", "tempo"])
    rows.append({"team_id": tid, "DEF_PPG_ALLOWED": def_ppg_allowed, "PACE_PROXY": pace_proxy})

ctx_df = pd.DataFrame(rows)
print("ctx_df shape:", ctx_df.shape)
ctx_df.head()

ctx_df shape: (3, 3)


,team_id,DEF_PPG_ALLOWED,PACE_PROXY
0,127,NaN,NaN
1,194,NaN,NaN
2,2509,NaN,NaN


In [ ]:
# ==============================
# CTX v2 — APPLY + EXPORT
# ==============================

def infer_player_is_home(r):
    if "HM_AW" in r.index and pd.notna(r["HM_AW"]):
        return float(r["HM_AW"]) >= 0.5
    return False

opp_ids=[]
for _, r in df.iterrows():
    player_home = infer_player_is_home(r)
    opp_ids.append(r["away_id"] if player_home else r["home_id"])

df["opp_team_id"] = opp_ids

df = df.merge(ctx_df, left_on="opp_team_id", right_on="team_id", how="left").drop(columns=["team_id"])
df.rename(columns={
    "DEF_PPG_ALLOWED":"ESPN_OPP_DEF_PPG_PROXY",
    "PACE_PROXY":"ESPN_PACE_PROXY"
}, inplace=True)

BASE_DEF_PPG = 72.0
BASE_PACE    = 70.0
CAP_DEF  = 0.025
CAP_PACE = 0.015

def clip01(x):
    return float(np.clip(x, 0.01, 0.99))

def is_over(outcome):
    return str(outcome).lower().strip().startswith("o")

def def_adj(def_ppg, over_flag):
    if pd.isna(def_ppg):
        return 0.0
    delta = (float(def_ppg) - BASE_DEF_PPG) / 20.0
    delta = float(np.clip(delta, -1, 1))
    adj = delta * CAP_DEF
    return adj if over_flag else -adj

def pace_adj(pace, over_flag):
    if pd.isna(pace):
        return 0.0
    delta = (float(pace) - BASE_PACE) / 20.0
    delta = float(np.clip(delta, -1, 1))
    adj = delta * CAP_PACE
    return adj if over_flag else -adj

df["MODEL_PROB_CTX_V2"] = [
    clip01(
        r["MODEL_PROB"]
        + def_adj(r["ESPN_OPP_DEF_PPG_PROXY"], is_over(r["OUTCOME"]))
        + pace_adj(r["ESPN_PACE_PROXY"], is_over(r["OUTCOME"]))
    )
    for _, r in df.iterrows()
]

df["EDGE_VS_BOOK_V2"] = df["MODEL_PROB_CTX_V2"] - df["BOOK_PROB"]

top10 = df.sort_values(
    ["MODEL_PROB_CTX_V2","EDGE_VS_BOOK_V2"],
    ascending=[False,False]
).head(10).copy()

top10.insert(0, "RANK", range(1, len(top10)+1))

top10.to_csv("hit_sheet_top10_ncaam_ctx_v2.csv", index=False)
df.to_csv("prop_engine_full_ncaam_ctx_v2.csv", index=False)

print("Saved: hit_sheet_top10_ncaam_ctx_v2.csv")
print("Saved: prop_engine_full_ncaam_ctx_v2.csv")

print(top10[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "AMERICAN_ODDS","MODEL_PROB_CTX_V2","EDGE_VS_BOOK_V2",
    "ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"
]])

Saved: hit_sheet_top10_ncaam_ctx_v2.csv
Saved: prop_engine_full_ncaam_ctx_v2.csv
    RANK             PLAYER        GAME                         PROP OUTCOME  \
0      1   Jeremy Fears Jr.  MSU at IND                      Assists    o3.5   
1      2           C.J. Cox  PUR at OSU                     Rebounds    o1.5   
2      3       Jaxon Kohler  MSU at IND          Three Pointers Made    u1.5   
4      4       Braden Smith  PUR at OSU  Points + Rebounds + Assists   u28.5   
7      5  Trey Kaufman-Renn  PUR at OSU                       Points   u14.5   
3      6          Nick Dorn  MSU at IND                     Rebounds    u2.5   
5      7    Lamar Wilkerson  MSU at IND                     Rebounds    u4.5   
6      8       Braden Smith  PUR at OSU                       Points   u15.5   
9      9     Bruce Thornton  PUR at OSU                     Rebounds    u5.5   
20    10        Oscar Cluff  PUR at OSU                     Rebounds    u6.5   

    AMERICAN_ODDS  MODEL_PROB_CTX_V2  

In [ ]:
print("home_id null:", df["home_id"].isna().sum(), " / ", len(df))
print("away_id null:", df["away_id"].isna().sum(), " / ", len(df))
print("opp_team_id null:", df["opp_team_id"].isna().sum(), " / ", len(df))

print("\nctx_df shape:", ctx_df.shape)
display(ctx_df.head(10))

home_id null: 51  /  100
away_id null: 0  /  100
opp_team_id null: 23  /  100

ctx_df shape: (3, 3)


,team_id,DEF_PPG_ALLOWED,PACE_PROXY
0,127,NaN,NaN
1,194,NaN,NaN
2,2509,NaN,NaN


In [ ]:
missing_home = sorted(df.loc[df["home_id"].isna(), "home_tok"].dropna().unique().tolist())
missing_away = sorted(df.loc[df["away_id"].isna(), "away_tok"].dropna().unique().tolist())

print("Missing home tokens:", missing_home)
print("Missing away tokens:", missing_away)

Missing home tokens: ['IND']
Missing away tokens: []


In [ ]:
import requests, pandas as pd, numpy as np, re
from difflib import SequenceMatcher
from datetime import datetime, timezone, timedelta

SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"
TEAM_SEARCH = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def parse_game_tokens(g):
    g = str(g).upper().strip().replace(" AT ", " at ")
    parts = [p.strip() for p in g.split(" at ")]
    if len(parts) != 2:
        return (None, None)
    return (parts[0], parts[1])

def norm(s):
    return re.sub(r"[^A-Z0-9]", "", str(s).upper().strip())

def ratio(a,b):
    return SequenceMatcher(None, a, b).ratio()

# Tokens we need
need=set()
for g in df["GAME"].astype(str).unique():
    a,h = parse_game_tokens(g)
    if a: need.add(a)
    if h: need.add(h)

# 1) Try scoreboard mapping across a few days
team_map={}
utc_now = datetime.now(timezone.utc)
dates = [(utc_now + timedelta(days=i)).strftime("%Y%m%d") for i in range(0, 6)]  # today+5 buffer

for d in dates:
    sb = get_json(SCOREBOARD, params={"dates": d, "limit": 300})
    for ev in sb.get("events", []) or []:
        comp = (ev.get("competitions") or [{}])[0]
        for c in comp.get("competitors", []) or []:
            t = c.get("team") or {}
            abbr = str(t.get("abbreviation","")).upper().strip()
            tid  = t.get("id")
            if abbr and tid and abbr in need and abbr not in team_map:
                team_map[abbr] = int(tid)

print("Scoreboard mapped:", len(team_map), "of", len(need))

# 2) Fallback: ESPN team search for missing tokens
def search_team(q):
    r = requests.get(TEAM_SEARCH, params={"limit": 50, "search": q}, timeout=30)
    r.raise_for_status()
    j = r.json()
    teams = j.get("sports",[{}])[0].get("leagues",[{}])[0].get("teams",[])
    out=[]
    for it in teams:
        t = it.get("team", {})
        out.append({
            "id": t.get("id"),
            "displayName": t.get("displayName",""),
            "abbreviation": t.get("abbreviation",""),
        })
    return pd.DataFrame(out)

missing = sorted(list(need - set(team_map.keys())))
print("Missing after scoreboard:", missing)

resolved = {}
for tok in missing:
    try:
        tdf = search_team(tok)
        if tdf.empty:
            continue

        # Prefer exact abbreviation match
        tdf["abbr_norm"] = tdf["abbreviation"].astype(str).map(norm)
        tok_norm = norm(tok)

        exact = tdf[tdf["abbr_norm"] == tok_norm]
        if len(exact) > 0:
            pick = exact.iloc[0]
        else:
            # Otherwise choose best fuzzy match between token and abbreviation/displayName
            tdf["score"] = tdf.apply(
                lambda r: max(ratio(tok_norm, norm(r["abbreviation"])), ratio(tok_norm, norm(r["displayName"]))),
                axis=1
            )
            pick = tdf.sort_values("score", ascending=False).iloc[0]

        if pd.notna(pick["id"]):
            resolved[tok] = int(pick["id"])
    except Exception as e:
        pass

team_map.update(resolved)

print("After team search mapped:", len(team_map), "of", len(need))
print("Still missing:", sorted(list(need - set(team_map.keys())))[:50])

# Apply mapping
df["away_tok"], df["home_tok"] = zip(*df["GAME"].astype(str).map(parse_game_tokens))
df["away_id"] = df["away_tok"].map(team_map)
df["home_id"] = df["home_tok"].map(team_map)

print("Missing home_id:", df["home_id"].isna().sum(), "| Missing away_id:", df["away_id"].isna().sum())

Scoreboard mapped: 3 of 4
Missing after scoreboard: ['IND']
After team search mapped: 4 of 4
Still missing: []
Missing home_id: 0 | Missing away_id: 0


In [ ]:
import pandas as pd, numpy as np
from functools import lru_cache
import requests
from difflib import get_close_matches

ESPN_TEAM_STATS = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics"

def safe_get(d, path, default=None):
    cur = d
    for p in path:
        if isinstance(cur, dict) and p in cur:
            cur = cur[p]
        else:
            return default
    return cur

@lru_cache(maxsize=1024)
def fetch_team_stats(team_id: int):
    url = ESPN_TEAM_STATS.format(team_id=int(team_id))
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    j = r.json()
    stats=[]
    cats = safe_get(j, ["results","stats","categories"], []) or []
    for cat in cats:
        for s in (cat.get("stats") or []):
            stats.append({"name": str(s.get("name","")), "value": s.get("value")})
    return pd.DataFrame(stats)

def find_stat(team_id: int, wanted_names):
    try:
        tdf = fetch_team_stats(int(team_id))
    except Exception:
        return np.nan
    if tdf.empty:
        return np.nan

    names = tdf["name"].astype(str).tolist()
    # exact
    for w in wanted_names:
        m = tdf.loc[tdf["name"].astype(str).str.lower() == w.lower(), "value"]
        if len(m):
            try: return float(m.iloc[0])
            except: pass
    # fuzzy
    for w in wanted_names:
        match = get_close_matches(w, names, n=1, cutoff=0.50)
        if match:
            m = tdf.loc[tdf["name"].astype(str) == match[0], "value"]
            if len(m):
                try: return float(m.iloc[0])
                except: pass
    return np.nan

needed_team_ids = sorted({int(x) for x in pd.concat([df["home_id"], df["away_id"]]).dropna().unique()})

rows=[]
for tid in needed_team_ids:
    def_ppg_allowed = find_stat(tid, ["pointsAllowedPerGame","oppPointsPerGame","opponentPointsPerGame","pointsAllowed"])
    pace_proxy      = find_stat(tid, ["possessionsPerGame","pace","tempo","adjTempo"])
    rows.append({"team_id": tid, "DEF_PPG_ALLOWED": def_ppg_allowed, "PACE_PROXY": pace_proxy})

ctx_df = pd.DataFrame(rows)

print("ctx_df shape:", ctx_df.shape)
print("DEF_PPG_ALLOWED null:", ctx_df["DEF_PPG_ALLOWED"].isna().sum(), " / ", len(ctx_df))
print("PACE_PROXY null:", ctx_df["PACE_PROXY"].isna().sum(), " / ", len(ctx_df))
display(ctx_df.head(10))

ctx_df shape: (4, 3)
DEF_PPG_ALLOWED null: 0  /  4
PACE_PROXY null: 4  /  4


,team_id,DEF_PPG_ALLOWED,PACE_PROXY
0,87,2161.0,NaN
1,127,2194.0,NaN
2,194,2227.0,NaN
3,2509,2313.0,NaN


In [ ]:
import pandas as pd, numpy as np, requests
from datetime import datetime, timezone, timedelta

SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def pull_scoreboard_totals(dates):
    rows=[]
    for d in dates:
        sb = get_json(SCOREBOARD, params={"dates": d, "limit": 300})
        for ev in sb.get("events", []) or []:
            comp = (ev.get("competitions") or [{}])[0]
            comps = comp.get("competitors") or []
            if len(comps) < 2:
                continue
            # team ids
            tids=[]
            pts=[]
            for c in comps:
                t = c.get("team") or {}
                tid = t.get("id")
                score = c.get("score")
                if tid is None:
                    continue
                tids.append(int(tid))
                try:
                    pts.append(float(score) if score is not None else np.nan)
                except:
                    pts.append(np.nan)
            if len(tids) != 2:
                continue
            total_pts = np.nan
            if len(pts)==2 and (not np.isnan(pts[0])) and (not np.isnan(pts[1])):
                total_pts = pts[0] + pts[1]

            # record for both teams (pace proxy uses total points)
            for tid in tids:
                rows.append({"team_id": tid, "date": d, "total_points": total_pts})
    return pd.DataFrame(rows)

# Use today + last 3 days buffer (helps if games aren’t final yet)
utc_now = datetime.now(timezone.utc)
dates = [(utc_now - timedelta(days=i)).strftime("%Y%m%d") for i in range(0, 4)]

totals_df = pull_scoreboard_totals(dates)

# Pace proxy: average total points (fallback baseline if NaN)
pace_proxy = (
    totals_df.groupby("team_id")["total_points"]
    .mean()
    .reset_index()
    .rename(columns={"total_points":"PACE_PROXY"})
)

print("pace_proxy rows:", pace_proxy.shape)
display(pace_proxy.head(10))

# Merge into ctx_df (overwrite NaN pace)
ctx_df = ctx_df.drop(columns=["PACE_PROXY"], errors="ignore").merge(pace_proxy, on="team_id", how="left")

print("\nAfter merge ctx_df:")
print("ctx_df shape:", ctx_df.shape)
print("PACE_PROXY null:", ctx_df["PACE_PROXY"].isna().sum(), "/", len(ctx_df))
display(ctx_df)

pace_proxy rows: (38, 2)


,team_id,PACE_PROXY
0,8,188.0
1,12,145.0
2,30,149.0
3,38,164.0
4,41,138.0
5,57,188.0
6,66,155.0
7,84,0.0
8,96,168.0
9,97,155.0



After merge ctx_df:
ctx_df shape: (4, 3)
PACE_PROXY null: 1 / 4


,team_id,DEF_PPG_ALLOWED,PACE_PROXY
0,87,2161.0,NaN
1,127,2194.0,75.0
2,194,2227.0,0.0
3,2509,2313.0,75.0


In [ ]:
import pandas as pd, numpy as np

# Use the totals_df we already built (team_id, total_points), but rebuild a richer frame from scoreboard pulls:
# We need team points + opponent points per game.

def pull_scoreboard_team_rows(dates):
    rows=[]
    for d in dates:
        sb = get_json(SCOREBOARD, params={"dates": d, "limit": 300})
        for ev in sb.get("events", []) or []:
            comp = (ev.get("competitions") or [{}])[0]
            comps = comp.get("competitors") or []
            if len(comps) < 2:
                continue

            # grab the two teams
            t0, t1 = comps[0], comps[1]

            def tid_and_pts(c):
                t = c.get("team") or {}
                tid = t.get("id")
                score = c.get("score")
                try:
                    pts = float(score) if score is not None else np.nan
                except:
                    pts = np.nan
                return (int(tid) if tid is not None else None, pts)

            id0, p0 = tid_and_pts(t0)
            id1, p1 = tid_and_pts(t1)
            if id0 is None or id1 is None:
                continue

            total = np.nan
            if not np.isnan(p0) and not np.isnan(p1):
                total = p0 + p1

            # row for each team: points_for, points_against, total
            rows.append({"team_id": id0, "date": d, "pf": p0, "pa": p1, "total": total})
            rows.append({"team_id": id1, "date": d, "pf": p1, "pa": p0, "total": total})
    return pd.DataFrame(rows)

team_games_df = pull_scoreboard_team_rows(dates)

# Compute proxies:
# DEF_PPG_ALLOWED proxy = avg points against (pa)
# PACE proxy = avg total points
def_proxy = team_games_df.groupby("team_id")["pa"].mean().reset_index().rename(columns={"pa":"DEF_PPG_ALLOWED_PROXY"})
pace_proxy2 = team_games_df.groupby("team_id")["total"].mean().reset_index().rename(columns={"total":"PACE_PROXY"})

# Merge into ctx_df and override the bad ESPN stat pull
ctx_df = ctx_df.drop(columns=["DEF_PPG_ALLOWED","PACE_PROXY"], errors="ignore") \
               .merge(def_proxy, on="team_id", how="left") \
               .merge(pace_proxy2, on="team_id", how="left")

# Fill baselines if missing
ctx_df["DEF_PPG_ALLOWED_PROXY"] = ctx_df["DEF_PPG_ALLOWED_PROXY"].fillna(72.0)
ctx_df["PACE_PROXY"] = ctx_df["PACE_PROXY"].replace(0, np.nan).fillna(145.0)

# Rename to match downstream CTX cell expectations
ctx_df = ctx_df.rename(columns={"DEF_PPG_ALLOWED_PROXY":"DEF_PPG_ALLOWED"})

print("ctx_df after scoreboard-derived proxies:")
display(ctx_df)

ctx_df after scoreboard-derived proxies:


,team_id,DEF_PPG_ALLOWED,PACE_PROXY
0,87,72.0,145.0
1,127,37.0,75.0
2,194,0.0,145.0
3,2509,38.0,75.0


In [ ]:
import pandas as pd, numpy as np
from datetime import datetime, timezone, timedelta

SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def pull_final_team_rows(dates):
    rows=[]
    for d in dates:
        sb = get_json(SCOREBOARD, params={"dates": d, "limit": 300})
        for ev in sb.get("events", []) or []:
            comp = (ev.get("competitions") or [{}])[0]

            # FINAL ONLY filter
            status = comp.get("status", {}).get("type", {})
            if not status.get("completed", False):
                continue

            comps = comp.get("competitors") or []
            if len(comps) < 2:
                continue

            def tid_and_pts(c):
                t = c.get("team") or {}
                tid = t.get("id")
                score = c.get("score")
                try:
                    pts = float(score) if score is not None else np.nan
                except:
                    pts = np.nan
                return (int(tid) if tid is not None else None, pts)

            id0, p0 = tid_and_pts(comps[0])
            id1, p1 = tid_and_pts(comps[1])
            if id0 is None or id1 is None:
                continue
            if np.isnan(p0) or np.isnan(p1):
                continue

            total = p0 + p1
            rows.append({"team_id": id0, "pf": p0, "pa": p1, "total": total})
            rows.append({"team_id": id1, "pf": p1, "pa": p0, "total": total})
    return pd.DataFrame(rows)

# look back 14 days so we definitely have finals for most teams
utc_now = datetime.now(timezone.utc)
dates = [(utc_now - timedelta(days=i)).strftime("%Y%m%d") for i in range(0, 14)]

final_df = pull_final_team_rows(dates)
print("Final games rows:", final_df.shape)

# proxies from FINALS ONLY
def_proxy = final_df.groupby("team_id")["pa"].mean().reset_index().rename(columns={"pa":"DEF_PPG_ALLOWED"})
pace_proxy = final_df.groupby("team_id")["total"].mean().reset_index().rename(columns={"total":"PACE_PROXY"})

# Build ctx_df for the teams we actually need today
needed_team_ids = sorted({int(x) for x in pd.concat([df["home_id"], df["away_id"]]).dropna().unique()})
ctx_df = pd.DataFrame({"team_id": needed_team_ids}).merge(def_proxy, on="team_id", how="left").merge(pace_proxy, on="team_id", how="left")

# baselines if no finals found
ctx_df["DEF_PPG_ALLOWED"] = ctx_df["DEF_PPG_ALLOWED"].fillna(72.0)
ctx_df["PACE_PROXY"] = ctx_df["PACE_PROXY"].fillna(145.0)

# sanity
print("ctx_df shape:", ctx_df.shape)
print("DEF range:", (ctx_df["DEF_PPG_ALLOWED"].min(), ctx_df["DEF_PPG_ALLOWED"].max()))
print("PACE range:", (ctx_df["PACE_PROXY"].min(), ctx_df["PACE_PROXY"].max()))
display(ctx_df.head(10))

Final games rows: (266, 4)
ctx_df shape: (4, 3)
DEF range: (64.33333333333333, 100.0)
PACE range: (139.0, 159.33333333333334)


,team_id,DEF_PPG_ALLOWED,PACE_PROXY
0,87,100.000000,156.000000
1,127,64.333333,139.000000
2,194,67.500000,140.500000
3,2509,77.000000,159.333333


In [ ]:
def discord_hit_sheet_top10(df_top10):
    lines = []
    lines.append("NCAA PROP ENGINE — TOP 10 MOST LIKELY (CTX v2)")
    lines.append("—")
    for _, r in df_top10.iterrows():
        odds = f"({int(r.AMERICAN_ODDS)})" if pd.notna(r.AMERICAN_ODDS) else ""
        lines.append(
            f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n"
            f"{r.PROP} {r.OUTCOME} {odds}\n"
            f"Model: {r.MODEL_PROB_CTX_V2*100:.1f}% | Edge: {r.EDGE_VS_BOOK_V2*100:+.1f} pts | Opp DEF PPG: {r.ESPN_OPP_DEF_PPG_PROXY:.1f} | Pace: {r.ESPN_PACE_PROXY:.1f}"
        )
    return "\n".join(lines)

print(discord_hit_sheet_top10(top10))

NCAA PROP ENGINE — TOP 10 MOST LIKELY (CTX v2)
—
1) Jeremy Fears Jr. — MSU at IND
Assists o3.5 (100)
Model: 85.8% | Edge: +35.8 pts | Opp DEF PPG: nan | Pace: nan
2) C.J. Cox — PUR at OSU
Rebounds o1.5 (-200)
Model: 74.0% | Edge: +7.3 pts | Opp DEF PPG: nan | Pace: nan
3) Jaxon Kohler — MSU at IND
Three Pointers Made u1.5 (-139)
Model: 70.8% | Edge: +12.7 pts | Opp DEF PPG: nan | Pace: nan
4) Braden Smith — PUR at OSU
Points + Rebounds + Assists u28.5 (-105)
Model: 69.0% | Edge: +17.8 pts | Opp DEF PPG: nan | Pace: nan
5) Trey Kaufman-Renn — PUR at OSU
Points u14.5 (-102)
Model: 67.2% | Edge: +16.7 pts | Opp DEF PPG: nan | Pace: nan
6) Nick Dorn — MSU at IND
Rebounds u2.5 (117)
Model: 66.7% | Edge: +20.6 pts | Opp DEF PPG: nan | Pace: nan
7) Lamar Wilkerson — MSU at IND
Rebounds u4.5 (-145)
Model: 66.1% | Edge: +6.9 pts | Opp DEF PPG: nan | Pace: nan
8) Braden Smith — PUR at OSU
Points u15.5 (-115)
Model: 65.8% | Edge: +12.3 pts | Opp DEF PPG: nan | Pace: nan
9) Bruce Thornton — PUR at

In [ ]:
# ==============================
# REAPPLY CTX USING NEW ctx_df
# ==============================

# 1) rebuild opponent id cleanly
def infer_player_is_home(r):
    if "HM_AW" in r.index and pd.notna(r["HM_AW"]):
        return float(r["HM_AW"]) >= 0.5
    return False

df["opp_team_id"] = [
    r["away_id"] if infer_player_is_home(r) else r["home_id"]
    for _, r in df.iterrows()
]

# 2) drop any old CTX columns
for col in [
    "ESPN_OPP_DEF_PPG_PROXY",
    "ESPN_PACE_PROXY",
    "MODEL_PROB_CTX_V2",
    "EDGE_VS_BOOK_V2"
]:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

# 3) merge the FINAL-only ctx_df
df = df.merge(ctx_df, left_on="opp_team_id", right_on="team_id", how="left")
df.rename(columns={
    "DEF_PPG_ALLOWED": "ESPN_OPP_DEF_PPG_PROXY",
    "PACE_PROXY": "ESPN_PACE_PROXY"
}, inplace=True)
df.drop(columns=["team_id"], inplace=True)

# 4) CTX adjustment (total points pace baseline)
BASE_DEF_PPG = 72.0
BASE_PACE    = 145.0
CAP_DEF  = 0.025
CAP_PACE = 0.015

def clip01(x):
    return float(np.clip(x, 0.01, 0.99))

def is_over(outcome):
    return str(outcome).lower().strip().startswith("o")

def def_adj(def_ppg, over_flag):
    delta = (float(def_ppg) - BASE_DEF_PPG) / 20.0
    delta = float(np.clip(delta, -1, 1))
    adj = delta * CAP_DEF
    return adj if over_flag else -adj

def pace_adj(pace, over_flag):
    delta = (float(pace) - BASE_PACE) / 20.0
    delta = float(np.clip(delta, -1, 1))
    adj = delta * CAP_PACE
    return adj if over_flag else -adj

df["MODEL_PROB_CTX_V2"] = [
    clip01(
        r["MODEL_PROB"]
        + def_adj(r["ESPN_OPP_DEF_PPG_PROXY"], is_over(r["OUTCOME"]))
        + pace_adj(r["ESPN_PACE_PROXY"], is_over(r["OUTCOME"]))
    )
    for _, r in df.iterrows()
]

df["EDGE_VS_BOOK_V2"] = df["MODEL_PROB_CTX_V2"] - df["BOOK_PROB"]

# 5) rebuild top10
top10 = df.sort_values(
    ["MODEL_PROB_CTX_V2","EDGE_VS_BOOK_V2"],
    ascending=[False,False]
).head(10).copy()

top10.insert(0, "RANK", range(1, len(top10)+1))

print(top10[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "MODEL_PROB_CTX_V2",
    "EDGE_VS_BOOK_V2",
    "ESPN_OPP_DEF_PPG_PROXY",
    "ESPN_PACE_PROXY"
]])

    RANK             PLAYER        GAME                         PROP OUTCOME  \
0      1   Jeremy Fears Jr.  MSU at IND                      Assists    o3.5   
1      2           C.J. Cox  PUR at OSU                     Rebounds    o1.5   
2      3       Jaxon Kohler  MSU at IND          Three Pointers Made    u1.5   
3      4          Nick Dorn  MSU at IND                     Rebounds    u2.5   
5      5    Lamar Wilkerson  MSU at IND                     Rebounds    u4.5   
4      6       Braden Smith  PUR at OSU  Points + Rebounds + Assists   u28.5   
7      7  Trey Kaufman-Renn  PUR at OSU                       Points   u14.5   
6      8       Braden Smith  PUR at OSU                       Points   u15.5   
26     9   Jeremy Fears Jr.  MSU at IND                       Points   u15.5   
9     10     Bruce Thornton  PUR at OSU                     Rebounds    u5.5   

    MODEL_PROB_CTX_V2  EDGE_VS_BOOK_V2  ESPN_OPP_DEF_PPG_PROXY  \
0            0.843979         0.343979               

In [ ]:
# ==============================
# FINAL CTX REAPPLY (KEEP THIS CELL)
# ==============================

import numpy as np
import pandas as pd

def infer_player_is_home(r):
    if "HM_AW" in r.index and pd.notna(r["HM_AW"]):
        return float(r["HM_AW"]) >= 0.5
    return False

# opponent id (uses HM_AW if present; otherwise defaults to home opponent)
df["opp_team_id"] = [
    r["away_id"] if infer_player_is_home(r) else r["home_id"]
    for _, r in df.iterrows()
]

# drop stale ctx columns so we don't carry NaNs forward
for col in ["ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY","MODEL_PROB_CTX_V2","EDGE_VS_BOOK_V2"]:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

# merge latest ctx_df
df = df.merge(ctx_df, left_on="opp_team_id", right_on="team_id", how="left")
df.rename(columns={
    "DEF_PPG_ALLOWED": "ESPN_OPP_DEF_PPG_PROXY",
    "PACE_PROXY": "ESPN_PACE_PROXY"
}, inplace=True)
df.drop(columns=["team_id"], inplace=True)

# CTX adjustment params
BASE_DEF_PPG = 72.0
BASE_PACE    = 145.0   # total-points pace proxy baseline
CAP_DEF  = 0.025
CAP_PACE = 0.015

def clip01(x): return float(np.clip(x, 0.01, 0.99))
def is_over(outcome): return str(outcome).lower().strip().startswith("o")

def def_adj(def_ppg, over_flag):
    delta = (float(def_ppg) - BASE_DEF_PPG) / 20.0
    delta = float(np.clip(delta, -1, 1))
    adj = delta * CAP_DEF
    return adj if over_flag else -adj

def pace_adj(pace, over_flag):
    delta = (float(pace) - BASE_PACE) / 20.0
    delta = float(np.clip(delta, -1, 1))
    adj = delta * CAP_PACE
    return adj if over_flag else -adj

# recompute ctx prob + edge
df["MODEL_PROB_CTX_V2"] = [
    clip01(
        r["MODEL_PROB"]
        + def_adj(r["ESPN_OPP_DEF_PPG_PROXY"], is_over(r["OUTCOME"]))
        + pace_adj(r["ESPN_PACE_PROXY"], is_over(r["OUTCOME"]))
    )
    for _, r in df.iterrows()
]
df["EDGE_VS_BOOK_V2"] = df["MODEL_PROB_CTX_V2"] - df["BOOK_PROB"]

# top10
top10 = df.sort_values(["MODEL_PROB_CTX_V2","EDGE_VS_BOOK_V2"], ascending=[False,False]).head(10).copy()
top10.insert(0, "RANK", range(1, len(top10)+1))

display(top10[[
    "RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "MODEL_PROB_CTX_V2","EDGE_VS_BOOK_V2","ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"
]])

# exports
top10.to_csv("hit_sheet_top10_ncaam_ctx_v2.csv", index=False)
df.to_csv("prop_engine_full_ncaam_ctx_v2.csv", index=False)

print("Saved: hit_sheet_top10_ncaam_ctx_v2.csv")
print("Saved: prop_engine_full_ncaam_ctx_v2.csv")

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,MODEL_PROB_CTX_V2,EDGE_VS_BOOK_V2,ESPN_OPP_DEF_PPG_PROXY,ESPN_PACE_PROXY
0,1,Jeremy Fears Jr.,MSU at IND,Assists,o3.5,100,0.843979,0.343979,64.333333,139.000000
1,2,C.J. Cox,PUR at OSU,Rebounds,o1.5,-200,0.756750,0.090083,77.000000,159.333333
2,3,Jaxon Kohler,MSU at IND,Three Pointers Made,u1.5,-139,0.722283,0.140693,64.333333,139.000000
3,4,Nick Dorn,MSU at IND,Rebounds,u2.5,117,0.680833,0.220004,64.333333,139.000000
5,5,Lamar Wilkerson,MSU at IND,Rebounds,u4.5,-145,0.675083,0.083247,64.333333,139.000000
4,6,Braden Smith,PUR at OSU,Points + Rebounds + Assists,u28.5,-105,0.673500,0.161305,77.000000,159.333333
7,7,Trey Kaufman-Renn,PUR at OSU,Points,u14.5,-102,0.654750,0.149800,77.000000,159.333333
6,8,Braden Smith,PUR at OSU,Points,u15.5,-115,0.641000,0.106116,77.000000,159.333333
26,9,Jeremy Fears Jr.,MSU at IND,Points,u15.5,-117,0.625283,0.086113,64.333333,139.000000
9,10,Bruce Thornton,PUR at OSU,Rebounds,u5.5,-115,0.615050,0.080166,77.000000,159.333333


Saved: hit_sheet_top10_ncaam_ctx_v2.csv
Saved: prop_engine_full_ncaam_ctx_v2.csv


In [ ]:
# ==============================
# PROJECTION LAYER (mean μ) + PROBABILITY
# Converts historical hit-rate at the line into a projected mean,
# then computes hit probability from a distribution model.
# ==============================

import numpy as np
import pandas as pd
import re

# Try scipy for stable CDF/PPF. If not available, use a lightweight approximation fallback.
try:
    from scipy.stats import norm
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False
    import math
    # Normal CDF approx
    def _norm_cdf(x):
        return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))
    # Inverse CDF approx (Acklam)
    def _norm_ppf(p):
        # clamp
        p = min(max(p, 1e-12), 1-1e-12)
        # Coefficients in rational approximations
        a = [-3.969683028665376e+01, 2.209460984245205e+02, -2.759285104469687e+02,
             1.383577518672690e+02, -3.066479806614716e+01, 2.506628277459239e+00]
        b = [-5.447609879822406e+01, 1.615858368580409e+02, -1.556989798598866e+02,
             6.680131188771972e+01, -1.328068155288572e+01]
        c = [-7.784894002430293e-03, -3.223964580411365e-01, -2.400758277161838e+00,
             -2.549732539343734e+00, 4.374664141464968e+00, 2.938163982698783e+00]
        d = [7.784695709041462e-03, 3.224671290700398e-01, 2.445134137142996e+00,
             3.754408661907416e+00]
        plow = 0.02425
        phigh = 1 - plow
        if p < plow:
            q = math.sqrt(-2*math.log(p))
            return (((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                   ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
        if p > phigh:
            q = math.sqrt(-2*math.log(1-p))
            return -(((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                    ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
        q = p - 0.5
        r = q*q
        return (((((a[0]*r + a[1])*r + a[2])*r + a[3])*r + a[4])*r + a[5]) * q / \
               (((((b[0]*r + b[1])*r + b[2])*r + b[3])*r + b[4])*r + 1)

def parse_line(outcome_str):
    # outcome like "o3.5" or "u28.5"
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

def stat_sigma(prop_name, line):
    # Simple, stable sigma heuristics by market type
    p = str(prop_name).lower()
    if "points + rebounds + assists" in p or "pra" in p:
        return max(3.8, 0.18*line)
    if "points + rebounds" in p or "pr" in p:
        return max(3.2, 0.17*line)
    if "points + assists" in p or "pa" in p:
        return max(3.0, 0.16*line)
    if "rebounds + assists" in p or "ra" in p:
        return max(2.6, 0.16*line)
    if "points" in p:
        return max(2.8, 0.18*line)
    if "rebounds" in p:
        return max(1.8, 0.22*max(line, 1.0))
    if "assists" in p:
        return max(1.6, 0.22*max(line, 1.0))
    if "three pointers" in p or "3" in p:
        return max(0.9, 0.35*max(line, 1.0))
    # fallback
    return max(2.0, 0.18*max(line, 1.0))

def implied_mu_from_prob(line, p_over, sigma):
    """
    Assume X ~ Normal(mu, sigma).
    With a continuity correction:
      P(X > line) ≈ 1 - Φ((line+0.5 - mu)/sigma)
    Solve for mu given p_over.
    """
    p_over = float(np.clip(p_over, 0.02, 0.98))
    if SCIPY_OK:
        z = norm.ppf(1 - p_over)  # because 1-Φ(z)=p_over -> z=Φ^{-1}(1-p_over)
    else:
        z = _norm_ppf(1 - p_over)
    mu = (line + 0.5) - sigma * z
    return float(mu)

def prob_over_from_mu(line, mu, sigma):
    # continuity correction: P(X > line) = 1 - Φ((line+0.5 - mu)/sigma)
    x = ((line + 0.5) - mu) / sigma
    if SCIPY_OK:
        return float(1 - norm.cdf(x))
    else:
        return float(1 - _norm_cdf(x))

# Build projection mean and projection probability
sides, lines = zip(*df["OUTCOME"].map(parse_line))
df["SIDE"] = sides
df["LINE"] = lines

# Base "historical" probability at the line:
# Use your MODEL_PROB as the probability of the selected outcome (over/under) hitting.
df["HIST_PROB"] = df["MODEL_PROB"].astype(float)

# Convert HIST_PROB into an implied mean μ under a Normal model
sigmas = []
mu_base = []
p_over_base = []
for _, r in df.iterrows():
    line = r["LINE"]
    sigma = stat_sigma(r["PROP"], line)
    sigmas.append(sigma)

    # Need p_over for inversion; if the pick is UNDER, p_over = 1 - HIST_PROB
    if r["SIDE"] == "over":
        p_over = r["HIST_PROB"]
    else:
        p_over = 1 - r["HIST_PROB"]
    p_over = float(np.clip(p_over, 0.02, 0.98))
    p_over_base.append(p_over)

    mu = implied_mu_from_prob(line, p_over, sigma)
    mu_base.append(mu)

df["SIGMA"] = sigmas
df["PROJ_MU_BASE"] = mu_base
df["P_OVER_BASE"] = p_over_base

# Projection probability for the chosen side (before context)
proj_prob = []
for _, r in df.iterrows():
    p_over = prob_over_from_mu(r["LINE"], r["PROJ_MU_BASE"], r["SIGMA"])
    proj_prob.append(p_over if r["SIDE"] == "over" else (1 - p_over))
df["PROJ_PROB_BASE"] = proj_prob

print("Projection layer built.")
print(df[["PLAYER","PROP","OUTCOME","LINE","SIGMA","HIST_PROB","PROJ_MU_BASE","PROJ_PROB_BASE"]].head(8))

Projection layer built.
              PLAYER                         PROP OUTCOME  LINE  SIGMA  \
0   Jeremy Fears Jr.                      Assists    o3.5   3.5   1.60   
1           C.J. Cox                     Rebounds    o1.5   1.5   1.80   
2       Jaxon Kohler          Three Pointers Made    u1.5   1.5   0.90   
3          Nick Dorn                     Rebounds    u2.5   2.5   1.80   
4       Braden Smith  Points + Rebounds + Assists   u28.5  28.5   5.13   
5    Lamar Wilkerson                     Rebounds    u4.5   4.5   1.80   
6       Braden Smith                       Points   u15.5  15.5   2.80   
7  Trey Kaufman-Renn                       Points   u14.5  14.5   2.80   

   HIST_PROB  PROJ_MU_BASE  PROJ_PROB_BASE  
0   0.858062      5.714648        0.858062  
1   0.739750      3.156635        0.739750  
2   0.708200      1.506680        0.708200  
3   0.666750      2.224278        0.666750  
4   0.690500     26.449015        0.690500  
5   0.661000      4.252651        0.661

In [ ]:
# ==============================
# APPLY CTX TO PROJECTION MEAN μ (NOT PROB) + BLEND WITH HIST
# ==============================

# Sanity: require opponent features already present from your fixed run
req = ["ESPN_OPP_DEF_PPG_PROXY", "ESPN_PACE_PROXY"]
missing = [c for c in req if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns for CTX mean adjustment: {missing}. Run your ctx merge fix first.")

BASE_DEF_PPG = 72.0
BASE_PACE_TOT = 145.0  # total points proxy baseline

# Mean-shift coefficients by market (heuristics; stable + capped)
# Units are "stat units per 20 points of DEF delta" and "stat units per 20 points of pace(total) delta"
BETA = {
    "points":      (1.00, 0.60),
    "rebounds":    (0.35, 0.25),
    "assists":     (0.30, 0.22),
    "three":       (0.12, 0.10),
    "pra":         (1.35, 0.85),
    "default":     (0.60, 0.40),
}

def market_key(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p:
        return "pra"
    if "three pointers" in p or "3" in p:
        return "three"
    if "points" in p and ("rebounds" in p or "assists" in p):
        return "pra"
    if "points" in p:
        return "points"
    if "rebounds" in p:
        return "rebounds"
    if "assists" in p:
        return "assists"
    return "default"

# Caps on mean shifts so we don’t overreact
MU_SHIFT_CAP = {
    "points": 2.0,
    "rebounds": 1.2,
    "assists": 1.1,
    "three": 0.6,
    "pra": 3.0,
    "default": 1.5,
}

def clip(v, lo, hi):
    return float(max(lo, min(hi, v)))

# Apply CTX to mean μ
mu_ctx = []
for _, r in df.iterrows():
    key = market_key(r["PROP"])
    b_def, b_pace = BETA.get(key, BETA["default"])
    cap = MU_SHIFT_CAP.get(key, MU_SHIFT_CAP["default"])

    d_def = float(r["ESPN_OPP_DEF_PPG_PROXY"]) - BASE_DEF_PPG
    d_pace = float(r["ESPN_PACE_PROXY"]) - BASE_PACE_TOT

    # scale deltas to per-20 units to match our beta definition
    shift = (b_def * (d_def/20.0)) + (b_pace * (d_pace/20.0))
    shift = clip(shift, -cap, cap)

    mu_ctx.append(float(r["PROJ_MU_BASE"]) + shift)

df["PROJ_MU_CTX"] = mu_ctx

# Recompute projection probability using the context-adjusted μ
proj_prob_ctx = []
for _, r in df.iterrows():
    p_over = prob_over_from_mu(r["LINE"], r["PROJ_MU_CTX"], r["SIGMA"])
    proj_prob_ctx.append(p_over if r["SIDE"] == "over" else (1 - p_over))
df["PROJ_PROB_CTX"] = proj_prob_ctx

# Blend with historical hit-rate probability
# Weight can be tuned; this is a stable default.
W_HIST = 0.60
W_PROJ = 0.40
df["BLEND_PROB"] = (W_HIST * df["HIST_PROB"]) + (W_PROJ * df["PROJ_PROB_CTX"])
df["BLEND_PROB"] = df["BLEND_PROB"].clip(0.01, 0.99)

# Recompute edge vs book using blended prob
df["EDGE_BLEND"] = df["BLEND_PROB"] - df["BOOK_PROB"]

# Rank output (blend-first)
top10_blend = df.sort_values(["BLEND_PROB","EDGE_BLEND"], ascending=[False, False]).head(10).copy()
top10_blend.insert(0, "RANK", range(1, len(top10_blend)+1))

display(top10_blend[[
    "RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "HIST_PROB","PROJ_MU_BASE","PROJ_MU_CTX","PROJ_PROB_CTX","BLEND_PROB",
    "EDGE_BLEND","ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"
]])

# Exports
top10_blend.to_csv("hit_sheet_top10_ncaam_blend_ctx_mu.csv", index=False)
df.to_csv("prop_engine_full_ncaam_blend_ctx_mu.csv", index=False)

print("Saved: hit_sheet_top10_ncaam_blend_ctx_mu.csv")
print("Saved: prop_engine_full_ncaam_blend_ctx_mu.csv")

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,HIST_PROB,PROJ_MU_BASE,PROJ_MU_CTX,PROJ_PROB_CTX,BLEND_PROB,EDGE_BLEND,ESPN_OPP_DEF_PPG_PROXY,ESPN_PACE_PROXY
0,1,Jeremy Fears Jr.,MSU at IND,Assists,o3.5,100,0.858062,5.714648,5.533648,0.831102,0.847278,0.347278,64.333333,139.000000
1,2,C.J. Cox,PUR at OSU,Rebounds,o1.5,-200,0.739750,3.156635,3.423301,0.785447,0.758029,0.091362,77.000000,159.333333
2,3,Jaxon Kohler,MSU at IND,Three Pointers Made,u1.5,-139,0.708200,1.506680,1.430680,0.736495,0.719518,0.137928,64.333333,139.000000
3,4,Nick Dorn,MSU at IND,Rebounds,u2.5,117,0.666750,2.224278,2.015112,0.707866,0.683196,0.222367,64.333333,139.000000
5,5,Lamar Wilkerson,MSU at IND,Rebounds,u4.5,-145,0.661000,4.252651,4.043484,0.702428,0.677571,0.085735,64.333333,139.000000
4,6,Braden Smith,PUR at OSU,Points + Rebounds + Assists,u28.5,-105,0.690500,26.449015,27.395681,0.622758,0.663403,0.151208,77.000000,159.333333
26,7,Jeremy Fears Jr.,MSU at IND,Points,u15.5,-117,0.611200,15.209146,14.645812,0.685679,0.640992,0.101821,64.333333,139.000000
7,8,Trey Kaufman-Renn,PUR at OSU,Points,u14.5,-102,0.671750,13.754698,14.434698,0.580000,0.635050,0.130100,77.000000,159.333333
6,9,Braden Smith,PUR at OSU,Points,u15.5,-115,0.658000,14.860370,15.540370,0.565195,0.620878,0.085994,77.000000,159.333333
14,10,Carson Cooper,MSU at IND,Rebounds,u7.5,-141,0.597050,7.557706,7.348540,0.641295,0.614748,0.029686,64.333333,139.000000


Saved: hit_sheet_top10_ncaam_blend_ctx_mu.csv
Saved: prop_engine_full_ncaam_blend_ctx_mu.csv


In [ ]:
# ==============================
# DISTRIBUTION-BY-MARKET PROJECTION PROB (μ-shift CTX) + BLEND
# ==============================

import numpy as np
import pandas as pd
import re

# SciPy recommended for stable Poisson/NB CDF. If missing, we fall back to Normal everywhere.
try:
    from scipy.stats import norm, poisson, nbinom
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

def market_key(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p:
        return "pra"
    if "three pointers" in p or "3" in p:
        return "three"
    if "steals" in p:
        return "steals"
    if "blocks" in p:
        return "blocks"
    if "assists" in p:
        return "assists"
    if "rebounds" in p:
        return "rebounds"
    if "points" in p:
        return "points"
    return "default"

# --- distribution selection ---
def dist_for_market(mkey):
    if not SCIPY_OK:
        return "normal"
    if mkey in ["three", "steals", "blocks"]:
        return "poisson"
    if mkey in ["assists", "rebounds"]:
        return "nbinom"
    if mkey in ["points", "pra"]:
        return "normal"
    return "normal"

# --- Normal helpers (continuity correction) ---
def normal_sigma(mkey, line):
    line = float(line)
    if mkey == "pra":     return max(3.8, 0.18*line)
    if mkey == "points":  return max(2.8, 0.18*line)
    if mkey == "rebounds":return max(1.8, 0.22*max(line,1.0))
    if mkey == "assists": return max(1.6, 0.22*max(line,1.0))
    if mkey == "three":   return max(0.9, 0.35*max(line,1.0))
    return max(2.0, 0.18*max(line,1.0))

def normal_prob_over(line, mu, sigma):
    # P(X > line) with continuity correction
    x = ((line + 0.5) - mu)/sigma
    return float(1 - norm.cdf(x))

def normal_mu_from_pover(line, p_over, sigma):
    p_over = float(np.clip(p_over, 0.02, 0.98))
    z = norm.ppf(1 - p_over)
    return float((line + 0.5) - sigma*z)

# --- Poisson helpers (integer count markets) ---
def poisson_prob_over(line, lam):
    # for counts: P(X > line) = 1 - F(floor(line))
    k = int(np.floor(line))
    return float(1 - poisson.cdf(k, mu=lam))

def poisson_lam_from_pover(line, p_over):
    # solve for lambda by binary search
    p_over = float(np.clip(p_over, 0.02, 0.98))
    lo, hi = 0.05, max(10.0, line + 10.0)
    for _ in range(60):
        mid = (lo+hi)/2
        p = poisson_prob_over(line, mid)
        if p > p_over:
            hi = mid
        else:
            lo = mid
    return float((lo+hi)/2)

# --- Negative Binomial helpers (overdispersed counts) ---
# parameterization: NB(n, p) with mean = n*(1-p)/p
# We'll build via mean (mu) and dispersion (k): Var = mu + mu^2/k
def nb_params_from_mu_k(mu, k):
    mu = max(0.05, float(mu))
    k = max(0.5, float(k))
    p = k/(k+mu)
    n = k
    return n, p

def nb_prob_over(line, mu, k):
    # counts: P(X > floor(line))
    n, p = nb_params_from_mu_k(mu, k)
    cut = int(np.floor(line))
    return float(1 - nbinom.cdf(cut, n=n, p=p))

def nb_mu_from_pover(line, p_over, k):
    # solve for mu by binary search given dispersion k
    p_over = float(np.clip(p_over, 0.02, 0.98))
    lo, hi = 0.05, max(15.0, line + 15.0)
    for _ in range(60):
        mid = (lo+hi)/2
        p = nb_prob_over(line, mid, k)
        if p > p_over:
            hi = mid
        else:
            lo = mid
    return float((lo+hi)/2)

def nb_k_for_market(mkey, line):
    # Higher k = closer to Poisson; lower k = more variance.
    # These are stable defaults; tune later with backtests.
    if mkey == "assists": return 4.0
    if mkey == "rebounds": return 5.0
    return 4.5

# -----------------------
# Build SIDE/LINE/HIST_PROB
# -----------------------
df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))
df["HIST_PROB"] = df["MODEL_PROB"].astype(float)

# -----------------------
# Implied μ from HIST_PROB at the line (distribution-specific)
# -----------------------
mu_base = []
dist_used = []
aux_param = []  # sigma for normal; k for nbinom; nan for poisson

for _, r in df.iterrows():
    mkey = market_key(r["PROP"])
    dist = dist_for_market(mkey)
    dist_used.append(dist)

    # Need p_over (convert from picked side)
    if r["SIDE"] == "over":
        p_over = r["HIST_PROB"]
    else:
        p_over = 1 - r["HIST_PROB"]
    p_over = float(np.clip(p_over, 0.02, 0.98))

    line = float(r["LINE"])

    if dist == "poisson":
        lam = poisson_lam_from_pover(line, p_over)
        mu_base.append(lam)
        aux_param.append(np.nan)

    elif dist == "nbinom":
        k = nb_k_for_market(mkey, line)
        mu = nb_mu_from_pover(line, p_over, k)
        mu_base.append(mu)
        aux_param.append(k)

    else:  # normal
        sigma = normal_sigma(mkey, line)
        mu = normal_mu_from_pover(line, p_over, sigma)
        mu_base.append(mu)
        aux_param.append(sigma)

df["DIST_USED"] = dist_used
df["PROJ_MU_BASE"] = mu_base
df["AUX_PARAM"] = aux_param  # sigma or k

# -----------------------
# Apply CTX to μ (same idea you asked for)
# -----------------------
BASE_DEF_PPG = 72.0
BASE_PACE_TOT = 145.0

# mean-shift coefficients (stat units per 20 DEF delta, per 20 pace delta)
BETA = {
    "points":   (1.00, 0.60),
    "rebounds": (0.35, 0.25),
    "assists":  (0.30, 0.22),
    "three":    (0.12, 0.10),
    "steals":   (0.08, 0.06),
    "blocks":   (0.08, 0.06),
    "pra":      (1.35, 0.85),
    "default":  (0.60, 0.40),
}
MU_SHIFT_CAP = {
    "points": 2.0, "rebounds": 1.2, "assists": 1.1,
    "three": 0.6, "steals": 0.4, "blocks": 0.4,
    "pra": 3.0, "default": 1.5
}
def clip(v, lo, hi): return float(max(lo, min(hi, v)))

# Require ctx columns
for col in ["ESPN_OPP_DEF_PPG_PROXY", "ESPN_PACE_PROXY"]:
    if col not in df.columns:
        raise ValueError(f"Missing {col}. Run your ESPN ctx merge fix cell first.")

mu_ctx = []
mu_shift = []
for _, r in df.iterrows():
    mkey = market_key(r["PROP"])
    b_def, b_pace = BETA.get(mkey, BETA["default"])
    cap = MU_SHIFT_CAP.get(mkey, MU_SHIFT_CAP["default"])

    d_def = float(r["ESPN_OPP_DEF_PPG_PROXY"]) - BASE_DEF_PPG
    d_pace = float(r["ESPN_PACE_PROXY"]) - BASE_PACE_TOT

    shift = (b_def*(d_def/20.0)) + (b_pace*(d_pace/20.0))
    shift = clip(shift, -cap, cap)

    mu_shift.append(shift)
    mu_ctx.append(float(r["PROJ_MU_BASE"]) + shift)

df["PROJ_MU_SHIFT"] = mu_shift
df["PROJ_MU_CTX"] = mu_ctx

# -----------------------
# Recompute PROJ_PROB_CTX from μ_ctx (distribution-specific)
# -----------------------
proj_prob_ctx = []
for _, r in df.iterrows():
    mkey = market_key(r["PROP"])
    dist = r["DIST_USED"]
    line = float(r["LINE"])
    mu = float(r["PROJ_MU_CTX"])

    if dist == "poisson":
        p_over = poisson_prob_over(line, mu)
    elif dist == "nbinom":
        k = float(r["AUX_PARAM"]) if pd.notna(r["AUX_PARAM"]) else nb_k_for_market(mkey, line)
        p_over = nb_prob_over(line, mu, k)
    else:
        sigma = float(r["AUX_PARAM"]) if pd.notna(r["AUX_PARAM"]) else normal_sigma(mkey, line)
        p_over = normal_prob_over(line, mu, sigma)

    proj_prob_ctx.append(p_over if r["SIDE"] == "over" else (1 - p_over))

df["PROJ_PROB_CTX"] = np.clip(proj_prob_ctx, 0.01, 0.99)

# -----------------------
# Blend with historical + edge
# -----------------------
W_HIST = 0.60
W_PROJ = 0.40
df["BLEND_PROB"] = np.clip(W_HIST*df["HIST_PROB"] + W_PROJ*df["PROJ_PROB_CTX"], 0.01, 0.99)
df["EDGE_BLEND"] = df["BLEND_PROB"] - df["BOOK_PROB"]

top10_blend = df.sort_values(["BLEND_PROB","EDGE_BLEND"], ascending=[False,False]).head(10).copy()
top10_blend.insert(0, "RANK", range(1, len(top10_blend)+1))

display(top10_blend[[
    "RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "DIST_USED","PROJ_MU_BASE","PROJ_MU_SHIFT","PROJ_MU_CTX",
    "HIST_PROB","PROJ_PROB_CTX","BLEND_PROB","EDGE_BLEND",
    "ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"
]])

top10_blend.to_csv("hit_sheet_top10_ncaam_blend_ctx_mu_dist.csv", index=False)
df.to_csv("prop_engine_full_ncaam_blend_ctx_mu_dist.csv", index=False)

print("Saved: hit_sheet_top10_ncaam_blend_ctx_mu_dist.csv")
print("Saved: prop_engine_full_ncaam_blend_ctx_mu_dist.csv")

# Discord-ready text
def discord_top10(df_top10):
    lines=[]
    lines.append("NCAA PROP ENGINE — TOP 10 MOST LIKELY (BLEND μ+CTX, dist)")
    lines.append("—")
    for _, r in df_top10.iterrows():
        odds = f"({int(r.AMERICAN_ODDS)})" if pd.notna(r.AMERICAN_ODDS) else ""
        lines.append(
            f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n"
            f"{r.PROP} {r.OUTCOME} {odds}\n"
            f"Blend: {r.BLEND_PROB*100:.1f}% | Edge: {r.EDGE_BLEND*100:+.1f} pts | "
            f"μ_ctx: {r.PROJ_MU_CTX:.2f} ({r.DIST_USED}) | Opp DEF: {r.ESPN_OPP_DEF_PPG_PROXY:.1f} | Pace: {r.ESPN_PACE_PROXY:.1f}"
        )
    return "\n".join(lines)

print(discord_top10(top10_blend))

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,DIST_USED,PROJ_MU_BASE,PROJ_MU_SHIFT,PROJ_MU_CTX,HIST_PROB,PROJ_PROB_CTX,BLEND_PROB,EDGE_BLEND,ESPN_OPP_DEF_PPG_PROXY,ESPN_PACE_PROXY
0,1,Jeremy Fears Jr.,MSU at IND,Assists,o3.5,100,nbinom,8.826083,-0.181000,8.645083,0.858062,0.851813,0.855563,0.355563,64.333333,139.000000
1,2,C.J. Cox,PUR at OSU,Rebounds,o1.5,-200,nbinom,3.106917,0.266667,3.373584,0.739750,0.771178,0.752321,0.085655,77.000000,159.333333
2,3,Jaxon Kohler,MSU at IND,Three Pointers Made,u1.5,-139,poisson,1.074980,-0.076000,0.998980,0.708200,0.736134,0.719374,0.137784,64.333333,139.000000
3,4,Nick Dorn,MSU at IND,Rebounds,u2.5,117,nbinom,2.055039,-0.209167,1.845873,0.666750,0.714660,0.685914,0.225084,64.333333,139.000000
5,5,Lamar Wilkerson,MSU at IND,Rebounds,u4.5,-145,nbinom,3.818332,-0.209167,3.609166,0.661000,0.692029,0.673411,0.081575,64.333333,139.000000
4,6,Braden Smith,PUR at OSU,Points + Rebounds + Assists,u28.5,-105,normal,26.449015,0.946667,27.395681,0.690500,0.622758,0.663403,0.151208,77.000000,159.333333
26,7,Jeremy Fears Jr.,MSU at IND,Points,u15.5,-117,normal,15.209146,-0.563333,14.645812,0.611200,0.685679,0.640992,0.101821,64.333333,139.000000
7,8,Trey Kaufman-Renn,PUR at OSU,Points,u14.5,-102,normal,13.754698,0.680000,14.434698,0.671750,0.580000,0.635050,0.130100,77.000000,159.333333
6,9,Braden Smith,PUR at OSU,Points,u15.5,-115,normal,14.860370,0.680000,15.540370,0.658000,0.565195,0.620878,0.085994,77.000000,159.333333
9,10,Bruce Thornton,PUR at OSU,Rebounds,u5.5,-115,nbinom,4.927092,0.266667,5.193759,0.632050,0.599448,0.619009,0.084125,77.000000,159.333333


Saved: hit_sheet_top10_ncaam_blend_ctx_mu_dist.csv
Saved: prop_engine_full_ncaam_blend_ctx_mu_dist.csv
NCAA PROP ENGINE — TOP 10 MOST LIKELY (BLEND μ+CTX, dist)
—
1) Jeremy Fears Jr. — MSU at IND
Assists o3.5 (100)
Blend: 85.6% | Edge: +35.6 pts | μ_ctx: 8.65 (nbinom) | Opp DEF: 64.3 | Pace: 139.0
2) C.J. Cox — PUR at OSU
Rebounds o1.5 (-200)
Blend: 75.2% | Edge: +8.6 pts | μ_ctx: 3.37 (nbinom) | Opp DEF: 77.0 | Pace: 159.3
3) Jaxon Kohler — MSU at IND
Three Pointers Made u1.5 (-139)
Blend: 71.9% | Edge: +13.8 pts | μ_ctx: 1.00 (poisson) | Opp DEF: 64.3 | Pace: 139.0
4) Nick Dorn — MSU at IND
Rebounds u2.5 (117)
Blend: 68.6% | Edge: +22.5 pts | μ_ctx: 1.85 (nbinom) | Opp DEF: 64.3 | Pace: 139.0
5) Lamar Wilkerson — MSU at IND
Rebounds u4.5 (-145)
Blend: 67.3% | Edge: +8.2 pts | μ_ctx: 3.61 (nbinom) | Opp DEF: 64.3 | Pace: 139.0
6) Braden Smith — PUR at OSU
Points + Rebounds + Assists u28.5 (-105)
Blend: 66.3% | Edge: +15.1 pts | μ_ctx: 27.40 (normal) | Opp DEF: 77.0 | Pace: 159.3
7) Je

In [ ]:
# ==============================
# BEST PROPS BOARD (Hit rater + μ projections + matchup + injury sensitivity)
# ==============================

import numpy as np
import pandas as pd
import re

# Require that the dist-based blend cell has been run
req = ["BLEND_PROB","EDGE_BLEND","PROJ_MU_CTX","DIST_USED","LINE","SIDE","DEC_ODDS","BOOK_PROB",
       "ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]
missing = [c for c in req if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns. Run the dist+blend cell first. Missing: {missing}")

# --- Helpers ---
def american_to_decimal(o):
    if pd.isna(o): return np.nan
    o=float(o)
    return 1+o/100 if o>0 else 1+100/(-o)

if "DEC_ODDS" not in df.columns or df["DEC_ODDS"].isna().all():
    df["DEC_ODDS"] = df["AMERICAN_ODDS"].apply(american_to_decimal)

def ev_1u(p, dec):
    if pd.isna(p) or pd.isna(dec) or dec <= 1: return np.nan
    return p*(dec-1) - (1-p)

df["EV_BLEND_1U"] = [ev_1u(p,d) for p,d in zip(df["BLEND_PROB"], df["DEC_ODDS"])]

# Market key for injury multipliers
def market_key(prop):
    p=str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p: return "pra"
    if "three pointers" in p or "3" in p: return "three"
    if "assists" in p: return "assists"
    if "rebounds" in p: return "rebounds"
    if "points" in p: return "points"
    return "other"

df["MKT"] = df["PROP"].map(market_key)

# Injury sensitivity scenarios:
# We don’t know actual injuries from the CSV reliably, so we create robust scenario shifts:
# - STAR OUT on player's team (more usage/mins) -> μ up (except unders)
# - PLAYER Q (reduced mins) -> μ down
# These are conservative default multipliers by market type.
UP_MULT  = {"points":1.06, "assists":1.07, "rebounds":1.04, "three":1.05, "pra":1.06, "other":1.05}
DOWN_MULT= {"points":0.92, "assists":0.90, "rebounds":0.93, "three":0.92, "pra":0.91, "other":0.92}

def apply_mu_scenario(mu, mkt, scenario):
    if pd.isna(mu): return np.nan
    if scenario == "up":
        return float(mu) * UP_MULT.get(mkt, 1.05)
    if scenario == "down":
        return float(mu) * DOWN_MULT.get(mkt, 0.92)
    return float(mu)

# Convert μ -> prob for the chosen side using the dist already selected
# (We reuse scipy stats if available from your previous cell; otherwise use the already computed PROJ_PROB_CTX as baseline)
try:
    from scipy.stats import norm, poisson, nbinom
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False

def nb_params_from_mu_k(mu, k):
    mu = max(0.05, float(mu))
    k = max(0.5, float(k))
    p = k/(k+mu)
    n = k
    return n, p

def prob_from_mu(line, mu, dist_used, side, aux_param=np.nan):
    line=float(line)
    if not SCIPY_OK:
        # fallback: scale baseline probability by mean ratio (conservative)
        return np.nan

    if dist_used == "poisson":
        k = int(np.floor(line))
        p_over = float(1 - poisson.cdf(k, mu=mu))
    elif dist_used == "nbinom":
        kdisp = float(aux_param) if pd.notna(aux_param) else 4.5
        n,p = nb_params_from_mu_k(mu, kdisp)
        cut=int(np.floor(line))
        p_over = float(1 - nbinom.cdf(cut, n=n, p=p))
    else:
        sigma = float(aux_param) if pd.notna(aux_param) else max(2.0, 0.18*max(line,1.0))
        x = ((line + 0.5) - mu)/sigma
        p_over = float(1 - norm.cdf(x))

    return p_over if side == "over" else (1 - p_over)

# Use AUX_PARAM if you have it; if not, set NaN
if "AUX_PARAM" not in df.columns:
    df["AUX_PARAM"] = np.nan

# Compute scenario probabilities + scenario EV deltas (if SciPy available)
p_up=[]; p_down=[]
ev_up=[]; ev_down=[]

for _, r in df.iterrows():
    mu0 = float(r["PROJ_MU_CTX"])
    mu_up = apply_mu_scenario(mu0, r["MKT"], "up")
    mu_dn = apply_mu_scenario(mu0, r["MKT"], "down")

    if SCIPY_OK:
        pu = prob_from_mu(r["LINE"], mu_up, r["DIST_USED"], r["SIDE"], r["AUX_PARAM"])
        pdn = prob_from_mu(r["LINE"], mu_dn, r["DIST_USED"], r["SIDE"], r["AUX_PARAM"])
    else:
        pu = np.nan
        pdn = np.nan

    p_up.append(pu); p_down.append(pdn)
    ev_up.append(ev_1u(pu, r["DEC_ODDS"]) if pd.notna(pu) else np.nan)
    ev_down.append(ev_1u(pdn, r["DEC_ODDS"]) if pd.notna(pdn) else np.nan)

df["PROJ_PROB_UP"] = p_up
df["PROJ_PROB_DOWN"] = p_down
df["EV_UP_1U"] = ev_up
df["EV_DOWN_1U"] = ev_down

# Injury Sensitivity Score: how much probability moves under plausible injury scenarios
# If SciPy not available, this will be NaN; still fine.
df["INJ_SENS_PTS"] = (df["PROJ_PROB_UP"] - df["PROJ_PROB_DOWN"]) * 100

# Final ranking score:
# prioritize EV + stability; penalty for being too sensitive unless user wants volatile plays
df["SCORE"] = (
    (df["EV_BLEND_1U"].fillna(0) * 100)  # EV as %
    + (df["EDGE_BLEND"].fillna(0) * 50)  # edge reinforcement
    - (df["INJ_SENS_PTS"].fillna(0) * 0.15) # mild penalty for sensitivity
)

best = df.sort_values("SCORE", ascending=False).head(20).copy()

cols = ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
        "BLEND_PROB","EV_BLEND_1U","EDGE_BLEND",
        "PROJ_MU_CTX","DIST_USED",
        "ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY",
        "INJ_SENS_PTS","SCORE"]
display(best[cols])
best.to_csv("best_props_board_top20.csv", index=False)
print("Saved: best_props_board_top20.csv")

,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,BLEND_PROB,EV_BLEND_1U,EDGE_BLEND,PROJ_MU_CTX,DIST_USED,ESPN_OPP_DEF_PPG_PROXY,ESPN_PACE_PROXY,INJ_SENS_PTS,SCORE
0,Jeremy Fears Jr.,MSU at IND,Assists,o3.5,100,0.855563,0.711126,0.355563,8.645083,nbinom,64.333333,139.000000,5.447951,88.073511
3,Nick Dorn,MSU at IND,Rebounds,u2.5,117,0.685914,0.488433,0.225084,1.845873,nbinom,64.333333,139.000000,-4.712380,60.804403
4,Braden Smith,PUR at OSU,Points + Rebounds + Assists,u28.5,-105,0.663403,0.295216,0.151208,27.395681,normal,77.000000,159.333333,-28.928126,41.421212
8,Devin Royal,PUR at OSU,Three Pointers Made,u0.5,137,0.559940,0.327057,0.137999,0.550521,poisson,67.500000,140.500000,-4.162058,40.229948
7,Trey Kaufman-Renn,PUR at OSU,Points,u14.5,-102,0.635050,0.257648,0.130100,14.434698,normal,77.000000,159.333333,-27.326858,36.368807
2,Jaxon Kohler,MSU at IND,Three Pointers Made,u1.5,-139,0.719374,0.236909,0.137784,0.998980,poisson,64.333333,139.000000,-4.773470,31.296054
26,Jeremy Fears Jr.,MSU at IND,Points,u15.5,-117,0.640992,0.188848,0.101821,14.645812,normal,64.333333,139.000000,-24.907939,27.711991
29,Jaxon Kohler,MSU at IND,Points,u9.5,108,0.571068,0.187821,0.090299,9.157921,normal,64.333333,139.000000,-17.146311,25.868984
6,Braden Smith,PUR at OSU,Points,u15.5,-115,0.620878,0.160772,0.085994,15.540370,normal,77.000000,159.333333,-29.550507,24.809476
20,Oscar Cluff,PUR at OSU,Rebounds,u6.5,-107,0.610990,0.182008,0.094081,6.195645,nbinom,77.000000,159.333333,-7.235564,23.990196


Saved: best_props_board_top20.csv


In [ ]:
top10_best = best.head(10).copy()
lines=[]
lines.append("NCAA PROP ENGINE — BEST PROPS BOARD (Hit rater + μ proj + matchup + injury)")
lines.append("—")
for i, r in enumerate(top10_best.itertuples(index=False), start=1):
    odds = f"({int(r.AMERICAN_ODDS)})"
    lines.append(
        f"{i}) {r.PLAYER} — {r.GAME}\n"
        f"{r.PROP} {r.OUTCOME} {odds}\n"
        f"Blend: {r.BLEND_PROB*100:.1f}% | EV: {r.EV_BLEND_1U*100:.1f}% | "
        f"μ_ctx: {r.PROJ_MU_CTX:.2f} ({r.DIST_USED}) | Opp DEF: {r.ESPN_OPP_DEF_PPG_PROXY:.1f} | Pace: {r.ESPN_PACE_PROXY:.1f} | "
        f"Injury sens: {r.INJ_SENS_PTS:.1f} pts"
    )
print("\n".join(lines))

NCAA PROP ENGINE — BEST PROPS BOARD (Hit rater + μ proj + matchup + injury)
—
1) Jeremy Fears Jr. — MSU at IND
Assists o3.5 (100)
Blend: 85.6% | EV: 71.1% | μ_ctx: 8.65 (nbinom) | Opp DEF: 64.3 | Pace: 139.0 | Injury sens: 5.4 pts
2) Nick Dorn — MSU at IND
Rebounds u2.5 (117)
Blend: 68.6% | EV: 48.8% | μ_ctx: 1.85 (nbinom) | Opp DEF: 64.3 | Pace: 139.0 | Injury sens: -4.7 pts
3) Braden Smith — PUR at OSU
Points + Rebounds + Assists u28.5 (-105)
Blend: 66.3% | EV: 29.5% | μ_ctx: 27.40 (normal) | Opp DEF: 77.0 | Pace: 159.3 | Injury sens: -28.9 pts
4) Devin Royal — PUR at OSU
Three Pointers Made u0.5 (137)
Blend: 56.0% | EV: 32.7% | μ_ctx: 0.55 (poisson) | Opp DEF: 67.5 | Pace: 140.5 | Injury sens: -4.2 pts
5) Trey Kaufman-Renn — PUR at OSU
Points u14.5 (-102)
Blend: 63.5% | EV: 25.8% | μ_ctx: 14.43 (normal) | Opp DEF: 77.0 | Pace: 159.3 | Injury sens: -27.3 pts
6) Jaxon Kohler — MSU at IND
Three Pointers Made u1.5 (-139)
Blend: 71.9% | EV: 23.7% | μ_ctx: 1.00 (poisson) | Opp DEF: 64.3 |

In [ ]:
df["BLEND_PROB_SHRUNK"] = 0.85*df["BLEND_PROB"] + 0.15*df["BOOK_PROB"]
df["EV_SHRUNK"] = df["BLEND_PROB_SHRUNK"]*(df["DEC_ODDS"]-1) - (1-df["BLEND_PROB_SHRUNK"])

In [ ]:
# ==============================
# SAFE / CONSISTENT BOARD (Mode A)
# - shrink probs toward market
# - heavily penalize injury sensitivity (fragility)
# - require minimum probability + cap extreme edges
# ==============================

import numpy as np
import pandas as pd

# Market-anchor shrink (prevents runaway 80%+ at plus money unless truly consistent)
SHRINK_TO_BOOK = 0.25   # 25% weight to market
df["SAFE_PROB"] = (1 - SHRINK_TO_BOOK)*df["BLEND_PROB"] + SHRINK_TO_BOOK*df["BOOK_PROB"]
df["SAFE_PROB"] = df["SAFE_PROB"].clip(0.01, 0.99)

# Safe EV
df["EV_SAFE_1U"] = df["SAFE_PROB"]*(df["DEC_ODDS"]-1) - (1-df["SAFE_PROB"])

# Additional safety gates
MIN_SAFE_PROB = 0.56          # avoid coinflips
MAX_INJ_SENS  = 15.0          # “consistent” filter (pts of swing)
MAX_SHIFT_MAG = 1.75          # avoid props that depend on huge μ shifts

df["INJ_SENS_ABS"] = df["INJ_SENS_PTS"].abs()
df["MU_SHIFT_ABS"] = df["PROJ_MU_SHIFT"].abs()

safe_pool = df[
    (df["SAFE_PROB"] >= MIN_SAFE_PROB) &
    (df["INJ_SENS_ABS"] <= MAX_INJ_SENS) &
    (df["MU_SHIFT_ABS"] <= MAX_SHIFT_MAG)
].copy()

# Stability-first score:
# - EV matters, but stability matters more
# - penalize injury swing + large μ shift (fragility)
safe_pool["SAFE_SCORE"] = (
    (safe_pool["EV_SAFE_1U"].fillna(-999) * 100)          # EV%
    - (safe_pool["INJ_SENS_ABS"].fillna(0) * 1.25)        # heavy penalty
    - (safe_pool["MU_SHIFT_ABS"].fillna(0) * 6.0)         # penalty for mean shift reliance
)

safe_top20 = safe_pool.sort_values("SAFE_SCORE", ascending=False).head(20).copy()
safe_top20.insert(0, "RANK", range(1, len(safe_top20)+1))

display(safe_top20[[
    "RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "SAFE_PROB","EV_SAFE_1U","SAFE_SCORE",
    "BLEND_PROB","BOOK_PROB","EDGE_BLEND",
    "DIST_USED","PROJ_MU_CTX","PROJ_MU_SHIFT",
    "INJ_SENS_PTS","ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"
]])

safe_top20.to_csv("best_props_safe_top20.csv", index=False)
print("Saved: best_props_safe_top20.csv")

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,SAFE_PROB,EV_SAFE_1U,SAFE_SCORE,BLEND_PROB,BOOK_PROB,EDGE_BLEND,DIST_USED,PROJ_MU_CTX,PROJ_MU_SHIFT,INJ_SENS_PTS,ESPN_OPP_DEF_PPG_PROXY,ESPN_PACE_PROXY
0,1,Jeremy Fears Jr.,MSU at IND,Assists,o3.5,100,0.766672,0.533344,45.438483,0.855563,0.500000,0.355563,nbinom,8.645083,-0.181000,5.447951,64.333333,139.000000
3,2,Nick Dorn,MSU at IND,Rebounds,u2.5,117,0.629643,0.366325,29.487018,0.685914,0.460829,0.225084,nbinom,1.845873,-0.209167,-4.712380,64.333333,139.000000
2,3,Jaxon Kohler,MSU at IND,Three Pointers Made,u1.5,-139,0.684928,0.177681,11.345301,0.719374,0.581590,0.137784,poisson,0.998980,-0.076000,-4.773470,64.333333,139.000000
20,4,Oscar Cluff,PUR at OSU,Rebounds,u6.5,-107,0.587469,0.136506,3.006139,0.610990,0.516908,0.094081,nbinom,6.195645,0.266667,-7.235564,77.000000,159.333333
1,5,C.J. Cox,PUR at OSU,Rebounds,o1.5,-200,0.730908,0.096362,2.803840,0.752321,0.666667,0.085655,nbinom,3.373584,0.266667,4.185850,77.000000,159.333333
5,6,Lamar Wilkerson,MSU at IND,Rebounds,u4.5,-145,0.653018,0.103375,1.659992,0.673411,0.591837,0.081575,nbinom,3.609166,-0.209167,-5.937992,64.333333,139.000000
9,7,Bruce Thornton,PUR at OSU,Rebounds,u5.5,-115,0.597978,0.117958,1.523338,0.619009,0.534884,0.084125,nbinom,5.193759,0.266667,-6.938003,77.000000,159.333333
15,8,Braden Smith,PUR at OSU,Assists,u8.5,-107,0.565411,0.093832,-5.861080,0.581578,0.516908,0.064670,nbinom,8.463615,0.232667,-11.078648,77.000000,159.333333
10,9,Fletcher Loyer,PUR at OSU,Three Pointers Made,u2.5,-134,0.592760,0.035119,-7.185877,0.599464,0.572650,0.026814,poisson,2.348254,0.101667,-8.070202,77.000000,159.333333
31,10,Braden Smith,PUR at OSU,Three Pointers Made,o1.5,-132,0.580903,0.020980,-7.442657,0.584882,0.568966,0.015916,poisson,2.028222,0.101667,7.144554,77.000000,159.333333


Saved: best_props_safe_top20.csv


In [ ]:
top10_safe = safe_top20.head(10).copy()

lines=[]
lines.append("NCAA PROP ENGINE — SAFEST EDGES (Mode A: Consistent)")
lines.append("—")
for r in top10_safe.itertuples(index=False):
    odds = f"({int(r.AMERICAN_ODDS)})"
    lines.append(
        f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n"
        f"{r.PROP} {r.OUTCOME} {odds}\n"
        f"SafeProb: {r.SAFE_PROB*100:.1f}% | EV: {r.EV_SAFE_1U*100:.1f}% | "
        f"Injury swing: {abs(r.INJ_SENS_PTS):.1f} pts | μ shift: {abs(r.PROJ_MU_SHIFT):.2f} | "
        f"Opp DEF: {r.ESPN_OPP_DEF_PPG_PROXY:.1f} | Pace: {r.ESPN_PACE_PROXY:.1f}"
    )

print("\n".join(lines))

NCAA PROP ENGINE — SAFEST EDGES (Mode A: Consistent)
—
1) Jeremy Fears Jr. — MSU at IND
Assists o3.5 (100)
SafeProb: 76.7% | EV: 53.3% | Injury swing: 5.4 pts | μ shift: 0.18 | Opp DEF: 64.3 | Pace: 139.0
2) Nick Dorn — MSU at IND
Rebounds u2.5 (117)
SafeProb: 63.0% | EV: 36.6% | Injury swing: 4.7 pts | μ shift: 0.21 | Opp DEF: 64.3 | Pace: 139.0
3) Jaxon Kohler — MSU at IND
Three Pointers Made u1.5 (-139)
SafeProb: 68.5% | EV: 17.8% | Injury swing: 4.8 pts | μ shift: 0.08 | Opp DEF: 64.3 | Pace: 139.0
4) Oscar Cluff — PUR at OSU
Rebounds u6.5 (-107)
SafeProb: 58.7% | EV: 13.7% | Injury swing: 7.2 pts | μ shift: 0.27 | Opp DEF: 77.0 | Pace: 159.3
5) C.J. Cox — PUR at OSU
Rebounds o1.5 (-200)
SafeProb: 73.1% | EV: 9.6% | Injury swing: 4.2 pts | μ shift: 0.27 | Opp DEF: 77.0 | Pace: 159.3
6) Lamar Wilkerson — MSU at IND
Rebounds u4.5 (-145)
SafeProb: 65.3% | EV: 10.3% | Injury swing: 5.9 pts | μ shift: 0.21 | Opp DEF: 64.3 | Pace: 139.0
7) Bruce Thornton — PUR at OSU
Rebounds u5.5 (-115)

In [ ]:
# ==============================
# ELITE CARD (3–5) + KELLY UNITS
# Clean Discord Output – No Model Lingo
# ==============================

import numpy as np
import pandas as pd

# Filters for "Elite / consistent"
MIN_PROB = 0.60
MIN_EV   = 0.08
MAX_INJ  = 10

elite = safe_top20[
    (safe_top20["SAFE_PROB"] >= MIN_PROB) &
    (safe_top20["EV_SAFE_1U"] >= MIN_EV) &
    (safe_top20["INJ_SENS_PTS"].abs() <= MAX_INJ)
].copy()

elite = elite.sort_values("SAFE_SCORE", ascending=False).head(5)

# --- Kelly sizing (Half Kelly), capped to your unit band ---
HALF_KELLY = 0.50
MIN_U = 0.25
MAX_U = 1.00

def american_to_decimal(o):
    o = float(o)
    return 1 + (o/100.0) if o > 0 else 1 + (100.0/abs(o))

def half_kelly_units(p, dec_odds, half=0.5, min_u=0.25, max_u=1.0):
    """
    Kelly fraction f* = (b*p - q)/b where b=dec-1, q=1-p.
    Half-Kelly -> half * f*. Convert to units in [min_u, max_u].
    If f* <= 0 -> 0 units (filtered out later if needed).
    """
    if pd.isna(p) or pd.isna(dec_odds) or dec_odds <= 1:
        return 0.0
    b = float(dec_odds) - 1.0
    p = float(p)
    q = 1.0 - p
    f = (b*p - q) / b
    f = max(0.0, f) * float(half)
    # Convert fraction to "units" directly; cap to band
    u = min(max(f, 0.0), max_u)
    # If it's positive but tiny, bump to min_u for card consistency
    if 0 < u < min_u:
        u = min_u
    return float(round(u + 1e-9, 2))

# Ensure DEC_ODDS exists
if "DEC_ODDS" not in elite.columns or elite["DEC_ODDS"].isna().all():
    elite["DEC_ODDS"] = elite["AMERICAN_ODDS"].apply(american_to_decimal)

elite["UNITS"] = [
    half_kelly_units(p, d, half=HALF_KELLY, min_u=MIN_U, max_u=MAX_U)
    for p, d in zip(elite["SAFE_PROB"], elite["DEC_ODDS"])
]

# Drop any 0u (should be rare given filters)
elite = elite[elite["UNITS"] > 0].copy()

display(elite[[
    "PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS"
]])

# ---- Clean Discord Snippet ----
lines = []
lines.append("NCAA ELITE PROP CARD")
lines.append("—")

for i, r in enumerate(elite.itertuples(index=False), start=1):
    odds = f"({int(r.AMERICAN_ODDS)})"
    lines.append(
        f"{i}) {r.PLAYER} — {r.GAME}\n"
        f"{r.PROP} {r.OUTCOME} {odds} — {r.UNITS:.2f}u"
    )

print("\n".join(lines))

,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS
0,Jeremy Fears Jr.,MSU at IND,Assists,o3.5,100,0.27
3,Nick Dorn,MSU at IND,Rebounds,u2.5,117,0.25
2,Jaxon Kohler,MSU at IND,Three Pointers Made,u1.5,-139,0.25
1,C.J. Cox,PUR at OSU,Rebounds,o1.5,-200,0.25
5,Lamar Wilkerson,MSU at IND,Rebounds,u4.5,-145,0.25


NCAA ELITE PROP CARD
—
1) Jeremy Fears Jr. — MSU at IND
Assists o3.5 (100) — 0.27u
2) Nick Dorn — MSU at IND
Rebounds u2.5 (117) — 0.25u
3) Jaxon Kohler — MSU at IND
Three Pointers Made u1.5 (-139) — 0.25u
4) C.J. Cox — PUR at OSU
Rebounds o1.5 (-200) — 0.25u
5) Lamar Wilkerson — MSU at IND
Rebounds u4.5 (-145) — 0.25u


In [2]:
from google.colab import files
uploaded = files.upload()

Saving NCAA prop 3:2.csv to NCAA prop 3:2.csv


In [5]:
import pandas as pd, numpy as np, re, os
from datetime import datetime, timedelta

CSV_PATH = "NCAA prop 3:2.csv"

raw = pd.read_csv(CSV_PATH, header=None, dtype=str, keep_default_na=False)
header_idx = None
for i in range(min(10, len(raw))):
    row = [str(x).strip().upper() for x in raw.iloc[i].tolist()]
    if "PLAYER" in row and "GAME" in row and "PROP" in row and "OUTCOME" in row:
        header_idx = i
        break
if header_idx is None:
    raise Exception("Could not find header row. Check CSV export format.")

hdr = [str(x).strip() for x in raw.iloc[header_idx].tolist()]
df = raw.iloc[header_idx+1:].copy()
df.columns = hdr
df = df.replace("", np.nan).dropna(how="all").copy()

# Normalize column names
df = df.rename(columns={
    "IM PROB %": "IM_PROB_%",
    "DVP L5":"DVP_L5",
    "DVP L10":"DVP_L10",
    "DVP 2025":"DVP_2025",
    "DVP HM/AW":"DVP_HM/AW",
})

# Odds column → AMERICAN_ODDS (ODDS looks like "BET\n-130")
def extract_american_odds(x):
    s = str(x).replace("−","-")
    m = re.search(r"([+-]\d{2,4})", s)
    return int(m.group(1)) if m else np.nan

df["AMERICAN_ODDS"] = df["ODDS"].apply(extract_american_odds)

# Outcome → SIDE/LINE
def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

# Book prob / decimal odds
def american_to_prob(o):
    o = float(o)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(o):
    o = float(o)
    return 1 + (100/abs(o)) if o < 0 else 1 + (o/100)

df = df[df["AMERICAN_ODDS"].notna() & df["SIDE"].notna() & df["LINE"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)
df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

print("Loaded:", CSV_PATH)
print("Shape:", df.shape)
display(df.head(10)[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS"]])

Loaded: NCAA prop 3:2.csv
Shape: (98, 22)


/tmp/ipython-input-646/2904018323.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("", np.nan).dropna(how="all").copy()


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS
2,Joshua Jefferson,ISU at ARIZ,Three Pointers Made,o0.5,-172
3,Jaden Bradley,ISU at ARIZ,Three Pointers Made,u0.5,106
4,Brayden Burries,ISU at ARIZ,Points + Rebounds,o20.5,-130
5,Quadir Copeland,DUKE at NCST,Three Pointers Made,u0.5,-139
6,Ven-Allen Lubin,DUKE at NCST,Rebounds,o6.5,-115
7,Milan Momcilovic,ISU at ARIZ,Rebounds,u3.5,-170
8,Brayden Burries,ISU at ARIZ,Rebounds,u5.5,-129
9,Milan Momcilovic,ISU at ARIZ,Three Pointers Made,o2.5,-146
10,Jaden Bradley,ISU at ARIZ,Points,u14.5,-115
11,Tamin Lipsey,ISU at ARIZ,Points,o11.5,-122


In [4]:
import os
print("Current working directory:", os.getcwd())
print("Files in cwd:")
print(os.listdir())

Current working directory: /content
Files in cwd:
['.config', 'NCAA prop 3:2.csv', 'sample_data']


In [6]:
import numpy as np

def pct_to_prob(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().replace("%","")
    try:
        v = float(s)
        return v/100.0 if v > 1 else v
    except:
        return np.nan

hit_cols = ["L5","L10","2025","HM/AW","H2H","DVP_L5","DVP_L10","DVP_2025","DVP_HM/AW"]
for c in hit_cols:
    if c in df.columns:
        df[c] = df[c].apply(pct_to_prob)

weights = {
    "L5":0.20, "L10":0.25, "2025":0.20, "HM/AW":0.10, "H2H":0.05,
    "DVP_L5":0.08, "DVP_L10":0.07, "DVP_2025":0.04, "DVP_HM/AW":0.01
}

num = 0.0
den = 0.0
for c,w in weights.items():
    v = df[c].astype(float)
    num += w * v
    den += w * v.notna().astype(float)

df["HIST_PROB"] = (num/den).clip(0.01,0.99)

print("HIST_PROB null:", df["HIST_PROB"].isna().sum(), "/", len(df))
display(df.head(10)[["PLAYER","PROP","OUTCOME","HIST_PROB"]])

HIST_PROB null: 43 / 98


,PLAYER,PROP,OUTCOME,HIST_PROB
2,Joshua Jefferson,Three Pointers Made,o0.5,0.63190
3,Jaden Bradley,Three Pointers Made,u0.5,0.61214
4,Brayden Burries,Points + Rebounds,o20.5,NaN
5,Quadir Copeland,Three Pointers Made,u0.5,0.55360
6,Ven-Allen Lubin,Rebounds,o6.5,0.51168
7,Milan Momcilovic,Rebounds,u3.5,0.70727
8,Brayden Burries,Rebounds,u5.5,NaN
9,Milan Momcilovic,Three Pointers Made,o2.5,0.55496
10,Jaden Bradley,Points,u14.5,0.66406
11,Tamin Lipsey,Points,o11.5,0.59764


In [7]:
from scipy.stats import norm, poisson, nbinom

def market_key(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p: return "pra"
    if "points + rebounds" in p or "pr" in p: return "pr"
    if "points + assists" in p or "pa" in p: return "pa"
    if "rebounds + assists" in p or "ra" in p: return "ra"
    if "three" in p: return "three"
    if "steals" in p: return "steals"
    if "blocks" in p: return "blocks"
    if "assists" in p: return "assists"
    if "rebounds" in p: return "rebounds"
    if "points" in p: return "points"
    return "other"

df["MKT"] = df["PROP"].map(market_key)

def pick_dist(mkt, line):
    # same style as 3/1 NCAA run
    if mkt in ["three","steals","blocks"]:
        return "poisson"
    if mkt in ["assists","rebounds"] and line <= 6.5:
        return "nbinom"
    if mkt in ["pa","pr","ra","pra"] or line >= 12:
        return "normal"
    return "nbinom" if line <= 8.5 else "normal"

df["DIST_USED"] = df.apply(lambda r: pick_dist(r.MKT, float(r.LINE)), axis=1)

def prob_from_mu(line, mu, dist, side):
    line=float(line); mu=float(mu)
    if dist=="normal":
        sd=max(1.0, mu*0.35)
        return (1-norm.cdf(line, loc=mu, scale=sd)) if side=="over" else norm.cdf(line, loc=mu, scale=sd)
    if dist=="poisson":
        k=int(np.floor(line))
        return (1-poisson.cdf(k, mu)) if side=="over" else poisson.cdf(k, mu)
    if dist=="nbinom":
        r=6.0
        p=r/(r+mu)
        k=int(np.floor(line))
        return (1-nbinom.cdf(k, r, p)) if side=="over" else nbinom.cdf(k, r, p)
    sd=max(1.0, mu*0.35)
    return (1-norm.cdf(line, loc=mu, scale=sd)) if side=="over" else norm.cdf(line, loc=mu, scale=sd)

def invert_mu(target_p, line, dist, side):
    target_p=float(target_p); line=float(line)
    lo, hi = 0.05, max(2.0, line*3.0 + 10)
    for _ in range(55):
        mid=(lo+hi)/2
        pmid = prob_from_mu(line, mid, dist, side)
        if pmid < target_p:
            if side=="over": lo=mid
            else: hi=mid
        else:
            if side=="over": hi=mid
            else: lo=mid
    return (lo+hi)/2

df["PROJ_MU_BASE"] = df.apply(lambda r: invert_mu(r.HIST_PROB, r.LINE, r.DIST_USED, r.SIDE), axis=1)

display(df.head(10)[["PLAYER","GAME","PROP","OUTCOME","HIST_PROB","DIST_USED","PROJ_MU_BASE"]])

,PLAYER,GAME,PROP,OUTCOME,HIST_PROB,DIST_USED,PROJ_MU_BASE
2,Joshua Jefferson,ISU at ARIZ,Three Pointers Made,o0.5,0.63190,poisson,0.999401
3,Jaden Bradley,ISU at ARIZ,Three Pointers Made,u0.5,0.61214,poisson,0.490794
4,Brayden Burries,ISU at ARIZ,Points + Rebounds,o20.5,NaN,normal,0.050000
5,Quadir Copeland,DUKE at NCST,Three Pointers Made,u0.5,0.55360,poisson,0.591313
6,Ven-Allen Lubin,DUKE at NCST,Rebounds,o6.5,0.51168,nbinom,7.175705
7,Milan Momcilovic,ISU at ARIZ,Rebounds,u3.5,0.70727,nbinom,2.691386
8,Brayden Burries,ISU at ARIZ,Rebounds,u5.5,NaN,nbinom,26.500000
9,Milan Momcilovic,ISU at ARIZ,Three Pointers Made,o2.5,0.55496,poisson,2.903993
10,Jaden Bradley,ISU at ARIZ,Points,u14.5,0.66406,normal,12.627921
11,Tamin Lipsey,ISU at ARIZ,Points,o11.5,0.59764,normal,12.589428


In [8]:
import requests, time, re
import pandas as pd, numpy as np
from datetime import datetime, timedelta

# ---- CONFIG ----
RUN_DATE = "20260302"          # YYYYMMDD (3/2)
LOOKBACK_DAYS = 30             # same style as 3/1 (can lower if slow)

session = requests.Session()
session.headers.update({"User-Agent":"Mozilla/5.0"})

def get_json(url, tries=6, timeout=25):
    last=None
    for i in range(tries):
        try:
            r=session.get(url, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last=e
            time.sleep(1.2*(i+1))
    raise last

SPORT_BASE = "https://site.web.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball"
SCOREBOARD = f"{SPORT_BASE}/scoreboard"

def parse_game_tokens(g):
    s=str(g).upper().strip()
    m=re.match(r"^\s*([A-Z0-9]{2,5})\s+(?:AT|@)\s+([A-Z0-9]{2,5})\s*$", s)
    return (m.group(1), m.group(2)) if m else (None,None)

df["AWAY_ABBR"], df["HOME_ABBR"] = zip(*df["GAME"].map(parse_game_tokens))

need_abbrs = set(df["AWAY_ABBR"].dropna().astype(str)) | set(df["HOME_ABBR"].dropna().astype(str))

# Build abbr -> team_id from scoreboard (covers weird tokens better than team directory)
def build_abbr_to_id_for_date(datestr):
    js = get_json(SCOREBOARD, params={"dates": datestr} if "params" in dir(requests) else None)
    # NOTE: ESPN scoreboard supports ?dates=YYYYMMDD; if your requests wrapper doesn't allow params,
    # replace get_json call with: get_json(SCOREBOARD + f"?dates={datestr}")
    return js

# Robust: call scoreboard with querystring (avoids params confusion)
sb_today = get_json(SCOREBOARD + f"?dates={RUN_DATE}")

abbr_to_id = {}
for ev in sb_today.get("events", []):
    for comp in ev.get("competitions", []):
        for teamwrap in comp.get("competitors", []):
            t = teamwrap.get("team", {})
            ab = str(t.get("abbreviation","")).upper()
            tid = t.get("id", None)
            if ab and tid:
                abbr_to_id[ab] = str(tid)

df["HOME_ID"] = df["HOME_ABBR"].map(abbr_to_id)
df["AWAY_ID"] = df["AWAY_ABBR"].map(abbr_to_id)

missing_home = sorted(df[df["HOME_ID"].isna()]["HOME_ABBR"].dropna().unique().tolist())
missing_away = sorted(df[df["AWAY_ID"].isna()]["AWAY_ABBR"].dropna().unique().tolist())
print("Missing home tokens:", missing_home)
print("Missing away tokens:", missing_away)

# ---- Lookback scoreboard scraping for DEF/PACE proxies ----
start = datetime.strptime(RUN_DATE, "%Y%m%d")
dates = [(start - timedelta(days=i)).strftime("%Y%m%d") for i in range(1, LOOKBACK_DAYS+1)]

games_rows = []
for d in dates:
    js = get_json(SCOREBOARD + f"?dates={d}")
    for ev in js.get("events", []):
        for comp in ev.get("competitions", []):
            comps = comp.get("competitors", [])
            if len(comps) != 2:
                continue
            a = comps[0]; b = comps[1]
            ta = a.get("team", {}); tb = b.get("team", {})
            sa = a.get("score", None); sb = b.get("score", None)
            try:
                sa = float(sa); sb = float(sb)
            except:
                continue
            ida = ta.get("id"); idb = tb.get("id")
            if not ida or not idb:
                continue
            games_rows.append({"team_id": str(ida), "pts_for": sa, "pts_against": sb, "total_pts": sa+sb})
            games_rows.append({"team_id": str(idb), "pts_for": sb, "pts_against": sa, "total_pts": sa+sb})

games_df = pd.DataFrame(games_rows)
print("Final games rows:", games_df.shape)

team_proxy = (games_df.groupby("team_id")
              .agg(DEF_PPG_ALLOWED=("pts_against","mean"),
                   PACE_PROXY=("total_pts","mean"))
              .reset_index())

# Attach per-row opponent proxy:
# We don't have player team, so we use game-average opponent proxy like 3/1
# (HOME+AWAY)/2 keeps it stable and still contextual.
home_ctx = team_proxy.rename(columns={"team_id":"HOME_ID",
                                      "DEF_PPG_ALLOWED":"HOME_DEF",
                                      "PACE_PROXY":"HOME_PACE"})
away_ctx = team_proxy.rename(columns={"team_id":"AWAY_ID",
                                      "DEF_PPG_ALLOWED":"AWAY_DEF",
                                      "PACE_PROXY":"AWAY_PACE"})

df = df.merge(home_ctx, on="HOME_ID", how="left").merge(away_ctx, on="AWAY_ID", how="left")

df["ESPN_OPP_DEF_PPG_PROXY"] = (df["HOME_DEF"] + df["AWAY_DEF"]) / 2.0
df["ESPN_PACE_PROXY"] = (df["HOME_PACE"] + df["AWAY_PACE"]) / 2.0

# Medians as fallback (same spirit as 3/1)
df["ESPN_OPP_DEF_PPG_PROXY"] = df["ESPN_OPP_DEF_PPG_PROXY"].fillna(df["ESPN_OPP_DEF_PPG_PROXY"].median())
df["ESPN_PACE_PROXY"] = df["ESPN_PACE_PROXY"].fillna(df["ESPN_PACE_PROXY"].median())

print("DEF range:", (df["ESPN_OPP_DEF_PPG_PROXY"].min(), df["ESPN_OPP_DEF_PPG_PROXY"].max()))
print("PACE range:", (df["ESPN_PACE_PROXY"].min(), df["ESPN_PACE_PROXY"].max()))

Missing home tokens: ['NCST']
Missing away tokens: []
Final games rows: (576, 4)
DEF range: (68.1875, 68.1875)
PACE range: (144.8125, 144.8125)


In [9]:
import numpy as np

# z-scores for context
def_z = (df["ESPN_OPP_DEF_PPG_PROXY"] - df["ESPN_OPP_DEF_PPG_PROXY"].median()) / df["ESPN_OPP_DEF_PPG_PROXY"].std()
pace_z = (df["ESPN_PACE_PROXY"] - df["ESPN_PACE_PROXY"].median()) / df["ESPN_PACE_PROXY"].std()

# Coefs (same “conservative” 3/1 style)
DEF_COEF  = -0.06
PACE_COEF =  0.04

df["PROJ_MU_SHIFT"] = (DEF_COEF*def_z + PACE_COEF*pace_z) * df["PROJ_MU_BASE"]
df["PROJ_MU_CTX"]   = (df["PROJ_MU_BASE"] + df["PROJ_MU_SHIFT"]).clip(lower=0.05)

df["PROJ_PROB_CTX"] = df.apply(lambda r: prob_from_mu(r.LINE, r.PROJ_MU_CTX, r.DIST_USED, r.SIDE), axis=1).clip(0.01,0.99)

# Blend (proj slightly heavier, like 3/1/NCAA today)
W_PROJ = 0.60
df["BLEND_PROB"] = (W_PROJ*df["PROJ_PROB_CTX"] + (1-W_PROJ)*df["HIST_PROB"]).clip(0.01,0.99)

# Injury sensitivity proxy (μ ±7%)
def apply_mu(mu, direction="up"):
    mu=float(mu)
    bump = mu*0.07
    return mu + bump if direction=="up" else max(0.05, mu - bump)

p_up=[]
p_dn=[]
for r in df.itertuples():
    p_up.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX,"up"), r.DIST_USED, r.SIDE))
    p_dn.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX,"down"), r.DIST_USED, r.SIDE))

p_up=np.array(p_up, dtype=float)
p_dn=np.array(p_dn, dtype=float)

df["INJ_SENS_ABS"] = np.abs(p_up - p_dn) * 100.0
df["MU_SHIFT_ABS"] = np.abs(df["PROJ_MU_SHIFT"])

# Mode A safe prob
df["SAFE_PROB"] = (df["BLEND_PROB"]
                   - (df["INJ_SENS_ABS"]/100.0)*0.18
                   - (df["MU_SHIFT_ABS"]/df["PROJ_MU_BASE"].clip(0.1))*0.10
                  ).clip(0.01,0.99)

df["EV_SAFE_1U"] = df["SAFE_PROB"]*df["DEC_ODDS"] - 1

display(df.head(10)[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","HIST_PROB","PROJ_MU_BASE","PROJ_MU_CTX",
                     "PROJ_PROB_CTX","BLEND_PROB","SAFE_PROB","EV_SAFE_1U","ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]])

,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,HIST_PROB,PROJ_MU_BASE,PROJ_MU_CTX,PROJ_PROB_CTX,BLEND_PROB,SAFE_PROB,EV_SAFE_1U,ESPN_OPP_DEF_PPG_PROXY,ESPN_PACE_PROXY
0,Joshua Jefferson,ISU at ARIZ,Three Pointers Made,o0.5,-172,0.63190,0.999401,NaN,NaN,NaN,NaN,NaN,68.1875,144.8125
1,Jaden Bradley,ISU at ARIZ,Three Pointers Made,u0.5,106,0.61214,0.490794,NaN,NaN,NaN,NaN,NaN,68.1875,144.8125
2,Brayden Burries,ISU at ARIZ,Points + Rebounds,o20.5,-130,NaN,0.050000,NaN,NaN,NaN,NaN,NaN,68.1875,144.8125
3,Quadir Copeland,DUKE at NCST,Three Pointers Made,u0.5,-139,0.55360,0.591313,NaN,NaN,NaN,NaN,NaN,68.1875,144.8125
4,Ven-Allen Lubin,DUKE at NCST,Rebounds,o6.5,-115,0.51168,7.175705,NaN,NaN,NaN,NaN,NaN,68.1875,144.8125
5,Milan Momcilovic,ISU at ARIZ,Rebounds,u3.5,-170,0.70727,2.691386,NaN,NaN,NaN,NaN,NaN,68.1875,144.8125
6,Brayden Burries,ISU at ARIZ,Rebounds,u5.5,-129,NaN,26.500000,NaN,NaN,NaN,NaN,NaN,68.1875,144.8125
7,Milan Momcilovic,ISU at ARIZ,Three Pointers Made,o2.5,-146,0.55496,2.903993,NaN,NaN,NaN,NaN,NaN,68.1875,144.8125
8,Jaden Bradley,ISU at ARIZ,Points,u14.5,-115,0.66406,12.627921,NaN,NaN,NaN,NaN,NaN,68.1875,144.8125
9,Tamin Lipsey,ISU at ARIZ,Points,o11.5,-122,0.59764,12.589428,NaN,NaN,NaN,NaN,NaN,68.1875,144.8125


In [10]:
import numpy as np
from datetime import datetime

def half_kelly(p, dec):
    b = dec - 1.0
    k = (p*b - (1-p)) / b
    return max(0.0, k/2.0)

df["KELLY_HALF"] = [half_kelly(p, d) for p,d in zip(df["SAFE_PROB"], df["DEC_ODDS"])]
df["UNITS"] = df["KELLY_HALF"].clip(0, 0.50)

# Strict filter (same as 3/1 style)
df_card = df[(df["EV_SAFE_1U"] > 0) & (df["SAFE_PROB"] >= df["BOOK_PROB"] + 0.03)].copy()
df_card["SCORE"] = (df_card["EV_SAFE_1U"]*100) + (df_card["SAFE_PROB"]*10) - (df_card["INJ_SENS_ABS"]*0.5)
df_card = df_card.sort_values("SCORE", ascending=False).copy()

top10 = df_card.head(10).copy().reset_index(drop=True)
top10["RANK"] = np.arange(1, len(top10)+1)

display(top10[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS",
               "SAFE_PROB","EV_SAFE_1U","ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]])

# Save outputs
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
full_path = f"prop_engine_full_ncaam_{RUN_DATE}_{ts}.csv"
top_path  = f"hit_sheet_top10_ncaam_{RUN_DATE}_{ts}.csv"
df.to_csv(full_path, index=False)
top10.to_csv(top_path, index=False)
print("Saved:", full_path)
print("Saved:", top_path)

# Discord output (no “model lingo”)
print("\nNCAA ELITE PROP CARD")
print("—")
for r in top10.head(5).itertuples():
    print(f"{r.RANK}) {r.PLAYER} — {r.GAME}")
    print(f"{r.PROP} {r.OUTCOME} ({r.AMERICAN_ODDS}) — {r.UNITS:.2f}u")

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,EV_SAFE_1U,ESPN_OPP_DEF_PPG_PROXY,ESPN_PACE_PROXY


Saved: prop_engine_full_ncaam_20260302_20260302_151508.csv
Saved: hit_sheet_top10_ncaam_20260302_20260302_151508.csv

NCAA ELITE PROP CARD
—


In [11]:
import numpy as np
import pandas as pd

print("Total rows:", len(df))
print("EV>0 count:", int((df["EV_SAFE_1U"] > 0).sum()))
print("SafeProb>BookProb count:", int((df["SAFE_PROB"] > df["BOOK_PROB"]).sum()))
print("Edge >= +3 pts count:", int((df["SAFE_PROB"] >= df["BOOK_PROB"] + 0.03).sum()))
print("Units > 0 count:", int((df["UNITS"] > 0).sum()))

# Show distribution so we can see if it's just a tight slate
print("\nSAFE_PROB quantiles:")
print(df["SAFE_PROB"].quantile([.5,.6,.7,.8,.9,.95]).round(4))
print("\nEV_SAFE_1U quantiles:")
print(df["EV_SAFE_1U"].quantile([.5,.6,.7,.8,.9,.95]).round(4))
print("\nTop 10 by EV_SAFE_1U (raw):")
display(df.sort_values("EV_SAFE_1U", ascending=False).head(10)[
    ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS","SAFE_PROB","BOOK_PROB","EV_SAFE_1U",
     "ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]
])

# --- Fallback ladder (ONLY relax final gate, keep all model layers) ---
# Gate A: original strict (yours)
gateA = df[(df["EV_SAFE_1U"] > 0) & (df["SAFE_PROB"] >= df["BOOK_PROB"] + 0.03)].copy()

# Gate B: still +EV, but edge >= +2 pts
gateB = df[(df["EV_SAFE_1U"] > 0) & (df["SAFE_PROB"] >= df["BOOK_PROB"] + 0.02)].copy()

# Gate C: +EV only (no edge requirement)
gateC = df[(df["EV_SAFE_1U"] > 0)].copy()

# Gate D: last resort (if slate is brutal): best edges even if tiny EV (still positive units)
gateD = df[(df["UNITS"] > 0)].copy()

def build_card(dfx, label):
    if len(dfx)==0:
        return None, label
    dfx = dfx.copy()
    dfx["SCORE"] = (dfx["EV_SAFE_1U"]*100) + (dfx["SAFE_PROB"]*10) - (dfx["INJ_SENS_ABS"]*0.5)
    dfx = dfx.sort_values("SCORE", ascending=False)
    top = dfx.head(10).reset_index(drop=True)
    top["RANK"] = np.arange(1, len(top)+1)
    return top, label

top10, used = None, None
for dfx, label in [(gateA,"Gate A (EV>0 & edge≥3pts)"),
                   (gateB,"Gate B (EV>0 & edge≥2pts)"),
                   (gateC,"Gate C (EV>0 only)"),
                   (gateD,"Gate D (Units>0 only)")]:
    top10, used = build_card(dfx, label)
    if top10 is not None and len(top10)>0:
        break

print("\nUSING:", used)
display(top10[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS","SAFE_PROB","BOOK_PROB","EV_SAFE_1U",
              "ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]])

print("\nNCAA ELITE PROP CARD")
print("—")
for r in top10.head(5).itertuples():
    print(f"{r.RANK}) {r.PLAYER} — {r.GAME}")
    print(f"{r.PROP} {r.OUTCOME} ({r.AMERICAN_ODDS}) — {r.UNITS:.2f}u")

Total rows: 98
EV>0 count: 0
SafeProb>BookProb count: 0
Edge >= +3 pts count: 0
Units > 0 count: 0

SAFE_PROB quantiles:
0.50   NaN
0.60   NaN
0.70   NaN
0.80   NaN
0.90   NaN
0.95   NaN
Name: SAFE_PROB, dtype: float64

EV_SAFE_1U quantiles:
0.50   NaN
0.60   NaN
0.70   NaN
0.80   NaN
0.90   NaN
0.95   NaN
Name: EV_SAFE_1U, dtype: float64

Top 10 by EV_SAFE_1U (raw):


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,BOOK_PROB,EV_SAFE_1U,ESPN_OPP_DEF_PPG_PROXY,ESPN_PACE_PROXY
0,Joshua Jefferson,ISU at ARIZ,Three Pointers Made,o0.5,-172,0.0,NaN,0.632353,NaN,68.1875,144.8125
1,Jaden Bradley,ISU at ARIZ,Three Pointers Made,u0.5,106,0.0,NaN,0.485437,NaN,68.1875,144.8125
2,Brayden Burries,ISU at ARIZ,Points + Rebounds,o20.5,-130,0.0,NaN,0.565217,NaN,68.1875,144.8125
3,Quadir Copeland,DUKE at NCST,Three Pointers Made,u0.5,-139,0.0,NaN,0.581590,NaN,68.1875,144.8125
4,Ven-Allen Lubin,DUKE at NCST,Rebounds,o6.5,-115,0.0,NaN,0.534884,NaN,68.1875,144.8125
5,Milan Momcilovic,ISU at ARIZ,Rebounds,u3.5,-170,0.0,NaN,0.629630,NaN,68.1875,144.8125
6,Brayden Burries,ISU at ARIZ,Rebounds,u5.5,-129,0.0,NaN,0.563319,NaN,68.1875,144.8125
7,Milan Momcilovic,ISU at ARIZ,Three Pointers Made,o2.5,-146,0.0,NaN,0.593496,NaN,68.1875,144.8125
8,Jaden Bradley,ISU at ARIZ,Points,u14.5,-115,0.0,NaN,0.534884,NaN,68.1875,144.8125
9,Tamin Lipsey,ISU at ARIZ,Points,o11.5,-122,0.0,NaN,0.549550,NaN,68.1875,144.8125



USING: Gate D (Units>0 only)


TypeError: 'NoneType' object is not subscriptable

In [12]:
import numpy as np
import pandas as pd

# 1) Quick diagnosis counts
cols_check = ["HIST_PROB","PROJ_MU_BASE","PROJ_MU_CTX","PROJ_PROB_CTX","BLEND_PROB","SAFE_PROB","EV_SAFE_1U"]
for c in cols_check:
    if c in df.columns:
        print(c, "null:", int(df[c].isna().sum()), "/", len(df))
    else:
        print(c, "MISSING")

# 2) If HIST_PROB is broken, rebuild it from the original hit columns again (robust)
hit_cols = ["L5","L10","2025","HM/AW","H2H","DVP_L5","DVP_L10","DVP_2025","DVP_HM/AW"]

def pct_to_prob_robust(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return np.nan
    s = s.replace("%","")
    try:
        v = float(s)
        return v/100.0 if v > 1 else v
    except:
        return np.nan

for c in hit_cols:
    if c in df.columns:
        df[c] = df[c].apply(pct_to_prob_robust).astype(float)

weights = {"L5":0.20,"L10":0.25,"2025":0.20,"HM/AW":0.10,"H2H":0.05,"DVP_L5":0.08,"DVP_L10":0.07,"DVP_2025":0.04,"DVP_HM/AW":0.01}

num = np.zeros(len(df), dtype=float)
den = np.zeros(len(df), dtype=float)
for c,w in weights.items():
    if c in df.columns:
        v = df[c].to_numpy(dtype=float)
        mask = ~np.isnan(v)
        num[mask] += w*v[mask]
        den[mask] += w

df["HIST_PROB"] = np.where(den>0, num/den, np.nan)
df["HIST_PROB"] = pd.Series(df["HIST_PROB"]).clip(0.01,0.99)

print("\nAfter rebuild HIST_PROB null:", int(pd.isna(df["HIST_PROB"]).sum()), "/", len(df))

# 3) Recompute μ + probs chain (same model layers, just re-running them cleanly)

# Ensure LINE numeric
df["LINE"] = df["LINE"].astype(float)

# Re-run PROJ_MU_BASE
df["PROJ_MU_BASE"] = df.apply(lambda r: invert_mu(r.HIST_PROB, r.LINE, r.DIST_USED, r.SIDE), axis=1)

# Re-run context μ shift (uses your ESPN proxies already)
def_z = (df["ESPN_OPP_DEF_PPG_PROXY"] - df["ESPN_OPP_DEF_PPG_PROXY"].median()) / df["ESPN_OPP_DEF_PPG_PROXY"].std()
pace_z = (df["ESPN_PACE_PROXY"] - df["ESPN_PACE_PROXY"].median()) / df["ESPN_PACE_PROXY"].std()
DEF_COEF, PACE_COEF = -0.06, 0.04

df["PROJ_MU_SHIFT"] = (DEF_COEF*def_z + PACE_COEF*pace_z) * df["PROJ_MU_BASE"]
df["PROJ_MU_CTX"] = (df["PROJ_MU_BASE"] + df["PROJ_MU_SHIFT"]).clip(lower=0.05)

# PROJ_PROB_CTX
df["PROJ_PROB_CTX"] = df.apply(lambda r: prob_from_mu(r.LINE, r.PROJ_MU_CTX, r.DIST_USED, r.SIDE), axis=1)
df["PROJ_PROB_CTX"] = df["PROJ_PROB_CTX"].astype(float).clip(0.01,0.99)

# Blend
W_PROJ = 0.60
df["BLEND_PROB"] = (W_PROJ*df["PROJ_PROB_CTX"] + (1-W_PROJ)*df["HIST_PROB"]).clip(0.01,0.99)

# Injury sens + SAFE_PROB + EV + Units (same as your stack)
def apply_mu(mu, direction="up"):
    mu=float(mu)
    bump = mu*0.07
    return mu + bump if direction=="up" else max(0.05, mu - bump)

p_up=[]
p_dn=[]
for r in df.itertuples():
    p_up.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX,"up"), r.DIST_USED, r.SIDE))
    p_dn.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX,"down"), r.DIST_USED, r.SIDE))

p_up=np.array(p_up, dtype=float)
p_dn=np.array(p_dn, dtype=float)

df["INJ_SENS_ABS"] = np.abs(p_up - p_dn) * 100.0
df["MU_SHIFT_ABS"] = np.abs(df["PROJ_MU_SHIFT"])

df["SAFE_PROB"] = (df["BLEND_PROB"]
                   - (df["INJ_SENS_ABS"]/100.0)*0.18
                   - (df["MU_SHIFT_ABS"]/df["PROJ_MU_BASE"].clip(0.1))*0.10
                  ).clip(0.01,0.99)

df["EV_SAFE_1U"] = df["SAFE_PROB"]*df["DEC_ODDS"] - 1

def half_kelly(p, dec):
    b=dec-1.0
    k=(p*b-(1-p))/b
    return max(0.0,k/2.0)

df["UNITS"] = [half_kelly(p,d) for p,d in zip(df["SAFE_PROB"],df["DEC_ODDS"])]
df["UNITS"] = df["UNITS"].clip(0,0.50)

print("\nPost-fix SAFE_PROB null:", int(df["SAFE_PROB"].isna().sum()), "/", len(df))
print("Post-fix EV>0 count:", int((df["EV_SAFE_1U"]>0).sum()))
print("Post-fix Units>0 count:", int((df["UNITS"]>0).sum()))

display(df.sort_values("EV_SAFE_1U", ascending=False).head(10)[
    ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS","SAFE_PROB","BOOK_PROB","EV_SAFE_1U",
     "ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]
])

HIST_PROB null: 43 / 98
PROJ_MU_BASE null: 0 / 98
PROJ_MU_CTX null: 98 / 98
PROJ_PROB_CTX null: 98 / 98
BLEND_PROB null: 98 / 98
SAFE_PROB null: 98 / 98
EV_SAFE_1U null: 98 / 98

After rebuild HIST_PROB null: 0 / 98

Post-fix SAFE_PROB null: 98 / 98
Post-fix EV>0 count: 0
Post-fix Units>0 count: 0


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,BOOK_PROB,EV_SAFE_1U,ESPN_OPP_DEF_PPG_PROXY,ESPN_PACE_PROXY
0,Joshua Jefferson,ISU at ARIZ,Three Pointers Made,o0.5,-172,0.0,NaN,0.632353,NaN,68.1875,144.8125
1,Jaden Bradley,ISU at ARIZ,Three Pointers Made,u0.5,106,0.0,NaN,0.485437,NaN,68.1875,144.8125
2,Brayden Burries,ISU at ARIZ,Points + Rebounds,o20.5,-130,0.0,NaN,0.565217,NaN,68.1875,144.8125
3,Quadir Copeland,DUKE at NCST,Three Pointers Made,u0.5,-139,0.0,NaN,0.581590,NaN,68.1875,144.8125
4,Ven-Allen Lubin,DUKE at NCST,Rebounds,o6.5,-115,0.0,NaN,0.534884,NaN,68.1875,144.8125
5,Milan Momcilovic,ISU at ARIZ,Rebounds,u3.5,-170,0.0,NaN,0.629630,NaN,68.1875,144.8125
6,Brayden Burries,ISU at ARIZ,Rebounds,u5.5,-129,0.0,NaN,0.563319,NaN,68.1875,144.8125
7,Milan Momcilovic,ISU at ARIZ,Three Pointers Made,o2.5,-146,0.0,NaN,0.593496,NaN,68.1875,144.8125
8,Jaden Bradley,ISU at ARIZ,Points,u14.5,-115,0.0,NaN,0.534884,NaN,68.1875,144.8125
9,Tamin Lipsey,ISU at ARIZ,Points,o11.5,-122,0.0,NaN,0.549550,NaN,68.1875,144.8125


In [13]:
import numpy as np

# --- Robust z-score: if std==0 (or NaN), return zeros (no context adjustment) ---
def safe_z(series: "pd.Series"):
    s = series.astype(float)
    sd = float(s.std())
    if (not np.isfinite(sd)) or sd == 0.0:
        return np.zeros(len(s), dtype=float)
    return ((s - float(s.median())) / sd).to_numpy(dtype=float)

def_z  = safe_z(df["ESPN_OPP_DEF_PPG_PROXY"])
pace_z = safe_z(df["ESPN_PACE_PROXY"])

# Same conservative coefs as your 3/1 run
DEF_COEF, PACE_COEF = -0.06, 0.04

# Recompute context shift on μ (not prob) — identical layer, now numerically safe
df["PROJ_MU_SHIFT"] = (DEF_COEF*def_z + PACE_COEF*pace_z) * df["PROJ_MU_BASE"]
df["PROJ_MU_CTX"]   = (df["PROJ_MU_BASE"] + df["PROJ_MU_SHIFT"]).clip(lower=0.05)

# Recompute probabilities & downstream layers
df["PROJ_PROB_CTX"] = df.apply(lambda r: prob_from_mu(r.LINE, r.PROJ_MU_CTX, r.DIST_USED, r.SIDE), axis=1).astype(float).clip(0.01,0.99)

W_PROJ = 0.60
df["BLEND_PROB"] = (W_PROJ*df["PROJ_PROB_CTX"] + (1-W_PROJ)*df["HIST_PROB"]).clip(0.01,0.99)

# Injury sensitivity proxy (μ ±7%)
def apply_mu(mu, direction="up"):
    mu=float(mu)
    bump = mu*0.07
    return mu + bump if direction=="up" else max(0.05, mu - bump)

p_up=[]
p_dn=[]
for r in df.itertuples():
    p_up.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX,"up"), r.DIST_USED, r.SIDE))
    p_dn.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX,"down"), r.DIST_USED, r.SIDE))

p_up=np.array(p_up, dtype=float)
p_dn=np.array(p_dn, dtype=float)

df["INJ_SENS_ABS"] = np.abs(p_up - p_dn) * 100.0
df["MU_SHIFT_ABS"] = np.abs(df["PROJ_MU_SHIFT"])

df["SAFE_PROB"] = (df["BLEND_PROB"]
                   - (df["INJ_SENS_ABS"]/100.0)*0.18
                   - (df["MU_SHIFT_ABS"]/df["PROJ_MU_BASE"].clip(0.1))*0.10
                  ).clip(0.01,0.99)

df["EV_SAFE_1U"] = df["SAFE_PROB"]*df["DEC_ODDS"] - 1

def half_kelly(p, dec):
    b=dec-1.0
    k=(p*b-(1-p))/b
    return max(0.0,k/2.0)

df["UNITS"] = [half_kelly(p,d) for p,d in zip(df["SAFE_PROB"],df["DEC_ODDS"])]
df["UNITS"] = df["UNITS"].clip(0,0.50)

print("SAFE_PROB null:", int(df["SAFE_PROB"].isna().sum()), "/", len(df))
print("EV>0 count:", int((df["EV_SAFE_1U"]>0).sum()))
print("Units>0 count:", int((df["UNITS"]>0).sum()))

display(df.sort_values("EV_SAFE_1U", ascending=False).head(10)[
    ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS","SAFE_PROB","BOOK_PROB","EV_SAFE_1U",
     "ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]
])

SAFE_PROB null: 0 / 98
EV>0 count: 25
Units>0 count: 25


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,BOOK_PROB,EV_SAFE_1U,ESPN_OPP_DEF_PPG_PROXY,ESPN_PACE_PROXY
6,Brayden Burries,ISU at ARIZ,Rebounds,u5.5,-129,0.220726,0.756092,0.563319,0.342210,68.1875,144.8125
12,Blake Buchanan,ISU at ARIZ,Rebounds,u5.5,114,0.139480,0.615894,0.467290,0.318014,68.1875,144.8125
22,Joshua Jefferson,ISU at ARIZ,Rebounds,u6.5,100,0.138014,0.638014,0.500000,0.276029,68.1875,144.8125
11,Cameron Boozer,DUKE at NCST,Points,u22.5,-120,0.163175,0.693795,0.545455,0.271958,68.1875,144.8125
14,Cameron Boozer,DUKE at NCST,Points + Rebounds + Assists,u36.5,-110,0.142507,0.659530,0.523810,0.259103,68.1875,144.8125
1,Jaden Bradley,ISU at ARIZ,Three Pointers Made,u0.5,106,0.115759,0.604568,0.485437,0.245409,68.1875,144.8125
8,Jaden Bradley,ISU at ARIZ,Points,u14.5,-115,0.106748,0.634184,0.534884,0.185648,68.1875,144.8125
19,Koa Peat,ISU at ARIZ,Rebounds,u5.5,-136,0.118974,0.677097,0.576271,0.174962,68.1875,144.8125
21,Joshua Jefferson,ISU at ARIZ,Points,u16.5,-114,0.092586,0.619239,0.532710,0.162431,68.1875,144.8125
18,Patrick Ngongba II,DUKE at NCST,Rebounds,u6.5,-150,0.118769,0.695015,0.600000,0.158358,68.1875,144.8125


In [14]:
import numpy as np
import pandas as pd
from datetime import datetime

# Strict gate (same as 3/1 style)
df_card = df[(df["EV_SAFE_1U"] > 0) & (df["SAFE_PROB"] >= df["BOOK_PROB"] + 0.03)].copy()

# If strict gate is too tight, fallback to EV>0 only (still not skipping layers)
if len(df_card) == 0:
    df_card = df[df["EV_SAFE_1U"] > 0].copy()
    gate_used = "EV>0 only (fallback)"
else:
    gate_used = "EV>0 & edge≥3pts (strict)"

# Score (same spirit as before)
df_card["SCORE"] = (df_card["EV_SAFE_1U"]*100) + (df_card["SAFE_PROB"]*10) - (df_card["INJ_SENS_ABS"]*0.5)

# Dedupe by player: keep best scoring prop per player
df_card = df_card.sort_values("SCORE", ascending=False).drop_duplicates(subset=["PLAYER"], keep="first")

top10 = df_card.head(10).copy().reset_index(drop=True)
top10["RANK"] = np.arange(1, len(top10)+1)

print("Gate used:", gate_used)
display(top10[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS","SAFE_PROB","BOOK_PROB","EV_SAFE_1U"]])

# Save files
RUN_DATE="20260302"
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
top_path  = f"hit_sheet_top10_ncaam_{RUN_DATE}_{ts}.csv"
full_path = f"prop_engine_full_ncaam_{RUN_DATE}_{ts}.csv"
top10.to_csv(top_path, index=False)
df.to_csv(full_path, index=False)
print("Saved:", top_path)
print("Saved:", full_path)

# Discord output (no model lingo)
print("\nNCAA ELITE PROP CARD")
print("—")
for r in top10.head(5).itertuples():
    print(f"{r.RANK}) {r.PLAYER} — {r.GAME}")
    print(f"{r.PROP} {r.OUTCOME} ({r.AMERICAN_ODDS}) — {r.UNITS:.2f}u")

Gate used: EV>0 & edge≥3pts (strict)


,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,BOOK_PROB,EV_SAFE_1U
0,1,Brayden Burries,ISU at ARIZ,Rebounds,u5.5,-129,0.220726,0.756092,0.563319,0.342210
1,2,Blake Buchanan,ISU at ARIZ,Rebounds,u5.5,114,0.139480,0.615894,0.467290,0.318014
2,3,Joshua Jefferson,ISU at ARIZ,Rebounds,u6.5,100,0.138014,0.638014,0.500000,0.276029
3,4,Jaden Bradley,ISU at ARIZ,Three Pointers Made,u0.5,106,0.115759,0.604568,0.485437,0.245409
4,5,Cameron Boozer,DUKE at NCST,Points,u22.5,-120,0.163175,0.693795,0.545455,0.271958
5,6,Koa Peat,ISU at ARIZ,Rebounds,u5.5,-136,0.118974,0.677097,0.576271,0.174962
6,7,Patrick Ngongba II,DUKE at NCST,Rebounds,u6.5,-150,0.118769,0.695015,0.600000,0.158358
7,8,Milan Momcilovic,ISU at ARIZ,Rebounds,u3.5,-170,0.087779,0.694651,0.629630,0.103269
8,9,Tamin Lipsey,ISU at ARIZ,Three Pointers Made,o0.5,-174,0.063054,0.681061,0.635036,0.072476
9,10,Isaiah Evans,DUKE at NCST,Three Pointers Made,u2.5,100,0.042478,0.542478,0.500000,0.084955


Saved: hit_sheet_top10_ncaam_20260302_20260302_152045.csv
Saved: prop_engine_full_ncaam_20260302_20260302_152045.csv

NCAA ELITE PROP CARD
—
1) Brayden Burries — ISU at ARIZ
Rebounds u5.5 (-129) — 0.22u
2) Blake Buchanan — ISU at ARIZ
Rebounds u5.5 (114) — 0.14u
3) Joshua Jefferson — ISU at ARIZ
Rebounds u6.5 (100) — 0.14u
4) Jaden Bradley — ISU at ARIZ
Three Pointers Made u0.5 (106) — 0.12u
5) Cameron Boozer — DUKE at NCST
Points u22.5 (-120) — 0.16u


In [15]:
df_card = df[(df["EV_SAFE_1U"] > 0) & (df["SAFE_PROB"] >= df["BOOK_PROB"] + 0.03)].copy()

df_card["SCORE"] = (df_card["EV_SAFE_1U"]*100) + (df_card["SAFE_PROB"]*10) - (df_card["INJ_SENS_ABS"]*0.5)
df_card = df_card.sort_values("SCORE", ascending=False)

final_rows = []
game_count = {}
stat_count = {}

for r in df_card.itertuples():
    game = r.GAME
    stat = r.PROP.lower()

    if game_count.get(game,0) >= 2:
        continue

    if stat_count.get(stat,0) >= 2:
        continue

    if any(x.PLAYER == r.PLAYER for x in final_rows):
        continue

    final_rows.append(r)
    game_count[game] = game_count.get(game,0)+1
    stat_count[stat] = stat_count.get(stat,0)+1

    if len(final_rows) == 5:
        break

final = pd.DataFrame(final_rows)
final["RANK"] = range(1,len(final)+1)

display(final[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS","SAFE_PROB","EV_SAFE_1U"]])

print("\nNCAA ELITE PROP CARD")
print("—")
for r in final.itertuples():
    print(f"{r.RANK}) {r.PLAYER} — {r.GAME}")
    print(f"{r.PROP} {r.OUTCOME} ({r.AMERICAN_ODDS}) — {r.UNITS:.2f}u")

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,EV_SAFE_1U
0,1,Brayden Burries,ISU at ARIZ,Rebounds,u5.5,-129,0.220726,0.756092,0.342210
1,2,Blake Buchanan,ISU at ARIZ,Rebounds,u5.5,114,0.139480,0.615894,0.318014
2,3,Cameron Boozer,DUKE at NCST,Points,u22.5,-120,0.163175,0.693795,0.271958
3,4,Isaiah Evans,DUKE at NCST,Three Pointers Made,u2.5,100,0.042478,0.542478,0.084955



NCAA ELITE PROP CARD
—
1) Brayden Burries — ISU at ARIZ
Rebounds u5.5 (-129) — 0.22u
2) Blake Buchanan — ISU at ARIZ
Rebounds u5.5 (114) — 0.14u
3) Cameron Boozer — DUKE at NCST
Points u22.5 (-120) — 0.16u
4) Isaiah Evans — DUKE at NCST
Three Pointers Made u2.5 (100) — 0.04u
